# Quantum Circuits | Source & Exact Compilation

This notebook builds the complete submission from readable code and a small public numerical dataset. V4 combines the **signed-Givens four-address lookup** and **curvature checkpoint** published by Kozykorpesh with our full-domain arithmetic compiler, certified rotation calibration and numerical factor gauge. The source decoder includes Dmitrii Gluzdov's V6 memory schedule and Kozykorpesh's exact symbolic lowering, building on Gu et al.

The changes address three different resource objectives:

| Component | Priced resource | This release |
|---|---|---|
| Inverse square root | Qubits times Toffoli-equivalent cost | 90Q schedule, five exact gate replacements, calibrated rotations |
| DQI | Repetition penalty times qubits squared times mean Toffolis | 1789Q source-built decoder and conservative symbolic lowering |
| Chemistry | Effective normalization times total Toffolis times total qubits | Signed angle gauge, four-address banks and five-scratch pebbling |

Every default Run All executes all three **unchanged official full verifiers**. Integer identities, phase certificates and physical checks have different scopes; none substitutes for the final checks.

The comparison snapshot is **24 October 2026**: Kozykorpesh V3 scored **6.68108**, and Mizero's new calibration notebook scored **6.37294**. Our previous V3 produced a full-verifier projection of **6.280714**, without a competition submission. The new result below is also labelled as a projection until its exact notebook version is submitted. The native badge may initially show this notebook's original V1 score **5.67004**.

All inputs are public. No submitted circuit archive or large Toolbox binary is downloaded. The development salt is fixed, and circuit hashes or salts are never searched for favorable samples.



In [ ]:
# notebook: hide_input
from __future__ import annotations

import gc
import hashlib
import importlib.util
import json
import math
import os
from pathlib import Path
import platform
import shutil
import sys
import time
import zipfile

os.environ.update(OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Resolve local inputs before changing directory so Run All also works when
# started from the project checkout. Kaggle keeps its usual default paths.
PROJECT_ROOT = Path.cwd().resolve()
for candidate_root in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate_root / 'data/competition/verifiers').is_dir():
        PROJECT_ROOT = candidate_root
        break
INPUT = Path(os.environ.get('QCC_INPUT', '/kaggle/input')).expanduser().resolve()
default_work = Path('/kaggle/working') if Path('/kaggle/input').exists() else PROJECT_ROOT / 'runs/notebook'
WORK = Path(os.environ.get('QCC_WORK', str(default_work))).expanduser().resolve()
COMP = next((p for p in [INPUT,
                         INPUT / 'competitions/google-verifiable-quantum-circuit-compilation',
                         INPUT / 'google-verifiable-quantum-circuit-compilation',
                         INPUT / 'competition',
                         PROJECT_ROOT / 'data/competition'] if (p / 'verifiers').exists()), None)
if COMP is None:
    raise FileNotFoundError('Attach the competition data listed in the notebook Input panel.')
PUBLIC = WORK / 'build'
FINAL = WORK / 'submission'
REPORTS = WORK / 'reports'
FIGURES = WORK / 'figures'
SRC = WORK / 'src'
for directory in [WORK, PUBLIC, FINAL, REPORTS, FIGURES, SRC, SRC / 'gen_dqi_v6/ref']:
    directory.mkdir(parents=True, exist_ok=True)
sys.path[:0] = [str(SRC), str(SRC / 'gen_dqi_v6'), str(COMP / 'kickmix'), str(COMP / 'verifiers')]
os.chdir(WORK)
import kickmix as km
from invsqrt import verify_submission_invsqrt as IV
from dqi import verify_submission_dqi as DV
from dfthcsos.verify import verify as verify_chemistry

RUN_FULL_VERIFICATION = True
SALT = 'vqcc-signed-digit-memory-v1-20261023'
WORKERS = min(3, os.cpu_count() or 1)

REFERENCE_COSTS = {'invsqrt': 2076774.233715282, 'dqi': 24439473914136.668, 'dfthcsos': 3355599588.3155365}

def digest(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def save_json(path, value):
    def encode(item):
        if isinstance(item, np.ndarray):
            return item.tolist()
        if isinstance(item, np.generic):
            return item.item()
        if isinstance(item, Path):
            return str(item)
        raise TypeError(type(item).__name__)
    Path(path).write_text(json.dumps(value, indent=2, default=encode), encoding='utf-8')

def measured(name, function):
    started = time.monotonic()
    result = function()
    result['wall_seconds'] = time.monotonic() - started
    save_json(REPORTS / f'{name}.json', result)
    return result

environment = {'python': sys.version, 'numpy': np.__version__, 'kickmix': getattr(km, '__version__', 'unknown'),
               'platform': platform.platform(), 'workers': WORKERS,
               'full_verification': RUN_FULL_VERIFICATION}
save_json(REPORTS / 'environment.json', environment)
print(json.dumps(environment, indent=2))

COLORS = {'ink': '#142B43', 'blue': '#207F9E', 'orange': '#D98D38', 'green': '#29876A', 'muted': '#9BAAB7'}
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titleweight': 'bold',
                     'text.color': COLORS['ink'], 'axes.labelcolor': COLORS['ink'],
                     'figure.facecolor': '#FFFFFF', 'savefig.facecolor': '#FFFFFF'})


## Reproducible inputs

The numerical input is [`femoco_public_words.npz`, public dataset V2](https://www.kaggle.com/datasets/dmitriigluzdov/vqcc-femoco-public-factorization/versions/2), released under Apache 2.0. The 1,350,093-byte file contains 3,976 rows of 75 ten-bit angle words, weights `W[65,60,30]`, and identity offsets `H2_I[65,30]`. Its exact SHA-256 and array shapes are checked below. The competition Input supplies the fixed Hamiltonian shifts and official circuit/verifier libraries.

No precompiled decoder or previous submission archive is needed. The decoder is regenerated for the competition's field polynomial, and the chemistry vectors are reconstructed from the checked angle words. All large circuit files are produced inside this run.


In [ ]:
# notebook: hide_input
paths = list(INPUT.rglob('femoco_public_words.npz')) if INPUT.exists() else []
if not paths:
    local_input = Path(os.environ.get('QCC_PUBLIC_WORDS', str(PROJECT_ROOT / 'research/dmitrii_source_20261023/public_data/femoco_public_words.npz')))
    paths = [local_input] if local_input.exists() else []
if len(paths) != 1:
    raise FileNotFoundError('Attach the public FeMoCo factorization dataset listed in Input.')
PUBLIC_WORDS = paths[0]
assert digest(PUBLIC_WORDS) == '9afd0ca5e90dd1740fe713e4ac1c26736184ea9bc4d4b57a6eeefafd608ee193'
with np.load(PUBLIC_WORDS, allow_pickle=False) as data:
    assert data['words'].shape == (3976, 75) and data['W'].shape == (65, 60, 30)
    assert data['H2_I'].shape == (65, 30)
save_json(REPORTS / 'public_input.json', {'source': 'dmitriigluzdov/vqcc-femoco-public-factorization',
    'file': PUBLIC_WORDS.name, 'sha256': digest(PUBLIC_WORDS), 'bytes': PUBLIC_WORDS.stat().st_size})
print('Checked public input: 3,976 addresses, 75 angles, physical weights and offsets.')


## 1. Arithmetic: erase, reuse, and certify

The oracle applies $e^{2\pi i/\sqrt{x}}$ while restoring its 22-bit input. We retain the explicit calibrated 256-row coefficient table credited to Kozykorpesh, and the Sumit/Rudra/Alex arithmetic lineage.

Kozykorpesh's new schedule measures two Gray-coded curvature bits after their last forward use. Their wires can serve as carry workspace during the outer polynomial calculation. Small Boolean circuits reconstruct the bits for reversal; the measurement phases are repaired together at the end. The normalized input's known top bit also supplies clean workspace. These are reversible lifetime changes, rather than a new polynomial fit.

Our exact-domain compiler evaluates reachable Boolean functions on every allowed input, $1\leq x\leq3,133,452$. It accepts an identity only after full truth-vector comparison. Five Clifford replacements reduce the structural schedule from 1454 to 1449 static Toffolis while preserving instruction order and measurement cleanup.

Following Mizero and lightningv08, angle calibration adjusts the phase linearly through each rotation's activation bit. This release applies fixed integer corrections, bound to the source and output circuit hashes. A fresh exhaustive scan checks the rewritten phase and two reproducible measurement traces. It removes one synthesized rotation, leaving **44**. The optional fitter remains in the source; the default build applies the selected corrections deterministically.

The complete 200,540,928-shot official check below separately checks phase, input restoration and scratch cleanup. Resource counts and the final cost come from that check.



In [ ]:
%%writefile src/signed_digit_invsqrt_v3.py
# SPDX-License-Identifier: Apache-2.0
# New derivative-split integration; original upstream notices retained below.
# Sumit Chongder: centered signed-digit generator and integer pipeline,361357149.
# Rudra Bhaskar: derivative split, top-carry reuse, static-high checkpoint,361445283.
# https://www.kaggle.com/code/rudrabhaskar/highest-public-score-direct-polynomial-phasing?scriptVersionId=361445283
# Alex Chilton: shared prefix and empty-target temporary-AND shifts,361472684.
# Kirill Tymchishin: exact Rudra/Alex integration used as the95Q source baseline,361491814.
# https://www.kaggle.com/code/kirilltym/quantum-circuits-qrom-tower42?scriptVersionId=361491814
# Kozykorpesh: explicit8,19,27,1 coefficient model,datasetVersionId21136269.
# Our changes: generic exact coefficient widths/left shifts, normalized-input
# borrowing on this derivative schedule, and the proven original-input rewrite.
# The derivative split and its precision calibrations are credited upstream.

# SPDX-License-Identifier: Apache-2.0
# Arithmetic research candidate. Retained upstream attribution:
# Sumit Chongder: readable centered signed-digit generator and integer pipeline.
# https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149
# anazemcev / Artem1410: early high-word phase, folded carry and normalized-input borrowing.
# https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory?scriptVersionId=361422248
# Kozykorpesh: exact reachable-domain calibration, Cfg(8,19,27,1), explicit table.
# https://www.kaggle.com/code/kozykappa/quantum-ai-compact-arithmetic-banked-qrom?scriptVersionId=361456406
# Quantum Circuit Toolbox datasetVersionId 21136269, code/domain_tables.py and coefficients.json.
# Alex Chilton: shared prefix flags and temporary-AND moves into zero destinations.
# https://www.kaggle.com/code/alexchilton/where-the-toffolis-go-anatomy-a-cheaper-invsqrt?scriptVersionId=361472684
# This revision combines credited methods. No upstream gain is claimed as original.
# Added own exact rewrite: the original input bound proves x[21]&x[20]==0.
# Linearize the first conjunction in normalization and the final input phase fixup.

# Attribution: Sumit Chongder, Apache-2.0 as published.
# https://www.kaggle.com/code/sumitchongder/notebook2f196e646b
# Read-only source downloaded from latest version 2. Kirill's v9 cites v1.
# This isolated lab regenerates the circuit from readable source; no binaries or
# remote notebook outputs are inputs. Our added exact integer certificate and
# reporting helpers and memory-schedule adaptations follow the original generator.
"""Builds an optimized kickmix circuit for the invsqrt benchmark (Benchmark I).

|x>|0> -> exp(2 pi i / sqrt(x)) |x>|0>,   x in [1, 3133452], 22-bit register 'x'.

Design (all arithmetic exactly mirrored by the numpy model below):
  1. Normalize x by single-bit binary stages (16, 8, 4, 2, 1) into M = x_norm / 2^21 in [1, 2);
     flags f0..f4 hold the shift count s' in binary.  e = f0 (parity), s = s' >> 1.
     1/sqrt(x) = 2^(s-10) * y,  y = 2^((e-1)/2) / sqrt(M)  in (0.5, 1].
  2. One 256-entry QROM indexed by (7 mantissa bits, e) XORs quadratic coefficients directly into
     the working registers (no separate coefficient copies).
  3. Centered signed-digit Horner: delta_c = sum_i (2 d_i - 1) 2^(i-22) over the 14 low bits.
     R = K1 - C2*delta_c,  y = K0 - R*delta_c.  Each row is an UNcontrolled Gidney addition
     wrapped in CNOT flips (Y - A = ~(~Y + A)), with a rounding carry-in.
  4. The y register is split: rows only touch a 21-bit low register L; the constant high part
     sits in a separate register and the overflow bit is merged in before phasing.
  5. Binary de-normalization (shift by s), one Z_POW per bit, then exact mirror uncompute.
     Lookup uncompute is measurement based (del_lookup); normalization flags are cleared by HMR
     with classically conditioned phase fixups.
Usage: python signed_digit_lab.py OUT.kmb [--baseline] (requires kickmix and NumPy)
"""
import sys, math, functools, types
from fractions import Fraction
import numpy as np
import kickmix as km

W = 22; MB = 21; DB = 14; NSEG = 128
F2, F1, FYR, GY, DROP2 = 8, 19, 27, 1, (5,)     # precision configuration (verified)

# ----------------------------------------------------------------- numpy model / tables
@functools.lru_cache(None)
def fitc(lo, hi, mid, scale, deg=2, iters=80):
    f = lambda M: scale / np.sqrt(M)
    d = np.linspace(lo - mid, hi - mid, 3001)
    A = np.vander(d, deg + 1, increasing=True)
    w = np.sqrt(mid + d) / scale
    wt = np.ones_like(d)
    for _ in range(iters):                       # Lawson iterations -> weighted minimax
        sw = w * np.sqrt(wt)
        coef = np.linalg.lstsq(A * sw[:, None], f(mid + d) * sw, rcond=None)[0]
        err = np.abs((A @ coef - f(mid + d)) * w)
        wt = wt * err; wt /= wt.sum()
    return coef

class Cfg:
    def __init__(self, f2, f1, fyr, gy, drop2=()):
        self.f2, self.f1, self.fyr, self.gy = f2, f1, fyr, gy
        self.fy = fyr + gy
        self.w2 = f2 - 1
        self.sh2 = [f2 + 22 - i - f1 for i in range(DB)]
        self.sh1 = [f1 + 22 - i - self.fy for i in range(DB)]
        self.drop2 = set(drop2)
    def rrow_list(self):
        return [i for i in reversed(range(DB)) if i not in self.drop2 and self.sh2[i] < self.w2]

def rd(v, s):
    if s < 0: return v << (-s)
    r = v >> s
    if s >= 1: r = r + ((v >> (s - 1)) & 1)
    return r

def pipeline(cfg, K2, K1, d):
    sig = [2 * ((d >> i) & 1) - 1 for i in range(DB)]
    R = np.full_like(d, K1)
    for i in cfg.rrow_list():
        R = R - sig[i] * rd(K2, cfg.sh2[i])
    Q = np.zeros_like(d)
    for i in reversed(range(DB)):
        Q = Q + sig[i] * rd(R, cfg.sh1[i])
    return R, Q

MAX_X = 3_133_452

def valid_residuals(e,a):
    d=np.arange(1<<DB,dtype=np.int64);xn=(1<<MB)+(a<<DB)+d
    mask=(d%2==0) if e else ((xn<=MAX_X)|(d%4==0))
    return d[mask]

def calibrate_segment(cfg,e,a,c_radius=1,b_radius=3):
    d=valid_residuals(e,a);lo=1+a*2.**(DB-MB);mid=lo+((1<<DB)-1)/2**(MB+1)
    scale=2**((e-1)/2);co=fitc(lo,lo+2.**(DB-MB),mid,scale)
    f=scale/np.sqrt(lo+d*2.**-MB);target=f*2**cfg.fy;w=1/f
    K2c=round(co[2]*2**cfg.f2);K1c=round(-co[1]*2**cfg.f1);best=None;quant=1<<cfg.gy
    for K2 in range(max(0,K2c-c_radius),min((1<<cfg.w2)-1,K2c+c_radius)+1):
        for K1 in range(K1c-b_radius,K1c+b_radius+1):
            R,Q=pipeline(cfg,K2,K1,d)
            if R.min()<0:continue
            vals=target+Q;center=.5*(vals.min()+vals.max())+.5*(quant-1)
            k0s=np.arange(math.floor(center)-quant-1,math.ceil(center)+quant+2,dtype=np.int64)
            # The stored high anchor and the carry must keep the leading one fixed.
            k0s=k0s[k0s <= (1<<cfg.fy)-1+int(Q.min())]
            if len(k0s)==0:continue
            y=((k0s[:,None]-Q)>>cfg.gy)<<cfg.gy
            scores=np.max(abs(y-target)*w,axis=1)/2**cfg.fy;i=int(scores.argmin())
            v=float(scores[i]);K0=int(k0s[i])
            if best is None or v<best[0]:best=(v,K2,K1,K0)
    if best is None:raise RuntimeError('No legal coefficient triple')
    return best

def regenerate_tables_exact(cfg,refine_threshold=.95):
    tabs={};ratios=[]
    for e in(0,1):
        for a in range(128):
            best=calibrate_segment(cfg,e,a)
            if best[0]*2**24>refine_threshold:
                alt=calibrate_segment(cfg,e,a,c_radius=4,b_radius=7)
                if alt[0]<best[0]:best=alt
            tabs[(e,a)]=best[1:];ratios.append(best[0]*2**24)
    return tabs,dict(worst_ratio=max(ratios),worst_key=[int(np.argmax(ratios))//128,int(np.argmax(ratios))%128],reachable_normalized_inputs=sum(len(valid_residuals(e,a)) for e in(0,1) for a in range(128)))

def build_tables(cfg, search=2):
    """Optional credited exact-domain coefficient regeneration.

    The radius policy is the published Kozykorpesh calibration, rather than
    the old continuous-domain fitter. The default selected build uses the
    explicitly published numeric table below, checked on every original x.
    """
    tables, report = regenerate_tables_exact(cfg)
    return tables, report['worst_ratio'] / 2**24

def layout(cfg, tabs, LB=21, S=20):
    d = np.arange(2**DB, dtype=np.int64)
    assert max(t[0] for t in tabs.values()) < (1 << cfg.w2)
    Rmax = 0; per = {}; Ymax = 0
    for key, (K2, K1, K0) in tabs.items():
        R, Q = pipeline(cfg, K2, K1, d)
        assert R.min() >= 0
        Rmax = max(Rmax, int(R.max()), K1)
        per[key] = (int(Q.min()), int(Q.max()))
        Ymax = max(Ymax, K0 - per[key][0])
    w1 = Rmax.bit_length()
    lo_tab, hi_tab = {}, {}
    for key, (K2, K1, K0) in tabs.items():
        qmin, qmax = per[key]
        base = K0 % (1 << S); j = 0
        while base + j * (1 << S) - qmax < 0: j += 1
        k0lo = base + j * (1 << S)
        assert k0lo - qmin < (1 << LB), 'layout infeasible'
        lo_tab[key] = k0lo; hi_tab[key] = (K0 - k0lo) >> S
        assert hi_tab[key] >= 0
    whi = max(max(hi_tab.values()).bit_length(), 1)
    return dict(w1=w1, LB=LB, S=S, whi=whi, lo=lo_tab, hi=hi_tab)

# ----------------------------------------------------------------- circuit pieces
def and_all_into(b, controls, target):
    k = len(controls)
    if k == 1: b.cx(controls[0], target); return
    if k == 2: b.ccx(controls[0], controls[1], target); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.ccx(anc[k - 3], controls[k - 1], target)
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

def phase_all(b, controls):
    k = len(controls)
    if k == 1: b.z(controls[0]); return
    if k == 2: b.cz(controls[0], controls[1]); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.cz(anc[k - 3], controls[k - 1])
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

NSTAGES = [(4, 16), (3, 8), (2, 4), (1, 2), (0, 1)]

def _baseline_normalize(b, x, f):
    for (k, d) in NSTAGES:
        top = x[W - d:W]
        if d == 1:
            b.cx(x[W - 1], f[0]); b.x(f[0])
        else:
            b.x(top); and_all_into(b, list(top), f[k]); b.x(top)
        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])


def _combined_normalize(b, x, f):
    """v3 step 1: f4, f3, f2 (the binary leading-zero count div 4) from one prefix-AND chain on the unshifted x.
    chain[m] = [x[W-m-2:W] == 0] (top m+2 bits zero); p_j = chain[4j-2] = [lz >= 4j].  f4 = p4, f3 = p2 ^ p4,
    f2 = p1 ^ p2 ^ p3 ^ p4 ^ p5 (nested prefixes, so XOR = bit 2 of lz).  19 ANDs instead of 15 + 7 + 3."""
    bits = [x[W - 1 - m] for m in range(20)]                  # x21, x20, ..., x2
    b.x(km.array(bits))
    chain = b.alloc_qubits(19)
    b.init_and(bits[0], bits[1], chain[0])
    for m in range(1, 19): b.init_and(chain[m - 1], bits[m + 1], chain[m])
    p = {j: chain[4 * j - 2] for j in range(1, 6)}
    b.cx(p[4], f[4])
    b.cx(p[2], f[3]); b.cx(p[4], f[3])
    for j in range(1, 6): b.cx(p[j], f[2])
    for m in reversed(range(1, 19)): b.del_and(chain[m - 1], bits[m + 1], chain[m])
    b.del_and(bits[0], bits[1], chain[0])
    b.free(chain)
    b.x(km.array(bits))
    for (k, d) in NSTAGES:
        top = x[W - d:W]
        if d == 1:
            b.cx(x[W - 1], f[0]); b.x(f[0])
        elif d == 2:
            b.x(top); b.ccx(top[0], top[1], f[1]); b.x(top)
        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])

# MAX_X < 3*2**20 proves the original two top bits are never both one.
# After inversion, their AND equals XOR with one, on every promised input.
def normalize(b, x, f):
    """v3 step 1: f4, f3, f2 (the binary leading-zero count div 4) from one prefix-AND chain on the unshifted x.
    chain[m] = [x[W-m-2:W] == 0] (top m+2 bits zero); p_j = chain[4j-2] = [lz >= 4j].  f4 = p4, f3 = p2 ^ p4,
    f2 = p1 ^ p2 ^ p3 ^ p4 ^ p5 (nested prefixes, so XOR = bit 2 of lz).  19 ANDs instead of 15 + 7 + 3."""
    bits = [x[W - 1 - m] for m in range(20)]                  # x21, x20, ..., x2
    b.x(km.array(bits))
    chain = b.alloc_qubits(19)
    b.x(chain[0]); b.cx(bits[0],chain[0]); b.cx(bits[1],chain[0])
    for m in range(1, 19): b.init_and(chain[m - 1], bits[m + 1], chain[m])
    p = {j: chain[4 * j - 2] for j in range(1, 6)}
    b.cx(p[4], f[4])
    b.cx(p[2], f[3]); b.cx(p[4], f[3])
    for j in range(1, 6): b.cx(p[j], f[2])
    for m in reversed(range(1, 19)): b.del_and(chain[m - 1], bits[m + 1], chain[m])
    b.cx(bits[1],chain[0]); b.cx(bits[0],chain[0]); b.x(chain[0])
    b.free(chain)
    b.x(km.array(bits))
    for (k, d) in NSTAGES:
        top = x[W - d:W]
        if d == 1:
            b.cx(x[W - 1], f[0]); b.x(f[0])
        elif d == 2:
            b.x(top); b.ccx(top[0], top[1], f[1]); b.x(top)
        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])

def _baseline_unnormalize(b, x, f):
    for (k, d) in reversed(NSTAGES):
        for i in range(0, W - d): b.cswap(f[k], x[i], x[i + d])
        top = x[W - d:W]
        if d == 1:
            b.x(f[0]); b.cx(x[W - 1], f[0])
        else:
            m = b.alloc_bits(1)[0]
            b.hmr(f[k], m)
            b.push_condition(m)                     # fixup runs on half the shots on average
            b.x(top); phase_all(b, list(top)); b.x(top)
            b.pop_condition()
            b.free(m)

def promise_phase_original_top(b,controls):
    """Phase the original top16-zero predicate using its promised bit relation."""
    assert len(controls)==16
    controls=list(reversed(controls))
    k=len(controls)
    anc=b.alloc_qubits(k-2)
    b.x(anc[0]); b.cx(controls[0],anc[0]); b.cx(controls[1],anc[0])
    for j in range(1,k-2):
        b.init_and(anc[j-1],controls[j+1],anc[j])
    b.cz(anc[k-3],controls[k-1])
    for j in reversed(range(1,k-2)):
        b.del_and(anc[j-1],controls[j+1],anc[j])
    b.cx(controls[1],anc[0]); b.cx(controls[0],anc[0]); b.x(anc[0])
    b.free(anc)

def unnormalize(b, x, f):
    for (k, d) in reversed(NSTAGES):
        for i in range(0, W - d): b.cswap(f[k], x[i], x[i + d])
        top = x[W - d:W]
        if d == 1:
            b.x(f[0]); b.cx(x[W - 1], f[0])
        else:
            m = b.alloc_bits(1)[0]
            b.hmr(f[k], m)
            b.push_condition(m)                     # fixup runs on half the shots on average
            b.x(top)
            if d==16: promise_phase_original_top(b,list(top))
            else: phase_all(b,list(top))
            b.x(top)
            b.pop_condition()
            b.free(m)

def gadd(b, a, t, cin=None, sub=False):
    """t += a + cin (or t -= a + cin), Gidney adder: len(t)-1 Toffolis, uncompute by HMR."""
    a = list(a); t = list(t); o = len(a); n = len(t)
    assert o <= n
    if sub: b.x(km.array(t))
    c = b.alloc_qubits(n - 1) if n > 1 else km.array([])
    def C(i): return cin if i == 0 else c[i - 1]
    for i in range(0, n - 1):
        if i == 0 and cin is None:
            if o: b.init_and(a[0], t[0], C(1))
        elif i < o:
            b.cx(C(i), a[i]); b.cx(C(i), t[i]); b.init_and(a[i], t[i], C(i + 1)); b.cx(C(i), C(i + 1))
        else:
            b.init_and(C(i), t[i], C(i + 1))
    if n - 1 < o: b.cx(a[n - 1], t[n - 1])
    if n > 1 or cin is not None: b.cx(C(n - 1), t[n - 1])
    for i in reversed(range(0, n - 1)):
        if i == 0 and cin is None:
            if o: b.del_and(a[0], t[0], C(1)); b.cx(a[0], t[0])
        elif i < o:
            b.cx(C(i), C(i + 1)); b.del_and(a[i], t[i], C(i + 1)); b.cx(C(i), a[i]); b.cx(a[i], t[i])
        else:
            b.del_and(C(i), t[i], C(i + 1)); b.cx(C(i), t[i])
    if n > 1: b.free(c)
    if sub: b.x(km.array(t))

def _baseline_shift_stages(b, f, reg, L, inverse):
    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]
    occ = []; o = L
    for (k, d) in stages: occ.append(o); o = min(o + d, len(reg))
    seq = list(enumerate(stages))
    if not inverse:
        for idx, (k, d) in seq:
            for i in reversed(range(0, occ[idx])):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])
    else:
        for idx, (k, d) in reversed(seq):
            for i in range(0, occ[idx]):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])


def shift_stages(b, f, reg, L, inverse):
    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]
    occ = []; o = L
    for (k, d) in stages: occ.append(o); o = min(o + d, len(reg))
    seq = list(enumerate(stages))
    if not inverse:
        for idx, (k, d) in seq:
            for i in reversed(range(0, occ[idx])):
                if i + d >= len(reg): continue
                if i + d >= occ[idx]:                       # v2: target is |0> for every f
                    b.init_and(f[k], reg[i], reg[i + d]); b.cx(reg[i + d], reg[i])
                else:
                    b.cswap(f[k], reg[i], reg[i + d])
    else:
        for idx, (k, d) in reversed(seq):
            for i in range(0, occ[idx]):
                if i + d >= len(reg): continue
                if i + d >= occ[idx]:
                    b.cx(reg[i + d], reg[i]); b.del_and(f[k], reg[i], reg[i + d])
                else:
                    b.cswap(f[k], reg[i], reg[i + d])

def build_circuit(cfg, tabs, lay, adder=None):
    # The default preserves the public generator exactly. An injected adder
    # supports isolated, explicitly tested memory experiments.
    if adder is None: adder=gadd
    w2, w1, LB, S, whi = cfg.w2, lay['w1'], lay['LB'], lay['S'], lay['whi']
    gy, fy = cfg.gy, cfg.fy
    b = km.CircuitBuilder()
    x = b.create_quantum_register(W, name='x')
    f = b.alloc_qubits(5)
    c2 = b.alloc_qubits(w2); R = b.alloc_qubits(w1); Lr = b.alloc_qubits(LB); Yh = b.alloc_qubits(whi)
    tgt = km.array(list(Lr) + list(R) + list(c2) + list(Yh))
    table = [0] * 256
    for (e, a), (K2, K1, K0) in tabs.items():
        table[a + 128 * e] = lay['lo'][(e, a)] | (K1 << LB) | (K2 << (LB + w1)) | (lay['hi'][(e, a)] << (LB + w1 + w2))
    addr = km.array(list(x[14:21]) + [f[0]])
    delta = x[0:DB]
    def rrows(inverse):
        seq = cfg.rrow_list()
        if inverse: seq = list(reversed(seq))
        for i in seq:
            sh = cfg.sh2[i]
            if sh >= 0: off = c2[sh:w2]; t = R[0:w1]; cin = c2[sh - 1] if sh >= 1 else None
            else: off = c2[0:w2]; t = R[-sh:w1]; cin = None
            b.cx(delta[i], t); adder(b, off, t, cin, sub=inverse); b.cx(delta[i], t)
    def yrows(inverse):
        seq = list(reversed(range(DB)))
        if inverse: seq.reverse()
        for i in seq:
            sh = cfg.sh1[i]; assert 0 <= sh < w1
            t = Lr[0:LB]; cin = R[sh - 1] if sh >= 1 else None
            b.cx(delta[i], t); adder(b, R[sh:w1], t, cin, sub=inverse); b.cx(delta[i], t)
    normalize(b, x, f)
    b.init_lookup(table=table, address=addr, target=tgt)
    rrows(False); yrows(False)
    ext = b.alloc_qubits(10)
    if LB > S: adder(b, Lr[S:LB], Yh)
    Cx = km.array(list(Lr[gy:S]) + list(Yh) + list(ext))
    shift_stages(b, f, Cx, S - gy + whi, False)
    for j in range(len(Cx)):
        e = gy + j - fy - 10                       # bit weight 2^e turns
        if e < 0: b.z_pow(Cx[j], Fraction(2)**(e + 1))
    shift_stages(b, f, Cx, S - gy + whi, True)
    if LB > S: adder(b, Lr[S:LB], Yh, sub=True)
    b.free(ext)
    yrows(True); rrows(True)
    b.del_lookup(table=table, address=addr, target=tgt)
    b.free(tgt)
    unnormalize(b, x, f)
    b.free(f)
    return b.finish_circuit()

def main(out):
    cfg = Cfg(F2, F1, FYR, GY, DROP2)
    tabs, worst = build_tables(cfg)
    print(f'model worst relative error / 2^-24 = {worst * 2**24:.3f}')
    lay = layout(cfg, tabs)
    c = build_circuit(cfg, tabs, lay)
    s = c.operation_statistics()
    print('qubits', c.num_qubits, 'toffoli(static)', s.num_toffoli_like,
          'rotations', s.num_z_pow(min_clifford_hierarchy_level=4))
    c.to_file(out, format='kmb')
    print('wrote', out)


def selected_cfg():
    return Cfg(F2,F1,FYR,GY,DROP2)


def config_record(cfg):
    return dict(f2=cfg.f2,f1=cfg.f1,fyr=cfg.fyr,gy=cfg.gy,drop2=sorted(cfg.drop2))


_PUBLISHED_COEFFICIENT_ROWS = ((67, 184284, 189442910), (66, 182154, 188710050), (65, 180064, 187985630), (63, 178014, 187269499), (62, 176002, 186561483), (61, 174029, 185861435), (60, 172092, 185169210), (59, 170191, 184484662), (58, 168323, 183807650), (57, 166490, 183138038), (56, 164690, 182475692), (55, 162923, 181820481), (54, 161186, 181172277), (53, 159480, 180530959), (52, 157805, 179896401), (51, 156158, 179268491), (50, 154540, 178647108), (49, 152949, 178032144), (48, 151386, 177423486), (48, 149849, 176821020), (47, 148338, 176224658), (46, 146852, 175634293), (45, 145391, 175049819), (44, 143954, 174471143), (44, 142540, 173898157), (43, 141150, 173330791), (42, 139782, 172768943), (42, 138435, 172212512), (41, 137110, 171661436), (40, 135806, 171115614), (40, 134523, 170574956), (39, 133260, 170039402), (39, 132016, 169508850), (38, 130792, 168983243), (37, 129587, 168462497), (37, 128400, 167946525), (36, 127231, 167435275), (36, 126080, 166928657), (35, 124945, 166426619), (35, 123828, 165929073), (34, 122728, 165435976), (34, 121643, 164947236), (33, 120574, 164462814), (33, 119521, 163982624), (32, 118483, 163506626), (32, 117460, 163034740), (31, 116452, 162566926), (31, 115458, 162103106), (30, 114478, 161643244), (30, 113512, 161187263), (29, 112560, 160735129), (29, 111621, 160286771), (29, 110694, 159842141), (28, 109781, 159401203), (28, 108879, 158963885), (28, 107991, 158530145), (27, 107114, 158099945), (27, 106249, 157673220), (26, 105396, 157249942), (26, 104554, 156830046), (26, 103723, 156413494), (25, 102902, 156000254), (25, 102094, 155590262), (25, 101295, 155183486), (24, 100506, 154779894), (24, 99729, 154379426), (24, 98960, 153982048), (23, 98202, 153587734), (23, 97454, 153196425), (23, 96715, 152808089), (23, 95985, 152422695), (22, 95263, 152040209), (22, 94552, 151660581), (22, 93849, 151283781), (22, 93155, 150909775), (21, 92468, 150538541), (21, 91791, 150170021), (21, 91122, 149804197), (20, 90461, 149441041), (20, 89807, 149080504), (20, 89162, 148722566), (20, 88525, 148367192), (20, 87894, 148014354), (19, 87272, 147664030), (19, 86657, 147316174), (19, 86048, 146970764), (19, 85447, 146627772), (18, 84854, 146287182), (18, 84266, 145948942), (18, 83686, 145613040), (18, 83112, 145279444), (18, 82544, 144948132), (17, 81983, 144619086), (17, 81429, 144292261), (17, 80881, 143967641), (17, 80338, 143645203), (17, 79802, 143324921), (16, 79272, 143006783), (16, 78748, 142690743), (16, 78229, 142376789), (16, 77716, 142064901), (16, 77208, 141755051), (16, 76707, 141447221), (15, 76210, 141141395), (15, 75719, 140837537), (15, 75233, 140535633), (15, 74753, 140235661), (15, 74277, 139937603), (15, 73806, 139641437), (14, 73340, 139347152), (14, 72880, 139054712), (14, 72424, 138764106), (14, 71973, 138475312), (14, 71526, 138188314), (14, 71084, 137903096), (14, 70647, 137619634), (13, 70214, 137337922), (13, 69785, 137057924), (13, 69361, 136779634), (13, 68941, 136503030), (13, 68525, 136228098), (13, 68113, 135954820), (13, 67707, 135683180), (13, 67303, 135413164), (12, 66903, 135144762), (12, 66508, 134877938), (12, 66116, 134612690), (12, 65728, 134349002), (95, 260618, 267912731), (93, 257604, 266876315), (91, 254649, 265851837), (90, 251750, 264839056), (88, 248905, 263837774), (86, 246114, 262847765), (85, 243374, 261868805), (83, 240685, 260900715), (82, 238045, 259943272), (80, 235453, 258996304), (79, 232907, 258059600), (77, 230407, 257133000), (76, 227952, 256216297), (75, 225540, 255309329), (73, 223170, 254411937), (72, 220841, 253523932), (71, 218553, 252645160), (70, 216303, 251775464), (68, 214093, 250914700), (67, 211918, 250062693), (66, 209782, 249219307), (65, 207680, 248384399), (64, 205614, 247557825), (63, 203581, 246739448), (62, 201582, 245929136), (61, 199616, 245126756), (60, 197681, 244332176), (59, 195777, 243545275), (58, 193903, 242765929), (57, 192059, 241994017), (56, 190245, 241229423), (55, 188459, 240472030), (54, 186700, 239721726), (54, 184968, 238978392), (53, 183264, 238241942), (52, 181585, 237512258), (51, 179932, 236789237), (51, 178303, 236072769), (50, 176700, 235362777), (49, 175120, 234659153), (48, 173563, 233961801), (47, 172029, 233270629), (47, 170518, 232585538), (46, 169029, 231906458), (45, 167561, 231233292), (45, 166114, 230565944), (44, 164688, 229904352), (44, 163283, 229248411), (43, 161897, 228598063), (42, 160531, 227953219), (42, 159184, 227313793), (41, 157855, 226679725), (41, 156545, 226050925), (40, 155253, 225427340), (39, 153979, 224808886), (39, 152722, 224195486), (38, 151482, 223587090), (38, 150259, 222983610), (37, 149052, 222385000), (37, 147861, 221791176), (37, 146686, 221202083), (36, 145526, 220617669), (35, 144382, 220037861), (35, 143252, 219462595), (35, 142138, 218891815), (34, 141037, 218325477), (34, 139951, 217763501), (33, 138878, 217205852), (33, 137820, 216652456), (32, 136775, 216103278), (32, 135742, 215558246), (32, 134723, 215017318), (31, 133716, 214480450), (31, 132722, 213947574), (31, 131741, 213418652), (30, 130770, 212893641), (30, 129812, 212372477), (29, 128866, 211855133), (29, 127931, 211341539), (29, 127007, 210831665), (28, 126095, 210325473), (28, 125193, 209822899), (28, 124302, 209323909), (27, 123422, 208828473), (27, 122551, 208336530), (27, 121691, 207848046), (26, 120841, 207362994), (26, 120001, 206881310), (26, 119170, 206402968), (26, 118350, 205927930), (25, 117538, 205456166), (25, 116736, 204987618), (25, 115943, 204522262), (24, 115158, 204060072), (24, 114383, 203600991), (24, 113616, 203144995), (24, 112858, 202692049), (23, 112108, 202242127), (23, 111366, 201795179), (23, 110632, 201351183), (23, 109907, 200910105), (22, 109189, 200471921), (22, 108480, 200036585), (22, 107778, 199604069), (22, 107083, 199174349), (21, 106395, 198747401), (21, 105716, 198323176), (21, 105044, 197901658), (21, 104377, 197482814), (20, 103720, 197066632), (20, 103068, 196653058), (20, 102423, 196242078), (20, 101785, 195833664), (20, 101154, 195427788), (19, 100528, 195024434), (19, 99910, 194623560), (19, 99297, 194225148), (19, 98691, 193829172), (19, 98092, 193435607), (19, 97497, 193044431), (18, 96909, 192655627), (18, 96328, 192269155), (18, 95751, 191884999), (18, 95180, 191503135), (18, 94616, 191123543), (17, 94056, 190746209), (17, 93502, 190371093), (17, 92954, 189998179))


def load_or_build_tables(path=None, regenerate=False):
    """Read explicit public numeric coefficients; optional open regeneration.

    No circuit binary is reused. The 256-row table is Kozykorpesh's published
    exact-domain calibration, pinned to datasetVersionId 21136269. Keeping
    these ordinary integer triples explicit avoids numerical-environment
    changes silently modifying the scored model. The notebook independently
    certifies all 3,133,452 original inputs before building circuits.
    """
    import json
    from pathlib import Path
    cfg = selected_cfg()
    expected_rows = [list(row) for row in _PUBLISHED_COEFFICIENT_ROWS]
    worst = 0.9628198889674466 / 2**24
    if regenerate:
        tables, worst = build_tables(cfg)
        regenerated = [list(tables[(e,a)]) for e in range(2) for a in range(128)]
        assert regenerated == expected_rows, 'Regenerated numeric coefficients differ from the explicit public table.'
    else:
        tables = {(i//128, i%128):tuple(row) for i,row in enumerate(expected_rows)}
    path = Path(path) if path else None
    if path and path.exists():
        saved = json.loads(path.read_text())
        assert saved['config'] == config_record(cfg)
        assert saved['rows'] == expected_rows, 'Cached coefficient table differs from the explicitly selected model.'
    elif path:
        path.write_text(json.dumps({'source':'Kozykorpesh exact-domain calibration; Atlas datasetVersionId21136269',
            'config':config_record(cfg), 'normalized_worst_relative_error':worst,
            'regeneration_is_optional':True, 'rows':expected_rows},indent=2))
    return cfg, tables, worst


def integer_model(xs,cfg,tabs):
    """Exact emitted arithmetic in units 2^-(cfg.fy+10) turns, before mod 1."""
    xs=np.asarray(xs,dtype=np.int64)
    if not ((xs>=1)&(xs<=3133452)).all():
        raise ValueError('The supported competition domain is x=1..3133452')
    # All inputs fit exactly in float64; frexp gives the exact binary exponent.
    shifts=21-(np.frexp(xs)[1].astype(np.int64)-1)
    norm=xs << shifts
    assert ((norm >> 21)==1).all(), 'Normalized top-bit borrowing requires x[21]=1'
    parity=shifts&1
    index=((norm >> 14)&127)+128*parity
    table=np.asarray([tabs[(e,a)] for e in range(2) for a in range(128)],dtype=np.int64)
    K2,K1,K0=(table[index,j] for j in range(3))
    d=norm&((1 << DB)-1)
    signs=[2*((d >> i)&1)-1 for i in range(DB)]
    R=K1.copy()
    for i in cfg.rrow_list():
        R-=signs[i]*rd(K2,cfg.sh2[i])
    Q=np.zeros_like(xs)
    for i in reversed(range(DB)):
        Q+=signs[i]*rd(R,cfg.sh1[i])
    Y=((K0-Q) >> cfg.gy) << cfg.gy
    if (R<0).any() or (Y<0).any():
        raise OverflowError('An unsigned intermediate is negative')
    return Y << (shifts >> 1)


def exhaustive_certificate(cfg,tabs,chunk_size=131072):
    """Every promised basis input, exact integer target and tolerance.

    This is an arithmetic certificate. Ancillas, restored inputs and phase
    feed-forward require fresh circuit simulation and the official verifier.
    """
    output_fraction=cfg.fy+10
    worst_ratio,worst_x,min_margin=0.,0,1 << 64
    for start in range(1,3133453,chunk_size):
        xs=np.arange(start,min(start+chunk_size,3133453),dtype=np.int64)
        values=integer_model(xs,cfg,tabs).astype(np.uint64)
        phase=values << np.uint64(64-output_fraction)
        roots=[math.isqrt((1 << 130)//int(x)) for x in xs]
        target=np.array([((r+1) >> 1)&((1 << 64)-1) for r in roots],dtype=np.uint64)
        tolerance=np.array([min(r >> 25,1 << 63) for r in roots],dtype=np.uint64)
        error=np.abs((phase-target).view(np.int64)).view(np.uint64)
        ratio=error/tolerance
        i=int(np.argmax(ratio))
        if float(ratio[i])>worst_ratio:
            worst_ratio,worst_x=float(ratio[i]),int(xs[i])
        if (error>tolerance).any():
            i=int(np.flatnonzero(error>tolerance)[0])
            return {'pass':False,'first_bad_x':int(xs[i]),'ratio':float(ratio[i])}
        min_margin=min(min_margin,int((tolerance-error).min()))
    return {'pass':True,'tested_inputs':3133452,'worst_ratio':worst_ratio,
            'worst_x':worst_x,'min_margin_units':min_margin,
            'phase_fraction_bits':output_fraction}


def conditional_toffoli_discount(circuit):
    """Mean discount for this emitted family's fair HMR conditional branches."""
    depth=0
    total=0
    discount=0.
    for line in str(circuit).splitlines():
        opcode=line.split(' ',1)[0]
        if opcode=='PUSH_CONDITION':depth+=1
        elif opcode=='POP_CONDITION':depth-=1
        elif opcode in ['CCX','CSWAP']:
            total+=1
            if depth:
                assert depth==1,'Expected one fair measurement bit in a Toffoli branch.'
                discount+=.5
    assert total==int(circuit.operation_statistics().num_toffoli_like)
    assert depth==0
    return discount

def estimated_resources(circuit,certificate):
    """Model-based cost; full unchanged official verification remains required."""
    assert certificate['pass'] and certificate['min_margin_units']>0
    stats=circuit.operation_statistics()
    rotations=int(stats.num_z_pow(min_clifford_hierarchy_level=4))
    epsilon=math.pi*(certificate['min_margin_units']/2**64)/rotations
    per_rotation=0.56*math.log2(1/epsilon)+5.3
    static_toffoli=int(stats.num_toffoli_like)
    # Fair flag measurements activate the emitted normalization phase-fixup
    # Toffolis on half the shots. Read their actual branch counts so the
    # original schedules discount 11 and the promise rewrite discounts 10.5.
    normalization_discount=conditional_toffoli_discount(circuit)
    expected_toffoli=static_toffoli-normalization_discount
    t_like=int(stats.num_t_like)
    extra=(t_like+rotations*per_rotation)/4
    return {'qubits':circuit.num_qubits,'static_toffoli':static_toffoli,
            'expected_mean_toffoli':expected_toffoli,'t_like':t_like,
            'rotations':rotations,'rotation_epsilon':epsilon,
            'estimated_expected_cost':circuit.num_qubits*(expected_toffoli+extra),
            'static_cost_upper_estimate':circuit.num_qubits*(static_toffoli+extra),
            'full_official_verification_required':True}




# ---- Our isolated phase scheduling, folded carry and borrowed workspace ----
def folded_gadd(b,a,t,cin=None,sub=False,borrowed=()):
    a,t=list(a),list(t)
    o,n=len(a),len(t)
    assert o<=n and n>=1
    if sub:
        for q in t:b.x(q)
    if n==1:
        if o:b.cx(a[0],t[0])
        if cin is not None:b.cx(cin,t[0])
        if sub:
            for q in t:b.x(q)
        return
    borrowed=list(borrowed)[:max(0,n-2)]
    count=max(0,n-2-len(borrowed))
    allocated=b.alloc_qubits(count) if count else []
    c=borrowed+list(allocated)
    def C(i):return cin if i==0 else c[i-1]
    for i in range(n-2):
        if i==0 and cin is None:
            if o:b.init_and(a[0],t[0],C(1))
        elif i<o:
            b.cx(C(i),a[i]);b.cx(C(i),t[i])
            b.init_and(a[i],t[i],C(i+1));b.cx(C(i),C(i+1))
        else:
            b.init_and(C(i),t[i],C(i+1))
    i=n-2
    carry=C(i)
    if carry is None:
        if i<o:
            b.ccx(a[i],t[i],t[-1]);b.cx(a[i],t[i])
    elif i<o:
        b.cx(carry,a[i]);b.cx(carry,t[i])
        b.ccx(a[i],t[i],t[-1]);b.cx(carry,t[-1])
        b.cx(carry,a[i]);b.cx(a[i],t[i])
    else:
        b.ccx(carry,t[i],t[-1]);b.cx(carry,t[i])
    if n-1<o:b.cx(a[n-1],t[n-1])
    for i in reversed(range(n-2)):
        if i==0 and cin is None:
            if o:b.del_and(a[0],t[0],C(1));b.cx(a[0],t[0])
        elif i<o:
            b.cx(C(i),C(i+1));b.del_and(a[i],t[i],C(i+1))
            b.cx(C(i),a[i]);b.cx(a[i],t[i])
        else:
            b.del_and(C(i),t[i],C(i+1));b.cx(C(i),t[i])
    if count:b.free(allocated)
    if sub:
        for q in t:b.x(q)


def truth_table_check(max_width=6,borrowed_count=0):
    class Bits:
        def __init__(self):self.v=[]
        def alloc_qubits(self,n):
            ids=list(range(len(self.v),len(self.v)+n));self.v.extend([0]*n);return ids
        def x(self,q):self.v[q]^=1
        def cx(self,a,t):self.v[t]^=self.v[a]
        def ccx(self,a,b,t):self.v[t]^=self.v[a]&self.v[b]
        def init_and(self,a,b,t):
            assert self.v[t]==0
            self.v[t]=self.v[a]&self.v[b]
        def del_and(self,a,b,t):
            assert self.v[t]==(self.v[a]&self.v[b])
            self.v[t]=0
        def free(self,reg):assert not any(self.v[q] for q in reg)
        def put(self,reg,value):
            for i,q in enumerate(reg):self.v[q]=(value >> i)&1
        def read(self,reg):return sum(self.v[q] << i for i,q in enumerate(reg))
    count=0
    for n in range(1,max_width+1):
        for m in range(n+1):
            for a in range(1 << m):
                for value in range(1 << n):
                    for cin_value in (None,0,1):
                        for sub in (False,True):
                            b=Bits();areg=b.alloc_qubits(m);target=b.alloc_qubits(n)
                            cin=b.alloc_qubits(1)[0] if cin_value is not None else None
                            if cin is not None:b.v[cin]=cin_value
                            b.put(areg,a);b.put(target,value)
                            borrowed=b.alloc_qubits(borrowed_count)
                            folded_gadd(b,areg,target,cin,sub,borrowed=borrowed)
                            delta=a+(cin_value or 0)
                            expected=(value-delta if sub else value+delta) % (1 << n)
                            assert b.read(target)==expected,(n,m,a,value,cin_value,sub)
                            assert b.read(areg)==a
                            assert cin is None or b.v[cin]==cin_value
                            assert not any(b.v[q] for q in borrowed)
                            count+=1
    return count


def build_split_circuit(cfg,tabs,lay,adder=None,borrow_top=False):
    if adder is None:adder=gadd
    w2,w1,LB,S,whi=cfg.w2,lay['w1'],lay['LB'],lay['S'],lay['whi']
    b=km.CircuitBuilder()
    x=b.create_quantum_register(W,name='x')
    flags=b.alloc_qubits(5)
    normalize(b,x,flags)
    # Every promised positive input normalizes to x[21]=1. The arithmetic
    # address uses x[14:21] and delta x[:14], so this wire is idle in the core.
    if borrow_top:
        b.x(x[-1])
        raw_adder=adder
        def adder(builder,a,t,cin=None,sub=False):
            return raw_adder(builder,a,t,cin,sub,borrowed=(x[-1],))
    c2=b.alloc_qubits(w2)
    R=b.alloc_qubits(w1)
    Lr=b.alloc_qubits(LB)
    Yh=b.alloc_qubits(whi)
    remaining=km.array(list(Lr)+list(R)+list(c2))
    target=km.array(list(remaining)+list(Yh))
    table=[0]*256
    rest_table=[0]*256
    high_table=[0]*256
    for (e,a),(K2,K1,K0) in tabs.items():
        idx=a+128*e
        rest_table[idx]=lay['lo'][(e,a)] | (K1 << LB) | (K2 << (LB+w1))
        high_table[idx]=lay['hi'][(e,a)]
        table[idx]=rest_table[idx] | (high_table[idx] << (LB+w1+w2))
    address=km.array(list(x[14:21])+[flags[0]])
    delta=x[:DB]
    def rrows(inverse):
        seq=cfg.rrow_list()
        if inverse:seq=list(reversed(seq))
        for i in seq:
            sh=cfg.sh2[i]
            if sh>=0:off=c2[sh:w2];t=R[:w1];cin=c2[sh-1] if sh>=1 else None
            else:off=c2[:w2];t=R[-sh:w1];cin=None
            b.cx(delta[i],t);adder(b,off,t,cin,sub=inverse);b.cx(delta[i],t)
    def yrows(inverse):
        seq=list(reversed(range(DB)))
        if inverse:seq.reverse()
        for i in seq:
            sh=cfg.sh1[i]
            assert 0<=sh<w1
            cin=R[sh-1] if sh>=1 else None
            b.cx(delta[i],Lr);adder(b,R[sh:w1],Lr,cin,sub=inverse);b.cx(delta[i],Lr)
    def phase(reg,base_bit):
        ext=b.alloc_qubits(10)
        word=km.array(list(reg)+list(ext))
        shift_stages(b,flags,word,len(reg),False)
        for j in range(len(word)):
            exponent=base_bit+j-cfg.fy-10
            if exponent<0:b.z_pow(word[j],Fraction(2)**(exponent+1))
        shift_stages(b,flags,word,len(reg),True)
        b.free(ext)
    b.init_lookup(table=table,address=address,target=target)
    phase(Yh,S)
    b.del_lookup(table=high_table,address=address,target=Yh)
    b.free(Yh)
    rrows(False)
    yrows(False)
    phase(Lr[cfg.gy:LB],cfg.gy)
    yrows(True)
    rrows(True)
    b.del_lookup(table=rest_table,address=address,target=remaining)
    b.free(remaining)
    if borrow_top:b.x(x[-1])
    unnormalize(b,x,flags)
    b.free(flags)
    return b.finish_circuit()



def build_previous_schedule(cfg,tabs,lay=None):
    """Regenerate 91Q circuit from the explicit model and credited schedules."""
    if lay is None: lay=layout(cfg,tabs)
    return build_split_circuit(cfg,tabs,lay,adder=folded_gadd,borrow_top=True)


# The derivative-split version of this API is defined below.

# The derivative-split version of this API is defined below.



# The derivative-split version of this API is defined below.



# The derivative-split version of this API is defined below.


def cli():
    import argparse
    import json
    from pathlib import Path
    parser = argparse.ArgumentParser(description='Generate the credited signed-digit circuit and memory-schedule adaptation.')
    parser.add_argument('output', nargs='?', default='invsqrt.kmb')
    parser.add_argument('--baseline', action='store_true')
    args = parser.parse_args()
    cfg, tabs, _ = load_or_build_tables()
    certificate = exhaustive_certificate(cfg, tabs)
    if not certificate['pass']:
        raise ArithmeticError(certificate)
    lay = layout(cfg, tabs)
    circuit = build_memory_baseline(cfg, tabs, lay) if args.baseline else build_selected(cfg, tabs, lay)
    circuit.to_file(args.output, format='kmb')
    report = estimated_resources(circuit, certificate)
    report['arithmetic_certificate'] = certificate
    report['variant'] = 'sumit_baseline' if args.baseline else selected_descriptor()['variant']
    Path(args.output + '.resources.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))





def _rudra_gadd_top_reused(b, source, target, cin=None, sub=False):
    """Gidney addition with n-2 clean carries; the last carry targets the MSB."""
    import kickmix as km
    a, t = list(source), list(target)
    n, m = len(t), len(a)
    assert n >= 1 and m <= n
    if sub:
        b.x(km.array(t))
    carries = b.alloc_qubits(n - 2) if n > 2 else km.array([])
    for i in range(n - 1):
        previous = cin if i == 0 else carries[i - 1]
        last = i == n - 2
        out = t[-1] if last else carries[i]
        if previous is None:
            if i < m:
                (b.ccx if last else b.init_and)(a[i], t[i], out)
        elif i < m:
            b.cx(previous, a[i]); b.cx(previous, t[i])
            (b.ccx if last else b.init_and)(a[i], t[i], out)
            b.cx(previous, out)
        else:
            (b.ccx if last else b.init_and)(previous, t[i], out)
    if n - 1 < m:
        b.cx(a[n - 1], t[n - 1])
    if n == 1 and cin is not None:
        b.cx(cin, t[0])
    for i in reversed(range(n - 1)):
        previous = cin if i == 0 else carries[i - 1]
        last = i == n - 2
        if previous is None:
            if i < m:
                if not last:
                    b.del_and(a[i], t[i], carries[i])
                b.cx(a[i], t[i])
        elif i < m:
            if not last:
                b.cx(previous, carries[i]); b.del_and(a[i], t[i], carries[i])
            b.cx(previous, a[i]); b.cx(a[i], t[i])
        else:
            if not last:
                b.del_and(previous, t[i], carries[i])
            b.cx(previous, t[i])
    if n > 2:
        b.free(carries)
    if sub:
        b.x(km.array(t))

def prepare_derivative_split(reference, cfg, tables):
    d = np.arange(1 << reference.DB, dtype=np.int64)
    coarse, residual = {}, {}
    bounds = {'normalized_addresses_checked': 0, 'min_residual': 1 << 30,
              'max_residual': 0, 'max_table_residual': 0, 'max_coarse': 0}
    for key, (k2, k1, _) in tables.items():
        derivative, _ = reference.pipeline(cfg, k2, k1, d)
        base = int(derivative.min()) >> 11
        low = derivative - (base << 11)
        coarse[key], residual[key] = base, k1 - (base << 11)
        assert 0 <= residual[key] < 1 << 12
        assert (low >= 0).all() and (low < 1 << 12).all()
        assert ((base + (low >> 11)) < 1 << 7).all()
        joined = (low & 2047) + ((base + (low >> 11)) << 11)
        assert np.array_equal(joined, derivative)
        bounds['normalized_addresses_checked'] += len(d)
        bounds['min_residual'] = min(bounds['min_residual'], int(low.min()))
        bounds['max_residual'] = max(bounds['max_residual'], int(low.max()))
        bounds['max_table_residual'] = max(bounds['max_table_residual'], residual[key])
        bounds['max_coarse'] = max(bounds['max_coarse'], base)
    assert cfg.w2 >= 6 and len(cfg.rrow_list()) == 8
    return {'coarse': coarse, 'residual': residual}, bounds

def _build_derivative_circuit(reference, cfg, tables, layout, split, borrow_top=False):
    import kickmix as km
    assert (layout['w1'], layout['LB'], layout['S'], layout['whi']) == (18, 21, 20, 8)
    b = km.CircuitBuilder()
    x = b.create_quantum_register(22, name='x')
    flags = b.alloc_qubits(5)
    # These coefficients are still zero during normalization; allocate afterward.
    reference.normalize(b, x, flags)
    if borrow_top: b.x(x[-1])
    def adder(builder,a,t,cin=None,sub=False):
        if borrow_top: return folded_gadd(builder,a,t,cin,sub,borrowed=(x[-1],))
        return _rudra_gadd_top_reused(builder,a,t,cin,sub)
    c2, residual, coarse, low, high = [b.alloc_qubits(n) for n in [cfg.w2, 12, 7, 21, 8]]
    full_target = km.array(list(low) + list(residual) + list(c2) + list(high) + list(coarse))
    rest_target = km.array(list(low) + list(residual) + list(c2) + list(coarse))
    full_table, rest_table, high_table = [0] * 256, [0] * 256, [0] * 256
    for (parity, address), (k2, _, _) in tables.items():
        key, index = (parity, address), address + 128 * parity
        common = layout['lo'][key] | (split['residual'][key] << 21) | (k2 << 33)
        full_table[index] = common | (layout['hi'][key] << (33+cfg.w2)) | (split['coarse'][key] << (41+cfg.w2))
        rest_table[index] = common | (split['coarse'][key] << (33+cfg.w2))
        high_table[index] = layout['hi'][key]
    address = km.array(list(x[14:21]) + [flags[0]])
    delta = x[:14]

    def phase_register(register, low_bit):
        ext = b.alloc_qubits(10)
        word = km.array(list(register) + list(ext))
        reference.shift_stages(b, flags, word, len(register), False)
        for i, qubit in enumerate(word):
            exponent = low_bit + i - cfg.fy - 10
            if exponent < 0:
                b.z_pow(qubit, Fraction(2) ** (exponent + 1))
        reference.shift_stages(b, flags, word, len(register), True)
        b.free(ext)

    def inner(inverse):
        rows = cfg.rrow_list()
        for i in reversed(rows) if inverse else rows:
            shift = cfg.sh2[i]
            # A narrower curvature coefficient can require an exact left shift.
            # Add to the corresponding residual suffix, leaving lower bits intact.
            if shift>=0:
                addend=c2[shift:];target=residual
                carry=c2[shift-1] if shift else None
            else:
                addend=c2;target=residual[-shift:];carry=None
            b.cx(delta[i],target)
            adder(b,addend,target,carry,sub=inverse)
            b.cx(delta[i],target)

    def outer(derivative, inverse):
        for i in range(14) if inverse else reversed(range(14)):
            shift = cfg.sh1[i]
            carry = derivative[shift - 1] if shift else None
            b.cx(delta[i], low)
            adder(b, derivative[shift:], low, carry, sub=inverse)
            b.cx(delta[i], low)

    b.init_lookup(table=full_table, address=address, target=full_target)
    phase_register(high, 20)
    # High and the shift workspace have been restored. For every X-measurement
    # mask m, unlookup contributes (-1)^(m.high(address)); its phase correction
    # contributes the same sign. Their product is +1, preserving our phase.
    # Address and its parity flag stay live and unchanged until after erasure.
    b.del_lookup(table=high_table, address=address, target=high)
    b.free(high)
    inner(False)
    adder(b, residual[11:], coarse)
    derivative = residual[:11] + coarse
    outer(derivative, False)
    # Include bit20 here. There is no output join: floor((L+2^20H)/4)
    # equals floor(L/4)+2^18H, so the two diagonal phases add exactly.
    phase_register(low[cfg.gy:], cfg.gy)
    outer(derivative, True)
    adder(b, residual[11:], coarse, sub=True)
    inner(True)
    b.del_lookup(table=rest_table, address=address, target=rest_target)
    b.free(rest_target)
    if borrow_top: b.x(x[-1])
    reference.unnormalize(b, x, flags)
    b.free(flags)
    return b.finish_circuit()


def _derivative_reference(promise):
    return types.SimpleNamespace(**dict(globals(),
        normalize=normalize if promise else _combined_normalize,
        unnormalize=unnormalize if promise else _baseline_unnormalize))


def _build_derivative_variant(cfg,tabs,lay,borrow,promise):
    if lay is None:lay=layout(cfg,tabs)
    reference=_derivative_reference(promise)
    split,bounds=prepare_derivative_split(reference,cfg,tabs)
    return _build_derivative_circuit(reference,cfg,tabs,lay,split,borrow_top=borrow)


def build_selected(cfg,tabs,lay=None):
    """Regenerate selected92Q derivative split with borrowed carry and domain rewrite."""
    return _build_derivative_variant(cfg,tabs,lay,borrow=True,promise=True)


def build_memory_baseline(cfg,tabs,lay=None):
    """Same Kozykorpesh coefficient model, Rudra/Kirill derivative schedule:93Q."""
    return _build_derivative_variant(cfg,tabs,lay,borrow=False,promise=False)


def build_borrowed_baseline(cfg,tabs,lay=None):
    """Same derivative schedule with the normalized input bit borrowed:92Q."""
    return _build_derivative_variant(cfg,tabs,lay,borrow=True,promise=False)


def build_combined_baseline(cfg,tabs,lay=None):
    """Compatibility alias for the borrowed-carry comparison."""
    return build_borrowed_baseline(cfg,tabs,lay)


def comparison_report(cfg,tabs,certificate,lay=None):
    """One fixed coefficient table: derivative schedule, borrow, domain rewrite."""
    if lay is None:lay=layout(cfg,tabs)
    return {
        'baseline':estimated_resources(build_memory_baseline(cfg,tabs,lay),certificate),
        'borrowed':estimated_resources(build_borrowed_baseline(cfg,tabs,lay),certificate),
        'selected':estimated_resources(build_selected(cfg,tabs,lay),certificate),
        'selected_descriptor':selected_descriptor()}


def selected_descriptor():
    return {
        'variant':'kozy_table_rudra_split_alex_schedule_borrow_and_domain',
        'source_author':'Sumit Chongder; Rudra Bhaskar; Alex Chilton; Kozykorpesh; Kirill Tymchishin; anazemcev / Artem1410',
        'source_url':'https://www.kaggle.com/code/kirilltym/quantum-circuits-qrom-tower42?scriptVersionId=361491814',
        'coefficient_source':'Kozykorpesh Quantum Circuit Toolbox datasetVersionId21136269, code/coefficients.json; same256-row model as our preceding91Q revision',
        'configuration':{'f2':8,'f1':19,'fyr':27,'gy':1,'drop2':[5]},
        'license_as_published':'Apache-2.0, subject to retained upstream notices',
        'baseline_definition':'Same fixed Kozykorpesh table with the Rudra derivative split and Alex normalization/shifts, before normalized-input carry borrowing:93Q',
        'baseline_canonical_sha256':'9e67c00941ff0623cfaf22f133fe7f70740f8b0a163a4ba64c6fdcd1c9446295',
        'borrowed_canonical_sha256':'f3f8457058aecdcf2650e00b411234899cb01933b79a1f6519c0bf28b49591bc',
        'selected_canonical_sha256':'63f70e623dd995f4f87ca13254f46f991a5ae545d4c20c80e8510f5ff0a49bbd',
        'historical_kirill95_sha256':'497b161bc468654b5b2570cfaa006ce336dec9b8c89f82f048fcca4e6e8f7919',
        'previous_same_table91_sha256':'e3db0cc65b4771b640fbcd73d770d2015c232e9f54bd69479786fb5445ca76b9',
        'measured_counts':{'qubits':92,'static_toffoli':1472,'rotations':45},
        'expected_mean_toffoli':1461.5,
        'estimated_cost':164645.89641938862,
        'own_changes_in_this_revision':[
            'Combine the published derivative-split representation with the previously selected explicit Kozykorpesh coefficient model',
            'Parameterize curvature storage and exact left-shift rows without changing the integer pipeline',
            'Borrow normalized x[21] as clean adder carry, restoring it before inverse normalization',
            'Retain our exact original-input top-bit mutual-exclusion rewrite'],
        'upstream_contributions':'Rudra derivative split/top-carry/static-high phase checkpoint; Alex prefix flags/empty-target shift; Kirill exact integration; Sumit model; Kozykorpesh precision calibration',
        'full_official_verification':'Required separately; actual unchanged full-verifier report identifies the selected hash'}


if __name__ == '__main__':
    cli()


In [ ]:
%%writefile src/portable_exact_domain.py
"""Portable exact-domain inverse-square-root Clifford compiler.

Requires the credited signed_digit_invsqrt_v3 module on sys.path, NumPy and
kickmix. The default entry point regenerates from readable source and explicit
coefficient data; it has no filesystem layout, fixture or network dependency.
Full truth-vector equality certifies every selected replacement over every
x=1..3,133,452. Classical branches join exact equal functions or unknown.
"""
from __future__ import annotations
import os
os.environ.update(OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
from collections import Counter
import json
import re
import numpy as np
import kickmix as km

MAX_X = 3_133_452

class Known:
    """Immutable truth vector. Samples only reject matches; equality is exact."""
    __slots__ = ('v', 'signature')
    sample = None

    def __init__(self, values):
        self.v = values
        self.signature = values[Known.sample].tobytes()

class DomainCompiler:
    def __init__(self, nqubits, max_x=MAX_X, linear=False, mbu=False, temp_bit=0):
        self.max_x = max_x
        self.nwords = (max_x + 63) // 64
        Known.sample = np.unique(np.r_[np.arange(min(32, self.nwords)),
            np.linspace(0, self.nwords - 1, min(96, self.nwords), dtype=int)])
        self.zero = Known(np.zeros(self.nwords, dtype=np.uint64))
        ones = np.full(self.nwords, np.uint64(2**64 - 1))
        if max_x % 64:
            ones[-1] = np.uint64((1 << (max_x % 64)) - 1)
        self.one = Known(ones)
        xs = np.arange(1, max_x + 1, dtype=np.uint32)
        self.original = []
        for bit in range(22):
            values = np.zeros(self.nwords * 8, dtype=np.uint8)
            packed = np.packbits(((xs >> bit) & 1).astype(np.uint8), bitorder='little')
            values[:len(packed)] = packed
            self.original.append(Known(values.view(np.uint64)))
        self.q = self.original[:] + [self.zero] * (nqubits - 22)
        self.stack = []
        self.replacements = []
        self.counts = Counter()
        self.join_unknown = 0
        self.output = []
        self.linear = linear
        self.linear_false_candidates = 0
        self.mbu = mbu
        self.temp_bit = temp_bit

    @staticmethod
    def equal(a, b):
        if a is b:
            return True
        if a is None or b is None or a.signature != b.signature:
            return False
        return bool(np.array_equal(a.v, b.v))

    def xor(self, a, b):
        if a is None or b is None:
            return None
        if a is self.zero:
            return b
        if b is self.zero:
            return a
        if a is b:
            return self.zero
        return Known(a.v ^ b.v)

    def band(self, a, b):
        if a is self.zero or b is self.zero:
            return self.zero
        if a is None or b is None:
            return None
        if a is self.one:
            return b
        if b is self.one or a is b:
            return a
        return Known(a.v & b.v)

    def literal(self, function, excluded):
        """Return an untouched live literal equal to the full truth vector."""
        if function is None:
            return None
        for q, value in enumerate(self.q):
            if q not in excluded and self.equal(function, value):
                return (q, False)
        complement = self.xor(function, self.one)
        for q, value in enumerate(self.q):
            if q not in excluded and self.equal(complement, value):
                return (q, True)
        return None

    def linear_terms(self, function, excluded):
        """Find an affine GF(2) expression, then prove it with ALL truth bits.

        Gaussian elimination over sampled bits only proposes a combination.
        It cannot certify a rewrite: the combination is always compared with
        the full domain vector before being accepted. Unsampled dependencies
        can cause missed opportunities, never invalid replacements.
        """
        if not self.linear or function is None:
            return None
        variables = [q for q, value in enumerate(self.q)
                     if q not in excluded and value is not None]
        values = [self.q[q] for q in variables] + [self.one]
        extra_words = []
        # A failed full proof supplies another exact input row to the linear
        # system. Refinement removes sample-induced dependencies rather than
        # stopping at the first invalid proposed combination.
        while True:
            def signature(value):
                raw = value.signature
                if extra_words:
                    raw += value.v[extra_words].tobytes()
                return int.from_bytes(raw, 'little')
            basis = {}
            for index, value in enumerate(values):
                vector = signature(value)
                combination = 1 << index
                while vector:
                    pivot = vector.bit_length()
                    if pivot not in basis:
                        basis[pivot] = vector, combination
                        break
                    existing, coefficients = basis[pivot]
                    vector ^= existing
                    combination ^= coefficients
            vector = signature(function)
            combination = 0
            while vector:
                pivot = vector.bit_length()
                if pivot not in basis:
                    return None
                existing, coefficients = basis[pivot]
                vector ^= existing
                combination ^= coefficients
            selected = [i for i in range(len(values)) if combination >> i & 1]
            exact = self.zero
            for i in selected:
                exact = self.xor(exact, values[i])
            if self.equal(exact, function):
                constant = len(variables) in selected
                return [variables[i] for i in selected if i < len(variables)], constant
            self.linear_false_candidates += 1
            witness = int(np.flatnonzero(exact.v != function.v)[0])
            assert witness not in extra_words
            extra_words.append(witness)

    def record(self, line_number, original, emitted, reason):
        self.replacements.append({'line': line_number, 'original': original,
            'replacement': emitted, 'reason': reason, 'condition_depth': len(self.stack)})
        self.counts[reason] += 1

    def execute(self, line, number):
        parts = line.split()
        if not parts:
            return
        op = parts[0]
        qs = [int(s[1:]) for s in parts[1:] if re.fullmatch(r'q\d+', s)]
        emitted = [line]
        # Conditional inline phases have no effect on computational functions.
        if op in {'Z', 'CZ', 'Z_POW', 'NEG', 'INFO_TAG', 'REGISTER',
                  'APPEND_TO_REGISTER', 'BIT_STORE0', 'BIT_INVERT'}:
            pass
        elif op == 'PUSH_CONDITION':
            self.stack.append(self.q[:])
        elif op == 'POP_CONDITION':
            before = self.stack.pop()
            for q, old in enumerate(before):
                if not self.equal(old, self.q[q]):
                    self.q[q] = None
                    self.join_unknown += 1
        elif op in {'HMR', 'R'}:
            assert 'if' not in parts, 'Conditional measurement/reset needs a joined branch.'
            self.q[qs[0]] = self.zero
        elif op == 'X':
            assert 'if' not in parts, 'Inline basis condition is unsupported.'
            self.q[qs[0]] = self.xor(self.q[qs[0]], self.one)
        elif op == 'CX':
            assert 'if' not in parts, 'Inline basis condition is unsupported.'
            a, b = qs
            self.q[b] = self.xor(self.q[b], self.q[a])
        elif op == 'SWAP':
            assert 'if' not in parts, 'Inline basis condition is unsupported.'
            a, b = qs
            self.q[a], self.q[b] = self.q[b], self.q[a]
        elif op == 'CCX':
            assert 'if' not in parts, 'Inline basis condition is unsupported.'
            a, b, t = qs
            product = self.band(self.q[a], self.q[b])
            reason = None
            if self.equal(product, self.zero):
                emitted, reason = [], 'ccx_zero_product'
            elif self.equal(product, self.one):
                emitted, reason = [f'X q{t}'], 'ccx_constant_one'
            else:
                literal = self.literal(product, {t})
                if literal is not None:
                    q, inverted = literal
                    emitted = ([f'X q{t}'] if inverted else []) + [f'CX q{q} q{t}']
                    reason = 'ccx_live_complement' if inverted else 'ccx_live_literal'
                else:
                    terms = self.linear_terms(product, {t})
                    if terms is not None:
                        variables, constant = terms
                        emitted = ([f'X q{t}'] if constant else [])
                        emitted += [f'CX q{q} q{t}' for q in variables]
                        reason = 'ccx_affine_live_span'
            if reason is None and self.mbu:
                result = self.xor(self.q[t], product)
                terms = None
                if self.equal(result, self.zero):
                    terms = [], False
                elif self.equal(result, self.one):
                    terms = [], True
                else:
                    literal = self.literal(result, {t})
                    if literal is not None:
                        q, inverted = literal
                        terms = [q], inverted
                    elif self.linear:
                        terms = self.linear_terms(result, {t})
                if terms is not None:
                    variables, constant = terms
                    prepare = ([f'X q{t}'] if constant else [])
                    prepare += [f'CX q{q} q{t}' for q in variables]
                    # Before: t = a*b XOR L(other wires).  XOR L makes t=a*b.
                    # HMR contributes (-1)**(m*a*b); conditioned CZ cancels it.
                    # The same Clifford expression then restores t=L.
                    emitted = prepare + [f'HMR q{t} b{self.temp_bit}',
                        f'CZ q{a} q{b} if b{self.temp_bit}'] + prepare
                    reason = 'ccx_output_affine_mbu'
            if reason:
                self.record(number, line, emitted, reason)
            self.q[t] = self.xor(self.q[t], product)
        elif op == 'CSWAP':
            assert 'if' not in parts, 'Inline basis condition is unsupported.'
            c, a, b = qs
            delta = self.xor(self.q[a], self.q[b])
            product = self.band(self.q[c], delta)
            reason = None
            if self.equal(product, self.zero):
                emitted, reason = [], 'cswap_zero_delta'
            elif self.equal(product, delta) and delta is not None:
                emitted, reason = [f'SWAP q{a} q{b}'], 'cswap_unconditional'
            else:
                literal = self.literal(product, {a, b})
                if literal is not None:
                    q, inverted = literal
                    emitted = ([f'X q{a}', f'X q{b}'] if inverted else [])
                    emitted += [f'CX q{q} q{a}', f'CX q{q} q{b}']
                    reason = 'cswap_live_complement' if inverted else 'cswap_live_literal'
                else:
                    terms = self.linear_terms(product, {a, b})
                    if terms is not None:
                        variables, constant = terms
                        emitted = ([f'X q{a}', f'X q{b}'] if constant else [])
                        emitted += [gate for q in variables
                                    for gate in (f'CX q{q} q{a}', f'CX q{q} q{b}')]
                        reason = 'cswap_affine_live_span'
            if reason:
                self.record(number, line, emitted, reason)
            self.q[a] = self.xor(self.q[a], product)
            self.q[b] = self.xor(self.q[b], product)
        else:
            raise AssertionError(f'Unsupported operation {op}: {line}')
        self.output.extend(emitted)

    def compile(self, circuit):
        for number, line in enumerate(str(circuit).splitlines(), 1):
            self.execute(line, number)
            if number % 5000 == 0:
                print(json.dumps({'line': number, 'replacements': len(self.replacements),
                    'unknown_qubits': sum(v is None for v in self.q)}), flush=True)
        assert not self.stack
        assert all(self.equal(self.q[q], self.original[q]) for q in range(22)), 'Input function lost.'
        assert all(self.equal(v, self.zero) for v in self.q[22:]), 'Scratch function is not zero.'
        return km.Circuit('\n'.join(self.output) + '\n')

def optimize_circuit(circuit, linear=False, mbu=False):
    """Open in-memory compiler API; no circuit binary or network is required."""
    compiler = DomainCompiler(circuit.num_qubits, linear=linear, mbu=mbu,
                              temp_bit=circuit.num_bits)
    return compiler.compile(circuit), compiler

def build_selected(cfg=None, tables=None, layout=None):
    """Regenerate the credited frozen arithmetic, then prove five reductions.

    This entry point depends only on the readable frozen source and its
    explicit coefficient data. The CLI's existing KMB is an independent
    canonical regression fixture rather than a generator dependency.
    """
    import signed_digit_invsqrt_v3 as core
    if cfg is None or tables is None:
        assert cfg is None and tables is None
        cfg, tables, _ = core.load_or_build_tables()
    source = core.build_selected(cfg, tables, layout)
    circuit, compiler = optimize_circuit(source)
    assert len(compiler.replacements) == 5
    return circuit


In [ ]:
%%writefile src/scheduled_arithmetic90.py
# SPDX-License-Identifier: Apache-2.0
# Kozykorpesh public V3/run361677445, https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445
# Source-only adaptation: explicit coefficients supplied by caller; no Toolbox or binary input.
# Original notices and credited Sumit/Rudra/Alex/Kirill/Artem lineage retained below.
# SPDX-License-Identifier: Apache-2.0
# Retained source credits: Sumit Chongder (signed digits/Gidney arithmetic),
# Rudra Bhaskar (derivative split), Alex Chilton (prefix/AND shifts),
# Kirill Tymchishin (integration), Kozykorpesh (calibrated coefficients),
# anazemcev / Artem1410 (memory schedule and input-domain borrowing).
# This module retains the required arithmetic from the supplied 5.77242 notebook.
import math
from fractions import Fraction
import numpy as np
import kickmix as km
W=22; MB=21; DB=14; NSEG=128; MAX_X=3133452
NSTAGES=[(4,16),(3,8),(2,4),(1,2),(0,1)]


class Cfg:
    def __init__(self, f2, f1, fyr, gy, drop2=()):
        self.f2, self.f1, self.fyr, self.gy = f2, f1, fyr, gy
        self.fy = fyr + gy
        self.w2 = f2 - 1
        self.sh2 = [f2 + 22 - i - f1 for i in range(DB)]
        self.sh1 = [f1 + 22 - i - self.fy for i in range(DB)]
        self.drop2 = set(drop2)
    def rrow_list(self):
        return [i for i in reversed(range(DB)) if i not in self.drop2 and self.sh2[i] < self.w2]


def rd(v, s):
    if s < 0: return v << (-s)
    r = v >> s
    if s >= 1: r = r + ((v >> (s - 1)) & 1)
    return r


def pipeline(cfg, K2, K1, d):
    sig = [2 * ((d >> i) & 1) - 1 for i in range(DB)]
    R = np.full_like(d, K1)
    for i in cfg.rrow_list():
        R = R - sig[i] * rd(K2, cfg.sh2[i])
    Q = np.zeros_like(d)
    for i in reversed(range(DB)):
        Q = Q + sig[i] * rd(R, cfg.sh1[i])
    return R, Q


def layout(cfg, tabs, LB=21, S=20):
    d = np.arange(2**DB, dtype=np.int64)
    assert max(t[0] for t in tabs.values()) < (1 << cfg.w2)
    Rmax = 0; per = {}; Ymax = 0
    for key, (K2, K1, K0) in tabs.items():
        R, Q = pipeline(cfg, K2, K1, d)
        assert R.min() >= 0
        Rmax = max(Rmax, int(R.max()), K1)
        per[key] = (int(Q.min()), int(Q.max()))
        Ymax = max(Ymax, K0 - per[key][0])
    w1 = Rmax.bit_length()
    lo_tab, hi_tab = {}, {}
    for key, (K2, K1, K0) in tabs.items():
        qmin, qmax = per[key]
        base = K0 % (1 << S); j = 0
        while base + j * (1 << S) - qmax < 0: j += 1
        k0lo = base + j * (1 << S)
        assert k0lo - qmin < (1 << LB), 'layout infeasible'
        lo_tab[key] = k0lo; hi_tab[key] = (K0 - k0lo) >> S
        assert hi_tab[key] >= 0
    whi = max(max(hi_tab.values()).bit_length(), 1)
    return dict(w1=w1, LB=LB, S=S, whi=whi, lo=lo_tab, hi=hi_tab)


def phase_all(b, controls):
    k = len(controls)
    if k == 1: b.z(controls[0]); return
    if k == 2: b.cz(controls[0], controls[1]); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.cz(anc[k - 3], controls[k - 1])
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)


def normalize(b, x, f):
    """v3 step 1: f4, f3, f2 (the binary leading-zero count div 4) from one prefix-AND chain on the unshifted x.
    chain[m] = [x[W-m-2:W] == 0] (top m+2 bits zero); p_j = chain[4j-2] = [lz >= 4j].  f4 = p4, f3 = p2 ^ p4,
    f2 = p1 ^ p2 ^ p3 ^ p4 ^ p5 (nested prefixes, so XOR = bit 2 of lz).  19 ANDs instead of 15 + 7 + 3."""
    bits = [x[W - 1 - m] for m in range(20)]                  # x21, x20, ..., x2
    b.x(km.array(bits))
    chain = b.alloc_qubits(19)
    b.x(chain[0]); b.cx(bits[0],chain[0]); b.cx(bits[1],chain[0])
    for m in range(1, 19): b.init_and(chain[m - 1], bits[m + 1], chain[m])
    p = {j: chain[4 * j - 2] for j in range(1, 6)}
    b.cx(p[4], f[4])
    b.cx(p[2], f[3]); b.cx(p[4], f[3])
    for j in range(1, 6): b.cx(p[j], f[2])
    for m in reversed(range(1, 19)): b.del_and(chain[m - 1], bits[m + 1], chain[m])
    b.cx(bits[1],chain[0]); b.cx(bits[0],chain[0]); b.x(chain[0])
    b.free(chain)
    b.x(km.array(bits))
    for (k, d) in NSTAGES:
        top = x[W - d:W]
        if d == 1:
            b.cx(x[W - 1], f[0]); b.x(f[0])
        elif d == 2:
            b.x(top); b.ccx(top[0], top[1], f[1]); b.x(top)
        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])


def promise_phase_original_top(b,controls):
    """Phase the original top16-zero predicate using its promised bit relation."""
    assert len(controls)==16
    controls=list(reversed(controls))
    k=len(controls)
    anc=b.alloc_qubits(k-2)
    b.x(anc[0]); b.cx(controls[0],anc[0]); b.cx(controls[1],anc[0])
    for j in range(1,k-2):
        b.init_and(anc[j-1],controls[j+1],anc[j])
    b.cz(anc[k-3],controls[k-1])
    for j in reversed(range(1,k-2)):
        b.del_and(anc[j-1],controls[j+1],anc[j])
    b.cx(controls[1],anc[0]); b.cx(controls[0],anc[0]); b.x(anc[0])
    b.free(anc)


def unnormalize(b, x, f):
    for (k, d) in reversed(NSTAGES):
        for i in range(0, W - d): b.cswap(f[k], x[i], x[i + d])
        top = x[W - d:W]
        if d == 1:
            b.x(f[0]); b.cx(x[W - 1], f[0])
        else:
            m = b.alloc_bits(1)[0]
            b.hmr(f[k], m)
            b.push_condition(m)                     # fixup runs on half the shots on average
            b.x(top)
            if d==16: promise_phase_original_top(b,list(top))
            else: phase_all(b,list(top))
            b.x(top)
            b.pop_condition()
            b.free(m)


def shift_stages(b, f, reg, L, inverse):
    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]
    occ = []; o = L
    for (k, d) in stages: occ.append(o); o = min(o + d, len(reg))
    seq = list(enumerate(stages))
    if not inverse:
        for idx, (k, d) in seq:
            for i in reversed(range(0, occ[idx])):
                if i + d >= len(reg): continue
                if i + d >= occ[idx]:                       # v2: target is |0> for every f
                    b.init_and(f[k], reg[i], reg[i + d]); b.cx(reg[i + d], reg[i])
                else:
                    b.cswap(f[k], reg[i], reg[i + d])
    else:
        for idx, (k, d) in reversed(seq):
            for i in range(0, occ[idx]):
                if i + d >= len(reg): continue
                if i + d >= occ[idx]:
                    b.cx(reg[i + d], reg[i]); b.del_and(f[k], reg[i], reg[i + d])
                else:
                    b.cswap(f[k], reg[i], reg[i + d])


def integer_model(xs,cfg,tabs):
    """Exact emitted arithmetic in units 2^-(cfg.fy+10) turns, before mod 1."""
    xs=np.asarray(xs,dtype=np.int64)
    if not ((xs>=1)&(xs<=3133452)).all():
        raise ValueError('The supported competition domain is x=1..3133452')
    # All inputs fit exactly in float64; frexp gives the exact binary exponent.
    shifts=21-(np.frexp(xs)[1].astype(np.int64)-1)
    norm=xs << shifts
    assert ((norm >> 21)==1).all(), 'Normalized top-bit borrowing requires x[21]=1'
    parity=shifts&1
    index=((norm >> 14)&127)+128*parity
    table=np.asarray([tabs[(e,a)] for e in range(2) for a in range(128)],dtype=np.int64)
    K2,K1,K0=(table[index,j] for j in range(3))
    d=norm&((1 << DB)-1)
    signs=[2*((d >> i)&1)-1 for i in range(DB)]
    R=K1.copy()
    for i in cfg.rrow_list():
        R-=signs[i]*rd(K2,cfg.sh2[i])
    Q=np.zeros_like(xs)
    for i in reversed(range(DB)):
        Q+=signs[i]*rd(R,cfg.sh1[i])
    Y=((K0-Q) >> cfg.gy) << cfg.gy
    if (R<0).any() or (Y<0).any():
        raise OverflowError('An unsigned intermediate is negative')
    return Y << (shifts >> 1)


def conditional_toffoli_discount(circuit):
    """Mean discount for this emitted family's fair HMR conditional branches."""
    depth=0
    total=0
    discount=0.
    for line in str(circuit).splitlines():
        opcode=line.split(' ',1)[0]
        if opcode=='PUSH_CONDITION':depth+=1
        elif opcode=='POP_CONDITION':depth-=1
        elif opcode in ['CCX','CSWAP']:
            total+=1
            if depth:
                assert depth==1,'Expected one fair measurement bit in a Toffoli branch.'
                discount+=.5
    assert total==int(circuit.operation_statistics().num_toffoli_like)
    assert depth==0
    return discount


def estimated_resources(circuit,certificate):
    """Model-based cost; full unchanged official verification remains required."""
    assert certificate['pass'] and certificate['min_margin_units']>0
    stats=circuit.operation_statistics()
    rotations=int(stats.num_z_pow(min_clifford_hierarchy_level=4))
    epsilon=math.pi*(certificate['min_margin_units']/2**64)/rotations
    per_rotation=0.56*math.log2(1/epsilon)+5.3
    static_toffoli=int(stats.num_toffoli_like)
    # Fair flag measurements activate the emitted normalization phase-fixup
    # Toffolis on half the shots. Read their actual branch counts so the
    # original schedules discount 11 and the promise rewrite discounts 10.5.
    normalization_discount=conditional_toffoli_discount(circuit)
    expected_toffoli=static_toffoli-normalization_discount
    t_like=int(stats.num_t_like)
    extra=(t_like+rotations*per_rotation)/4
    return {'qubits':circuit.num_qubits,'static_toffoli':static_toffoli,
            'expected_mean_toffoli':expected_toffoli,'t_like':t_like,
            'rotations':rotations,'rotation_epsilon':epsilon,
            'estimated_expected_cost':circuit.num_qubits*(expected_toffoli+extra),
            'static_cost_upper_estimate':circuit.num_qubits*(static_toffoli+extra),
            'full_official_verification_required':True}


def folded_gadd(b,a,t,cin=None,sub=False,borrowed=()):
    a,t=list(a),list(t)
    o,n=len(a),len(t)
    assert o<=n and n>=1
    if sub:
        for q in t:b.x(q)
    if n==1:
        if o:b.cx(a[0],t[0])
        if cin is not None:b.cx(cin,t[0])
        if sub:
            for q in t:b.x(q)
        return
    borrowed=list(borrowed)[:max(0,n-2)]
    count=max(0,n-2-len(borrowed))
    allocated=b.alloc_qubits(count) if count else []
    c=borrowed+list(allocated)
    def C(i):return cin if i==0 else c[i-1]
    for i in range(n-2):
        if i==0 and cin is None:
            if o:b.init_and(a[0],t[0],C(1))
        elif i<o:
            b.cx(C(i),a[i]);b.cx(C(i),t[i])
            b.init_and(a[i],t[i],C(i+1));b.cx(C(i),C(i+1))
        else:
            b.init_and(C(i),t[i],C(i+1))
    i=n-2
    carry=C(i)
    if carry is None:
        if i<o:
            b.ccx(a[i],t[i],t[-1]);b.cx(a[i],t[i])
    elif i<o:
        b.cx(carry,a[i]);b.cx(carry,t[i])
        b.ccx(a[i],t[i],t[-1]);b.cx(carry,t[-1])
        b.cx(carry,a[i]);b.cx(a[i],t[i])
    else:
        b.ccx(carry,t[i],t[-1]);b.cx(carry,t[i])
    if n-1<o:b.cx(a[n-1],t[n-1])
    for i in reversed(range(n-2)):
        if i==0 and cin is None:
            if o:b.del_and(a[0],t[0],C(1));b.cx(a[0],t[0])
        elif i<o:
            b.cx(C(i),C(i+1));b.del_and(a[i],t[i],C(i+1))
            b.cx(C(i),a[i]);b.cx(a[i],t[i])
        else:
            b.del_and(C(i),t[i],C(i+1));b.cx(C(i),t[i])
    if count:b.free(allocated)
    if sub:
        for q in t:b.x(q)


def prepare_derivative_split(reference, cfg, tables):
    d = np.arange(1 << reference.DB, dtype=np.int64)
    coarse, residual = {}, {}
    bounds = {'normalized_addresses_checked': 0, 'min_residual': 1 << 30,
              'max_residual': 0, 'max_table_residual': 0, 'max_coarse': 0}
    for key, (k2, k1, _) in tables.items():
        derivative, _ = reference.pipeline(cfg, k2, k1, d)
        base = int(derivative.min()) >> 11
        low = derivative - (base << 11)
        coarse[key], residual[key] = base, k1 - (base << 11)
        assert 0 <= residual[key] < 1 << 12
        assert (low >= 0).all() and (low < 1 << 12).all()
        assert ((base + (low >> 11)) < 1 << 7).all()
        joined = (low & 2047) + ((base + (low >> 11)) << 11)
        assert np.array_equal(joined, derivative)
        bounds['normalized_addresses_checked'] += len(d)
        bounds['min_residual'] = min(bounds['min_residual'], int(low.min()))
        bounds['max_residual'] = max(bounds['max_residual'], int(low.max()))
        bounds['max_table_residual'] = max(bounds['max_table_residual'], residual[key])
        bounds['max_coarse'] = max(bounds['max_coarse'], base)
    assert cfg.w2 >= 6 and len(cfg.rrow_list()) == 8
    return {'coarse': coarse, 'residual': residual}, bounds



In [ ]:
%%writefile src/schedule_support90.py
# SPDX-License-Identifier: Apache-2.0
# Kozykorpesh public V3/run361677445, https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445
# Source-only adaptation: explicit coefficients supplied by caller; no Toolbox or binary input.
# Original notices and credited Sumit/Rudra/Alex/Kirill/Artem lineage retained below.
# Optimized shift adapted from Dmitrii Gluzdov V6/run361592338; original arithmetic credits retained.
import numpy as np

def optimized_shift(b,flags,word,L,inverse,prune=True):
 stages=[(1,1),(2,2),(3,4),(4,8)];occ=[];n=L
 for k,d in stages:
  occ.append(min(n,L+2) if prune and k==4 else n);n=min(n+d,len(word))
 seq=list(enumerate(stages));seq=seq[::-1] if inverse else seq
 for j,(k,d) in seq:
  indices=range(occ[j]) if inverse else reversed(range(occ[j]))
  for i in indices:
   if i+d<len(word):
    zero_boundary = i+d >= L+sum(amount for _,amount in stages[:j])
    if zero_boundary:
     if inverse:b.cx(word[i+d],word[i]);b.del_and(flags[k],word[i],word[i+d])
     else:b.init_and(flags[k],word[i],word[i+d]);b.cx(word[i+d],word[i])
    else:b.cswap(flags[k],word[i],word[i+d])


def mobius_transform(table):
    """Packed-output Boolean truth table <-> ANF coefficient table over GF(2)."""
    result = list(map(int, table))
    if not result or len(result) & (len(result) - 1):
        raise ValueError("truth-table length must be a power of two")
    n = (len(result) - 1).bit_length()
    for bit in range(n):
        step = 1 << bit
        for start in range(0, len(result), 2 * step):
            for offset in range(step):
                result[start + step + offset] ^= result[start + offset]
    return result



In [ ]:
%%writefile src/fusion_schedule90.py
# SPDX-License-Identifier: Apache-2.0
# Kozykorpesh public V3/run361677445, https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445
# Source-only adaptation: explicit coefficients supplied by caller; no Toolbox or binary input.
# Original notices and credited Sumit/Rudra/Alex/Kirill/Artem lineage retained below.
"""Deferred diagonal feed-forward and live-range reduction for VQCC.
SPDX-License-Identifier: Apache-2.0
Builds on the attributed signed_digit_lab and banked_lab implementations.
"""
from fractions import Fraction
from functools import lru_cache
import numpy as np
import kickmix as km
import scheduled_arithmetic90 as H
from schedule_support90 import optimized_shift
from schedule_support90 import mobius_transform


def xor_many_toffoli(b,a,c,targets):
    targets=list(targets)
    if not targets:return
    first=targets[0]
    for t in targets[1:]:b.cx(first,t)
    b.ccx(a,c,first)
    for t in reversed(targets[1:]):b.cx(first,t)


def init_anf(b,table,address,target,borrowed=()):
    """XOR lookup with leaf products written directly into dirty output wires."""
    address,target=list(address),list(target);n=len(address)
    coef=mobius_transform(table)
    required=max(0,n-2);loan=list(borrowed)[:required]
    extra=list(b.alloc_qubits(required-len(loan)))
    scratch=loan+extra
    def emit(mask,control):
        ts=[target[j] for j in range(len(target)) if (coef[mask]>>j)&1]
        if ts:
            if control is None:b.x(km.array(ts))
            else:b.cx(control,km.array(ts))
    def walk(mask,last,control):
        emit(mask,control)
        for bit in range(last+1,n):
            child=mask|(1<<bit)
            if not mask:walk(child,bit,address[bit])
            elif bit==n-1:
                ts=[target[j] for j in range(len(target)) if (coef[child]>>j)&1]
                xor_many_toffoli(b,control,address[bit],ts)
            else:
                q=scratch[mask.bit_count()-1]
                b.init_and(control,address[bit],q);walk(child,bit,q)
                b.del_and(control,address[bit],q)
    walk(0,-1,None);b.free(km.array(extra))


def phase_from_records(b,table,address,records):
    """Correct sum_i m_i f_i(address) once, after all address-preserving work."""
    address,records=list(address),list(records);n=len(address);k=n//2
    coef=mobius_transform(table)
    left={1<<j:address[j] for j in range(k)}
    masks=[m for m in range(1,1<<k) if m.bit_count()>1]
    scratch=list(b.alloc_qubits(len(masks)+n-k-1))
    for mask,q in zip(masks,scratch):
        bit=mask.bit_length()-1;b.init_and(left[mask^(1<<bit)],address[bit],q);left[mask]=q
    carry=scratch[len(masks):];parity=b.alloc_bits(1)[0]
    def emit(high,hq):
        for low in range(1<<k):
            v=coef[low|(high<<k)]
            if not v:continue
            b.bit_store(parity,False)
            while v:
                bit=(v&-v).bit_length()-1;b.cx(records[bit],parity);v&=v-1
            if not high and not low:b.neg_if(parity);continue
            b.push_condition(parity)
            if not low:b.z(hq)
            elif not high:b.z(left[low])
            else:b.cz(left[low],hq)
            b.pop_condition()
    def walk(mask,last,q):
        emit(mask,q)
        for j in range(last+1,n-k):
            child=mask|(1<<j)
            if not mask:walk(child,j,address[k+j])
            else:
                t=carry[mask.bit_count()-1];b.init_and(q,address[k+j],t)
                walk(child,j,t);b.del_and(q,address[k+j],t)
    walk(0,-1,None)
    for mask in reversed(masks):
        bit=mask.bit_length()-1;b.del_and(left[mask^(1<<bit)],address[bit],left[mask])
    b.free(km.array(scratch));b.free(parity)


def boolean_plan(values,order=(7,4,2,3,0,1,6,5),polarity=104):
    """Positive Davio XOR/AND graph. Fixed deterministic algebra, no sampling."""
    n=len(order);indices=np.arange(1<<n)
    original=sum(((indices>>j)&1)<<order[j] for j in range(n))^polarity
    truth=int.from_bytes(np.packbits(np.asarray(values,dtype=np.uint8)[original],bitorder='little').tobytes(),'little')
    nodes=[];node_ids={}
    def xor(a,b):return(a[0]^b[0],a[1]^b[1])
    @lru_cache(None)
    def rec(t,n):
        one=(1<<(1<<n))-1
        if t==0:return (0,frozenset())
        if t==one:return (1,frozenset())
        half=(1<<(1<<(n-1)))-1;lo=t&half;hi=t>>(1<<(n-1))
        left=rec(lo,n-1);right=rec(lo^hi,n-1)
        variable=(0,frozenset((n-1,)))
        if not right[1]:return xor(left,variable) if right[0] else left
        key=(n-1,right)
        if key not in node_ids:node_ids[key]=len(order)+len(nodes);nodes.append(key)
        return xor(left,(0,frozenset((node_ids[key],))))
    result=rec(truth,n)
    return dict(order=tuple(order),polarity=polarity,nodes=nodes,result=result)


def restore_boolean(b,plan,address,target):
    address=list(address);n=len(address);order=plan['order'];pol=plan['polarity']
    flipped=[address[j] for j in range(n) if(pol>>j)&1]
    if flipped:b.x(km.array(flipped))
    wires=[address[j] for j in order]
    nodes=list(b.alloc_qubits(len(plan['nodes'])));parity=b.alloc_qubits(1)[0]
    wires+=nodes
    def form(f):
        if f[0]:b.x(parity)
        for j in sorted(f[1]):b.cx(wires[j],parity)
    for i,(variable,right) in enumerate(plan['nodes']):
        form(right);b.init_and(wires[variable],parity,nodes[i]);form(right)
    constant,terms=plan['result']
    if constant:b.x(target)
    for term in sorted(terms):b.cx(wires[term],target)
    for i in reversed(range(len(nodes))):
        variable,right=plan['nodes'][i];form(right)
        b.del_and(wires[variable],parity,nodes[i]);form(right)
    b.free(km.array(nodes));b.free(parity)
    if flipped:b.x(km.array(flipped))


def build(cfg,tables,*,fused=True,anf=True,borrow_phase=True,erase_curvature=True,
          delay_high=False,gray_checkpoint=False,late_coarse=0,curve_count=2,boolean_choices=None):
    boolean_choices=boolean_choices or {}
    assert 0<=late_coarse<=3 and 1<=curve_count<=7
    def chosen_plan(values,name,default_order=(7,4,2,3,0,1,6,5),default_pol=104):
        choice=boolean_choices.get(name,{})
        return boolean_plan(values,order=choice.get("order",default_order),polarity=choice.get("polarity",default_pol))
    extra_reload=0
    lay=H.layout(cfg,tables)
    assert (lay['w1'],lay['LB'],lay['S'],lay['whi'])==(18,21,20,8)
    split,bounds=H.prepare_derivative_split(H,cfg,tables)
    b=km.CircuitBuilder();x=b.create_quantum_register(22,name='x');flags=b.alloc_qubits(5)
    H.normalize(b,x,flags);b.x(x[-1])
    c2,residual,coarse,low,high=[b.alloc_qubits(n) for n in [cfg.w2,12,7-late_coarse,21,8]]
    full_target=km.array(list(low)+list(residual)+list(c2)+list(high)+list(coarse))
    rest_target=km.array(list(residual)+list(c2)+list(coarse))
    full_table=[0]*256;rest_table=[0]*256;high_table=[0]*256;low_table=[0]*256;c2bit_table=[0]*256
    for (e,a),(k2,_,_) in tables.items():
        key=(e,a);idx=a+128*e
        common=lay['lo'][key]|split['residual'][key]<<21|k2<<33
        full_table[idx]=common|lay['hi'][key]<<(33+cfg.w2)|split['coarse'][key]<<(41+cfg.w2)
        rest_table[idx]=split['residual'][key]|k2<<12|split['coarse'][key]<<(12+cfg.w2)
        high_table[idx]=lay['hi'][key];low_table[idx]=lay['lo'][key];c2bit_table[idx]=(k2>>6)&1
    address=km.array(list(x[14:21])+[flags[0]]);delta=x[:14]
    measured=[]
    def erase(reg,table):
        ms=b.alloc_bits(len(reg));b.hmr(reg,ms)
        measured.append((list(ms),list(table)))
        return list(ms)
    def phase(reg,low_bit):
        n_ext=10-int(borrow_phase)
        ext=b.alloc_qubits(n_ext)
        word=km.array(list(reg)+list(ext)+([x[-1]] if borrow_phase else []))
        optimized_shift(b,flags,word,len(reg),False,prune=True)
        for i,q in enumerate(word):
            exponent=low_bit+i-cfg.fy-10
            if exponent<0:b.z_pow(q,Fraction(2)**(exponent+1))
        optimized_shift(b,flags,word,len(reg),True,prune=True);b.free(ext)
    extra_borrow=[]
    def adder(a,t,cin=None,sub=False):H.folded_gadd(b,a,t,cin,sub,borrowed=(x[-1],*extra_borrow))
    def inner(inverse):
        rows=cfg.rrow_list()
        for i in reversed(rows) if inverse else rows:
            shift=cfg.sh2[i]
            if shift>=0:addend=c2[shift:];target=residual;carry=c2[shift-1] if shift else None
            else:addend=c2;target=residual[-shift:];carry=None
            b.cx(delta[i],target);adder(addend,target,carry,sub=inverse);b.cx(delta[i],target)
    def outer(derivative,inverse):
        for i in range(14) if inverse else reversed(range(14)):
            shift=cfg.sh1[i];carry=derivative[shift-1] if shift else None
            b.cx(delta[i],low);adder(derivative[shift:],low,carry,sub=inverse);b.cx(delta[i],low)
    if anf:init_anf(b,full_table,address,full_target,borrowed=(x[-1],))
    else:b.init_lookup(table=full_table,address=address,target=full_target)
    if not delay_high:
        phase(high,20)
        if fused:high_ms=erase(high,high_table)
        else:b.del_lookup(table=high_table,address=address,target=high)
        b.free(high)
    else:
        assert all(v>>7==1 for v in high_table),'Constant-high carry borrowing proof failed'
        b.x(high[-1]);extra_borrow.append(high[-1])
    inner(False)
    if delay_high:
        extra_borrow.clear();b.x(high[-1])
    erased=None
    if erase_curvature:
        assert cfg.w2==7
        checkpoint_start=7-curve_count if gray_checkpoint else 6
        if gray_checkpoint:
            for j in range(checkpoint_start,6):b.cx(c2[j+1],c2[j])
        erased=km.array(list(c2[checkpoint_start:]))
        checkpoint_table=[((tables[(i//128,i%128)][0] ^ (tables[(i//128,i%128)][0]>>1)) if gray_checkpoint else tables[(i//128,i%128)][0])>>checkpoint_start for i in range(256)]
        checkpoint_ms=erase(erased,checkpoint_table);b.free(erased)
    if delay_high:
        phase(high,20)
        if fused:high_ms=erase(high,high_table)
        else:b.del_lookup(table=high_table,address=address,target=high)
        b.free(high)
    if late_coarse:
        new=b.alloc_qubits(late_coarse)
        coarse=km.array(list(coarse)+list(new))
        cv=np.array([split['coarse'][(i//128,i%128)] for i in range(256)],dtype=np.int64)
        for j in reversed(range(7-late_coarse,7)):
            pp=chosen_plan(((cv^(cv>>1))>>j)&1,f'coarse{j}')
            restore_boolean(b,pp,address,coarse[j]);extra_reload+=len(pp['nodes'])
            if j<6:b.cx(coarse[j+1],coarse[j])
    adder(residual[11:],coarse);derivative=residual[:11]+coarse
    outer(derivative,False);phase(low[cfg.gy:],cfg.gy);outer(derivative,True)
    if fused:low_ms=erase(low,low_table);b.free(low)
    if erase_curvature:
        replacement=b.alloc_qubits(7-checkpoint_start)
        c2=km.array(list(c2[:checkpoint_start])+list(replacement))
        cv=np.array([tables[(i//128,i%128)][0] for i in range(256)],dtype=np.int64)
        gray_cv=cv^(cv>>1) if gray_checkpoint else cv
        for j in reversed(range(checkpoint_start,7)):
            pp=chosen_plan((gray_cv>>j)&1,f'curve{j}')
            restore_boolean(b,pp,address,c2[j]);extra_reload+=len(pp['nodes'])
            if gray_checkpoint and j<6:b.cx(c2[j+1],c2[j])
    adder(residual[11:],coarse,sub=True);inner(True)
    rest_target=km.array(list(residual)+list(c2)+list(coarse))
    if fused:
        rest_ms=erase(rest_target,rest_table);b.free(rest_target)
        # Fold repeated coefficient measurements into the final classical mask.
        # Gray checkpoint g5=c5 XOR c6 contributes m5*c5+(m5 XOR m6)*c6.
        if erase_curvature:
            for k,m in enumerate(checkpoint_ms):
                j=checkpoint_start+k;b.cx(m,rest_ms[12+j])
                if gray_checkpoint and j<6:b.cx(m,rest_ms[12+j+1])
        records=low_ms+rest_ms[:12+cfg.w2]+high_ms+rest_ms[12+cfg.w2:]
        assert len(records)==len(full_target)+late_coarse
        phase_from_records(b,full_table,address,records)
    else:
        # The curvature checkpoint also needs its deferred phase correction.
        if erase_curvature:raise ValueError('Curvature erasure requires fused correction')
        target=km.array(list(low)+list(rest_target));table=[low_table[j]|(rest_table[j]<<21) for j in range(256)]
        b.del_lookup(table=table,address=address,target=target);b.free(target)
    b.x(x[-1]);H.unnormalize(b,x,flags);b.free(flags)
    circuit=b.finish_circuit()
    return circuit,dict(variant='deferred_feedforward',fused=fused,anf=anf,borrow_phase=borrow_phase,
                        erase_curvature=erase_curvature,delay_high=delay_high,gray_checkpoint=gray_checkpoint,
                        boolean_reload_toffoli=extra_reload,late_coarse=late_coarse,curve_count=curve_count,
                        qubits=circuit.num_qubits,static_toffoli=int(circuit.operation_statistics().num_toffoli_like),split_bounds=bounds)


In [ ]:
%%writefile src/selected_schedule90.py
# SPDX-License-Identifier: Apache-2.0
# Kozykorpesh signed-curvature lifetime schedule, V3/run361677445.
# https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445
# Caller supplies the explicit public Kozy coefficient table, already retained
# in our attributed signed_digit_invsqrt_v3 module. No Toolbox data is required.
import hashlib
import io
import json
from pathlib import Path
import fusion_schedule90 as F


SELECTED_OPTIONS = dict(delay_high=True, gray_checkpoint=True,
                        curve_count=2, late_coarse=0)


def build_selected(cfg, tables):
    """Return the raw selected public schedule and its provenance report.

    Apply the existing exact-domain compiler and fixed calibrated rotations
    separately; their canonical source/output hashes bind those transformations.
    """
    if (cfg.f2, cfg.f1, cfg.fyr, cfg.gy, sorted(cfg.drop2)) != (8, 19, 27, 1, [5]):
        raise ValueError("This schedule is pinned to the explicit 256-row Kozy model")
    choices_path = Path(__file__).with_name("boolean_choices90.json")
    choices = json.loads(choices_path.read_text())
    circuit, report = F.build(cfg, tables, boolean_choices=choices, **SELECTED_OPTIONS)
    stream = io.BytesIO()
    circuit.to_file(stream, format="kmb")
    report.update(upstream_version=3, upstream_run_id=361677445,
                  method="Gray curvature checkpoint with two early erased bits and deferred joint ANF correction",
                  boolean_choices_sha256=hashlib.sha256(choices_path.read_bytes()).hexdigest(),
                  raw_circuit_sha256=hashlib.sha256(stream.getvalue()).hexdigest(),
                  table_source="Caller-supplied explicit public Kozy integer triples",
                  selected_options=SELECTED_OPTIONS,
                  verification_status="Requires exact-domain compilation, phase certificate and full official check")
    return circuit, report


In [ ]:
%%writefile src/boolean_choices90.json
{
  "coarse4": {
    "ands": 16,
    "order": [
      4,
      5,
      0,
      7,
      1,
      2,
      6,
      3
    ],
    "polarity": 84
  },
  "coarse5": {
    "ands": 7,
    "order": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7
    ],
    "polarity": 0
  },
  "coarse6": {
    "ands": 11,
    "order": [
      7,
      6,
      5,
      3,
      0,
      1,
      2,
      4
    ],
    "polarity": 144
  },
  "curve4": {
    "ands": 15,
    "order": [
      4,
      5,
      3,
      0,
      2,
      1,
      6,
      7
    ],
    "polarity": 127
  },
  "curve5": {
    "ands": 8,
    "order": [
      0,
      7,
      6,
      5,
      1,
      2,
      3,
      4
    ],
    "polarity": 61
  },
  "curve6": {
    "ands": 9,
    "order": [
      5,
      4,
      7,
      2,
      1,
      0,
      3,
      6
    ],
    "polarity": 249
  }
}

In [ ]:
%%writefile src/upstream_functions.py
# SPDX-License-Identifier: Apache-2.0
# Original algorithm and functions: mizeroluckygall, Kaggle Rotation Calibration V3.
# Pinned primary source: https://www.kaggle.com/code/mizeroluckygall/quantum-circuits-rotation-calibration-39-char?scriptVersionId=361537580
# Upstream credits lightningv08, Quantum Circuits | Memory & Algebra, for exhaustive phase calibration.
# Exact function bodies extracted verbatim; notebook setup, Input scan and packaging omitted.
# Apache-2.0 confirmed from anonymous pinned Kaggle license footer (see upstream_primary_license.json).
from decimal import Decimal, localcontext
from fractions import Fraction
import math
import numpy as np
from scipy.optimize import linprog
UNIT = 2 ** 64

def parse_kmx(text):
    """Return (lines, rotations); rotation = dict(line, qubit, t, cond, counted)."""
    lines = text.splitlines()
    rots = []
    for i, line in enumerate(lines):
        tok = line.split('#')[0].split()
        if not tok or tok[0] != 'Z_POW':
            continue
        cond = tok[tok.index('if') + 1] if 'if' in tok else None
        t = Fraction(Decimal(tok[2]))
        counted = (t * 4).denominator != 1          # finer than T: a synthesized rotation
        rots.append(dict(line=i, qubit=tok[1], t=t, cond=cond, counted=counted))
    return lines, rots


def instrument(lines, rots, first_record_qubit):
    """Copy of the circuit where rotation j becomes CX qubit -> record_j."""
    out = list(lines)
    for j, r in enumerate(rots):
        rec = f'q{first_record_qubit + j}'
        out[r['line']] = f"CX {r['qubit']} {rec}" + (f" if {r['cond']}" if r['cond'] else '')
    return '\n'.join(out) + '\n'


def theta_units(t):
    """Z_POW exponent (half turns) -> phase units (2^-64 turn), exact if representable."""
    v = t * (UNIT // 2)
    return int(round(v)), v.denominator == 1


def exponent_text(t):
    """Exact decimal text for a dyadic exponent, in kickmix's no-exponent format."""
    with localcontext() as ctx:
        ctx.prec = 400
        d = Decimal(t.numerator) / Decimal(t.denominator)
    s = format(d, 'f')
    if '.' not in s:
        s += '.0'
    return s


def rewrite(lines, rots, deltas_units, dropped):
    out = list(lines)
    remove = set()
    for j, r in enumerate(rots):
        if j in dropped:
            remove.add(r['line'])
            continue
        if deltas_units.get(j, 0):
            t = r['t'] + Fraction(int(deltas_units[j]), UNIT // 2)
            out[r['line']] = f"Z_POW {r['qubit']} {exponent_text(t)}" + (f" if {r['cond']}" if r['cond'] else '')
    return '\n'.join(l for i, l in enumerate(out) if i not in remove) + '\n'


def fit(err_units, tol_units, bits, free, fixed_shift_units, *, bound_units, scale=2.0 ** 28,
        start=4096, add=400, max_iter=60, verbose=True):
    """Maximise m s.t. |err + shift + B[:,free] d| <= tol - m for every input.

    err_units: int64 signed error of the current circuit; tol_units: uint64;
    bits: (n_inputs, n_rot) uint8; fixed_shift_units: float array added to err
    (the effect of dropping rotations). Returns (deltas, margin_units).
    """
    n = len(err_units)
    e = (err_units.astype(np.float64) + fixed_shift_units) / scale
    tol = tol_units.astype(np.float64) / scale
    Bf = bits[:, free].astype(np.float64)
    k = len(free)
    active = set(np.linspace(0, n - 1, start, dtype=np.int64).tolist())
    active.update(np.argsort(tol - np.abs(e))[:start].tolist())
    for it in range(max_iter):
        ii = np.fromiter(sorted(active), dtype=np.int64)
        A = np.vstack([np.c_[Bf[ii], np.ones(len(ii))], np.c_[-Bf[ii], np.ones(len(ii))]])
        b = np.r_[tol[ii] - e[ii], tol[ii] + e[ii]]
        res = linprog(np.r_[np.zeros(k), -1.0], A_ub=A, b_ub=b,
                      bounds=[(-bound_units / scale, bound_units / scale)] * k + [(None, None)], method='highs')
        if not res.success:
            raise RuntimeError(res.message)
        d, m = res.x[:k], res.x[k]
        corrected = e + Bf @ d
        slack = tol - np.abs(corrected)
        worst = float(slack.min())
        if verbose:
            print(f'    LP iter {it}: rows {len(ii):6d}  lp margin {m*scale:.4g}  true margin {worst*scale:.4g} units', flush=True)
        if worst >= m - 1e-9 * max(1.0, abs(m)):
            break
        active.update(np.argpartition(slack, add)[:add].tolist())
    return np.rint(d * scale).astype(np.int64), worst * scale


def exact_margin(err_units, tol_units, bits, deltas_full_units):
    """Integer check of the final, rounded corrections over every input."""
    shift = bits.astype(np.int64) @ deltas_full_units.astype(np.int64)
    corrected = err_units.astype(np.int64) + shift
    slack = tol_units.astype(np.int64) - np.abs(corrected)
    return int(slack.min()), int(np.argmin(slack))


def collect(km, IV, spec, text, rots, *, batch=16384, seed=20261023, log=print):
    """Simulate every input: true phase error (units) and every rotation's bit."""
    c0 = km.Circuit(text)
    Q = c0.num_qubits
    ci = km.Circuit(instrument(text.splitlines(), rots, Q))
    xr, _ = IV.validate_registers(c0, spec)
    rec = km.array([km.q(Q + j) for j in range(len(rots))])
    xs = np.arange(1, spec.max_x + 1, dtype=np.uint64)
    n = len(xs)
    phase = np.empty(n, dtype=np.uint64)
    bits = np.empty((n, len(rots)), dtype=np.uint8)
    s0 = km.Simulator(batch_size=batch, seed=seed, debug_print=False)
    s1 = km.Simulator(batch_size=batch, seed=seed + 1, debug_print=False)
    for start in range(0, n, batch):
        x = np.resize(xs[start:start + batch], batch)
        take = min(batch, n - start)
        for s, c in ((s0, c0), (s1, ci)):
            s.clear_for_shot(); s.write_across_shots(xr, IV.to_bits(x, spec.width)); s.do(c)
        phase[start:start + take] = [IV.half_turns_to_units(s0.read_shot_phase(j)) for j in range(take)]
        bits[start:start + take] = s1.read_across_shots(rec)[:, :take].T
        if start // batch % 40 == 0:
            log(f'  simulated {start + take:,}/{n:,}')
    target, tol = IV.reference_values(spec, xs)
    err = (phase - np.asarray(target, dtype=np.uint64)).view(np.int64)
    return xs, err, np.asarray(tol, dtype=np.uint64), bits, Q


def search(err, tol, bits, rots, report, rotation_t_count, *, max_drop=8, log=print):
    """Try dropping 0..max_drop smallest rotations; return candidates with cost estimates."""
    counted = [j for j, r in enumerate(rots) if r['counted']]
    weight = {j: abs(float(min(rots[j]['t'] % 2, 2 - rots[j]['t'] % 2))) for j in counted}
    order = sorted(counted, key=lambda j: weight[j])
    Qn, tof, tcount = report['num_qubits'], report['toffoli_count'], report['t_count']
    theta = np.array([theta_units(r['t'])[0] for r in rots], dtype=np.float64)

    def estimate(n_rot, margin_units):
        if margin_units <= 0:
            return float('inf')
        eps = math.pi * (margin_units / UNIT) / n_rot
        return Qn * (tof + (tcount + n_rot * rotation_t_count(eps)) / 4)

    m0, _ = exact_margin(err, tol, bits, np.zeros(len(rots), dtype=np.int64))
    base = estimate(len(counted), m0)
    log(f'baseline: {len(counted)} rotations, margin {m0:.6g} units, estimated cost {base:,.3f} '
        f'(verifier reported {report["score"]:,.3f})')
    rows = [dict(drop=0, dropped=[], deltas=np.zeros(len(rots), dtype=np.int64), margin=m0, estimate=base, fitted=False)]
    for k in range(0, max_drop + 1):
        dropped = order[:k]
        free = [j for j in counted if j not in dropped]
        shift = -(bits[:, dropped].astype(np.float64) @ theta[dropped]) if dropped else np.zeros(len(err))
        bound = max([4 * abs(theta[j]) for j in dropped] + [2.0 ** 36])
        log(f'drop {k}: ' + (', '.join(f'{float(rots[j]["t"]):.3e}' for j in dropped) or 'none'))
        try:
            d_free, _ = fit(err, tol, bits, free, shift, bound_units=bound, verbose=False)
        except RuntimeError as exc:
            log(f'    LP failed: {exc}'); continue
        deltas = np.zeros(len(rots), dtype=np.int64)
        deltas[free] = d_free
        for j in dropped:
            deltas[j] = -int(theta_units(rots[j]['t'])[0])
        m, worst_i = exact_margin(err, tol, bits, deltas)
        est = estimate(len(free), m)
        log(f'    exact margin {m:.6g} units, estimated cost {est:,.3f}')
        rows.append(dict(drop=k, dropped=dropped, deltas=deltas, margin=m, estimate=est, fitted=True))
        if m <= 0:
            break
    return rows



In [ ]:
%%writefile src/portable_calibration.py
# SPDX-License-Identifier: Apache-2.0
# Adapted from mizeroluckygall, Quantum Circuits | Rotation Calibration, V3:
# https://www.kaggle.com/code/mizeroluckygall/quantum-circuits-rotation-calibration-39-char?scriptVersionId=361537580
# Upstream credits lightningv08, Quantum Circuits | Memory & Algebra, for
# exhaustive phase calibration. See upstream_functions.py for unchanged bodies.
# Local changes: explicit Circuit/spec API, strict supported-circuit audit,
# chunked LP scans and integer products, canonical hash checks, independent
# measurement replay, and report-only output without verifier/submission calls.
"""Exhaustive rotation calibration for a supported basis-state phase oracle.

``calibrate(circuit, IV, spec, output_path, baseline_report=None)`` accepts the
unchanged official inverse-square-root verifier module IV. It does not search
Inputs, read ZIPs, contact Kaggle, or call IV.verify. Its output requires a full
official verification before use. No verifier salt/hash is optimized.

Only unconditioned, exactly representable Z_POW rotations are calibrated.
Before the final rotation, classical conditions may guard diagonal phases;
arithmetic is deterministic on basis states and HMR resets its target.
After the final rotation the original cleanup may contain conditional arithmetic.
This excludes arbitrary quantum
circuits whose rotation controls can depend on measurement outcomes.
"""
from __future__ import annotations

import hashlib
import importlib.util
import io
import json
import math
from pathlib import Path
import time

import numpy as np
from scipy.optimize import linprog

_path = Path(__file__).with_name("upstream_functions.py")
_spec = importlib.util.spec_from_file_location("_vqcc_upstream_rotcal", _path)
_upstream = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_upstream)
UNIT = _upstream.UNIT
parse_kmx = _upstream.parse_kmx
instrument = _upstream.instrument
theta_units = _upstream.theta_units
exponent_text = _upstream.exponent_text
rewrite = _upstream.rewrite
UPSTREAM_URL = "https://www.kaggle.com/code/mizeroluckygall/quantum-circuits-rotation-calibration-39-char?scriptVersionId=361537580"


def _serialize(circuit, fmt):
    stream = io.BytesIO()
    circuit.to_file(stream, format=fmt)
    return stream.getvalue()


def audit_supported(text, rots):
    """Conservative structural guard for deterministic rotation activations.

    Before the final rotation, HMR results may only control diagonal gates.
    Therefore the Z_POW activation bits do not depend on their outcomes.
    Original cleanup after the final rotation cannot change earlier activation
    bits, so its conditional arithmetic is permitted. This alone does not prove
    MBU phase corrections; replay and the official verifier remain necessary.
    """
    metadata = {"REGISTER", "APPEND_TO_REGISTER", "INFO_TAG"}
    basis = {"X", "Y", "CX", "CY", "CCX", "CCY", "SWAP", "CSWAP", "R", "HMR"}
    diagonal = {"Z", "CZ", "CCZ", "Z_POW"}
    classical = {"BIT_STORE0", "BIT_STORE1", "BIT_INVERT", "BIT_XOR", "NEG"}
    depth = 0
    last_rotation_line = max((r["line"] for r in rots), default=-1)
    unknown = []
    conditional_nondiagonal = []
    conditional_cleanup = 0
    for i, line in enumerate(text.splitlines()):
        tok = line.split("#", 1)[0].split()
        if not tok:
            continue
        op = tok[0]
        if op == "PUSH_CONDITION":
            depth += 1
            continue
        if op == "POP_CONDITION":
            depth -= 1
            if depth < 0:
                raise ValueError("Unbalanced POP_CONDITION")
            continue
        if op not in metadata | basis | diagonal | classical:
            unknown.append((i + 1, op))
        conditioned = depth > 0 or "if" in tok
        if conditioned and op in basis:
            if i < last_rotation_line:
                conditional_nondiagonal.append((i + 1, op))
            else:
                conditional_cleanup += 1
        if conditioned and op == "Z_POW":
            raise ValueError("Conditioned Z_POW is outside this calibration scope")
    if depth:
        raise ValueError("Unbalanced PUSH_CONDITION")
    if unknown or conditional_nondiagonal:
        raise ValueError(f"Unsupported operations: {unknown[:8]}; conditioned arithmetic: {conditional_nondiagonal[:8]}")
    for rotation in rots:
        if rotation["cond"] is not None:
            raise ValueError("Conditioned rotations are unsupported")
        if not theta_units(rotation["t"])[1]:
            raise ValueError("Every original exponent must be exact on the 2^-64-turn lattice")
    return {"supported": True, "rotationsUnconditioned": True,
            "measurementIndependentActivationBits": True, "exactPhaseLattice": True,
            "conditionalCleanupOperationsAfterLastRotation": conditional_cleanup}


def _matvec(bits, vector, columns=None, *, chunk=16384, dtype=np.float64):
    """A small temporary replaces the upstream full float/int64 matrix."""
    result = np.empty(len(bits), dtype=dtype)
    vector = np.asarray(vector, dtype=dtype)
    for begin in range(0, len(bits), chunk):
        end = min(begin + chunk, len(bits))
        block = bits[begin:end]
        if columns is not None:
            block = block[:, columns]
        result[begin:end] = block.astype(dtype) @ vector
    return result


def exact_margin(err_units, tol_units, bits, deltas_full_units, *, chunk=16384):
    """Exact rounded-correction check on every collected input, without overflow."""
    deltas = np.asarray(deltas_full_units, dtype=np.int64)
    largest_error = max(abs(int(err_units.min())), abs(int(err_units.max())))
    if largest_error + sum(abs(int(d)) for d in deltas) >= 1 << 63:
        raise ValueError("Correction can overflow signed 64-bit phase arithmetic")
    if int(np.max(tol_units)) >= 1 << 63:
        raise ValueError("Half-turn tolerance needs a wider integer checker")
    minimum, worst = None, None
    for begin in range(0, len(bits), chunk):
        end = min(begin + chunk, len(bits))
        shift = bits[begin:end].astype(np.int64) @ deltas
        corrected = np.asarray(err_units[begin:end], dtype=np.int64) + shift
        slack = np.asarray(tol_units[begin:end], dtype=np.int64) - np.abs(corrected)
        local = int(np.argmin(slack))
        value = int(slack[local])
        if minimum is None or value < minimum:
            minimum, worst = value, begin + local
    return minimum, worst


def fit(err_units, tol_units, bits, free, fixed_shift_units, *, bound_units,
        scale=2.0 ** 28, start=4096, add=400, max_iter=60, chunk=16384, log=None):
    """Maximize worst slack using cutting planes; every iteration scans all x."""
    n, k = len(err_units), len(free)
    e = (err_units.astype(np.float64) + fixed_shift_units) / scale
    tol = tol_units.astype(np.float64) / scale
    active = set(np.linspace(0, n - 1, min(start, n), dtype=np.int64).tolist())
    initial = tol - np.abs(e)
    count = min(start, n)
    active.update(np.argpartition(initial, count - 1)[:count].tolist())
    converged = False
    for iteration in range(max_iter):
        ii = np.fromiter(sorted(active), dtype=np.int64)
        selected = bits[ii][:, free].astype(np.float64)
        A = np.vstack([np.c_[selected, np.ones(len(ii))],
                       np.c_[-selected, np.ones(len(ii))]])
        rhs = np.r_[tol[ii] - e[ii], tol[ii] + e[ii]]
        result = linprog(np.r_[np.zeros(k), -1.0], A_ub=A, b_ub=rhs,
                         bounds=[(-bound_units / scale, bound_units / scale)] * k + [(None, None)],
                         method="highs")
        if not result.success:
            raise RuntimeError(result.message)
        delta, margin = result.x[:k], result.x[k]
        corrected = e + _matvec(bits, delta, free, chunk=chunk)
        slack = tol - np.abs(corrected)
        actual = float(slack.min())
        if log:
            log(f"LP {iteration}: active={len(ii)}, lp_margin={margin*scale:.3f}, exhaustive_margin={actual*scale:.3f}")
        if actual >= margin - 1e-9 * max(1.0, abs(margin)):
            converged = True
            break
        count = min(add, n)
        new_rows = set(np.argpartition(slack, count - 1)[:count].tolist())
        if new_rows.issubset(active):
            break
        active.update(new_rows)
    return np.rint(delta * scale).astype(np.int64), actual * scale, {
        "iterations": iteration + 1, "activeInputs": len(active), "lpConverged": converged,
        "exhaustiveScans": iteration + 1,
    }


def collect(km, IV, spec, text, rots, *, batch=16384, seed=20261023,
            replay_measurements=True, log=print):
    """Visit every x; check input/scratch and optionally replay a second RNG trace.

    The fixed seeds provide reproducibility, never candidate selection. The
    structural audit proves activation-bit independence. Equal phases under two
    traces are a diagnostic; they are not a replacement for all official shots.
    """
    circuit = km.Circuit(text)
    Q = circuit.num_qubits
    recorded = km.Circuit(instrument(text.splitlines(), rots, Q))
    xr, scratch = IV.validate_registers(circuit, spec)
    rec = km.array([km.q(Q + j) for j in range(len(rots))])
    xs = np.arange(1, spec.max_x + 1, dtype=np.uint64)
    phase = np.empty(len(xs), dtype=np.uint64)
    bits = np.empty((len(xs), len(rots)), dtype=np.uint8)
    simulations = [km.Simulator(batch_size=batch, seed=seed + j, debug_print=False)
                   for j in range(3 if replay_measurements else 2)]
    for begin in range(0, len(xs), batch):
        take = min(batch, len(xs) - begin)
        values = np.resize(xs[begin:begin + batch], batch)
        for index, sim in enumerate(simulations):
            sim.clear_for_shot()
            sim.write_across_shots(xr, IV.to_bits(values, spec.width))
            sim.do(recorded if index == 1 else circuit)
            if not np.array_equal(IV.from_bits(sim.read_across_shots(xr))[:take], values[:take]):
                raise ValueError("Input register was not restored during exhaustive collection")
            if len(scratch) and sim.read_across_shots(scratch)[:, :take].any():
                raise ValueError("Original scratch was not cleared during exhaustive collection")
        phases = np.asarray([IV.half_turns_to_units(simulations[0].read_shot_phase(j))
                             for j in range(take)], dtype=np.uint64)
        if replay_measurements:
            replay = np.asarray([IV.half_turns_to_units(simulations[2].read_shot_phase(j))
                                 for j in range(take)], dtype=np.uint64)
            if not np.array_equal(phases, replay):
                raise ValueError("Phase depends on measurement trace; do not calibrate this model")
        phase[begin:begin + take] = phases
        bits[begin:begin + take] = simulations[1].read_across_shots(rec)[:, :take].T
        if log and begin // batch % 40 == 0:
            log(f"Collected {begin + take:,}/{len(xs):,} inputs")
    target, tolerance = IV.reference_values(spec, xs)
    error = (phase - np.asarray(target, dtype=np.uint64)).view(np.int64)
    return xs, error, np.asarray(tolerance, dtype=np.uint64), bits, Q


def search(err, tol, bits, rots, report, rotation_t_count, *, max_drop=8,
           bound_floor_units=2.0 ** 36, chunk=16384, log=print, fit_options=None,
           stop_on_invalid_margin=True):
    """Upstream smallest-angle drop prefixes, followed by exact integer checks."""
    counted = [j for j, r in enumerate(rots) if r["counted"]]
    order = sorted(counted, key=lambda j: abs(float(min(rots[j]["t"] % 2, 2 - rots[j]["t"] % 2))))
    # A Clifford Z_POW(1) is exactly 2^63 phase units. It is never varied or
    # dropped, so avoid putting that unsigned value into a signed int64 array.
    counted_set = set(counted)
    theta = np.array([theta_units(r["t"])[0] if j in counted_set else 0
                      for j, r in enumerate(rots)], dtype=np.int64)

    def estimate(n_rotations, margin):
        if margin <= 0:
            return math.inf
        eps = math.pi * (margin / UNIT) / n_rotations if n_rotations else None
        synthesis = n_rotations * rotation_t_count(eps) if n_rotations else 0
        return report["num_qubits"] * (report["toffoli_count"] + (report["t_count"] + synthesis) / 4)

    baseline, worst = exact_margin(err, tol, bits, np.zeros(len(rots), dtype=np.int64), chunk=chunk)
    rows = [{"drop": 0, "dropped": [], "deltas": np.zeros(len(rots), dtype=np.int64),
             "margin": baseline, "worstIndex": worst, "estimate": estimate(len(counted), baseline), "fitted": False}]
    for count in range(min(max_drop, len(counted)) + 1):
        dropped = order[:count]
        free = [j for j in counted if j not in dropped]
        shift = -_matvec(bits, theta[dropped], dropped, chunk=chunk, dtype=np.int64) if dropped else np.zeros(len(err), dtype=np.int64)
        bound = max([4 * abs(int(theta[j])) for j in dropped] + [bound_floor_units])
        options = dict(fit_options or {})
        try:
            delta, _, diagnostics = fit(err, tol, bits, free, shift,
                                         bound_units=bound, chunk=chunk, **options)
        except RuntimeError as exc:
            rows.append({"drop": count, "dropped": dropped, "fitted": False,
                         "estimate": math.inf, "margin": -1, "lpError": str(exc)})
            continue
        deltas = np.zeros(len(rots), dtype=np.int64)
        deltas[free] = delta
        deltas[dropped] = -theta[dropped]
        margin, worst = exact_margin(err, tol, bits, deltas, chunk=chunk)
        row = {"drop": count, "dropped": dropped, "deltas": deltas, "margin": margin,
               "worstIndex": worst, "estimate": estimate(len(free), margin), "fitted": True,
               **diagnostics}
        rows.append(row)
        if log:
            log(f"Drop {count}: exact margin={margin:,} units, estimated cost={row['estimate']:,.3f}")
        if margin <= 0 and stop_on_invalid_margin:
            break
    return rows


def _jsonable(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {k: _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def apply_fixed_calibration(circuit, fixed_deltas, output_path=None, *, km=None):
    """Reproduce a prior calibration without simulation or LP; bind both hashes.

    fixed_deltas is a dict or JSON path written by run_two_pinned.py. Its
    canonical source hash and candidate hash are mandatory. The output still
    inherits whatever verification status was independently established for
    that exact candidate; this helper does not run or replace verification.
    """
    if km is None:
        import kickmix as km
    if not isinstance(fixed_deltas, dict):
        fixed_deltas = json.loads(Path(fixed_deltas).read_text())
    source_hash = hashlib.sha256(_serialize(circuit, "kmb")).hexdigest()
    if source_hash != fixed_deltas["sourceCircuitSha256"]:
        raise ValueError("Fixed calibration source circuit hash mismatch")
    if fixed_deltas.get("phaseUnitsPerTurn") != UNIT:
        raise ValueError("Unexpected phase units")
    text = _serialize(circuit, "kmx").decode()
    lines, rots = parse_kmx(text)
    audit_supported(text, rots)
    values = fixed_deltas["deltasPhaseUnits"]
    if len(values) != len(rots):
        raise ValueError("Fixed calibration rotation count mismatch")
    dropped = set(fixed_deltas["droppedOriginalRotationIndices"])
    if not dropped.issubset(range(len(rots))):
        raise ValueError("Invalid dropped rotation index")
    replacement = rewrite(lines, rots, {j: int(v) for j, v in enumerate(values) if j not in dropped}, dropped)
    candidate = km.Circuit(replacement)
    if hashlib.sha256(_serialize(candidate, "kmb")).hexdigest() != fixed_deltas["candidateCircuitSha256"]:
        raise ValueError("Fixed calibration candidate hash mismatch")
    if candidate.num_qubits != circuit.num_qubits:
        raise ValueError("Fixed calibration changed qubit count")
    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        candidate.to_file(str(output_path), format="kmb")
    return candidate


def calibrate(circuit, IV, spec, output_path, baseline_report=None, *, km=None,
              max_drop=8, batch=16384, chunk=16384, seed=20261023,
              replay_measurements=True, fit_options=None, log=print):
    """Write one estimated-improving candidate and JSON; never run IV.verify.

    baseline_report may be the unchanged official full report for this canonical
    circuit. If omitted, static operation counts give an upper-bound resource
    estimate, sufficient to compare angle changes with fixed arithmetic. A report
    with a mismatched canonical hash is rejected. The selected candidate is
    recollected over the entire domain and its integer margin must reproduce.
    """
    if km is None:
        import kickmix as km
    started = time.perf_counter()
    output_path = Path(output_path)
    source_bytes = _serialize(circuit, "kmb")
    source_hash = hashlib.sha256(source_bytes).hexdigest()
    text = _serialize(circuit, "kmx").decode()
    lines, rots = parse_kmx(text)
    if not rots:
        raise ValueError("Circuit has no rotations to calibrate")
    audit = audit_supported(text, rots)
    IV.validate_registers(circuit, spec)
    resource_source = "static upper-bound operation counts"
    if baseline_report is not None:
        if baseline_report.get("circuit_sha256") != source_hash:
            raise ValueError("Baseline report does not bind this canonical circuit hash")
        if baseline_report.get("num_qubits") != circuit.num_qubits:
            raise ValueError("Baseline report qubit count mismatch")
        resources = baseline_report
        resource_source = "canonical-hash-bound caller report"
    else:
        stats = circuit.operation_statistics()
        resources = {"num_qubits": circuit.num_qubits,
                     "toffoli_count": stats.num_toffoli_like, "t_count": stats.num_t_like}
    xs, err, tol, bits, _ = collect(km, IV, spec, text, rots, batch=batch, seed=seed,
                                  replay_measurements=replay_measurements, log=log)
    rows = search(err, tol, bits, rots, resources, IV.rotation_t_count,
                  max_drop=max_drop, chunk=chunk, log=log, fit_options=fit_options)
    baseline = rows[0]
    best = min((r for r in rows if r.get("fitted") and r["margin"] > 0),
               key=lambda r: r["estimate"], default=None)
    selected = best is not None and best["estimate"] < baseline["estimate"]
    result = {"upstream": UPSTREAM_URL, "sourceCircuitSha256": source_hash,
              "resourceSource": resource_source, "structuralAudit": audit,
              "allInputsVisited": len(xs), "maxX": int(spec.max_x),
              "seedPurpose": "Fixed reproducibility only; no secret salt, hash search or seed selection",
              "measurementReplay": replay_measurements,
              "rows": rows, "selectedImprovement": selected,
              "officialVerificationRun": False, "officialStatus": "NOT_RUN"}
    if selected:
        dropped = set(best["dropped"])
        delta = {j: int(v) for j, v in enumerate(best["deltas"]) if j not in dropped}
        new_text = rewrite(lines, rots, delta, dropped)
        new_lines, new_rots = parse_kmx(new_text)
        audit_supported(new_text, new_rots)
        del xs, err, tol, bits
        xs2, err2, tol2, bits2, _ = collect(km, IV, spec, new_text, new_rots, batch=batch,
                                           seed=seed, replay_measurements=replay_measurements, log=log)
        actual_margin, worst = exact_margin(err2, tol2, bits2, np.zeros(len(new_rots), dtype=np.int64), chunk=chunk)
        if actual_margin <= 0 or actual_margin != best["margin"]:
            raise ValueError(f"Rewritten exact margin {actual_margin} differs from model {best['margin']}")
        candidate = km.Circuit(new_text)
        if candidate.num_qubits != circuit.num_qubits:
            raise ValueError("Rotation rewrite changed qubit count")
        output_path.parent.mkdir(parents=True, exist_ok=True)
        candidate.to_file(str(output_path), format="kmb")
        result.update({"candidatePath": str(output_path),
                       "candidateCircuitSha256": hashlib.sha256(_serialize(candidate, "kmb")).hexdigest(),
                       "candidateRotationCount": sum(r["counted"] for r in new_rots),
                       "candidateMarginUnits": actual_margin, "candidateWorstX": int(xs2[worst]),
                       "estimatedCost": best["estimate"], "baselineEstimatedCost": baseline["estimate"],
                       "status": "NEEDS_FULL_OFFICIAL_VERIFICATION"})
    else:
        result["status"] = "NO_ESTIMATED_IMPROVEMENT"
    result["elapsedSeconds"] = time.perf_counter() - started
    result = _jsonable(result)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    report_path = output_path.with_suffix(".calibration.json")
    report_path.write_text(json.dumps(result, indent=2, allow_nan=False), encoding="utf-8")
    result["reportPath"] = str(report_path)
    return result


In [ ]:
%%writefile src/invsqrt_calibration.json
{
  "sourceCircuitSha256": "343ae65e25032fca924c4c3d719478bba9871631ead110bfaa47bc52e01daa76",
  "candidateCircuitSha256": "97f12b109eb17346991ca33e5604963a11eff1b2098586e048d1fd103de55b7c",
  "droppedOriginalRotationIndices": [
    18
  ],
  "deltasPhaseUnits": [
    15503389,
    7204717,
    -18590236,
    14903209,
    -5832939,
    -13433149,
    97059903,
    93233282,
    -28395803,
    -282842922,
    -72641001,
    -1763289056,
    -1814996023,
    130343098,
    -17034870096,
    0,
    0,
    0,
    -134217728,
    8097276,
    1815847,
    4491724,
    -10991042,
    5870512,
    -17519882,
    -4641899,
    10843453,
    -2449327,
    6247377,
    -1806161,
    -5052034,
    23508591,
    -26351233,
    -10145084,
    -14057994,
    -4128602,
    10688249,
    -5051023,
    -37412180,
    729297823,
    -718326109,
    -678507807,
    -5547184257,
    -14622195669,
    -12662101646,
    -68719476736,
    -68719476736,
    -68719476736
  ],
  "phaseUnitsPerTurn": 18446744073709551616,
  "maxX": 3133452,
  "marginUnits": 73326605,
  "officialVerificationRun": false
}

In [ ]:
# notebook: hide_input
import signed_digit_invsqrt_v3 as core
import selected_schedule90
import portable_exact_domain as domain_compiler
import portable_calibration as phase_calibration
cfg,tables,table_origin=core.load_or_build_tables()
integer_certificate=core.exhaustive_certificate(cfg,tables)
assert integer_certificate['pass'] and integer_certificate['tested_inputs']==3133452
save_json(REPORTS/'invsqrt_integer_certificate.json',{**integer_certificate,
    'scope':'Fixed integer model before angle calibration; full quantum verification follows.'})
raw_inverse,schedule_report=selected_schedule90.build_selected(cfg,tables)
assert schedule_report['raw_circuit_sha256']=='5cb5837936858ccd63dd160dd4df4031abad5dbf057ce873434e9708cd337b9e'
structural_inverse,domain_report=domain_compiler.optimize_circuit(raw_inverse)
structural_path=PUBLIC/'inverse_structural.kmb'
structural_inverse.to_file(str(structural_path),format='kmb')
assert digest(structural_path)=='343ae65e25032fca924c4c3d719478bba9871631ead110bfaa47bc52e01daa76'
fixed_deltas=json.loads((SRC/'invsqrt_calibration.json').read_text())
inv_candidate=phase_calibration.apply_fixed_calibration(structural_inverse,fixed_deltas,
    FINAL/'invsqrt.kmb',km=km)
assert digest(FINAL/'invsqrt.kmb')=='97f12b109eb17346991ca33e5604963a11eff1b2098586e048d1fd103de55b7c'
stats=inv_candidate.operation_statistics()
assert (inv_candidate.num_qubits,stats.num_toffoli_like,stats.num_t_like,
        stats.num_z_pow(min_clifford_hierarchy_level=4))==(90,1449,1,44)
inverse_build_report={'selected_sha256':digest(FINAL/'invsqrt.kmb'),
    'source_before_calibration_sha256':digest(structural_path),
    'schedule':schedule_report,'exact_gate_replacements':len(domain_report.replacements),
    'compiler_replacements':domain_report.replacements,'compiler_counts':dict(domain_report.counts),
    'selected_resources':{'qubits':90,'static_toffoli':1449,'rotations':44},
    'full_official_verification':'Required below; this build report does not replace it.'}
save_json(REPORTS/'invsqrt_build.json',inverse_build_report)
display(pd.DataFrame([
    {'Variant':'Kozy checkpoint schedule','Qubits':90,'Static Toffolis':1454,'Rotations':45},
    {'Variant':'Full-domain exact compilation','Qubits':90,'Static Toffolis':1449,'Rotations':45},
    {'Variant':'Certified fixed angle calibration','Qubits':90,'Static Toffolis':1449,'Rotations':44}]))
del raw_inverse,domain_report
gc.collect()



In [ ]:
# notebook: hide_input
import io
phase_started=time.perf_counter()
fixed_bytes=(SRC/'invsqrt_calibration.json').read_bytes()
fixed_deltas=json.loads(fixed_bytes)
fixed_data_sha=hashlib.sha256(json.dumps(fixed_deltas,sort_keys=True,separators=(',',':')).encode()).hexdigest()
assert fixed_data_sha=='d7c16d21ae8f516b0d4b66e3057802c3e545048a1d3906dc8656e82ca41a8f37'
assert fixed_deltas['sourceCircuitSha256']==digest(PUBLIC/'inverse_structural.kmb')
assert fixed_deltas['candidateCircuitSha256']==digest(FINAL/'invsqrt.kmb')
assert fixed_deltas['phaseUnitsPerTurn']==2**64 and fixed_deltas['maxX']==3133452
stream=io.BytesIO();inv_candidate.to_file(stream,format='kmx')
phase_text=stream.getvalue().decode()
_,rotation_instructions=phase_calibration.parse_kmx(phase_text)
phase_audit=phase_calibration.audit_supported(phase_text,rotation_instructions)
phase_x,error_units,tolerance_units,activation_bits,phase_Q=phase_calibration.collect(
    km,IV,IV.ProblemSpecInvSqrt.for_n_p(10),phase_text,rotation_instructions,
    batch=16384,seed=20261023,replay_measurements=True,log=print)
margin,worst_index=phase_calibration.exact_margin(error_units,tolerance_units,
    activation_bits,np.zeros(len(rotation_instructions),dtype=np.int64))
assert margin==73326605 and int(phase_x[worst_index])==2489983 and len(phase_x)==3133452
rotation_calibration_report={'diagnostic_status':'PASS','full_domain_inputs':len(phase_x),
    'margin_units':margin,'worst_x':int(phase_x[worst_index]),'num_qubits':phase_Q,
    'source_circuit_sha256':fixed_deltas['sourceCircuitSha256'],
    'candidate_circuit_sha256':fixed_deltas['candidateCircuitSha256'],
    'fixed_delta_data_sha256':fixed_data_sha,'fixed_delta_file_sha256':digest(SRC/'invsqrt_calibration.json'),
    'static_toffoli_count':1449,'rotation_count':44,'raw_t_count':1,
    'phase_units_per_turn':2**64,'fixed_measurement_phase_traces_per_input':2,
    'structural_audit':phase_audit,'lp_refit':False,
    'rotation_eps_from_margin':math.pi*margin/2**64/44,
    'official_verifier_run_by_this_diagnostic':False,
    'elapsed_seconds':time.perf_counter()-phase_started}
save_json(REPORTS/'rotation_calibration.json',rotation_calibration_report)
print(f"Fresh all-domain phase certificate PASS: {len(phase_x):,} inputs; margin {margin} integer units.",flush=True)
del phase_x,error_units,tolerance_units,activation_bits,phase_text,rotation_instructions,structural_inverse
gc.collect()



## 2. Chemistry: a signed gauge creates cheaper banks

A Givens tree reconstructs each real orbital vector from quantized angles. Kozykorpesh observed that changing each reconstructed vector's overall sign preserves its outer product. Choosing consistent signed rotations makes angle bit9 equal bit8. Our adapter checks this identity and reconstructs **all 3976 vectors**, bounding their component drift up to a row sign. The physical verifier still checks both Hamiltonian error tolerances.

Four neighboring addresses are represented by one base word and three differences. Each difference needs only nine independent bits per angle, because its two high bits agree. Three angles therefore fit in 30 output wires plus 81 difference-bank wires. A depth-first Boolean evaluation uses five scratch wires, selectively recomputing monomials to stay within the **128Q cap**. Measurement-based ANF cleanup restores the bank while preserving dirty output targets.

The resulting 25 circuits each use 128Q and 1129 Toffolis. Together with the unchanged state-preparation and Givens overhead, the model gives **661 total qubits and 65244 Toffolis**. Our existing orthogonal numerical gauge preserves the augmented factor Gram and lowers the effective normalization slightly further. The original shifts and acceptance thresholds are retained.

Both the signed gauge and bank architecture are credited to Kozykorpesh. Our source adapter regenerates them from the small public dataset and requires fresh physical and full circuit verification.



In [ ]:
%%writefile src/banked_angles_v3.py
"""Truncated ANF bank loading and split ANF erasure for exact ANGLES XOR.

The contiguous prefix of1988 high addresses uses1976 nonlinear monomials
versus1986 unary products. All legal rows are certified unchanged. A precise
polynomial extension defines the other60 rows and is used for cleanup too.

Own follow-up: eleven-bit measurement correction uses 83 Toffoli instead of88,
using a split Boolean algebraic normal form and measurement-erased products.

SPDX-License-Identifier: Apache-2.0

Banked QROM construction adapted from Kozykorpesh / Sumit Chongder:
https://www.kaggle.com/code/kozykappa/quantum-ai-compact-arithmetic-banked-qrom?scriptVersionId=361433617
https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149
Read through Kirill Tym's public quantum-circuits-qrom-tower42 v10 source.

Our change: allocate only nonzero difference columns. Optionally conjugate the
target by a reversible CNOT change of basis, reducing the difference bank to
its GF(2) column rank. The same change is applied to BOTH lookup tables. Thus
the input target can be arbitrary: T^-1(T z XOR T f(a)) = z XOR f(a).
No coefficient, angle word, address, or measurement branch is approximated.

Numeric/profile operations work without kickmix. Circuit building/testing
requires the competition's Linux kickmix package.
"""
from __future__ import annotations

import argparse
import hashlib
import json
from math import ceil
from pathlib import Path

import numpy as np


def batch_deltas(words, J=5):
    words = np.asarray(words, dtype=np.int64)
    nv, na = words.shape
    padded = np.zeros((nv, ceil(na / J) * J), dtype=np.int64)
    padded[:, :na] = words
    batches = padded.reshape(nv, -1, J).transpose(1, 0, 2)
    result = batches.copy()
    result[1:] ^= batches[:-1]
    return result


def pack(row, brot):
    return sum(int(value) << (j * brot) for j, value in enumerate(row))


def _change_basis(values, operations):
    result = list(values)
    for source, target in operations:
        result = [value ^ (((value >> source) & 1) << target)
                  for value in result]
    return result


def difference_tables(delta, brot=10, mode="zero"):
    """Return packed d0, compact dd, and exact reversible output transforms."""
    if mode not in {"baseline", "zero", "linear"}:
        raise ValueError("mode must be baseline, zero, or linear")
    delta = np.asarray(delta, dtype=np.int64)
    if delta.ndim != 2 or len(delta) < 4:
        raise ValueError("expected at least four address rows")
    if np.any(delta < 0) or np.any(delta >= (1 << brot)):
        raise ValueError("angle word out of range")
    A, J = delta.shape
    width = J * brot
    d0 = [pack(row, brot) for row in delta[::2]]
    d1 = [pack(delta[2 * h + 1], brot) if 2 * h + 1 < A else value
          for h, value in enumerate(d0)]
    dd = [a ^ b for a, b in zip(d0, d1)]
    paired = delta[::2].copy()
    paired[:len(delta[1::2])] ^= delta[1::2]
    if A & 1:
        paired[-1] = 0
    bit_columns = ((paired[:, :, None] >> np.arange(brot)) & 1).reshape(H := len(d0), width).T
    packed_columns = np.packbits(bit_columns.astype(np.uint8), axis=1, bitorder="little")
    columns = [int.from_bytes(column.tobytes(), "little") for column in packed_columns]
    active = [bit for bit, column in enumerate(columns) if column]
    unique = len(set(columns) - {0})
    basis = {}
    operations = []
    reduced = columns.copy()
    for bit, original in enumerate(columns):
        column = original
        relation = 0
        while column:
            pivot = column.bit_length() - 1
            if pivot not in basis:
                basis[pivot] = (column, relation ^ (1 << bit))
                break
            value, combination = basis[pivot]
            column ^= value
            relation ^= combination
        if not column:
            # Independent coordinates retain their original values. A dependent
            # coordinate is zeroed by XORing only independent coordinates into it.
            # In particular, identical columns need just one CNOT each way.
            operations.extend((source, bit) for source in range(width)
                              if (relation >> source) & 1)
            reduced[bit] = 0
    rank = len(basis)
    if mode == "linear":
        d0 = _change_basis(d0, operations)
        transformed_dd = _change_basis(dd, operations)
        active = [bit for bit, column in enumerate(reduced) if column]
    else:
        transformed_dd = dd
        operations = []
        if mode == "baseline":
            active = list(range(width))
    compact = [sum(((value >> bit) & 1) << target
                   for target, bit in enumerate(active))
               for value in transformed_dd]
    # A complete integer certificate of the table identity for every legal a.
    reconstructed = [sum(((value >> target) & 1) << bit
                         for target, bit in enumerate(active)) for value in compact]
    restored = _change_basis(
        [d0[a // 2] ^ (reconstructed[a // 2] if a & 1 else 0)
         for a in range(A)], reversed(operations))
    assert restored == [pack(row, brot) for row in delta]
    return d0, compact, active, operations, {
        "rows": A, "width": width, "active_columns": sum(bool(c) for c in columns),
        "unique_nonzero_columns": unique, "gf2_rank": rank,
        "bank_width": len(active), "basis_cnot_count": 2 * len(operations),
        "integer_table_certificate": "PASS", "mode": mode,
    }


def mobius_transform(table):
    """Packed-output Boolean truth table <-> ANF coefficient table over GF(2)."""
    result = list(map(int, table))
    if not result or len(result) & (len(result) - 1):
        raise ValueError("truth-table length must be a power of two")
    n = (len(result) - 1).bit_length()
    for bit in range(n):
        step = 1 << bit
        for start in range(0, len(result), 2 * step):
            for offset in range(step):
                result[start + step + offset] ^= result[start + offset]
    return result


def del_lookup_anf(builder, *, table, address, target, low_bits=None):
    """Clear table[address] with exact conditional ANF phase correction.

    `target` must contain the given lookup value and remain allocated. Extra
    scratch is allocated only when the reset target is too narrow. Clean target
    wires are reused internally; all target and extra scratch finish at zero.
    """
    import kickmix as km
    address = list(address)
    target = list(target)
    n = len(address)
    if len(table) != 1 << n:
        raise ValueError("lookup table must cover the complete address space")
    if n < 2:
        raise ValueError("split ANF cleanup expects at least two address bits")
    low_bits = n // 2 if low_bits is None else low_bits
    if not 0 < low_bits < n:
        raise ValueError("both address halves must be nonempty")
    if any(value < 0 or value >= (1 << len(target)) for value in table):
        raise ValueError("lookup value does not fit the target")
    coefficients = mobius_transform(table)
    assert mobius_transform(coefficients) == list(table), "exact ANF identity failed"
    measurements = builder.alloc_bits(len(target))
    builder.hmr(km.array(target), measurements)
    parity = builder.alloc_bits(1)[0]

    low_count = 1 << low_bits
    high_bits = n - low_bits
    retained_masks = [mask for mask in range(1, low_count) if mask.bit_count() > 1]
    scratch_count = len(retained_masks) + high_bits - 1
    extra = builder.alloc_qubits(max(0, scratch_count - len(target)))
    scratch = target + list(extra)
    low = {1 << bit: address[bit] for bit in range(low_bits)}
    for mask, qubit in zip(retained_masks, scratch):
        bit = mask.bit_length() - 1
        parent = mask ^ (1 << bit)
        builder.init_and(low[parent], address[bit], qubit)
        low[mask] = qubit
    carries = scratch[len(retained_masks):scratch_count]

    def correct(high_mask, high_qubit):
        for low_mask in range(low_count):
            coefficient = coefficients[low_mask | (high_mask << low_bits)]
            if not coefficient:
                continue
            builder.bit_store(parity, False)
            while coefficient:
                bit = (coefficient & -coefficient).bit_length() - 1
                builder.cx(measurements[bit], parity)
                coefficient &= coefficient - 1
            if not low_mask and not high_mask:
                builder.neg_if(parity)
                continue
            builder.push_condition(parity)
            if not low_mask:
                builder.z(high_qubit)
            elif not high_mask:
                builder.z(low[low_mask])
            else:
                builder.cz(low[low_mask], high_qubit)
            builder.pop_condition()

    def walk(mask, highest_bit, control):
        correct(mask, control)
        for bit in range(highest_bit + 1, high_bits):
            child_mask = mask | (1 << bit)
            if not mask:
                walk(child_mask, bit, address[low_bits + bit])
            else:
                child = carries[mask.bit_count() - 1]
                builder.init_and(control, address[low_bits + bit], child)
                walk(child_mask, bit, child)
                builder.del_and(control, address[low_bits + bit], child)

    walk(0, -1, None)
    for mask in reversed(retained_masks):
        bit = mask.bit_length() - 1
        parent = mask ^ (1 << bit)
        builder.del_and(low[parent], address[bit], low[mask])
    builder.free(extra)
    return {"address_bits": n, "low_bits": low_bits, "high_bits": high_bits,
            "retained_products": len(retained_masks), "traversal_carries": high_bits - 1,
            "anf_toffoli": (1 << low_bits) - low_bits - 1 + (1 << high_bits) - high_bits - 1,
            "scratch_count": scratch_count, "anf_table_identity": "PASS"}


def init_banked_truncated_anf(builder, *, d0, dd, address, target, bank, scratch):
    """XOR d0[h] into target, dd[h] into bank; return the completed dd table.

    Requires `mobius_transform` in the surrounding module (anf_cleanup.py).
    Legal h ranges from zero through len(d0)-1. `scratch` is clean and remains
    clean. All target bits may start arbitrary; bank initialization is the
    caller's usual clean lookup bank.
    """
    import kickmix as km
    address, target, bank, scratch = map(list, (address, target, bank, scratch))
    H, n, W = len(d0), len(address), len(target)
    if len(dd) != H or H > (1 << n) or H <= (1 << (n - 1)):
        raise ValueError("expected a nonempty prefix using every address bit")
    if len(scratch) < n - 1:
        raise ValueError("need n-1 clean traversal carry wires")
    original = [int(first) | (int(second) << W) for first, second in zip(d0, dd)]
    coefficients = mobius_transform(original + [0] * ((1 << n) - H))
    coefficients[H:] = [0] * ((1 << n) - H)
    completed = mobius_transform(coefficients)
    assert completed[:H] == original, "truncated ANF changed a legal table row"
    assert mobius_transform(completed) == coefficients
    outputs = target + bank

    def load(mask, control):
        value = coefficients[mask]
        targets = [outputs[bit] for bit in range(len(outputs)) if (value >> bit) & 1]
        if targets:
            if control is None:
                builder.x(km.array(targets))
            else:
                builder.cx(control, km.array(targets))

    def walk(mask, highest_bit, control):
        load(mask, control)
        for bit in range(highest_bit + 1, n):
            child_mask = mask | (1 << bit)
            if child_mask >= H:
                continue
            if not mask:
                walk(child_mask, bit, address[bit])
            else:
                child = scratch[mask.bit_count() - 1]
                builder.init_and(control, address[bit], child)
                walk(child_mask, bit, child)
                builder.del_and(control, address[bit], child)

    walk(0, -1, None)
    return [value >> W for value in completed], {
        "legal_high_addresses": H, "address_bits": n,
        "nonlinear_products": H - 1 - n,
        "previous_unary_products": H - 2,
        "exact_legal_table_identity": "PASS",
        "extension_mobius_identity": "PASS",
        "completed_high_addresses": (1 << n) - H,
        "max_required_carries": n - 1,
    }


def build_qroam2(delta, brot=10, *, mode="zero", return_report=False):
    """Exact banked XOR oracle; same public register contract as upstream."""
    import kickmix as km
    delta = np.asarray(delta, dtype=np.int64)
    A, J = delta.shape
    nbits = (A - 1).bit_length()
    H = (A + 1) // 2
    hbits = nbits - 1
    W = J * brot
    d0, dd, active, transforms, report = difference_tables(delta, brot, mode)
    b = km.CircuitBuilder()
    sel = b.create_quantum_register(nbits, "selection")
    words = [b.create_quantum_register(brot, f"word_{j}") for j in range(J)]
    tgt = [q for word in words for q in word]
    for source, target in transforms:
        b.cx(tgt[source], tgt[target])
    B = b.alloc_qubits(len(active))
    hreg = sel[1:nbits]
    anc = b.alloc_qubits(hbits - 1)
    completed_dd, load_report = init_banked_truncated_anf(
        b, d0=d0, dd=dd, address=hreg, target=tgt, bank=B, scratch=anc)
    report["anf_bank_load"] = load_report
    b.free(anc)
    for bank_bit, target in enumerate(active):
        b.ccx(sel[0], B[bank_bit], tgt[target])
    if active:
        table = completed_dd
        report["anf_cleanup"] = del_lookup_anf(b, table=table, address=hreg, target=B)
    b.free(B)
    for source, target in reversed(transforms):
        b.cx(tgt[source], tgt[target])
    circuit = b.finish_circuit()
    report.update(num_qubits=circuit.num_qubits,
                  static_toffoli=int(circuit.operation_statistics().num_toffoli_like))
    return (circuit, report) if return_report else circuit


def profile(words, brot=10, Js=(4, 5, 6, 7)):
    return {str(J): [difference_tables(delta, brot, "linear")[-1]
                    for delta in batch_deltas(words, J)] for J in Js}


def test_random_targets(circuits, deltas, *, brot=10, repeats=8, seed=20261023):
    """Every legal address, random dirty targets, HMR phase and scratch checks.

    This is simulation evidence; the exhaustive integer identity is separately
    certified above. This test is not a proof over all measurement branches.
    The full official verifier remains the acceptance check for the family.
    """
    import kickmix as km
    from dfthcsos.scoreinfra.circuits import register_layout, _check_state
    rng = np.random.default_rng(seed)
    layout = register_layout(circuits)
    A = len(deltas[0])
    addresses = np.tile(np.arange(A, dtype=np.int64), repeats)
    address_bits = ((addresses[None, :] >> np.arange(len(layout.selection))[:, None]) & 1).astype(bool)
    sim = km.Simulator(batch_size=len(addresses), debug_print=False,
                       seed=seed, counting_operations=True)
    sim.use_same_registers_as(circuits[0])
    place_values = np.int64(1) << np.arange(brot)
    toffolis = []
    for k, (circuit, delta) in enumerate(zip(circuits, deltas)):
        sim.clear_for_shot()
        sim.write_across_shots(layout.selection, address_bits)
        dirty = rng.integers(0, 1 << brot, size=(len(addresses), len(layout.words)), dtype=np.int64)
        for j, word in enumerate(layout.words):
            sim.write_across_shots(word, ((dirty[:, j][None, :] >> np.arange(brot)[:, None]) & 1).astype(bool))
        before = sim.operation_statistics().num_toffoli_like
        sim.do(circuit)
        _check_state(sim, circuit, layout, address_bits, f"random-target ANGLE_{k}")
        result = np.stack([place_values @ sim.read_across_shots(word).astype(np.int64)
                           for word in layout.words], axis=1)
        assert np.array_equal(result, dirty ^ delta[addresses]), f"ANGLE_{k} wrong XOR"
        toffolis.append((sim.operation_statistics().num_toffoli_like - before) / len(addresses))
    return {"status": "PASS", "addresses": A, "repeats": repeats,
            "shots_per_circuit": len(addresses), "num_circuits": len(circuits),
            "random_target_xor": "PASS", "unchanged_selection": "PASS",
            "clean_workspace": "PASS", "zero_observed_phase_errors": True,
            "toffoli_per_circuit": toffolis, "toffoli_per_pass": sum(toffolis),
            "full_official_verification": "PENDING", "seed": seed}


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("words", type=Path)
    parser.add_argument("output", type=Path)
    parser.add_argument("--mode", choices=("baseline", "zero", "linear"), default="zero")
    parser.add_argument("--J", type=int, default=5)
    parser.add_argument("--brot", type=int, default=10)
    parser.add_argument("--profile-only", action="store_true")
    parser.add_argument("--repeats", type=int, default=8)
    args = parser.parse_args()
    words = np.loadtxt(args.words, delimiter=",", dtype=np.int64)
    args.output.mkdir(parents=True, exist_ok=True)
    report = {"source_csv": str(args.words), "source_csv_sha256": hashlib.sha256(args.words.read_bytes()).hexdigest(),
              "shape": list(words.shape), "brot": args.brot, "J": args.J, "mode": args.mode,
              "profiles": profile(words, args.brot)}
    if not args.profile_only:
        deltas = batch_deltas(words, args.J)
        circuits, resources = [], []
        for k, delta in enumerate(deltas):
            circuit, resource = build_qroam2(delta, args.brot, mode=args.mode, return_report=True)
            if circuit.num_qubits > 128:
                raise ValueError(f"ANGLE_{k} exceeds the 128-qubit cap")
            path = args.output / f"angle_k_{k}.kmb"
            circuit.to_file(path, format="kmb")
            resource.update(k=k, sha256=hashlib.sha256(path.read_bytes()).hexdigest(), bytes=path.stat().st_size)
            circuits.append(circuit)
            resources.append(resource)
            print(json.dumps(resource), flush=True)
        report["circuits"] = resources
        report["random_target_test"] = test_random_targets(circuits, deltas, brot=args.brot, repeats=args.repeats)
        toffoli = 8794 + 2 * report["random_target_test"]["toffoli_per_pass"]
        report["estimate_using_unchanged_c5_coefficients"] = {
            "full_system_qubits": 631 + args.J * args.brot,
            "full_system_toffoli": toffoli, "lambda_eff": 47.558542335159316,
            "estimated_cost": (631 + args.J * args.brot) * toffoli * 47.558542335159316,
            "label": "RESOURCE ESTIMATE; full official verifier pending"}
    (args.output / "report.json").write_text(json.dumps(report, indent=2), encoding="utf8")
    print(json.dumps({key: value for key, value in report.items() if key != "profiles" and key != "circuits"}, indent=2), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/gauge_generator.py
"""Reproducible exact DFTHC gauge from public numerical input, NumPy only.

SPDX-License-Identifier: Apache-2.0

Input and initial factorization: Dmitrii Gluzdov's public Apache-2.0 dataset
https://www.kaggle.com/datasets/dmitriigluzdov/vqcc-femoco-public-factorization/versions/2
The C30 factor and original sparse-factor work trace to k256.dev and
Kozykorpesh, as credited by the accompanying notebook/source notices.
The log-depth tree and vector decoding follow the organizers' reference.

Own optimization: simultaneous orthogonal Cayley moves reduce the squared
column L1 sum. The augmented [W; H2_I] Gram is preserved; no angle words,
Hamiltonian integrals, symmetry shifts, or acceptance thresholds are changed.
This file imports no research modules or previously saved factorization.
"""
from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path
import time

import numpy as np

PUBLIC_SHA256 = "9afd0ca5e90dd1740fe713e4ac1c26736184ea9bc4d4b57a6eeefafd608ee193"
SHIFTS_SHA256 = "7b7facf63fe35a9d9d3d00787ffdb700c863fa845fe4724d7f7c54778151cd87"
FACTORIZATION_KEYS = ("U", "W", "H2_I", "H1_I", "Hsym_lower", "tree_edges")


def log_depth_tree_edges(n=76):
    """Parent-before-child breadth-first order of the official Givens tree."""
    edges, level, depth = [], [0], 0
    while len(edges) < n-1:
        nxt = []
        for label in level:
            nxt.append(label)
            child = label+(1 << depth)
            if child < n:
                edges.append((label, child))
                nxt.append(child)
        level, depth = nxt, depth+1
    return np.asarray(edges, dtype=np.int64)


def decode_words(words, edges, brot=10):
    """Apply the ordered rotations to e0, matching official vector decoding."""
    angles = np.asarray(words, dtype=np.float64)/(1 << (brot-1))
    vectors = np.zeros((len(words), words.shape[1]+1), np.float64)
    vectors[:, 0] = 1.0
    for j, (parent, child) in enumerate(edges):
        parent_value, child_value = vectors[:, parent].copy(), vectors[:, child].copy()
        theta = np.pi*angles[:, j]
        cosine, sine = np.cos(theta), np.sin(theta)
        vectors[:, parent] = cosine*parent_value-sine*child_value
        vectors[:, child] = sine*parent_value+cosine*child_value
    return vectors


def column_l1_square(a, epsilon=0.0):
    absolute = np.sqrt(a*a+epsilon*epsilon) if epsilon else np.abs(a)
    return np.sum(absolute.sum(axis=1)**2, axis=1)


def cayley_stage(a, iterations, epsilon):
    """Orthogonal right rotations, with separate line searches for each rho."""
    r, _, c = a.shape
    eye = np.broadcast_to(np.eye(c), (r, c, c))
    best, best_loss = a.copy(), column_l1_square(a)
    rates = np.full(r, 0.5)
    history = []
    for iteration in range(iterations):
        absolute = np.sqrt(a*a+epsilon*epsilon)
        gradient = 2*absolute.sum(axis=1)[:, None, :]*a/absolute
        raw = a.swapaxes(1, 2) @ gradient
        skew = 0.5*(raw-raw.swapaxes(1, 2))
        initial = column_l1_square(a, epsilon)
        accepted = np.zeros(r, bool)
        next_a, trial_rates = a.copy(), rates.copy()
        for _ in range(12):
            matrix = skew*(0.5*trial_rates)[:, None, None]
            # For skew K, (I+K)^(-1)(I-K) is orthogonal.
            rotation = np.linalg.solve(eye+matrix, eye-matrix)
            trial = a @ rotation
            good = (column_l1_square(trial, epsilon) < initial-1e-13) & ~accepted
            next_a[good] = trial[good]
            accepted |= good
            trial_rates[~accepted] *= 0.5
            if accepted.all():
                break
        a = next_a
        rates = np.minimum(3.0, np.maximum(1e-5, trial_rates*1.2))
        actual = column_l1_square(a)
        improved = actual < best_loss
        best[improved], best_loss[improved] = a[improved], actual[improved]
        if iteration % 40 == 0:
            history.append({"iteration": iteration, "best_l1_square": float(best_loss.sum())})
    return best, history


def generate(public_npz, fixed_shifts_npz, *, copies=30, iterations=100):
    """Return (factorization, unchanged angle words, algebraic build report).

    Dependencies are NumPy, the checksummed public v2 NPZ, and the official
    fixed shifts NPZ. Fresh official verification is a separate caller step.
    """
    started = time.monotonic()
    source, shifts_source = Path(public_npz), Path(fixed_shifts_npz)
    assert hashlib.sha256(source.read_bytes()).hexdigest() == PUBLIC_SHA256
    assert hashlib.sha256(shifts_source.read_bytes()).hexdigest() == SHIFTS_SHA256
    assert copies == 30 and iterations == 100, "reviewed candidate uses C30 and100 iterations/stage"
    with np.load(source, allow_pickle=False) as archive:
        words = archive["words"].astype(np.int64)
        w, h2 = archive["W"].copy(), archive["H2_I"].copy()
    assert words.shape == (3976, 75) and w.shape == (65, 60, 30) and h2.shape == (65, 30)
    assert np.all((words >= 0) & (words < 1024))
    assert np.isfinite(w).all() and np.isfinite(h2).all()
    original_words = words.copy()
    edges = log_depth_tree_edges()
    u = decode_words(words[76:], edges).reshape(65, 60, 76)
    assert np.max(np.abs(np.linalg.norm(u, axis=-1)-1)) < 1e-12
    a = np.concatenate([w, h2[:, None, :]], axis=1)
    original_gram = a @ a.swapaxes(1, 2)
    baseline = float(column_l1_square(a).sum())
    stages = []
    for epsilon in (0.0005, 0.00005, 0.000005):
        a, history = cayley_stage(a, iterations, epsilon)
        stages.append({"epsilon": epsilon, "history": history,
                       "l1_square": float(column_l1_square(a).sum())})
    final = float(column_l1_square(a).sum())
    drift = float(np.max(np.abs(a @ a.swapaxes(1, 2)-original_gram)))
    assert drift < 1e-8 and final < baseline
    # h1 is unchanged. Retain its original encoded eigenvectors: a redundant
    # eigendecomposition could choose different signs and alter angle words.
    assert np.array_equal(words, original_words), "all3976x75 integer angle words must be unchanged"
    with np.load(shifts_source, allow_pickle=False) as archive:
        fixed = {key: archive[key].copy() for key in ("H1_I", "Hsym_lower")}
    data = {"U": u, "W": a[:, :-1, :], "H2_I": a[:, -1, :],
            "tree_edges": edges, **fixed}
    report = {"public_input_sha256": PUBLIC_SHA256, "fixed_shifts_sha256": SHIFTS_SHA256,
              "R_B_C": [65, 60, 30], "brot": 10, "iterations_per_stage": iterations,
              "baseline_l1_square": baseline, "final_l1_square": final,
              "augmented_gram_max_drift": drift,
              "all_3976x75_angle_words_unchanged": True,
              "angle_words_sha256_int64": hashlib.sha256(words.astype("<i8").tobytes()).hexdigest(),
              "stages": stages, "seconds": time.monotonic()-started,
              "numpy_version": np.__version__,
              "correctness_label": "Algebraic invariants checked; full official verifier not invoked by generate()"}
    return data, words, report


def save_factorization(path, data):
    """Save only the public factorization contract and its organizer tree."""
    np.savez(path, **{key: data[key] for key in FACTORIZATION_KEYS})


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("public_npz", type=Path)
    parser.add_argument("fixed_shifts_npz", type=Path)
    parser.add_argument("output_npz", type=Path)
    args = parser.parse_args()
    result, words, report = generate(args.public_npz, args.fixed_shifts_npz)
    args.output_npz.parent.mkdir(parents=True, exist_ok=True)
    save_factorization(args.output_npz, result)
    report["output_npz_sha256"] = hashlib.sha256(args.output_npz.read_bytes()).hexdigest()
    args.output_npz.with_suffix(".build.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2), flush=True)


In [ ]:
%%writefile src/signed_fourbank.py
"""Signed-Givens four-address QROM, adapted from Kozykorpesh V3.

SPDX-License-Identifier: Apache-2.0
Pinned primary source:
https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445
The signed-angle gauge, four-address decomposition and selective five-scratch
pebbles originate in that source. The exact ANF eraser is our credited
banked_angles_v3 (Sumit/Kozykorpesh/Kirill/anazemcev lineage).

Our adapter uses the small public words and our existing Gram-gauged factor;
no Toolbox binary, cached QROM output or training checkpoint is required.
The all-row sign certificate bounds the decoded-vector change. Full official
verification is a separate required check for the generated family.
"""
from functools import lru_cache
import numpy as np
import kickmix as km
import banked_angles_v3 as B


def canonicalize(data, angle_words, brot=10):
    """Return signed words/physical factor, preserving each vector up to sign.

    Adapted from Kozykorpesh gauge_qrom.canonicalize. Removing numerical
    zero branches is accepted only if every reconstructed row passes the
    explicit component-wise sign certificate below.
    """
    from dfthcsos.scoreinfra.dfthc_math import reconstruct_u_vectors
    out={key:np.array(value,copy=True) for key,value in data.items()}
    words_in=np.asarray(angle_words,dtype=np.int64)
    if brot!=10 or words_in.ndim!=2:
        raise ValueError('Expected a two-dimensional 10-bit angle table.')
    edges=np.asarray(data['tree_edges'],dtype=np.int64)
    u=reconstruct_u_vectors(words_in/(1<<(brot-1)),edges)
    n=u.shape[1]
    if len(words_in)!=n+int(np.prod(data['U'].shape[:-1])):
        raise ValueError('Angle rows do not match the factor dimensions.')
    assert np.array_equal(u[n:].reshape(data['U'].shape),data['U'])
    a=u.copy();angles=np.empty_like(words_in,dtype=np.float64)
    for j,(p,q) in reversed(list(enumerate(edges))):
        norm=np.hypot(a[:,p],a[:,q])
        sign=np.where(np.abs(a[:,p])<1e-12,np.where(a[:,q]>0,-1,1),np.where(a[:,p]<0,-1,1))
        angles[:,j]=np.where(norm<1e-12,0,np.arctan2(a[:,q]*sign,a[:,p]*sign))
        a[:,p]=np.where(norm<1e-12,0,norm*sign);a[:,q]=0
    words=np.rint(angles*((1<<(brot-1))/np.pi)).astype(np.int64)%(1<<brot)
    assert not np.any(((words>>9)^(words>>8))&1),'Signed-angle identity failed'
    decoded=reconstruct_u_vectors(words/(1<<(brot-1)),edges)
    row_sign=np.where((u*decoded).sum(axis=1)<0,-1,1)
    drift=float(np.max(np.abs(decoded*row_sign[:,None]-u)))
    if drift>1e-11:
        raise ArithmeticError(f'Signed Givens gauge drift {drift}')
    out['U']=decoded[n:].reshape(data['U'].shape)
    out['angle_words']=words
    report={'rows_checked':len(words),'angles_per_row':words.shape[1],
        'max_component_drift_up_to_row_sign':drift,
        'redundant_bit_identity':'bit9 = bit8',
        'W_unchanged':np.array_equal(out['W'],data['W']),
        'H2_I_unchanged':np.array_equal(out['H2_I'],data['H2_I']),
        'H1_I_unchanged':np.array_equal(out['H1_I'],data['H1_I']),
        'Hsym_lower_unchanged':np.array_equal(out['Hsym_lower'],data['Hsym_lower']),
        'source':'Kozykorpesh V3/run361677445, canonicalize adapted to public input',
        'physical_invariance':'Each one-body and factor vector is identical up to a row sign; its outer product is invariant. W and shifts are unchanged.',
        'full_official_verification':'PENDING'}
    return out,report


def xor_and(b,left,right,targets):
    targets=list(targets)
    if not targets:return
    q=targets[0]
    for t in targets[1:]:b.cx(q,t)
    b.ccx(left,right,q)
    for t in reversed(targets[1:]):b.cx(q,t)

def pebbled_init(b,table,address,outputs,scratch):
    address,outputs,scratch=map(list,(address,outputs,scratch))
    H,n=len(table),len(address)
    coefficients=B.mobius_transform(list(table)+[0]*((1<<n)-H))
    coefficients[H:]=[0]*((1<<n)-H)
    completed=B.mobius_transform(coefficients)
    assert completed[:H]==table
    stats={'monomial_ands':0,'recomputed_ands':0,'pebble_sites':0,'direct_leaf_ands':0,'scratch':len(scratch)}
    @lru_cache(None)
    def children(mask,last):return tuple((mask|(1<<j),j) for j in range(last+1,n) if (mask|(1<<j))<H)
    @lru_cache(None)
    def need(mask,last):
        return max([1+need(c,j) for c,j in children(mask,last) if children(c,j)]+[0])
    def targets(mask):
        v=coefficients[mask]
        return [outputs[j] for j in range(len(outputs)) if (v>>j)&1]
    def walk(mask,last,control,path,free):
        erased=[]
        if mask.bit_count()==5 and need(mask,last)>len(free):
            m=need(mask,last)-len(free)
            assert 0<m<=len(path)-1
            erased=path[:m]
            for a,c,t in reversed(erased):b.del_and(a,c,t);free.append(t)
            stats['pebble_sites']+=1
        ts=targets(mask)
        if ts:
            if control is None:b.x(km.array(ts))
            else:b.cx(control,km.array(ts))
        for child,j in children(mask,last):
            if not mask:
                walk(child,j,address[j],path,free)
            elif not children(child,j):
                ts=targets(child)
                if ts:xor_and(b,control,address[j],ts);stats['direct_leaf_ands']+=1
            else:
                assert free,('scratch exhausted',mask,j,len(path))
                t=free.pop();b.init_and(control,address[j],t);stats['monomial_ands']+=1
                walk(child,j,t,path+[(control,address[j],t)],free)
                b.del_and(control,address[j],t);free.append(t)
        for a,c,t in erased:
            free.remove(t);b.init_and(a,c,t);stats['recomputed_ands']+=1
    walk(0,-1,None,[],scratch.copy())
    return completed,stats

def build(delta,brot=10,return_report=False):
    delta=np.asarray(delta,dtype=np.int64);A,J=delta.shape
    assert brot==10 and J==3 and A%4==0
    assert not np.any(((delta>>9)^(delta>>8))&1)
    W=10*J;BW=9*J;H=A//4;n=(A-1).bit_length()
    packed=[B.pack(row,10) for row in delta]
    ds=[[],[],[]];base=[]
    compact=lambda row:sum(((row>>(j*10))&511)<<(j*9) for j in range(J))
    for h in range(H):
        v=packed[4*h:4*h+4];base.append(v[0])
        ds[0].append(compact(v[0]^v[1]));ds[1].append(compact(v[0]^v[2]))
        ds[2].append(compact(v[0]^v[1]^v[2]^v[3]))
    def expand(v):
        return sum((((v>>(9*j))&511)|(((v>>(9*j+8))&1)<<9))<<(10*j) for j in range(J))
    restored=[base[h]^expand((ds[0][h] if a&1 else 0)^(ds[1][h] if a&2 else 0)^(ds[2][h] if (a&3)==3 else 0)) for h in range(H) for a in range(4)]
    assert restored==packed
    values=[base[h]|(ds[0][h]<<W)|(ds[1][h]<<(W+BW))|(ds[2][h]<<(W+2*BW)) for h in range(H)]
    b=km.CircuitBuilder();sel=b.create_quantum_register(n,'selection')
    tgt=[q for j in range(J) for q in b.create_quantum_register(10,f'word_{j}')]
    bank=list(b.alloc_qubits(3*BW));scratch=b.alloc_qubits(5)
    completed,stats=pebbled_init(b,values,sel[2:],tgt+bank,scratch);b.free(scratch)
    both=b.alloc_qubits(1)[0];b.init_and(sel[0],sel[1],both)
    for s,ctl in enumerate((sel[0],sel[1],both)):
        for j in range(J):
            for k in range(9):
                ts=[tgt[j*10+k]]+([tgt[j*10+9]] if k==8 else [])
                xor_and(b,ctl,bank[s*BW+j*9+k],ts)
    b.del_and(sel[0],sel[1],both);b.free(both)
    erase=B.del_lookup_anf(b,table=[v>>W for v in completed],address=sel[2:],target=bank,low_bits=5)
    b.free(km.array(bank));c=b.finish_circuit()
    report={'num_qubits':c.num_qubits,'static_toffoli':int(c.operation_statistics().num_toffoli_like),'table_identity':'PASS','init':stats,'cleanup':erase}
    assert c.num_qubits<=128
    return (c,report) if return_report else c


In [ ]:
# notebook: hide_input
import gauge_generator
import signed_fourbank
import banked_angles_v3 as banked
fixed_shifts=COMP/'verifiers/dfthcsos/FeMoCo76o113e.fixed_shifts.npz'
factor,original_words,gauge_report=gauge_generator.generate(PUBLIC_WORDS,fixed_shifts,copies=30,iterations=100)
chem_data,sign_gauge_report=signed_fourbank.canonicalize(factor,original_words,brot=10)
angle_words=chem_data['angle_words']
assert sign_gauge_report['rows_checked']==3976
assert sign_gauge_report['max_component_drift_up_to_row_sign']<1e-11
assert np.array_equal(chem_data['W'],factor['W']) and np.array_equal(chem_data['H2_I'],factor['H2_I'])
chem_folder=FINAL/'dfthcsos';chem_folder.mkdir(exist_ok=True)
gauge_generator.save_factorization(chem_folder/'submission.npz',chem_data)
save_json(REPORTS/'gauge_build.json',gauge_report)
save_json(REPORTS/'signed_gauge.json',sign_gauge_report)
EXPECTED_ANGLE_COUNT=25
deltas=banked.batch_deltas(angle_words,3)
assert len(deltas)==EXPECTED_ANGLE_COUNT
angle_reports=[]
for index,delta in enumerate(deltas):
    circuit,report=signed_fourbank.build(delta,10,return_report=True)
    path=chem_folder/f'angle_k_{index}.kmb'
    circuit.to_file(str(path),format='kmb')
    assert circuit.num_qubits==128 and report['static_toffoli']==1129
    angle_reports.append({'index':index,'sha256':digest(path),**report})
    del circuit
save_json(REPORTS/'banked_build.json',{'angles':angle_reports,'angle_count':len(angle_reports),
    'max_angle_qubits':max(row['num_qubits'] for row in angle_reports),
    'method':'Signed-Givens gauge, four-address banks and five-scratch pebbling',
    'full_official_check_required':True})
print('Built 25 signed four-address lookups: 128Q and 1129 Toffolis each.',flush=True)
del factor,original_words,chem_data,angle_words,deltas
gc.collect()


## 3. DQI: source-level memory scheduling

The decoder is the published reversible Reed–Solomon construction of **Gu et al.**, ported to the competition field by **Dmitrii Gluzdov**. His V6 schedule borrows idle template words, streams sweep evaluation, and recreates a field norm rather than retaining it. Kozykorpesh's symbolic runtime folds identities using exact Boolean forms and conservative invalidation across native operations.

The pinned baseline rebuilds to **1789Q and 445229 static Toffolis**. Its canonical circuit hash must match before verification. Approximate decoder support assumptions remain part of the credited algorithm; they are not labelled as all-input correctness proofs. The full official verifier measures both value and phase errors and includes the resulting repetition penalty.

That penalty matters because the task objective is $R Q^2 T$. A structural reduction in $Q$ has a quadratic benefit. A different sampled error count alone does not establish a better decoder, so the development salt is fixed and hashes are never selected for favorable samples.



In [ ]:
%%writefile src/dqi_driver.py
"""Full source decoder build with exact symbolic lowering (Apache-2.0)."""
import os,sys,pathlib,json,time,gc

def build(destination,work):
    src=pathlib.Path(__file__).resolve().parent
    work=pathlib.Path(work);work.mkdir(parents=True,exist_ok=True)
    os.environ['DQI_BORROW']='affine'
    if str(src/'gen_dqi_v6') not in sys.path:sys.path.insert(0,str(src/'gen_dqi_v6'))
    import gu_formula_derivation as F
    t,_=F.run_b12();assert F.certify_kt(t['KT12'],F.FIELD)
    c=json.loads((src/'gen_dqi_v6/ref/gu_constants.json').read_text());c.update(t)
    path=work/'dqi_constants.json';path.write_text(json.dumps(c));os.environ['DQI_CONSTANTS']=str(path)
    import gu_port as G,run_gu_port as RG,round2_gadgets as R2
    import importlib
    G=importlib.reload(G);R2=importlib.reload(R2)
    import symbolic_runtime as S
    G.Builder=S.SymbolicBuilder;G.declare_registers=S.declare_registers
    G.borrow_template=RG.borrow_template
    R2.install(G,top_first=True,pebble_last=True,norm_pebble=True,compact_from=70)
    source=(src/'gen_dqi_v6/round2_gadgets.py').read_text()
    # Sweep emits native array operations. Mark every write conservatively.
    source=source[source.index('def lean_sweep('):]
    source=source.replace('b.physical[q]','b.wire(q)').replace('b.physical[root]','b.wire(root)').replace('b.physical[loc[0]]','b.wire(loc[0])').replace('b.physical[ps[h]]','b.wire(ps[h])')
    source=source.replace("raw.cx(arr([q for word in words for q in word]),arr([q for _ in words for q in target]))", "raw.cx(arr([q for word in words for q in word]),arr([q for _ in words for q in target]));b.invalidate(target)")
    source=source.replace("tr=A.open('tree',11);", "b.invalidate(val);tr=A.open('tree',11);")
    source=source.replace("linear=[('L',tuple(w[k] for w in ebw),0,(val[k],)) for k in range(12)]", "linear=[('L',tuple(w[k] for w in ebw),0,(val[k],)) for k in range(12)]")
    source=source.replace("raw.del_and(b.wire(root),arr(om),arr(loc))", "raw.del_and(b.wire(root),arr(om),arr(loc));b.forms[loc[0]]=S.ZERO")
    source=source.replace("raw.init_and(b.wire(root),arr(om),arr(loc))", "raw.init_and(b.wire(root),arr(om),arr(loc));b.invalidate(loc)")
    source=source.replace("raw.cx(arr([w[k] for w in omw]),arr(om*len(omw)))", "raw.cx(arr([w[k] for w in omw]),arr(om*len(omw)));b.invalidate(om)")
    source=source.replace("if factor!=1:raw.gf2_imul(factor,target=reg,field=gf)", "if factor!=1:raw.gf2_imul(factor,target=reg,field=gf)")
    ns={'S':S};exec(source,ns)
    def native_sweep(b,E,A,omw,ebw,abw):
        qs=[q for r in omw+ebw+abw for q in r]
        b.suspended=True
        ns['stream_sweep'](b,E,A,omw,ebw,abw)
        b.invalidate(qs)
        b.suspended=False
    G.emit_sweep=native_sweep
    start=time.monotonic();b,_=G.build();b.k.free(b.measurement)
    cir=b.k.finish_circuit();cir.to_file(str(destination),format='kmb')
    report={'num_qubits':cir.num_qubits,'toffoli_count':int(cir.operation_statistics().num_toffoli_like),'ops':len(cir),'build_seconds':time.monotonic()-start,'elided':b.elided,'folded':b.folded,'symbolic_products':len(b.products)}
    return report


In [ ]:
%%writefile src/selected_decoder.py
"""Notebook-facing wrapper for Kozykappa V3's readable DQI source.

Place upstream dqi_driver.py and its gen_dqi_v6/ directory beside this file.
The competition kickmix package must already be on sys.path. This wrapper
performs source generation only; use the unchanged official full verifier.
"""
from hashlib import sha256
from pathlib import Path

EXPECTED_SHA256 = '529b075009f7758abeda83a63407f0c1feeb51dd70631ae4335dfd28721b7e08'
EXPECTED_QUBITS = 1789
EXPECTED_TOFFOLIS = 445229


def build_selected(destination, work):
    import dqi_driver
    destination = Path(destination)
    destination.parent.mkdir(exist_ok=True, parents=True)
    report = dqi_driver.build(destination, Path(work))
    digest = sha256()
    with destination.open('rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            digest.update(block)
    actual = digest.hexdigest()
    assert actual == EXPECTED_SHA256, (actual, EXPECTED_SHA256)
    assert report['num_qubits'] == EXPECTED_QUBITS, report
    assert report['toffoli_count'] == EXPECTED_TOFFOLIS, report
    return report | {'sha256': actual, 'upstream': 'Kozykappa V3 / run361677445'}


In [ ]:
%%writefile src/gen_dqi_v6/port_runtime.py
# SPDX-License-Identifier: Apache-2.0
"""Small explicit adapter for Gu et al.'s published source to official kickmix.

Integer wire handles and LIFO ownership match the published builder. Physical
kickmix qubits stay allocated while this adapter accounts for their lifetimes;
every recycle emits the same reset as the authors' Builder. Named replay holds
are checked, so the journal cannot silently alias live scratch.
"""
from contextlib import contextmanager
import json, os, pathlib, time
import kickmix as km
from ref.gu_classical import GF2n, f2_matinv, f2_transpose, parity
from ref.gu_linear import linear_in_place


def load_contract(_):
    contract=json.loads((pathlib.Path(__file__).parent/'ref/gu_contract.json').read_text())
    # The paper uses 0x10eb. The competition uses 0x1053. All arithmetic
    # formulas are regenerated from this field; no field labels are conflated.
    contract['gf_poly']=km.GF2Field(12).modulus
    return contract


class Journal:
    """Gu et al., uncompute.bennett.Journal; pure conjugate choreography."""
    def __init__(self): self._steps=[];self.recording=True
    def mark(self):return len(self._steps)
    def step(self, fwd, rev=None):
        fwd()
        if self.recording:self._steps.append((fwd,rev if rev is not None else fwd))
    def defer(self, fwd, rev=None):self._steps.append((fwd,rev if rev is not None else fwd))
    def forward(self,since=0,until=None):
        for fwd,_ in self._steps[since:until]:fwd()
    def reverse(self,since=0,until=None):
        for _,rev in reversed(self._steps[since:until]):rev()


class Builder:
    def __init__(self):
        self.k=km.CircuitBuilder()
        self.physical=[]
        self.free_qubits=[];self.free_set=set();self.active=set()
        self.errors=[self.k.create_classical_register(12,name=f'E{j}') for j in range(4095)]
        # Published emitter uses each HMR outcome only for the immediately
        # following phase repayments, before the next HMR. Reusing one live
        # classical bit preserves that ordering and avoids dead classical ids.
        self.measurement=self.k.alloc_bits(1)[0]
        self.phase_stats=[];self.gate_counts={}
        self.phase_name='init';self.started=time.monotonic()
        self.owners={};self.live_profile={};self.current_phase=None
        self.progress=pathlib.Path(os.environ['BRIDGE_OUT'])/'build_progress.json' if 'BRIDGE_OUT' in os.environ else None

    def qubit(self):
        if self.free_qubits:
            q=self.free_qubits.pop();self.free_set.remove(q)
        else:
            q=len(self.physical);self.physical.append(self.k.alloc_qubits(1)[0])
        assert q not in self.active
        self.active.add(q)
        self._profile()
        return q
    def qubits(self,n):return [self.qubit() for _ in range(n)]
    def reclaim(self,qs):
        qs=list(qs);want=set(qs)
        assert len(want)==len(qs) and want<=self.free_set, ('reclaim live ids',want-self.free_set)
        self.free_qubits[:]=[q for q in self.free_qubits if q not in want]
        self.free_set-=want;self.active.update(want)
        self._profile()
    def recycle(self,q):
        assert q in self.active,('recycle not active',q)
        self.k.reset(self.physical[q])
        self.active.remove(q);self.free_qubits.append(q);self.free_set.add(q)
        self.owners.pop(q,None)
    def recycle_all(self,qs):
        for q in qs:self.recycle(q)
    def _profile(self,force=False):
        if self.current_phase is None:return
        p=self.live_profile.setdefault(self.current_phase,{'peak_active':0,'peak_physical':0})
        p['peak_physical']=max(p['peak_physical'],len(self.physical))
        if len(self.active)>p['peak_active'] or (force and len(self.active)==p['peak_active']):
            p['peak_active']=len(self.active)
            counts={}
            for q in self.active:
                name=self.owners.get(q,'unclaimed')
                counts[name]=counts.get(name,0)+1
            p['owners_at_peak']=counts
    def claim(self,name,qs):
        for q in qs:self.owners[q]=name
        self._profile(force=True)
    def release(self,name,qs):
        for q in qs:self.owners.pop(q,None)
    def release_recycle(self,name,qs):self.recycle_all(reversed(qs))
    def _gate(self,name,*qs,cond=None):
        if cond is not None:self.k.push_condition(cond)
        self.gate_counts[name]=self.gate_counts.get(name,0)+1
        getattr(self.k,name)(*[self.physical[q] for q in qs])
        if cond is not None:self.k.pop_condition()
    def x(self,q,cond=None):self._gate('x',q,cond=cond)
    def z(self,q,cond=None):self._gate('z',q,cond=cond)
    def cx(self,c,t,cond=None):self._gate('cx',c,t,cond=cond)
    def cz(self,a,b,cond=None):self._gate('cz',a,b,cond=cond)
    def ccx(self,a,b,t,cond=None):self._gate('ccx',a,b,t,cond=cond)
    def ccz(self,a,b,c,cond=None):self._gate('ccz',a,b,c,cond=cond)
    def swap(self,a,b,cond=None):self._gate('swap',a,b,cond=cond)
    def cswap(self,c,a,b,cond=None):self._gate('cswap',c,a,b,cond=cond)
    def neg(self,cond=None):
        assert cond is not None
        self.k.neg_if(cond)
    def hmr(self,q):
        self.k.hmr(self.physical[q],self.measurement)
        return self.measurement
    @contextmanager
    def phase(self,name):
        self.k.info_tag('@enter:'+name)
        self.phase_name=name
        prev=self.current_phase;self.current_phase=name
        self._profile()
        before=dict(self.gate_counts)
        t=time.monotonic()
        p={'phase':name,'elapsed':t-self.started,'physical_qubits':len(self.physical),'active_qubits':len(self.active)}
        print(json.dumps(p),flush=True)
        if self.progress:self.progress.write_text(json.dumps(p,indent=2))
        yield
        self.k.info_tag('@exit:'+name)
        self.phase_stats.append({'phase':name,'seconds':time.monotonic()-t,
                                 'gates':{k:v-before.get(k,0) for k,v in self.gate_counts.items()}})
        self.live_profile[name]['exit_active']=len(self.active)
        self.current_phase=prev


def declare_registers(b,decls):
    regs={}
    for index,(_,width) in enumerate(decls):
        assert not b.free_qubits
        start=len(b.physical)
        reg=b.k.create_quantum_register(width,name=f'S{index}')
        b.physical.extend(reg)
        qs=list(range(start,start+width));b.active.update(qs)
        regs[f'S{index}']=qs
    return regs


def assert_metrics(b,_):
    return b.gate_counts.get('ccx',0)+b.gate_counts.get('cswap',0)+b.gate_counts.get('ccz',0)


def emit_and_report(b):
    raise RuntimeError('Use run_gu_port.py; no implicit publication or emission destination.')

In [ ]:
%%writefile src/gen_dqi_v6/gu_formula_derivation.py
# SPDX-License-Identifier: Apache-2.0
# Gu et al., deterministic field formula derivation, adapted imports.
#!/usr/bin/env python3
"""Regenerate constants.json for the dqi-reed-solomon discovered circuit.

main() reruns every search and writes constants.json.  The searches, in
file order:

  Formulas     the multiplication formulas and tower maps (KT12, KT6,
               KT3, CUBE3Q, PAIR3, LIN).  Deterministic; seconds.
  BM windows   the per-round coefficient windows (IMAXV, KMSV, KMMV).
               search.c samples the full Berlekamp-Massey recursion over
               403.2M weight-law draws (56 fixed seeds, 7.2M draws each)
               and counts, per (round, slot), how often the slot changes
               state.  Each window is the smallest one whose clipped
               slots all have mass under 4e-5 per draw.  About 2.7
               CPU-hours; the seeds run on a thread pool (3.5 min on 56
               cores).
  Psi windows  the cofactor windows of the Psi route: per CR op, the
               minimal cyclic interval covering the union support where
               the op acts, over 400k law draws (16 fixed seeds) plus
               the w = 0 anchor and every w = 1 position.  BM runs under
               the freshly derived BM windows.

The models section holds the bit-exact classical models of the circuit
that the Psi window search runs: the fused-Q BM recursion and the Psi
route (stages 5-7).  The script derives the instance from contract.json
and does not import circuit.py, so it runs without constants.json.

Formula derivation.  The field is GF(2^12), poly 4331, x = 2
primitive.  The tower is GF(2^12) = GF(64)(t) = GF(8)(s)(t), read top
down: A = x + x^64, t = x / A, nu = x^65 / A^2, u = x^65,
A' = u + u^8, s = u / A', nu' = u^9 / A'^2, v = u^9.  Tower
coordinates are E12[i + 3j + 6k] = v^i s^j t^k.  The norm descends
both quadratic levels (N = x xbar in GF(64), then M = N Nbar in GF(8)),
so the only nonlinear inverse left is M -> M^3 in GF(8), a 3-product
quadratic form; M^-1 = (M^3)^2 is a free squaring.  A full inversion
costs N (15) + M (6) + M^3 (3) + Nbar M^-1 (12) + xbar N^-1 (30) = 66
Toffoli, and every stage maps 0 to 0.  Rejected towers: GF(64)^2 alone
needs N^31 (five 1-bits, so a generic GF(64) product on top of two
quadratic forms); GF(16)^3 pays a cubic norm; plain Itoh-Tsujii needs
at least four 42-Toffoli products.

  KT12    GF(2^12) = GF(4)[x]: a 6-term product over GF(4) by CRT at
          the four GF(4) points, infinity, and the irreducible
          quadratics x^2+x+w, x^2+x+w^2, x^2+wx+1.  14 GF(4) products
          x 3 = 42; the greedy prune removes nothing.  GF(8)-CRT and
          Chudnovsky-style curves over GF(4) tie at 42; Karatsuba
          variants give 45-51.
  KT6     GF(64) = GF(4)[u], 3 terms, CRT at four points + infinity:
          15 products, the best known bilinear complexity of GF(2^6).
  KT3     the 3-term formula M3TERMS reduced modulo v's minimal
          polynomial: 6 products.
  CUBE3Q  exhaustive search over product subsets: 3 is the minimum.
  PAIR3   elimination rank decomposition of Tr3(r s^2): rank 3.
  LIN     the tower's linear maps (pure linear algebra on the
          elements above).

RUN: uv run python gen_constants.py   (writes constants.json)
"""

import collections
import ctypes
import json
import os
import pathlib
import random
import sys
import time
from multiprocessing import Pool
from multiprocessing.pool import ThreadPool

import numpy as np

from port_runtime import GF2n, f2_matinv, f2_transpose, parity
from port_runtime import load_contract

HERE = pathlib.Path(__file__).resolve().parent
PATH = HERE / "constants.json"

# The instance (contract.json), derived as circuit.py derives it.
CONTRACT = load_contract(__file__)
M = CONTRACT["m"]  # code length = nonzero field points
N = CONTRACT["n_synd"]  # syndrome symbols = BM rounds
B = CONTRACT["b"]  # bits per symbol
W = CONTRACT["w_max"]  # unique-decoding radius n/2 = max locator degree
LAW_CUM = CONTRACT["weight_cum_2p16"]  # the grader's 2^16 cumulative weight law
NH = W // 2 + 1  # words per Euclid half E', O' (and per cofactor)
FIELD = GF2n(B, CONTRACT["gf_poly"])
EXP, LOG = FIELD.exp, FIELD.log
gmul, ginv, gsqrt, trace = FIELD.mul, FIELD.inv, FIELD.sqrt, FIELD.trace
DWB = (N - 2).bit_length() + 1  # the signed d counter dW: two's complement
DMAX = 1 << (DWB - 1)  # two's-complement range of dW


def bitlist(m, n):
    return [i for i in range(n) if (m >> i) & 1]


def bitmask(n, pred):
    return sum(1 << i for i in range(n) if pred(i))


DWMASK = (1 << DWB) - 1  # the dW register (two's complement, DWB bits)
MMASK = 31  # m = (dW + 2) >> 2 lives in 5 bits (dW[2:7])
# The cores this process may use (its CPU affinity set, not the whole node).
NCPU = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()

# #########################################################################
# Formulas
# #########################################################################
# Every table is deterministic; the module docstring gives the derivation.
# search_kt keeps the CRT formula only if certify_kt accepts it.

# ---------------------------------------------------------------------------
# small F2 helpers
# ---------------------------------------------------------------------------


def rows_apply(rows, x):
    """bit r of the result = parity(rows[r] & x)."""
    v = 0
    for r in range(len(rows)):
        if parity(rows[r] & x):
            v |= 1 << r
    return v


def compose_rows(A, B, nb):
    """Rows of A o B (apply B, then A); B has nb rows."""
    out = []
    for r in range(len(A)):
        m = 0
        for i in bitlist(A[r], nb):
            m ^= B[i]
        out.append(m)
    return out


# ---------------------------------------------------------------------------
# GF(2^b) arithmetic
# ---------------------------------------------------------------------------


def reduce_raw(raw):
    """Reduce a raw F2[x] product (degree <= 2B - 2) mod the field
    polynomial: x^k = EXP[k], since gamma = x.
    """
    v = 0
    for k in bitlist(raw, 2 * B - 1):
        v ^= EXP[k]
    return v


# ---------------------------------------------------------------------------
# polynomial-multiplication formulas (Karatsuba fallback, GF(8) terms)
# ---------------------------------------------------------------------------

M3TERMS = [
    (1, 1, 0b00111),
    (2, 2, 0b01110),
    (4, 4, 0b11100),
    (3, 3, 0b00010),
    (6, 6, 0b01000),
    (5, 5, 0b00100),
]


def kara(n):
    """Recursive symmetric Karatsuba formula for n-term F2 polynomial
    multiplication, [(Amask, Bmask, raw-product mask)].
    """
    if n == 1:
        return [(1, 1, 1)]
    if n == 3:
        return list(M3TERMS)
    n0 = n // 2
    n1 = n - n0
    m = max(n0, n1)
    out = []
    for A, B, C in kara(n0):
        out.append((A, B, C ^ (C << n0)))
    for A, B, C in kara(n1):
        out.append((A << n0, B << n0, (C << n0) ^ (C << (2 * n0))))
    for A, B, C in kara(m):
        Am = 0
        Bm = 0
        for i in range(m):
            if (A >> i) & 1:
                if i < n0:
                    Am ^= 1 << i
                if i < n1:
                    Am ^= 1 << (n0 + i)
            if (B >> i) & 1:
                if i < n0:
                    Bm ^= 1 << i
                if i < n1:
                    Bm ^= 1 << (n0 + i)
        out.append((Am, Bm, C << n0))
    return out


# ---------------------------------------------------------------------------
# GF(4)-tower CRT bilinear formulas (deterministic; KT / KT12 / KT6)
# ---------------------------------------------------------------------------


class F4:
    """GF(4) = {0, 1, w, w+1} inside fld, index-encoded (bit 0 = 1, bit 1 =
    w), so index addition is XOR.
    """

    def __init__(self, fld):
        gmul = fld.mul
        w3 = fld.exp[fld.order // 3]
        assert gmul(w3, w3) == (w3 ^ 1) and gmul(w3, w3 ^ 1) == 1
        self.w3 = w3
        self.E = [0, 1, w3, w3 ^ 1]
        self.I = {self.E[i]: i for i in range(4)}
        self.gmul = gmul

    def mul(self, a, bq):
        return self.I[self.gmul(self.E[a], self.E[bq])]


def f4_irreducible_quadratics(f4):
    """Monic irreducible x^2 + a x + b over GF(4), (a, b) in index order."""
    out = []
    for a in range(4):
        for bq in range(4):
            if all((f4.mul(r, r) ^ f4.mul(a, r) ^ bq) != 0 for r in range(4)):
                out.append((a, bq))
    assert len(out) == 6
    return out


def f4_lams(f4, k, nquad):
    """Return the CRT evaluation functionals for a k-term product over GF(4):
    evaluation at 0, 1, w, w^2, at infinity, then for each of the first
    nquad irreducible quadratics q the two reduction coordinates
    (x^i mod q = c0_i + c1_i x) and their sum (Karatsuba over GF(4)).
    Modulus degree = 4 + 1 + 2 nquad; the product has 2k - 1 coefficients.
    """
    assert 4 + 1 + 2 * nquad == 2 * k - 1, (k, nquad)
    lams = []
    for pt in range(4):
        lam, p = [], 1
        for _ in range(k):
            lam.append(p)
            p = f4.mul(p, pt)
        lams.append(lam)
    lams.append([0] * (k - 1) + [1])
    for a, bq in f4_irreducible_quadratics(f4)[:nquad]:
        L0, L1 = [], []
        c0, c1 = 1, 0
        for _ in range(k):
            L0.append(c0)
            L1.append(c1)
            c0, c1 = f4.mul(c1, bq), c0 ^ f4.mul(c1, a)
        lams.extend([L0, L1, [p ^ q for p, q in zip(L0, L1)]])
    return lams


def f4_digits(f4, fld, coords, gens, k):
    """GF(4)-digit extraction for an n = 2k dimensional coordinate system:
    gens[i] (i < k) are field elements whose GF(4)-span {gens[i], gens[i]
    w} is the space; coords(elt) -> n-bit coordinate vector.  Returns
    DIG[bq] = the k GF(4)-digit indices of coordinate basis vector bq.
    """
    n = 2 * k
    colsE = []
    for i in range(k):
        for j in range(2):
            e = gens[i]
            if j:
                e = fld.mul(e, f4.w3)
            colsE.append(coords(e))
    EXTR = f2_matinv(f2_transpose(colsE, n), n)
    DIG = []
    for bq in range(n):
        c = rows_apply(EXTR, 1 << bq)
        DIG.append([(c >> (2 * i)) & 3 for i in range(k)])
    return DIG


def solve_sym(n, prodlist, mulcoord):
    """Reconstruction solve for symmetric operand masks: target masks T[t]
    per product such that a*b = XOR_t parity(a&m_t) parity(b&m_t) T[t] on all
    n x n basis pairs (verified), or None.
    """
    piv = {}
    for i in range(n):
        for j in range(n):
            a = 0
            for t, mk in enumerate(prodlist):
                if ((mk >> i) & 1) and ((mk >> j) & 1):
                    a |= 1 << t
            r = mulcoord(i, j)
            while a:
                cc = a.bit_length() - 1
                if cc in piv:
                    pa, pr = piv[cc]
                    a ^= pa
                    r ^= pr
                else:
                    piv[cc] = (a, r)
                    a = 0
                    r = 0
            if r:
                return None
    T = [0] * len(prodlist)
    for cc in sorted(piv):
        a, r = piv[cc]
        acc = r
        aa = a & ~(1 << cc)
        while aa:
            c2 = aa.bit_length() - 1
            aa ^= 1 << c2
            acc ^= T[c2]
        T[cc] = acc
    for i in range(n):
        for j in range(n):
            acc = 0
            for t, mk in enumerate(prodlist):
                if ((mk >> i) & 1) and ((mk >> j) & 1):
                    acc ^= T[t]
            if acc != mulcoord(i, j):
                return None
    return T


def crt_f4_formula(n, DIG, k, lams, f4mul, mulcoord):
    """GF(4)-CRT bilinear formula on n = 2k coordinate bits: products =
    3 F2 parities per lambda, reconstruction solved by
    Gaussian elimination over all n x n basis pairs, deterministic greedy
    prune in index order.  Returns [(prodmask, targetmask)] (symmetric
    operand masks) or None.
    """
    prods = []
    for lam in lams:
        m0 = 0
        m1 = 0
        for bq in range(n):
            U = 0
            for i in range(k):
                U ^= f4mul(lam[i], DIG[bq][i])
            if U & 1:
                m0 |= 1 << bq
            if U & 2:
                m1 |= 1 << bq
        prods.extend([m0, m1, m0 ^ m1])
    assert len(prods) == 3 * len(lams)

    T = solve_sym(n, prods, mulcoord)
    if T is None:
        return None
    keep = list(range(len(prods)))
    for t in range(len(prods)):
        if t not in keep:
            continue
        trial = [prods[u] for u in keep if u != t]
        if solve_sym(n, trial, mulcoord) is not None:
            keep = [u for u in keep if u != t]
    plist = [prods[u] for u in keep]
    T = solve_sym(n, plist, mulcoord)
    if T is None:
        return None
    out = []
    for t, mk in enumerate(plist):
        if T[t] and mk:
            out.append((mk, T[t]))
    return out


def build_f4_crt_formula(fld):
    """GF(2^b) multiplication over the poly basis via GF(2^b) = GF(4)[x]:
    b/2 terms over GF(4), moduli = 4 points + infinity + (b/2 - 3)
    irreducible quadratics (b = 12: 42 products).
    """
    b = fld.n
    k = b // 2
    try:
        f4 = F4(fld)
        theta = 2
        gens = [fld.pow(theta, i) for i in range(k)]
        DIG = f4_digits(f4, fld, lambda e: e, gens, k)
        for v in range(1 << b):  # exhaustive digit round-trip
            dg = [0] * k
            for bq in bitlist(v, b):
                dg = [x ^ y for x, y in zip(dg, DIG[bq])]
            w = 0
            for i in range(k):
                w ^= fld.mul(gens[i], f4.E[dg[i]])
            assert w == v
        lams = f4_lams(f4, k, k - 3)
        return crt_f4_formula(b, DIG, k, lams, f4.mul, lambda i, j: fld.mul(1 << i, 1 << j))
    except Exception as ex:  # noqa: BLE001  (best-effort alternative: any failure falls back to kara(b))
        print("f4-crt formula construction failed:", repr(ex))
        return None


def search_kt(fld):
    """Return the GF(2^b) multiplication formula, constants.json shape
    [[Ab, Bb, Tb], ...].  Deterministic: the Karatsuba formula kara(b)
    (b = 12: 54 products), replaced by the GF(4)-tower CRT formula when
    that is smaller.
    """
    b = fld.n
    RAW = kara(b)
    # basis-exhaustive check of RAW (complete for a bilinear form)
    for i in range(b):
        for j in range(b):
            pr = 0
            for Am, Bm, Cm in RAW:
                if parity((1 << i) & Am) and parity((1 << j) & Bm):
                    pr ^= Cm
            assert reduce_raw(pr) == fld.mul(1 << i, 1 << j)
    KT = []
    for Am, Bm, Cm in RAW:
        tm = reduce_raw(Cm)
        if tm:
            KT.append([bitlist(Am, b), bitlist(Bm, b), bitlist(tm, b)])
    F4KT = build_f4_crt_formula(fld)
    if F4KT is not None:
        KT_NEW = [[bitlist(mk, b), bitlist(mk, b), bitlist(tm, b)] for (mk, tm) in F4KT]
        if certify_kt(KT_NEW, fld) and len(KT_NEW) < len(KT):
            KT = KT_NEW
    return KT


# ---------------------------------------------------------------------------
# The towers
# ---------------------------------------------------------------------------


class Tower:
    """The (2, 2, 3) tower of the module docstring.  GF(2^12) = GF(64)(t),
    GF(64) = GF(8)(s), GF(8) = F2[v]; E_list = [v^i s^j t^k] indexed
    i + 3j + 6k; the GF(64) coordinates are E6 = [v^i s^j] (i + 3j), the
    GF(8) coordinates E3 = [v^i].
    """

    def __init__(self, fld):
        self.fld = fld
        b = fld.n
        assert b == 12
        h = b // 2
        EXP, gmul, ginv, gpow = fld.exp, fld.mul, fld.inv, fld.pow
        Ael = EXP[1 << h] ^ 2  # x^(2^h) + x = Tr(x)
        t_el = gmul(2, ginv(Ael))  # t = x / A
        u_el = EXP[(1 << h) + 1]  # u = x^(2^h + 1) = N(x)
        nu_el = gmul(u_el, ginv(gmul(Ael, Ael)))  # nu = u / A^2
        assert gpow(u_el, 1 << h) == u_el and u_el != 0
        assert gpow(Ael, 1 << h) == Ael and Ael != 0
        assert gpow(nu_el, 1 << h) == nu_el
        assert gmul(t_el, t_el) ^ t_el == nu_el
        self.Ael, self.t_el, self.u_el, self.nu_el = Ael, t_el, u_el, nu_el
        self.h = h
        # sub-level GF(64) = GF(8)(s)
        Ap = gpow(u_el, 8) ^ u_el  # Tr_{6/3}(u)
        s_el = gmul(u_el, ginv(Ap))
        v_el = gpow(u_el, 9)  # N_{6/3}(u)
        nup_el = gmul(v_el, ginv(gmul(Ap, Ap)))
        assert gpow(Ap, 8) == Ap and Ap != 0
        assert gpow(v_el, 8) == v_el and gpow(v_el, 7) == 1 and v_el != 1
        assert gpow(nup_el, 8) == nup_el
        assert gmul(s_el, s_el) ^ s_el == nup_el
        self.Ap, self.s_el, self.v_el, self.nup_el = (Ap, s_el, v_el, nup_el)
        E3 = [gpow(v_el, i) for i in range(3)]
        E6 = E3 + [gmul(e, s_el) for e in E3]
        E_list = E6 + [gmul(e, t_el) for e in E6]
        self.E_sub = E6
        self.E3 = E3
        self.E_list = E_list
        self.MINVR = f2_matinv(f2_transpose(E_list, b), b)
        for v in range(1 << b):
            c = self.coords_full(v)
            w = 0
            for u in bitlist(c, b):
                w ^= E_list[u]
            assert w == v
        # projections poly -> (a0, a1) and the conjugate's constant part
        self.A0ROWS = self.MINVR[0:h]
        self.A1ROWS = self.MINVR[h:b]
        self.A01ROWS = [self.A0ROWS[r] ^ self.A1ROWS[r] for r in range(h)]
        # squaring and multiply-by-nu on subfield coords
        self.SQR = f2_transpose([self.coords_sub(gmul(e, e)) for e in self.E_sub], h)
        self.NUR = f2_transpose([self.coords_sub(gmul(nu_el, e)) for e in self.E_sub], h)
        NSR = compose_rows(self.NUR, self.SQR, h)
        # linear part of the norm N = a0 a1 + a0^2 + nu a1^2 on poly words
        self.LNROWS = []
        for r in range(h):
            m = 0
            for j in bitlist(self.SQR[r], h):
                m ^= self.A0ROWS[j]
            for j in bitlist(NSR[r], h):
                m ^= self.A1ROWS[j]
            self.LNROWS.append(m)
        # GF(8) level: coords3 = low 3 bits of the GF(64) coords of a
        # GF(8) element; RED3 = v^k for the KT3 reduction
        self.RED3 = [self.coords3(gpow(v_el, k)) for k in range(5)]
        self.SQ3R = f2_transpose([self.coords3(gmul(e, e)) for e in self.E3], 3)
        self.NU3R = f2_transpose([self.coords3(gmul(nup_el, e)) for e in self.E3], 3)
        NSR3 = compose_rows(self.NU3R, self.SQ3R, 3)
        # linear part of M = n0 n1 + n0^2 + nu' n1^2 on GF(64) coords
        self.LMROWS = [self.SQ3R[r] ^ (NSR3[r] << 3) for r in range(3)]
        self.FCUBE3 = [self.coords3(gpow(self.elt3(c), 3)) for c in range(8)]

    # -- coordinates --
    def coords_full(self, v):
        return rows_apply(self.MINVR, v)

    def coords_sub(self, v):
        c = self.coords_full(v)
        assert c < (1 << self.h), "not a subfield element"
        return c

    def coords3(self, v):
        c = self.coords_full(v)
        assert c < 8, "not a GF(8) element"
        return c

    def elt3(self, c):
        w = 0
        for i in bitlist(c, 3):
            w ^= self.E3[i]
        return w

    def lin_tables(self):
        """Return the LIN block of constants.json: the linear maps circuit.py
        reads.
        """
        return {
            "E12": list(self.E_list),
            "A0ROWS": list(self.A0ROWS),
            "A1ROWS": list(self.A1ROWS),
            "LNROWS": list(self.LNROWS),
            "SQ3R": list(self.SQ3R),
            "LMROWS": list(self.LMROWS),
        }


# ---------------------------------------------------------------------------
# subfield multiplication formulas
# ---------------------------------------------------------------------------


def search_kt6(fld, tower):
    """Return the GF(64) multiplication formula in tower coords (E6), 15
    products: GF(64) = GF(4)[u], 3 terms over GF(4), CRT at the four points
    + infinity, built directly on the 6-bit coordinate vectors.
    """
    f4 = F4(fld)
    u = tower.u_el
    gens = [fld.pow(u, i) for i in range(3)]
    DIG = f4_digits(f4, fld, tower.coords_sub, gens, 3)
    lams = f4_lams(f4, 3, 0)
    assert len(lams) == 5
    E6 = tower.E_sub
    F = crt_f4_formula(
        6, DIG, 3, lams, f4.mul, lambda i, j: tower.coords_sub(fld.mul(E6[i], E6[j]))
    )
    assert F is not None, "GF(64) CRT formula failed"
    return [[bitlist(mk, 6), bitlist(mk, 6), bitlist(tm, 6)] for (mk, tm) in F]


def search_kt3(tower):
    """Return the GF(8) multiplication formula in the v-basis, 6 products:
    M3TERMS reduced modulo the minimal polynomial of v.
    """
    KT3 = []
    for Am, Bm, Cm in M3TERMS:
        tm = 0
        for kk in range(5):
            if (Cm >> kk) & 1:
                tm ^= tower.RED3[kk]
        if tm:
            KT3.append([bitlist(Am, 3), bitlist(Bm, 3), bitlist(tm, 3)])
    return KT3


# ---------------------------------------------------------------------------
# quadratic-form formulas: exact solve and exhaustive search (CUBE3Q)
# ---------------------------------------------------------------------------


def qp_pairs(n):
    return [(i, j) for i in range(n) for j in range(i + 1, n)]


def _offvec(n, u, w):
    v = 0
    for idx, (i, j) in enumerate(qp_pairs(n)):
        if (((u >> i) & (w >> j)) ^ ((u >> j) & (w >> i))) & 1:
            v |= 1 << idx
    return v


def quad_solve(n, prods, fvals):
    """Given product pairs (u, w),
    solve for output masks + linear correction so the quadratic formula
    reproduces fvals on all 2^n elements; returns (tris, corr) or None.
    """
    pairs = qp_pairs(n)
    bas = {}
    for t_, (u, w) in enumerate(prods):
        x = _offvec(n, u, w)
        sel = 1 << t_
        while x:
            mb = x.bit_length() - 1
            if mb in bas:
                x ^= bas[mb][0]
                sel ^= bas[mb][1]
            else:
                bas[mb] = (x, sel)
                break
    tg = [0] * n
    for idx, (i, j) in enumerate(pairs):
        cdiff = fvals[(1 << i) | (1 << j)] ^ fvals[1 << i] ^ fvals[1 << j]
        for m in range(n):
            if (cdiff >> m) & 1:
                tg[m] |= 1 << idx
    outs = []
    for m in range(n):
        x = tg[m]
        sel = 0
        while x:
            mb = x.bit_length() - 1
            if mb not in bas:
                return None
            x ^= bas[mb][0]
            sel ^= bas[mb][1]
        outs.append(sel)
    tris = []
    for t_, (u, w) in enumerate(prods):
        om = 0
        for m in range(n):
            if (outs[m] >> t_) & 1:
                om |= 1 << m
        if om:
            tris.append((u, w, om))
    corr = [0] * n
    for i in range(n):
        cv = fvals[1 << i]
        for u, w, om in tris:
            if (u >> i) & 1 and (w >> i) & 1:
                cv ^= om
        for m in range(n):
            if (cv >> m) & 1:
                corr[m] |= 1 << i
    for x in range(1 << n):
        if quad_apply((tris, corr), x, n) != fvals[x]:
            return None
    return (tris, corr)


def quad_exhaustive(n, fvals):
    """EXHAUSTIVE minimum-product quadratic formula for fvals on n bits:
    candidate products are the unordered pairs u < w of nonzero masks (a
    pair (u, u) is linear and never needed); subsets are tried in
    increasing size, lexicographic order, and the first solvable one wins.
    Feasible for n <= 4; used for GF(8).
    """
    from itertools import combinations

    cands = [(u, w) for u in range(1, 1 << n) for w in range(u + 1, 1 << n)]
    for r in range(len(qp_pairs(n)) + 1):
        for sub in combinations(cands, r):
            res = quad_solve(n, list(sub), fvals)
            if res is not None:
                tris, corr = res
                return [[list(t) for t in tris], list(corr)]
    raise AssertionError("no quadratic formula (impossible)")


def search_cube3q(tower):
    """Exhaustive, deterministic: x -> x^3 in GF(8), v-basis."""
    return quad_exhaustive(3, tower.FCUBE3)


# ---------------------------------------------------------------------------
# pairing-form rank decomposition (elimination)
# ---------------------------------------------------------------------------


def pair_decomp(Brows, n):
    """B[u] = mask over v of the form on basis pair (u, v).  Returns
    [(p, qm), ...] with B(a, c) = XOR_k parity(a & p_k) & parity(c & qm_k).
    """
    Rw = list(Brows)
    out = []
    while True:
        u0 = None
        for u in range(n):
            if Rw[u]:
                u0 = u
                break
        if u0 is None:
            break
        qm = Rw[u0]
        v0 = (qm & -qm).bit_length() - 1
        p = bitmask(n, lambda u: (Rw[u] >> v0) & 1)
        for u in bitlist(p, n):
            Rw[u] ^= qm
        out.append((p, qm))
    chk = [0] * n
    for p, qm in out:
        for u in bitlist(p, n):
            chk[u] ^= qm
    assert chk == list(Brows)
    return out


def search_pair3(fld, tower):
    """Rank decomposition of B(r, s) = Tr3(r s^2) on GF(8) v-coords."""
    gmul = fld.mul
    E3 = tower.E3
    B33 = [bitmask(3, lambda j: trace(gmul(E3[i], gmul(E3[j], E3[j])), 3)) for i in range(3)]
    return [[p, qm] for (p, qm) in pair_decomp(B33, 3)]


# ---------------------------------------------------------------------------
# formula evaluation (search_kt's certify_kt, quad_solve's quad_apply)
# ---------------------------------------------------------------------------


def kt_mul_fn(KT):
    ktm = [
        (sum(1 << u for u in Ab), sum(1 << u for u in Bb), sum(1 << u for u in Tb))
        for (Ab, Bb, Tb) in KT
    ]

    def mul(a, bb):
        acc = 0
        for Am, Bm, Tm in ktm:
            if parity(a & Am) and parity(bb & Bm):
                acc ^= Tm
        return acc

    return mul


def certify_kt(KT, fld):
    """Return True iff KT multiplies exactly in GF(2^b) over the poly basis:
    checked on the b x b basis pairs, complete for a bilinear form.
    """
    kt_mul = kt_mul_fn(KT)
    return all(
        kt_mul(1 << i, 1 << j) == fld.mul(1 << i, 1 << j)
        for i in range(fld.n)
        for j in range(fld.n)
    )


def quad_apply(Q, x, n):
    tris, corr = Q
    acc = 0
    for u, w, om in tris:
        if parity(u & x) and parity(w & x):
            acc ^= om
    for m in range(n):
        if parity(corr[m] & x):
            acc ^= 1 << m
    return acc


def run_b12():
    """All formula searches (every one deterministic); returns (tables,
    timings) with tables in constants.json shape.
    """
    fld = FIELD
    tower = Tower(fld)
    tables, tm = {}, {}
    for name, search in (
        ("KT12", lambda: search_kt(fld)),
        ("KT6", lambda: search_kt6(fld, tower)),
        ("KT3", lambda: search_kt3(tower)),
        ("CUBE3Q", lambda: search_cube3q(tower)),
        ("PAIR3", lambda: search_pair3(fld, tower)),
    ):
        t0 = time.time()
        tables[name] = search()
        tm[name + "_s"] = time.time() - t0
    tables["LIN"] = tower.lin_tables()
    return tables, tm


# #########################################################################
# Models
# #########################################################################
# Bit-exact classical models of the circuit (Builder-free): the one Python
# model of its fused-Q BM recursion, and the Psi route (stages 5-7) op for
# op.  The Psi window search runs them.

# =========================================================================
# BM recursion (q-form conditional-swap, fused-Q B bank)
# =========================================================================
# The B bank rotates at bank-1.  A per-round hook instruments the loop.

In [ ]:
%%writefile src/gen_dqi_v6/gu_port.py
# SPDX-License-Identifier: Apache-2.0
# Adapted from andigu/uncompute by Gu et al.; see REPORT.md and ref/gu_README.md.
#!/usr/bin/env python3
"""dqi-reed-solomon discovered circuit: fused-Q truncated Berlekamp-Massey
decode at (m, n, b) = (4095, 70, 12), then a half-degree Bezout pass that
turns the per-point Forney division into a precomputed Psi = 1/sigma'
mod sigma, so the Chien/Forney sweep costs 23 Toffoli per point (root
test 11 + a rank-12 gated pairing 12) instead of 80.  Built natively on
uncompute.Builder.

The BM body runs the (2,2,3)-tower field arithmetic under the measured
window schedule and stores no accepted discrepancy: the scale identity
sigma_0 * DEL == 1 replaces it (README).
The Psi route (stages 5-7) holds the Bezout pair in the syndrome TAIL,
zeroed by extending the in-place omega transform to all 70 rows.

Layout (flat):
  circuit.py         the tables (field, formulas, schedule; loaded from
                    constants.json), the circuit layer (term lists,
                    emitter, Psi-route gadgets, sweep body), and build(),
                    the stage sequence with its phase marks and cost pins.
  constants.json    every searched value: the formula tables, the BM
                    windows (bm_windows), the Psi windows (psi_windows).
  gen_constants.py  reruns every search and writes constants.json; it
                    holds the bit-exact classical models the searches run.
  search.c          the C sampler behind the BM window census.

RUN: uv run python circuit.py -- builds, asserts its cost, and
streams to $UNCOMPUTE_OPS_PATH (else ./ops.bin).  Importing is cheap and
builds nothing; build() feeds the occupancy/liveness profilers.
Evaluate: python tools/eval.py dqi-reed-solomon.

README.md in this folder describes the construction.
"""

import functools
import json
import pathlib
from contextlib import contextmanager

import numpy as np

from port_runtime import (Builder, Journal, GF2n, f2_matinv, f2_transpose, parity,
    declare_registers, linear_in_place, assert_metrics, emit_and_report, load_contract)
import os

HERE = pathlib.Path(__file__).resolve().parent


# =========================================================================
# Field and instance
# =========================================================================

CONTRACT = load_contract(__file__)
POLY = CONTRACT["gf_poly"]
M = CONTRACT["m"]  # code length = nonzero field points
N = CONTRACT["n_synd"]  # syndrome symbols = BM rounds
B = CONTRACT["b"]  # bits per symbol
W = CONTRACT["w_max"]  # unique-decoding radius n/2 = max locator degree
LAW_CUM = CONTRACT["weight_cum_2p16"]  # the grader's 2^16 cumulative weight law
assert M == (1 << B) - 1 and N == 2 * W
NH = W // 2 + 1  # words per Euclid half E', O' (and per cofactor)
SQRT_EXP = (M + 1) // 2  # sqrt(x) = x^SQRT_EXP (2048)

# EXP/LOG tables (gamma = 2): EXP[k] = gamma^k for k < 2M, LOG its inverse.
# ginv is totalized (zero maps to zero), as the tower inversion chain.
FIELD = GF2n(B, POLY)
assert FIELD.gen == 2, "gamma = 2 not primitive?!"
EXP, LOG = FIELD.exp, FIELD.log
gmul, ginv, gpow, gsqrt = FIELD.mul, FIELD.inv, FIELD.pow, FIELD.sqrt
# trace(x) is the absolute trace; trace(x, 3) the GF(8) trace of a GF(8) element.
trace = FIELD.trace


def bitlist(m, n):
    return [i for i in range(n) if (m >> i) & 1]


def bitmask(n, pred):
    return sum(1 << i for i in range(n) if pred(i))


def gf_matinv(Mx):
    """Inverse of a square matrix over GF(2^B) (Gauss-Jordan)."""
    n = len(Mx)
    A = [row[:] + [1 if i == j else 0 for j in range(n)] for i, row in enumerate(Mx)]
    for col in range(n):
        piv = next(r for r in range(col, n) if A[r][col])
        A[col], A[piv] = A[piv], A[col]
        iv = ginv(A[col][col])
        A[col] = [gmul(iv, v) for v in A[col]]
        for r in range(n):
            if r != col and A[r][col]:
                f = A[r][col]
                A[r] = [A[r][c] ^ gmul(f, A[col][c]) for c in range(2 * n)]
    return [row[n:] for row in A]


# =========================================================================
# A. Fixup vector c (contract.json; task README section 3.2:
# counter-mode SHA-256 XOF over c_tag, grader-asserted)
# =========================================================================
_buf = bytes.fromhex(CONTRACT["c_stream_hex"])
CBITS = [(_buf[t >> 3] >> (t & 7)) & 1 for t in range(M * B)]
CMASKJ = [bitmask(B, lambda bp: CBITS[B * j + bp]) for j in range(M)]

# =========================================================================
# B. Frozen multiplication formulas + tower row masks (constants.json
# beside circuit.py, written by gen_constants.py).
# =========================================================================
CONSTS = json.loads(pathlib.Path(os.environ["DQI_CONSTANTS"]).read_bytes())
KT = [tuple(map(tuple, t)) for t in CONSTS["KT12"]]  # GF(2^12) mult: 42 products
KT6 = [tuple(map(tuple, t)) for t in CONSTS["KT6"]]  # GF(64) mult: 15 products
KT3 = [tuple(map(tuple, t)) for t in CONSTS["KT3"]]  # GF(8) mult: 6 products
# x^3 in GF(8): 3 products
CUBE3Q = tuple(map(tuple, CONSTS["CUBE3Q"][0])), tuple(CONSTS["CUBE3Q"][1])
PAIR3 = [tuple(pq) for pq in CONSTS["PAIR3"]]  # Tr3(r s^2) pairing: rank 3
LIN = CONSTS["LIN"]  # the tower's linear maps

# =========================================================================
# C. Omega schedule: in-place triangular product from middle-product
# blocks, each at its field-rank a+b-1 by evaluation-interpolation.
# Constant GF(2^B) maps are free CX networks, so evaluating both
# operands at a+b-1 fixed points and interpolating the results back
# costs only the pointwise quantum-quantum products; the recursion
# below (balanced binary) is the DP optimum of the alias-free schedule
# family: 147 products against the dense triangle's W(W+1)/2 = 630.
# Blocks read syn columns strictly below the rows they write, so the
# hi -> block -> lo emission order stays in-place.
# =========================================================================
_OMC = {1: 0}
_OMS = {1: None}
for _n in range(2, W + 1):
    _best = None
    _bh = None
    for _h in range(1, _n // 2 + 1):
        _v = _OMC[_h] + _OMC[_n - _h]
        if _best is None or _v < _best:
            _best, _bh = _v, _h
    assert _best is not None  # range(1, _n // 2 + 1) is non-empty
    _OMC[_n] = (_n - 1) + _best
    _OMS[_n] = _bh

MACS = []  # (r0, a, c0, b): reads syn cols [c0, c0+b), writes [r0, r0+a)


def _build_tri(lo, n):
    if n == 1:
        return
    h = _OMS[n]
    _build_tri(lo + h, n - h)
    MACS.append((lo + h, n - h, lo, h))
    _build_tri(lo, h)


# omega70: the same in-place transform extended to all N = 2W rows.  Rows
# W..N-1 of sigma_m*S mod z^N vanish on every correct decode (the key
# equation: deg omega < deg sigma <= W), which frees the 35-word syndrome
# TAIL as workspace for the Bezout pair (stage 5).  Emission
# order hi -> lo keeps it in place: the tail triangle (rows W..N-1 from
# rows W..N-2, diagonals 1..W-1), then one off-diagonal middle-product
# block (rows W..N-1 from rows 0..W-1; its Toeplitz diagonals W+1..N-1
# are zero -- sigma has W+1 coefficients -- and are clipped from the
# sigma-side evaluation), then the omega triangle.  The map is unit
# triangular on all N rows, so its literal reverse restores S on EVERY
# path, including approximate-BM paths where the tail is not zero.
_build_tri(W, W)
MACS.append((W, W, 0, W))
_build_tri(0, W)
assert len(MACS) == 2 * (W - 1) + 1
for _r0, _a, _c0, _b in MACS:
    # alias-free in place: every block reads rows strictly below the rows
    # it writes; the triangles span diagonals 1..W-1, the off-diagonal
    # block 1..N-1 (clipped to 1..W at emission)
    assert _c0 + _b == _r0, (_r0, _a, _c0, _b)
    assert _r0 + _a - 1 - _c0 <= (N - 1 if _b == W else W - 1), (_r0, _a, _c0, _b)

_VINV = {}


def vinv(npts):
    """(points, inverse Vandermonde) for evaluation at gamma^0..gamma^(npts-1),
    cached per size.
    """
    if npts not in _VINV:
        pts = [EXP[t] for t in range(npts)]
        _VINV[npts] = (
            pts,
            gf_matinv([[gpow(pts[t], k) for k in range(npts)] for t in range(npts)]),
        )
    return _VINV[npts]


@functools.cache
def cmrows(c):
    """Row masks of the constant multiplication x -> c*x (a free CX network)."""
    return f2_transpose([gmul(c, 1 << j) for j in range(B)], B)


# =========================================================================
# D. (2,2,3) tower: GF(2^12) = GF(64)(t) = GF(8)(s)(t), read top-down
# (gen_constants.py docstring).  A 12-bit tower word is a0 (low 6) + a1 (high 6) t with
# t^2 + t = nu, conjugate xbar = (a0 + a1) + a1 t; a GF(64) word is
# n0 (low 3) + n1 (high 3) s, conjugate Nbar = (n0 + n1) + n1 s.  The
# norm descends both quadratic levels (N = x xbar in GF(64) = a0 a1 +
# linear, M = N Nbar in GF(8) = n0 n1 + linear); the only nonlinear
# inverse left is M -> M^3 (a 3-product quadratic form), M^-1 = (M^3)^2
# a free squaring.  Coordinates: E12[i + 3j + 6k] = v^i s^j t^k; the
# GF(8) projection of a GF(64) word is a bit slice (no row parities).
# Every stage maps 0 to 0, so the zflag armor (monicize) carries over.
# local: gadgets.gf_tower.GF asserts q0 in {4, 8}; this tower is
# GF(8)(s)(t) and its in-circuit chain runs on A0/A1 row projections of
# 12-bit poly-basis words, never on tower-packed words.
# =========================================================================
E12 = LIN["E12"]  # the 12 tower basis elements (poly ints)
A0ROWS = LIN["A0ROWS"]  # poly(12) -> a0 (GF(64) coords)
A1ROWS = LIN["A1ROWS"]  # poly(12) -> a1
A01ROWS = [A0ROWS[r] ^ A1ROWS[r] for r in range(6)]  # a0 + a1 (xbar's constant part)
LNROWS = LIN["LNROWS"]  # poly(12) -> the linear part of N: a0^2 + nu a1^2
LMROWS = LIN["LMROWS"]  # GF(64) coords -> the linear part of M: n0^2 + nu' n1^2
SQ3R = LIN["SQ3R"]  # squaring on GF(8) coords
assert (len(E12), len(A0ROWS), len(A1ROWS), len(LNROWS), len(LMROWS), len(SQ3R)) == (
    12,
    6,
    6,
    6,
    3,
    3,
)
# the tower elements re-derived from the field (the recipe in the
# gen_constants.py docstring); E12 must be their products
_A_el = 2 ^ gpow(2, 64)
_t_el = gmul(2, ginv(_A_el))
_u_el = gpow(2, 65)
_Ap_el = _u_el ^ gpow(_u_el, 8)
_s_el = gmul(_u_el, ginv(_Ap_el))
_v_el = gpow(_u_el, 9)
assert E12 == [
    gmul(gmul(gpow(_v_el, i), gpow(_s_el, j)), gpow(_t_el, k))
    for k in range(2)
    for j in range(2)
    for i in range(3)
], "E12 != v^i s^j t^k"
MINVR = f2_matinv(f2_transpose(E12, B), B)
assert MINVR[0:6] == A0ROWS and MINVR[6:12] == A1ROWS, "A0/A1 rows != inverse basis matrix"

# =========================================================================
# E. Trace dual, free-Z interpolation split, GF(8) fixed pairing form
# =========================================================================
TRT = [trace(x) for x in range(1 << B)]  # absolute trace table
TINV = f2_matinv([bitmask(B, lambda c: TRT[gmul(1 << a, 1 << c)]) for a in range(B)], B)
MU = f2_transpose(TINV, B)
for _a in range(B):
    for _bb in range(B):
        assert TRT[gmul(1 << _a, MU[_bb])] == (1 if _a == _bb else 0)

LAMJ = [0] * M
for _j in range(M):
    _v = 0
    for _bpos in bitlist(CMASKJ[_j], B):
        _v ^= MU[_bpos]
    LAMJ[_j] = _v

# ---- move N sweep points into free Z gates (Vandermonde interpolation) --
KILLJ = [j for j in range(M) if LAMJ[j] == 0]
KILLJ += [j for j in range(M) if LAMJ[j] != 0][: N - len(KILLJ)]
assert len(KILLJ) == N and len(set(KILLJ)) == N
_tgt = [gmul(LAMJ[j], ginv(EXP[j])) for j in KILLJ]
_Ai = gf_matinv([[gpow(EXP[j], i) for i in range(N)] for j in KILLJ])
UCO = [0] * (N + 1)
for _r in range(N):
    _v = 0
    for _c in range(N):
        _v ^= gmul(_Ai[_r][_c], _tgt[_c])
    UCO[_r + 1] = _v


def U_eval(j):
    acc = 0
    for i in range(1, N + 1):
        if UCO[i]:
            acc ^= EXP[(LOG[UCO[i]] + i * j) % M]
    return acc


LAMJ2 = [LAMJ[j] ^ U_eval(j) for j in range(M)]
for _j in KILLJ:
    assert LAMJ2[_j] == 0, ("interpolation failed", _j)
NLIVE = sum(1 for j in range(M) if LAMJ2[j])
ZMASKI = [bitmask(B, lambda bb: TRT[gmul(UCO[i + 1], 1 << bb)]) for i in range(N)]


def _elt3(c):
    w = 0
    for i in bitlist(c, 3):
        w ^= gpow(_v_el, i)
    return w


def _tr3(x):
    t = trace(x, 3)
    assert t in (0, 1)
    return t


# the GF(8) pairing form B(r, s) = Tr3(r * s^2) as PAIR3 spells it: rank 3
for _r in range(8):
    for _s3 in range(8):
        _acc = 0
        for _p, _qm in PAIR3:
            _acc ^= parity(_r & _p) & parity(_s3 & _qm)
        assert _acc == _tr3(gmul(_elt3(_r), gmul(_elt3(_s3), _elt3(_s3)))), (
            "pairing form drifted",
            _r,
            _s3,
        )


def pair_rows(lam):
    """r_a(lam), a < B: parity(r_a & U) = Tr(e_a sqrt(lam U)), lam != 0."""
    assert lam
    sq = [gsqrt(gmul(lam, 1 << c)) for c in range(B)]
    return [bitmask(B, lambda c: TRT[gmul(1 << a_, sq[c])]) for a_ in range(B)]


# =========================================================================
# Constant linear maps of monicize and the Euclid pass
# =========================================================================
# MONB[k] = the (a, b) basis pairs whose product has bit k set
MONB = [[] for _ in range(B)]
for _a in range(B):
    for _b2 in range(B):
        _vv = gmul(1 << _a, 1 << _b2)
        for _k2 in range(B):
            if (_vv >> _k2) & 1:
                MONB[_k2].append((_a, _b2))
SQRT_ROWS = f2_transpose([gsqrt(1 << j) for j in range(B)], B)
SQUARE_ROWS = f2_transpose([gmul(1 << j, 1 << j) for j in range(B)], B)


# =========================================================================
# Truncation schedule
# =========================================================================
# Reversal-stack capacity (tagged slots; overflow shots form error
# class E2, the stack overflow that README.md describes).  Overflow rate
# by cap under the DQI weight law, measured under the derived windows
# (403.2M draws): cap 1 -> 1.02e-4 [1.01e-4, 1.03e-4], cap 2 -> 4.1e-7,
# cap 3 -> 0 seen; max pushes 3.
# Cap 1 saves STKW = 19 wires at the global peak (both slots materialize
# by round 4 and stay live through the sweep) and 19 Toffoli per push
# cascade forward and again in the reverse pass: 2,470 Toffoli and 19
# qubits against cap 2, a 1.5% lower TQ-cost.  The aggregate sign-error
# rate is ~1.4e-4 (E1 3.8e-5 + E2 1.02e-4), 7x under the 1e-3 allowance
# (cap 2: 3.8e-5, 26x).
STK_CAP = 1

# Per-round coefficient windows (MEASURED; constants.json bm_windows,
# derived by gen_constants.py from 403.2M draws under the DQI weight
# law).  IMAXV bounds the discrepancy convolution, KMSV the
# flag-controlled slot swaps, KMMV the update macs.  The rule: the
# smallest per-round windows such that every clipped slot has marginal
# mass < 4e-5; r=0 and r=N-1 are pinned by the circuit.  Events outside
# a window form error class E1, the cutoff crossings of README.md.
_WIN = CONSTS["bm_windows"]
IMAXV = list(_WIN["IMAXV"])
KMSV = list(_WIN["KMSV"])
KMMV = list(_WIN["KMMV"])
assert len(IMAXV) == len(KMSV) == len(KMMV) == N
for _r in range(N):
    # never wider than the reachable envelope, never narrower than
    # round 0's seed path needs
    assert IMAXV[_r] <= min(_r, W, (_r + 1) // 2 + 1)
    assert KMMV[_r] <= KMSV[_r] <= min(_r + 1, W, _r // 2 + 2)
assert IMAXV[N - 1] == W and KMSV[N - 1] == KMMV[N - 1] == W
assert KMSV[0] == 1 and KMMV[0] == 0

# Signed d counter: two's complement, wide enough for the all-zero
# syndrome's monotone climb (d = N-2 after the last increment) -- DWB-1
# magnitude bits + the sign bit dW[DWB-1].
DWB = (N - 2).bit_length() + 1
assert (1 << (DWB - 1)) - 1 >= N - 2
SGN = DWB - 1  # the sign bit index (pqc / ctl anchor)
DMAX = 1 << (DWB - 1)  # two's-complement range of dW
# Stack round tag: TAGB bits hold every pushing round 0..N-1.
TAGB = (N - 1).bit_length()
TAGMASK = (1 << TAGB) - 1
STKW = B + TAGB  # stack slot: B value lanes + the round tag
assert DWB == 8  # MBITS = dW[2:7] and search.c hard-code the 8-bit dW


def caps_schedule():
    """Exact reachable-(d, depth) DP over the d-automaton: d>=0 branches to
    d+1 (ctl=0, needs D==0) or -d (ctl=1); d<0 forces ctl=0 and a PUSH
    (pqc = dW[SGN]), d -> d+1.  r=N-1 pushes unconditionally.  CAPS[r] =
    max reachable stack depth AFTER a push at round r = the emitted
    cascade extent; CAPS[r]==0 => a push is unreachable => the cascade is
    elided.
    """
    states = {(0, 0)}
    caps = [0] * N
    for r in range(N):
        ns = set()
        mx = 0
        for d_, k_ in states:
            if r == N - 1:
                ns.add((d_, k_ + 1))
                mx = max(mx, k_ + 1)
            elif d_ < 0:
                ns.add((d_ + 1, k_ + 1))
                mx = max(mx, k_ + 1)
            else:
                ns.add((d_ + 1, k_))
                ns.add((-d_, k_))
        caps[r] = mx
        states = ns
    return caps


CAPS_FULL = caps_schedule()
assert max(CAPS_FULL) <= W, ("stack cap bound broken", CAPS_FULL)
assert CAPS_FULL[0] == 0 and CAPS_FULL[1] == 0
# reachable depth is W-1 before the unconditional r=N-1 push, W after
assert max(CAPS_FULL[: N - 1]) == W - 1 and CAPS_FULL[N - 1] == W
# CAP truncation: clamp every emitted cascade extent to STK_CAP.
# min(c, STK_CAP) == 0 iff c == 0, so the set of rounds carrying a cascade
# is unchanged; only extents shrink.  Overflow shots (depth would exceed
# STK_CAP) are the failure class E2 (README.md).  With
# STK_CAP = 1 every cascade is the single Dw <-> STK[0] Fredkin level.
CAPS = [min(c, STK_CAP) for c in CAPS_FULL]
assert max(CAPS) == STK_CAP

# Psi-route cofactor windows (cyclic intervals (lo, n) of the 18-word
# registers, array-index frame) per CR op of R: the measured union
# support, recorded only where the op acts (swap: flag = 1; rotation:
# c = 1; product: (q_lo, q_hi) != 0), over 400k weight-law draws plus the
# w = 0 / w = 1 anchors (constants.json psi_windows, searched by
# gen_constants.py).  A support outside a window only mis-sets the
# Psi value (a sign error on that shot): R^-1 mirrors R op for op, so the
# registers are restored exactly either way.
PSI_WIN_IV = {k: {int(j): tuple(v) for j, v in d.items()} for k, d in CONSTS["psi_windows"].items()}
assert sorted(PSI_WIN_IV) == ["mul", "rz", "swap"] and all(
    sorted(d) == list(range(NH)) for d in PSI_WIN_IV.values()
)
PSI_WIN = {(k, jj): v for k, d in PSI_WIN_IV.items() for jj, v in d.items()}


# =========================================================================
# Terms (Builder-free term-list construction)
# =========================================================================
# A term list is the build's unit of arithmetic: a list of normalized
# terms over hard wire ids,
#
#   ('L', ws, c, tws)             tws ^= parity(ws) ^ c      (free CX/X)
#   ('P', wa, ca, wb, cb, tws)    tws ^= (parity(wa) ^ ca) AND
#                                        (parity(wb) ^ cb)   (one CCX)
#
# built by folding the frozen formula tables (section B) through operand
# specs (one wire-set per operand bit).  Em emits a term list and
# erases it.  Everything here is pure construction: no gate is emitted.


def spec_plain(ws):
    return [frozenset((w,)) for w in ws]


def spec_rows(rows, ws):
    return [
        frozenset(ws[j] for j in range(len(ws)) if (rows[r] >> j) & 1) for r in range(len(rows))
    ]


def fold(spec, bits_):
    ws = frozenset()
    for i in bits_:
        ws = ws ^ spec[i]
    return tuple(sorted(ws))


def tmask(Tb):
    """Return a formula product's target index set as a B-bit mask."""
    u = 0
    for jj in Tb:
        u ^= 1 << jj
    return u


def xfold(pairs):
    """Operand spec bit from (word, row mask) pairs: the XOR (not union)
    of the selected wires, so repeated wires cancel.
    """
    s = set()
    for word, row in pairs:
        for j in bitlist(row, B):
            s ^= {word[j]}
    return frozenset(s)


def norm_term(wa, ca, wb, cb, tws):
    if (not wa) and ca == 0:
        return None
    if (not wb) and cb == 0:
        return None
    if not wa:
        return ("L", wb, cb, tws)
    if not wb:
        return ("L", wa, ca, tws)
    if wa == wb:
        if ca == cb:
            return ("L", wa, ca, tws)
        return None
    return ("P", wa, ca, wb, cb, tws)


def bil_terms(rt, table, specA, specB, tgt=None, tws_of=None):
    """Bilinear terms of one frozen formula table over operand specs:
    fold each product's operand bit-sets through the specs, normalize,
    append.  Targets come from tgt (indexed by the table's target set)
    or a custom tws_of(Tb).
    """
    for Ab, Bb, Tb in table:
        tws = tws_of(Tb) if tws_of else [tgt[jj] for jj in Tb]  # ty: ignore[not-subscriptable]  # tgt is given when tws_of is not
        tm = norm_term(fold(specA, Ab), 0, fold(specB, Bb), 0, tws)
        if tm:
            rt.append(tm)


def kt6_terms_poly(rt, specA, specB, tgt12, off):
    """KT6 terms whose GF(64)-coordinate targets land in the 12-bit poly
    word tgt12 through the tower basis: coord j -> E12[j + off].
    """

    def tws_of(Tb):
        pm = 0
        for u in Tb:
            pm ^= E12[u + off]
        return [tgt12[jj] for jj in bitlist(pm, B)]

    bil_terms(rt, KT6, specA, specB, tws_of=tws_of)


def quad_terms(rt, Qf, Xs, tgt, n):
    tris, corr = Qf
    sp = spec_plain(Xs)
    for u, w, om in tris:
        tm = norm_term(
            fold(sp, bitlist(u, n)), 0, fold(sp, bitlist(w, n)), 0, [tgt[m] for m in bitlist(om, n)]
        )
        if tm:
            rt.append(tm)
    for m in range(n):
        if corr[m]:
            rt.append(("L", tuple(sorted(Xs[i] for i in bitlist(corr[m], n))), 0, (tgt[m],)))


def lin_rows_terms(rt, rows, src, tgt):
    """Free linear rows: tgt[r] ^= parity(rows[r] & src)."""
    for rr in range(len(rows)):
        if rows[rr]:
            rt.append(
                ("L", tuple(sorted(src[jj] for jj in bitlist(rows[rr], len(src)))), 0, (tgt[rr],))
            )


def prod_rt(tgt, wa, wb):
    """tgt ^= wa * wb (KT12, 42 products) as a term list."""
    rt = []
    bil_terms(rt, KT, spec_plain(wa), spec_plain(wb), tgt)
    return rt


def one_rt(wa, ca, wb, cb, t):
    """t ^= (wa ^ ca) AND (wb ^ cb) over wire tuples as a one-term list."""
    return [norm_term(tuple(sorted(wa)), ca, tuple(sorted(wb)), cb, [t])]


# ---- AND ladders ----------------------------------------------------------
def ladder_rt(rungs, tree):
    """AND-ladder term lists over rungs [(wire tuple, polarity), ...]:
    tree[i-1] <- AND of the first i+1 rungs, each parity tested against
    its polarity bit (len(rungs) - 1 products).
    """
    (w0, c0), (w1, c1) = rungs[0], rungs[1]
    rts = [one_rt(w0, c0, w1, c1, tree[0])]
    for i in range(2, len(rungs)):
        ws, cc = rungs[i]
        rts.append(one_rt((tree[i - 2],), 0, ws, cc, tree[i - 1]))
    return rts


def eq_rungs(bits, val):
    """Rungs whose ladder top is [bits == val] (val = 0: the zero test)."""
    return [((w,), 1 - ((val >> i) & 1)) for i, w in enumerate(bits)]


# ---- the tower inversion chain -------------------------------------------
CHAIN_WINS = [("nqd", 6), ("m3d", 3), ("s3d", 3), ("yd", 6)]


def norm_chain_terms(src, a0, a1, nq, m3, s3):
    """Term lists for the norm chain on src: N (KT6 bilinear + linear
    rows into nq), M (KT3 on the halves of nq + linear rows into m3),
    M^3 (CUBE3Q into s3).
    """
    rtN = []
    bil_terms(rtN, KT6, a0, a1, nq)
    lin_rows_terms(rtN, LNROWS, src, nq)
    rtM = []
    bil_terms(rtM, KT3, spec_plain(nq[0:3]), spec_plain(nq[3:6]), m3)
    lin_rows_terms(rtM, LMROWS, nq, m3)
    rtS = []
    quad_terms(rtS, CUBE3Q, m3, s3, 3)
    return rtN, rtM, rtS


def chain_rts(src, wins, out):
    """Forward tower inversion-chain term lists for out = inv(src): the
    norm chain, N^-1 = Nbar M^-1 (two KT3 into yd, M^-1 = (M^3)^2 folded
    into the operand spec as SQ3R rows), then x^-1 = xbar N^-1 (two KT6)
    recombined into the 12-bit poly-basis out.  wins are the CHAIN_WINS
    scratch words; rts[:-1] write wins in order, rts[-1] writes out.
    """
    nq, m3, s3, yd = wins
    a0 = spec_rows(A0ROWS, src)
    a1 = spec_rows(A1ROWS, src)
    a01 = spec_rows(A01ROWS, src)
    rtN, rtM, rtS = norm_chain_terms(src, a0, a1, nq, m3, s3)
    m6 = spec_rows(SQ3R, s3)
    n0 = spec_plain(nq[0:3])
    n1 = spec_plain(nq[3:6])
    n01 = [n0[i] ^ n1[i] for i in range(3)]
    rtY = []
    bil_terms(rtY, KT3, n01, m6, yd[0:3])
    bil_terms(rtY, KT3, n1, m6, yd[3:6])
    y = spec_plain(yd)
    rtI = []
    kt6_terms_poly(rtI, a01, y, out, 0)
    kt6_terms_poly(rtI, a1, y, out, 6)
    return [rtN, rtM, rtS, rtY, rtI]


# ---- evaluation/interpolation products -----------------------------------
def polyprod_rt(Qw, Fw, tgt):
    """Evaluation/interpolation product: for the polynomials Q(y) =
    sum_i Qw[i] y^i and F(y) = sum_k Fw[k] y^k (lists of words), XOR
    coefficient n of Q*F into every word in tgt[n] (a list; empty = drop).
    Costs len(Qw) + len(Fw) - 1 products (one scalar product when both
    have one word); operand evaluations and the interpolation rows are
    free CX networks folded into the operand specs and target sets.
    """
    npts = len(Qw) + len(Fw) - 1
    assert len(tgt) == npts
    if npts == 1:
        rt = []
        for T in tgt[0]:
            bil_terms(rt, KT, spec_plain(Qw[0]), spec_plain(Fw[0]), T)
        return rt
    pts, Iv = vinv(npts)
    rt = []
    for t in range(npts):
        cq = [cmrows(gpow(pts[t], i)) for i in range(len(Qw))]
        cf = [cmrows(gpow(pts[t], k)) for k in range(len(Fw))]
        sA = [xfold([(Qw[i], cq[i][bb]) for i in range(len(Qw))]) for bb in range(B)]
        sB = [xfold([(Fw[k], cf[k][bb]) for k in range(len(Fw))]) for bb in range(B)]

        def tws_of(Tb, t=t):
            u = tmask(Tb)
            acc = {}
            for n in range(npts):
                if not tgt[n]:
                    continue
                v = gmul(Iv[n][t], u)
                for T in tgt[n]:
                    for mm in bitlist(v, B):
                        acc[T[mm]] = acc.get(T[mm], 0) ^ 1
            return [w for w in sorted(acc) if acc[w]]

        bil_terms(rt, KT, sA, sB, tws_of=tws_of)
    return [tm for tm in rt if tm[-1]]


def omega_rts(sig, syn):
    """Return the in-place omega70 block term lists (MACS order) for the
    monic sigma bank sig (words sig[1..W]) and the syndrome words syn.
    Each block folds its evaluation maps into the operand parity specs
    (sigma side over the bank, S side over the rows it reads) and its
    interpolation rows into the KT target sets, so the a+b-1 pointwise
    products are the only Toffoli gates.
    """
    out = []
    for r0, a_, c0, b_ in MACS:
        npts = a_ + b_ - 1
        pts, Iv = vinv(npts)
        rt = []
        for t in range(npts):
            cmA = [cmrows(Iv[k][t]) for k in range(npts)]
            cmB = [cmrows(gpow(pts[t], j)) for j in range(b_)]
            # diagonals beyond W are zero (clipped): sigma has W+1 coefficients
            sA = [
                frozenset(
                    sig[1 + k][j] for k in range(min(npts, W)) for j in bitlist(cmA[k][bb], B)
                )
                for bb in range(B)
            ]
            sB = [
                frozenset(
                    syn[c0 + b_ - 1 - j][jb] for j in range(b_) for jb in bitlist(cmB[j][bb], B)
                )
                for bb in range(B)
            ]
            wcoef = [gpow(pts[t], i2) for i2 in range(a_)]

            def om_tws(Tb, wcoef=wcoef, r0=r0, a_=a_):
                u = tmask(Tb)
                tws = []
                for i2 in range(a_):
                    tws.extend(syn[r0 + i2][m] for m in bitlist(gmul(wcoef[i2], u), B))
                return tws

            bil_terms(rt, KT, sA, sB, tws_of=om_tws)
        out.append(rt)
    return out


# =========================================================================
# Emission (term-list emitter, journaled vocabulary, allocators)
# =========================================================================
# Em emits the term lists of the Terms section through a live Builder: emit_mac
# writes a term list forward (one CCX per product; a shared multi-target
# product fans out through the sT scratch and erases it for free), and
# emit_free erases a written term list by demolition measurement + phase
# fixups, 0 Toffoli.  The journaled vocabulary (mac / xor / free / x /
# cx / cswap / swap / swaps, and the ladder, chain and increment helpers
# built on them) runs each step now and records its conjugate in a
# uncompute.bennett.Journal, so a later Journal.reverse() replays the
# conjugates over the same hard wire ids.
#
# Scratch comes from Em's current allocator, an Alloc: it round-trips
# windows through the Builder allocator and logs the window ids a journal
# touches, so a replay can hold exactly those ids (Alloc.replay).


def word_pairs(xs, ys):
    """Bitwise wire pairs of two word lists: word by word, bit 0 first."""
    return [(x[t], y[t]) for x, y in zip(xs, ys) for t in range(B)]


# NOJ never records: unjournaled emission through the journaled vocabulary.
NOJ = Journal()
NOJ.recording = False


class Em:
    """Term-list emitter over the held pivots sA, sB and the fan-out
    scratch sT, with the journaled vocabulary.  The vocabulary records
    into the current journal J (NOJ outside any using() block), and the
    scratch helpers draw from the current allocator A.
    """

    def __init__(self, b, sA, sB, sT, A):
        self.b = b
        self.sA = sA
        self.sB = sB
        self.sT = sT
        self.J = NOJ
        self.A = A

    @contextmanager
    def using(self, J=None, A=None):
        """Make J and/or A the current journal / allocator inside the block."""
        prev = self.J, self.A
        if J is not None:
            self.J = J
        if A is not None:
            self.A = A
        yield
        self.J, self.A = prev

    # ---- the unjournaled core --------------------------------------------
    def _build(self, ws, cc, s):
        if len(ws) == 1 and cc == 0:
            return ws[0], False
        for w in ws:
            self.b.cx(w, s)
        if cc:
            self.b.x(s)
        return s, True

    def _unbuild(self, ws, cc, s, built):
        if not built:
            return
        if cc:
            self.b.x(s)
        for w in reversed(ws):
            self.b.cx(w, s)

    def pivot_pair(self, wa, ca, wb, cb, mid):
        """Emit the shared pivot-operand sandwich: build both pivot operands
        into sA/sB, run the caller's middle 2-qubit op on the built
        pivots (ua, ub), then unbuild in reverse.  One core for the
        single-target CCX in emit_mac, the conditioned-CZ repayment in
        emit_free, and fan_pair's full sT window.
        """
        ua, ba = self._build(wa, ca, self.sA)
        ub, bb2 = self._build(wb, cb, self.sB)
        mid(ua, ub)
        self._unbuild(wb, cb, self.sB, bb2)
        self._unbuild(wa, ca, self.sA, ba)

    def fan_pair(self, wa, ca, wb, cb, mid):
        """Emit the product-pair skeleton over pivot_pair: CCX the pivots'
        AND into the sT fan-out scratch, run the caller's mid(sT) on it,
        then measure sT away with its phase repayment.
        """
        b = self.b

        def sandwich(ua, ub):
            b.ccx(ua, ub, self.sT)
            mid(self.sT)
            m = b.hmr(self.sT)
            b.cz(ua, ub, cond=m)

        self.pivot_pair(wa, ca, wb, cb, sandwich)

    def emit_mac(self, rt):
        b = self.b
        for t in rt:
            k = t[0]
            if k == "L":
                ws, cc, tws = t[1], t[2], t[3]
                for tw in tws:
                    for w in ws:
                        b.cx(w, tw)
                    if cc:
                        b.x(tw)
            else:
                wa, ca, wb, cb, tws = t[1], t[2], t[3], t[4], t[5]
                if len(tws) == 1:
                    self.pivot_pair(wa, ca, wb, cb, lambda ua, ub, tw=tws[0]: b.ccx(ua, ub, tw))
                else:

                    def mid(fan, tws=tws):
                        for tw in tws:
                            b.cx(fan, tw)

                    self.fan_pair(wa, ca, wb, cb, mid)

    def emit_free(self, rt):
        b = self.b
        per = {}
        order = []
        for t in rt:
            for tw in t[-1]:
                if tw not in per:
                    per[tw] = []
                    order.append(tw)
                per[tw].append(t)
        for tw in reversed(order):
            m = b.hmr(tw)
            for t in per[tw]:
                k = t[0]
                if k == "L":
                    for w in t[1]:
                        b.z(w, cond=m)
                    if t[2]:
                        b.neg(cond=m)
                else:
                    self.pivot_pair(t[1], t[2], t[3], t[4], lambda ua, ub: b.cz(ua, ub, cond=m))

    def swap_fn(self, c, pairs):
        """One closure applying cswap(c, x, y) over pairs in order (c None:
        plain swaps).
        """
        b = self.b
        if c is None:

            def fn():
                for x, y in pairs:
                    b.swap(x, y)
        else:

            def fn():
                for x, y in pairs:
                    b.cswap(c, x, y)

        return fn

    # ---- the journaled vocabulary ----------------------------------------
    def mac(self, rt):
        """Write a term list into ZERO targets; the conjugate erases it
        for free.
        """
        self.J.step(lambda: self.emit_mac(rt), lambda: self.emit_free(rt))
        return rt

    def xor(self, rt):
        """XOR-accumulate a term list into live targets: self-inverse."""
        self.J.step(lambda: self.emit_mac(rt))
        return rt

    def free(self, rt):
        """Measurement-erase targets that equal the term list (a claim);
        the conjugate recomputes them.
        """
        self.J.step(lambda: self.emit_free(rt), lambda: self.emit_mac(rt))

    def macs(self, rts):
        for rt in rts:
            self.mac(rt)

    def frees(self, rts):
        """Erase term lists in reverse order."""
        for rt in reversed(rts):
            self.free(rt)

    def x(self, q):
        self.J.step(lambda: self.b.x(q))

    def cx(self, c, t):
        self.J.step(lambda: self.b.cx(c, t))

    def cswap(self, c, x, y):
        self.J.step(lambda: self.b.cswap(c, x, y))

    def swap(self, x, y):
        self.J.step(lambda: self.b.swap(x, y))

    def swaps(self, c, pairs):
        """cswap(c, .) (c None: swap) over pairs as ONE journal step."""
        self.J.step(self.swap_fn(c, pairs))

    def ladder(self, rungs, tree):
        """Emit an AND ladder (ladder_rt) and return its term
        lists; the caller consumes the top and erases with frees().
        """
        rts = ladder_rt(rungs, tree)
        self.macs(rts)
        return rts

    def inc1(self, x, tree):
        """x += 1 (mod 2^len(x)) as a constant ripple: len(x) - 2 CCX
        ladder carries into tree (len(x) - 2 zero wires), each erased by
        measurement right after its use.
        """
        crts = self.ladder([((x[i],), 0) for i in range(len(x) - 1)], tree)
        for j in range(len(x) - 1, 1, -1):
            self.cx(tree[j - 2], x[j])
            self.free(crts[j - 2])
        self.cx(x[0], x[1])
        self.x(x[0])


def chain_open(E, src, out):
    """Open the inversion-chain scratch on E's allocator and build the
    chain term lists for out = inv(src) (chain_rts).
    """
    wins = [E.A.open(name, n) for (name, n) in CHAIN_WINS]
    return wins, chain_rts(src, wins, out)


def chain_close(E, wins):
    for (name, _n), win in zip(reversed(CHAIN_WINS), reversed(wins)):
        E.A.close(name, win)


def chain_fwd(E, src, out):
    wins, rts = chain_open(E, src, out)
    E.macs(rts)
    return wins, rts


def chain_undo(E, wins, rts):
    E.frees(rts)
    chain_close(E, wins)


@contextmanager
def chain_live(E, src, out):
    """out <- inv(src) with the chain intermediates live inside the block;
    exit erases out and the intermediates.
    """
    wins, rts = chain_fwd(E, src, out)
    yield
    chain_undo(E, wins, rts)


@contextmanager
def opened(E, name, n):
    """Open an n-wire scratch window on E's allocator; exit closes it."""
    qs = E.A.open(name, n)
    yield qs
    E.A.close(name, qs)


@contextmanager
def eq_test(E, bits, val):
    """Yield the top wire of a [bits == val] ladder; exit
    erases the ladder.
    """
    with opened(E, "tree", len(bits) - 1) as tr:
        rts = E.ladder(eq_rungs(bits, val), tr)
        yield tr[len(bits) - 2]
        E.frees(rts)


class Alloc:
    """Scratch windows through the Builder allocator: open allocs and
    claims a window, close releases it and frees it in reverse alloc
    order.  Journals replay closures over hard wire ids, so open records
    each window id into the current log set (logging()), and replay()
    re-materializes exactly a logged set via reclaim before a replay.
    """

    def __init__(self, b):
        self.b = b
        self.log = None  # the set open records window ids into, if any
        self.round = None  # BM round tag for bank bookkeeping
        self.bank_log = {}  # round -> bank registers first touched there

    def open(self, name, n):
        qs = self.b.qubits(n)
        if self.log is not None:
            self.log.update(qs)
        self.b.claim(name, qs)
        return qs

    def reopen(self, name, qs):
        """Re-acquire these exact (free) ids as a window."""
        self.b.reclaim(qs)
        self.b.claim(name, qs)

    def close(self, name, qs):
        self.b.release_recycle(name, qs)

    @contextmanager
    def logging(self, ids):
        """Record every window id opened inside the block into the set ids."""
        prev, self.log = self.log, ids
        yield
        self.log = prev

    @contextmanager
    def replay(self, ids, name="jscratch"):
        """Hold exactly the (free) ids inside the block, for a journal replay."""
        pool = sorted(ids)
        if pool:
            self.reopen(name, pool)
        yield
        if pool:
            self.close(name, pool)


class BankReg:
    """One lazy bank register: B wires materialized at the first operand
    touch (allocated, claimed into the bank's set, and logged with the
    touch round).  The fused-Q rotation relabels handles for free, so a
    physical slot's first touch is discovered mechanically instead of
    derived from the clipped-window schedule.  The mirror replay of the
    touch round returns the slot to |0> (post-replay state equals the
    forward state before that round), which is where the reverse pass
    frees it.
    """

    __slots__ = ("A", "name", "round", "width", "wires")

    def __init__(self, A, name, width=B):
        self.A = A
        self.name = name
        self.width = width
        self.wires = None
        self.round = None

    def mat(self):
        if self.wires is None:
            A = self.A
            self.wires = A.b.qubits(self.width)
            A.b.claim(self.name, self.wires)
            self.round = A.round
            A.bank_log.setdefault(A.round, []).append(self)
        return self.wires

    def __getitem__(self, k):
        return self.mat()[k]

    def __iter__(self):
        return iter(self.mat())

    def __len__(self):
        return self.width


# =========================================================================
# Psi route (half-degree Euclid F, R, cofactor recurrence)
# =========================================================================
# README.md, "Division-free Forney recovery".  The half-degree Euclid
# pass F compresses the padded locator halves (E', O') into quotients;
# R replays F backwards (restoring E', O') while the cofactor recurrence
# CR builds the Bezout pair (a, b), A E' + B O' = z^m, so
# Psi = B^2 + z A^2 = 1/sigma_m' at every root.  Every step runs
# through a uncompute.bennett.Journal over hard wire ids: F records into its
# journal JF; R is a deferred journal that replays JF's block segments
# backwards (interleaved with the CR steps), R^-1 replays them forwards,
# and F^-1 replays all of JF backwards.  The allocator logs every scratch
# id these journals touch (PSI_IDS), and R and R^-1..F^-1 hold exactly that
# set, so every replay reuses exactly the forward wire ids.  gen_constants.py holds the bit-exact
# classical model of these gadgets, op for op.


def flag_nz(E, word, q):
    """q ^= [word != 0]: a zero-test ladder, then NOT top into q (free:
    CX + X, one journal step).
    """
    b = E.b
    with eq_test(E, word, 0) as top:

        def fn():
            b.cx(top, q)
            b.x(q)

        E.J.step(fn)


@contextmanager
def inv_word(E, src):
    """Yield out = inv(src) (66 Toffoli), chain intermediates erased for
    free; exit erases out (36 Toffoli: the intermediates are recomputed).
    """
    with opened(E, "inv", B) as out:
        wins, rts = chain_fwd(E, src, out)
        chain_undo(E, wins, rts[:-1])
        yield out
        # the recompute runs rts built on the chain's exact ids
        for (name, _n), w in zip(CHAIN_WINS, wins):
            E.A.reopen(name, w)
        E.macs(rts[:-1])
        E.free(rts[-1])
        E.frees(rts[:-1])
        chain_close(E, wins)


def pad_barrel(E, mbits, ew, ow):
    """Shift both halves up by m = the 5-bit register mbits (non-cyclic:
    the barrel of cswap chains, one per set bit of m).
    """
    for sft in (16, 8, 4, 2, 1):
        ctl = mbits[sft.bit_length() - 1]
        for X in (ew, ow):
            for k in range(NH - 1 - sft, -1, -1):
                E.swaps(ctl, word_pairs([X[k]], [X[k + sft]]))


def psi_pre(E, ew, ow, pre):
    """pre <- [E'_17 == 0]; cswap the halves on pre (odd L with
    sigma_{L-1} = 0).  pre stays live until the literal reverse.
    """
    with eq_test(E, ew[17], 0) as top:
        E.cx(top, pre)
    E.swaps(pre, word_pairs(ew, ow))


def psi_forward(E, f, g, eps, idx):
    """F on the Euclid bank (f, g: lists of 18 words, top-aligned, f[17]
    the pivot): round 1, the block-0 zero-pivot repair, then 17 blocks
    s0..s9.  Mirrors gen_constants.py forward() op for op.  Every free
    erase claim holds whenever f17 != 0 at the elimination, i.e. on every
    correct decode (the pre-swap and padding guarantee deg E' = 17) and on
    every garbage locator the model has seen.  Returns the final (f, g)
    word arrays and the per-block records that build_R consumes: the
    flag / c / q wires and the journal marks where R interleaves the
    cofactor ops (start; swap, after the flag cswap; q, quotients ready;
    c, c set; end).
    """
    f, g = list(f), list(g)
    recs = {}
    # ---- round 1: q1 = g17/f17; g -= q1 f; store q1 in g17's word ----
    mk = {"start": E.J.mark(), "swap": E.J.mark()}  # block 0 swaps on pre, before F
    with inv_word(E, f[17]) as fi:
        with opened(E, "q1", B) as q1:
            E.mac(prod_rt(q1, g[17], fi))
            E.free(prod_rt(g[17], q1, f[17]))  # claim g17 == q1 f17
            E.swaps(None, word_pairs([q1], [g[17]]))  # g17's word <- q1
        mk["q"] = E.J.mark()  # q1 lives in g[17]'s word
        q1w = g[17]
        E.xor([op for k in range(17) for op in prod_rt(g[k], q1w, f[k])])
    g = g[17:] + g[:17]  # rot_up (relabel)
    # ---- block 0: zero-pivot test + repair (degree defect after round 1)
    with eq_test(E, g[17], 0) as top:
        E.cx(top, eps)  # eps <- [g17 == 0]
    mk["c"] = E.J.mark()
    for k in range(17, 1, -1):  # rot_up(g, lo=1) on eps
        E.swaps(eps, word_pairs([g[k]], [g[k - 1]]))
    mk["end"] = E.J.mark()
    recs[0] = {"marks": mk, "c": eps, "q": [q1w]}
    for j in range(1, 18):
        mk = {"start": E.J.mark()}
        f[j - 1], g[j - 1] = g[j - 1], f[j - 1]  # boundary swap (relabel)
        with opened(E, "flag", 1) as (flag,):
            # s0: set flag to [g17 != 0]
            flag_nz(E, g[17], flag)
            # s1: cswap window [j, 17] on flag
            E.swaps(flag, word_pairs([f[k] for k in range(j, 18)], [g[k] for k in range(j, 18)]))
            mk["swap"] = E.J.mark()
            # s2: erase flag against [g17 != 0] (post-swap)
            flag_nz(E, g[17], flag)
        # s3: invert f17 into fi; s9 (the with exit) erases it
        with inv_word(E, f[17]) as fi:
            # s4: q_hi, t, q_lo
            with opened(E, "qh", B) as qh, opened(E, "ql", B) as ql:
                E.mac(prod_rt(qh, g[17], fi))
                if j <= 16:
                    with opened(E, "t", B) as t:
                        trt = E.mac(prod_rt(t, qh, f[16]))
                        rtl = []
                        bil_terms(
                            rtl,
                            KT,
                            [frozenset((g[16][i], t[i])) for i in range(B)],
                            spec_plain(fi),
                            ql,
                        )
                        E.mac(rtl)
                        E.free(trt)
                else:
                    E.mac(prod_rt(ql, g[16], fi))
                mk["q"] = E.J.mark()
                # s5: block elimination: target word g[j-1+n] ^= coef_n of
                # (ql + qh y) * (f[j] + f[j+1] y + ... + f[16] y^(16-j)), n = 0..17-j
                if j <= 16:
                    E.xor(
                        polyprod_rt(
                            [ql, qh],
                            [f[k] for k in range(j, 17)],
                            [[g[j - 1 + n]] for n in range(18 - j)],
                        ),
                    )
                E.free(prod_rt(g[17], qh, f[17]))  # claim g17 == qh f17
                E.free(prod_rt(g[16], ql, f[17]))  # claim g16' == ql f17
                E.swaps(None, word_pairs([qh, ql], [g[17], g[16]]))
            g = g[16:] + g[:16]  # rot_up twice (relabel)
            with opened(E, "c", 1) as (c,):
                # s6: c = [g17 == 0] AND NOT eps
                with eq_test(E, g[17], 0) as top:
                    E.mac(one_rt((top,), 0, (eps,), 1, c))
                mk["c"] = E.J.mark()
                # s7: controlled rot_up(g, lo=j+1) on c; eps ^= c; idx ^= c*j
                for k in range(17, j + 1, -1):
                    E.swaps(c, word_pairs([g[k]], [g[k - 1]]))
                E.cx(c, eps)
                for bp in bitlist(j, 5):
                    E.cx(c, idx[bp])
                # s8: erase c == eps AND [idx == j]
                with eq_test(E, idx, j) as top:
                    E.free(one_rt((top,), 0, (eps,), 0, c))
        mk["end"] = E.J.mark()
        recs[j] = {"marks": mk, "flag": flag, "c": c, "q": [ql, qh]}
    return f, g, recs


def build_R(E, JF, recs, A, Bw, pre, win, eb):
    """R as a deferred Journal: JF's block segments replayed backwards,
    interleaved with the cofactor recurrence CR on the 18-word cyclic
    registers A (a) and Bw (b).  win maps ('mul'|'rz'|'swap', j) to the
    measured union support (lo, n) of the register(s) the op touches
    (array index frame).  R.forward() runs R; R.reverse() runs R^-1.
    Returns (R, a, b), the cofactor word lists in their final labels.
    """
    a, bb = list(A), list(Bw)
    R = Journal()

    def seg_rev(x, y):
        R.defer(lambda: JF.reverse(x, y), lambda: JF.forward(x, y))

    def cr_rz_ctl(c, j):
        # controlled cyclic rotation up by one (new ws[k] = old ws[k-1])
        # of the window words: len-1 word cswaps
        lo, n = win[("rz", j)]
        if n == 0:
            return
        ws = [bb[(lo + i) % NH] for i in range(min(n + 1, NH))]
        hi = [ws[k] for k in range(len(ws) - 1, 0, -1)]
        low = [ws[k - 1] for k in range(len(ws) - 1, 0, -1)]
        R.defer(E.swap_fn(c, word_pairs(hi, low)), E.swap_fn(c, word_pairs(hi[::-1], low[::-1])))

    def cr_mul(qws, j):
        # b <- z^len(qws) b (relabel); a ^= (sum_i q_i z^(i+1)) b_old,
        # with qws listed low -> high (ql, qh for a block; [q1] for round 1)
        b0 = list(bb)
        lo, n = win[("mul", j)]
        if n:
            Fw = [b0[(lo + i) % NH] for i in range(n)]
            tgt = [[a[(lo + k + 1) % NH]] for k in range(len(qws) + n - 1)]
            rt = polyprod_rt(list(qws), Fw, tgt)
            R.defer(lambda: E.emit_mac(rt))
        return b0[-len(qws) :] + b0[: -len(qws)]

    def cr_swap(c, j):
        lo, n = win[("swap", j)]
        ks = [(lo + i) % NH for i in range(n)]
        R.defer(E.swap_fn(c, word_pairs([a[k] for k in ks], [bb[k] for k in ks])))

    for j in range(17, -1, -1):
        r = recs[j]
        mk = r["marks"]
        seg_rev(mk["c"], mk["end"])
        cr_rz_ctl(r["c"], j)
        seg_rev(mk["q"], mk["c"])
        bb = cr_mul(r["q"], j)
        seg_rev(mk["swap"], mk["q"])
        cr_swap(pre if j == 0 else r["flag"], j)
        seg_rev(mk["start"], mk["swap"])
    # undo the pre-swap of the halves too: the sweep's root test needs E'
    # in the E words (psi_pre's swap itself stays live until the literal
    # reverse, which redoes this swap first)
    ew, ow = eb
    R.defer(E.swap_fn(pre, word_pairs(ew, ow)))
    return R, a, bb


def psi_seed(E, h, mbits, A):
    """A[(m+1) % 18] ^= inv(h), m = the 5-bit register mbits (0..17):
    one [m == v] ladder (4 Toffoli) and one controlled 12-bit copy per v
    (12 Toffoli, paid again by the literal reverse).
    """
    with inv_word(E, h) as ih:
        for v in range(NH):
            with eq_test(E, mbits, v) as top:
                # XOR-accumulate, recomputed on the reverse: the TAIL word is zero
                # only on a correct decode, so a measurement erase here would
                # break syndrome restoration on approximate-BM paths
                E.xor([norm_term((top,), 0, (ih[t],), 0, [A[(v + 1) % NH][t]]) for t in range(B)])


# =========================================================================
# Sweep (Chien/Forney root test + Psi pairing, one gated repeat body)
# =========================================================================
# Per point x = gamma^t (error position j = M - t): with E'_k, b_k read as
# if rotated by gamma^(kt) and O'_k, a_k by gamma^((k+SQRT_EXP)t) (parity sets
# over the unrotated words; no rotation is emitted),
#   XOR of the 36 EB words = E'(x) + sqrt(x) O'(x) = x^m sqrt(sigma_m(x)),
#   XOR of the 36 (a, b) words = V = B(x) + sqrt(x) A(x),  Psi(x) = V^2,
# so flag = [XOR of EB == 0] (11 Toffoli) and the Forney bit
# Tr(lam omega(x) / sigma_m'(x)) = Tr(lam omega(x) Psi(x)) = Tr(sqrt(lam U) V)
# = sum_a parity(r_a(lam) . U) V_a (U = omega(x), loaded into som): 12
# flag-gated rank-1 terms, one Toffoli each.  23 Toffoli per point.
#
# Loop form: the words never move, so every CX of every point acts on
# fixed wires; only WHICH stored bits a fan-in reads changes with t.  At
# point t, word k would hold g_k^t times its stored value, so bit bb of the
# rotated word is the parity of the stored bits selected by row bb of
# cmrows(g_k^t).  The body emits the union of each fan-in's possible bits
# in the per-point emission order, each CX data-gated on its membership
# bit at t, so a point's executed ops are exactly its per-point spelling
# (the Em core's pivot_pair / fan_pair / emit_free order).  Every other op
# is gated on the point being live (lam != 0).  A pairing row r_a(lam)
# with a single bit uses that som wire directly (no sA build), so the
# pairing CCX/CZ appear once per case (built sA, or som[c]), each gated on
# its case.

LOG_EB = list(range(NH)) + [(k + SQRT_EXP) % M for k in range(NH)]
LOG_AB = [(k + SQRT_EXP) % M for k in range(NH)] + list(range(NH))
CM_ARR = np.array(
    [[0] * B] + [cmrows(EXP[e]) for e in range(M)], dtype=np.int64
)  # CM_ARR[1 + e, bb]: cmrows(gamma^e)


def rot_member(logs):
    """mem[bb, k, j, t]: bit j of word k is in operand group bb at t."""
    t = np.arange(M, dtype=np.int64)
    mem = np.zeros((B, len(logs), B, M), dtype=bool)
    for k, lg in enumerate(logs):
        rows = CM_ARR[1 + (lg * t) % M]  # [t, bb]
        for j in range(B):
            mem[:, k, j, :] = ((rows >> j) & 1).T.astype(bool)
    return mem


LIVE = np.array([bool(LAMJ2[(M - t) % M]) for t in range(M)])
assert int(LIVE.sum()) == NLIVE


def sweep_plan(b, A, omw, ebw, abw):
    """Return the body as [(gate vector over t, emitter(s_))] in emission order,
    s_ the body's wire handles; a None vector emits ungated (the tree
    window open on allocator A).  Mirrors the per-point form: som fan-ins, the root-test
    ladder (pivot_pair builds, sorted operands), the rank-12 pairing
    (fan_pair), the ladder erase (emit_free, reversed), the som unwind.
    """
    m_om = rot_member(list(range(W))) & LIVE  # som fan-ins
    m_eb = rot_member(LOG_EB) & LIVE  # root-test groups
    m_v = rot_member(LOG_AB) & LIVE  # pairing V groups
    prow = np.zeros((B, M), dtype=np.int64)  # r_a(lam) masks
    for t in range(M):
        if LIVE[t]:
            prow[:, t] = pair_rows(LAMJ2[(M - t) % M])
    single = np.stack([prow == (1 << c) for c in range(B)], axis=1)  # [a, c, t]
    built = ~single.any(axis=1) & LIVE  # [a, t]
    m_sa = np.stack([((prow >> c) & 1).astype(bool) & built for c in range(B)], axis=1)  # [a, c, t]
    # the per-point spelling the body mirrors: every root-test group is
    # non-empty (both pivots build), every V group has >= 2 wires (it builds)
    assert (m_eb.any(axis=(1, 2)) | ~LIVE).all(), "empty root-test group"
    assert (m_v.sum(axis=(1, 2)) >= 2)[:, LIVE].all(), "degenerate V group"
    eb_sorted = sorted(
        (w, k, j) for k, word in enumerate(ebw) for j, w in enumerate(word)
    )  # terms sort operands
    assert len({w for w, _, _ in eb_sorted}) == len(eb_sorted)

    plan = []

    def op(vec, fn):
        plan.append((vec, fn))

    def eb_build(g, s, inverse):
        cands = [(w, m_eb[g, k, j]) for w, k, j in eb_sorted]
        if inverse:
            op(LIVE, lambda s_: b.x(s_[s]))
            cands.reverse()
        for w, v in cands:
            op(v, lambda s_, w=w: b.cx(w, s_[s]))
        if not inverse:
            op(LIVE, lambda s_: b.x(s_[s]))

    def ua(i, s_):
        return s_["sA"] if i == 1 else s_["tr"][i - 2]

    def rung(i, mid):
        """ladder rung i >= 1: wa = group 0 (i == 1, built in sA) or
        tree[i-2] (direct), wb = group i (built in sB).
        """
        if i == 1:
            eb_build(0, "sA", False)
        eb_build(i, "sB", False)
        op(LIVE, mid)
        eb_build(i, "sB", True)
        if i == 1:
            eb_build(0, "sA", True)

    om = [(bb, k, j) for bb in range(B) for k in range(W) for j in range(B)]
    for bb, k, j in om:
        op(m_om[bb, k, j], lambda s_, bb=bb, k=k, j=j: b.cx(omw[k][j], s_["som"][bb]))
    op(None, lambda s_: s_.__setitem__("tr", A.open("tree", B - 1)))
    for i in range(1, B):
        rung(i, lambda s_, i=i: b.ccx(ua(i, s_), s_["sB"], s_["tr"][i - 1]))
    vw = [(k, j) for k in range(len(abw)) for j in range(B)]
    for a_ in range(B):
        cases = [(built[a_], lambda s_: s_["sA"])] + [
            (single[a_, c] & LIVE, lambda s_, c=c: s_["som"][c]) for c in range(B)
        ]
        for c in range(B):
            op(m_sa[a_, c], lambda s_, c=c: b.cx(s_["som"][c], s_["sA"]))
        for k, j in vw:
            op(m_v[a_, k, j], lambda s_, k=k, j=j: b.cx(abw[k][j], s_["sB"]))
        for v, u in cases:
            op(v, lambda s_, u=u: b.ccx(u(s_), s_["sB"], s_["sT"]))
        op(LIVE, lambda s_: b.cz(s_["tr"][B - 2], s_["sT"]))
        op(LIVE, lambda s_: s_.__setitem__("m", b.hmr(s_["sT"])))
        for v, u in cases:
            op(v, lambda s_, u=u: b.cz(u(s_), s_["sB"], cond=s_["m"]))
        for k, j in reversed(vw):
            op(m_v[a_, k, j], lambda s_, k=k, j=j: b.cx(abw[k][j], s_["sB"]))
        for c in reversed(range(B)):
            op(m_sa[a_, c], lambda s_, c=c: b.cx(s_["som"][c], s_["sA"]))
    for i in range(B - 1, 0, -1):
        op(LIVE, lambda s_, i=i: s_.__setitem__("m", b.hmr(s_["tr"][i - 1])))
        rung(i, lambda s_, i=i: b.cz(ua(i, s_), s_["sB"], cond=s_["m"]))
    op(LIVE, lambda s_: A.close("tree", s_["tr"]))
    for bb, k, j in reversed(om):
        op(m_om[bb, k, j], lambda s_, bb=bb, k=k, j=j: b.cx(omw[k][j], s_["som"][bb]))
    return plan


def emit_sweep(b, E, A, omw, ebw, abw):
    """Emit the sweep: one gate-table row of M bits per distinct gate
    vector (the forward and unwind passes share rows; row r gates
    iteration t on bit r * M + t), then the repeat(M) body over the held
    pivots and the som / tree windows of allocator A.
    """
    plan = sweep_plan(b, A, omw, ebw, abw)
    rows = {}
    for vec, _ in plan:
        if vec is not None:
            rows.setdefault(vec.tobytes(), vec)
    rows.setdefault(LIVE.tobytes(), LIVE)
    row_of = {key: r for r, key in enumerate(rows)}
    table = b.declare_table(np.concatenate(list(rows.values())).astype(np.uint8).tolist())
    with b.repeat(M) as it:

        def gate(vec):
            return b.gated(table, it + row_of[vec.tobytes()] * M)

        s_ = {"sA": E.sA, "sB": E.sB, "sT": E.sT, "som": A.open("som", B)}
        for vec, fn in plan:
            if vec is None:
                fn(s_)
            else:
                with gate(vec):
                    fn(s_)
        with gate(LIVE):
            A.close("som", s_["som"])


# =========================================================================
# BM (stage 2 forward, stage 10 journaled reverse)
# =========================================================================
def flag_test(E, src, gate, ctl):
    """ctl <- [src != 0] AND parity(gate) (gate=() folds to the bare
    nonzero test): the tree window, the (B-1)-rung zero-test ladder, the
    flag AND, and the rung teardown.
    local: every zero-test tree here has gadgets.mcx's n-1 Toffoli cost,
    but must hold its flag live across later gates and journal each step
    for the reverse pass; mcx computes and releases inside one call and
    records nothing.
    """
    with eq_test(E, src, 0) as top:
        E.mac(one_rt((top,), 1, gate, 1, ctl))


class BM:
    """The q-form conditional-swap BM forward (stage 2) and its journaled
    reverse (stage 10).  Round 0 and round N-1 are peeled; the general
    round is a sequence of step methods over explicit round state.  run()
    makes the BM journal J current, so every step records
    into it; JROUND holds the
    journal index at the seed start and at every round start.
    """

    def __init__(self, E, A, syn, Dw, dW):
        self.b, self.E, self.A, self.J = E.b, E, A, Journal()
        self.J.recording = False
        self.pools = {}  # round -> window ids its journal touches
        self.syn, self.Dw, self.dW = syn, Dw, dW
        self.SIG = [BankReg(A, "sig") for _ in range(W + 1)]  # sigma_0..sigma_W
        self.BB = [BankReg(A, "bb") for _ in range(W + 2)]  # B bank (fused-Q: wrap at W+1)
        # STK_CAP-slot LIFO push stack, STKW wires per slot: B value lanes + a
        # TAGB-bit round tag.  The tag records the pushing round and gives
        # every round's control flag an erase anchor at the round tail --
        # after the d complement, ctl == dW[SGN] AND NOT [top tag == r] -- so
        # no flag outlives its round.
        self.STK = [BankReg(A, "stk", STKW) for _ in range(STK_CAP)]
        assert max(CAPS) <= len(self.STK), ("push cascade would overrun STK", max(CAPS))
        self.JROUND = [0]

    def round_head(self, r, bb_l):
        """Open round r (inside its bm/rNN phase block): journal mark, the
        round's cascade stack slots, and the fused-Q bank rotation (a
        relabel).  Returns the rotated bank.
        """
        self.A.round = r
        self.A.log = self.pools.setdefault(r, set())
        self.JROUND.append(self.J.mark())
        # Lazy-materialization invariant: a register touched mid-round may
        # only take free-list ids that strictly earlier rounds' windows freed
        # (those return before the descending replay reaches them).  At the
        # bank/ctl touch sites this holds because the round's own freed ids
        # are always reconsumed by the windows still open there; the cascade
        # runs after every window closed, so its stack depths materialize at
        # the round head instead.  reclaim fails closed on any breach.
        for s in self.STK[: CAPS[r]]:
            s.mat()
        return [bb_l[W + 1]] + bb_l[: W + 1]

    def discrepancy(self, r):
        """Dw <- sum_{i <= IMAXV[r]} sigma_i S_{r-i} (the truncated window)."""
        rtD = []
        for i in range(IMAXV[r] + 1):
            bil_terms(rtD, KT, spec_plain(self.SIG[i]), spec_plain(self.syn[r - i]), self.Dw)
        self.E.mac(rtD)

    @contextmanager
    def quotient(self, ctl, bb_l):
        """Yield the round quotient m without a stored accepted discrepancy.
        Invariant (README): sigma_0 * DEL == 1 on every
        correct-path round, DEL the accepted discrepancy of the q-form
        recursion, so the recursion's m = D'/DEL' is
          ctl=0: m = Delta * sigma_0 = p,
          ctl=1: m = DEL_old / Delta = 1/(Delta * sigma_0) = inv(p).
        Exit erases m (a function of p, inv(p), ctl), the chain and inv(p),
        then p.  Dw = Delta on every branch there; p = Delta * sigma_0_old,
        and sigma_0_old now sits in bb_l[0] (ctl=1) or SIG[0] (ctl=0, where
        Delta != 0 forces bb_l[0] == 0); on ctl=1 the update left SIG[0] =
        1/Delta, so Delta*(bb_l[0] + SIG[0]) = p + ctl:
          p == Dw * (bb_l[0] ^ SIG[0]) ^ ctl*[1]   (one KT claim).
        """
        E = self.E
        with opened(E, "pq", B) as pv:
            E.mac(prod_rt(pv, self.Dw, self.SIG[0]))
            with opened(E, "dinv", B) as dv, chain_live(E, pv, dv), opened(E, "m", B) as mq:
                # m = p ^ ctl*(p ^ inv(p)): B free copies + B CCX
                rtM = []
                for t_ in range(B):
                    rtM.append(("L", (pv[t_],), 0, (mq[t_],)))
                    rtM.append(norm_term((ctl,), 0, tuple(sorted((pv[t_], dv[t_]))), 0, (mq[t_],)))
                E.mac(rtM)
                yield mq
                E.free(rtM)
            rtPe = []
            bil_terms(
                rtPe,
                KT,
                spec_plain(self.Dw),
                [frozenset((bb_l[0][j], self.SIG[0][j])) for j in range(B)],
                pv,
            )
            rtPe.append(("L", (ctl,), 0, (pv[0],)))
            E.free(rtPe)

    def bank_swaps(self, r, ctl, bb_l):
        """cswap(ctl) of the sigma and B banks over slots 0..KMSV[r]."""
        for k2 in range(KMSV[r] + 1):
            if r == 1 and k2 == 1:
                # provable identity: both branches leave sigma_1 ==
                # B_1 entering this swap (2-branch enumeration)
                continue
            for t_ in range(B):
                self.E.cswap(ctl, self.SIG[k2][t_], bb_l[k2][t_])

    def update(self, r, k0, mq, bb_l):
        """sigma_k ^= m * B_k over slots k0..KMMV[r] (one KT12 each)."""
        for k2 in range(k0, KMMV[r] + 1):
            self.E.xor(prod_rt(self.SIG[k2], mq, bb_l[k2]))

    def push(self, r, ctl=True):
        """Emit the fused-Q push cascade on pqc = dW[SGN] (d<0 forces ctl=0);
        ctl=False spells it with plain swaps (the unconditional r=N-1
        push, 0 Toffoli, no tag).  Extent CAPS[r] = min(DP extent,
        STK_CAP): the CLAMPED Fredkin ladder still acts on pairwise-distinct
        qubits; on a full stack it rotates STK[STK_CAP-1] into Dw (overflow
        drop, the E2 class); on a not-full stack it is the identical
        shift as the uncapped ladder.  Inter-slot swaps carry all STKW
        lanes; the Dw level is value-only.
        """
        E, STK = self.E, self.STK
        capr = CAPS[r]
        if not capr:
            return
        pqc = self.dW[SGN]

        def sw(x, y):
            if ctl:
                E.cswap(pqc, x, y)
            else:
                E.swap(x, y)

        for i2 in range(capr - 2, -1, -1):
            for t_ in range(STKW):
                sw(STK[i2][t_], STK[i2 + 1][t_])
        for t_ in range(B):
            sw(self.Dw[t_], STK[0][t_])
        if not ctl:
            return
        # tag the incoming entry with its round: the vacated slot's tag lanes
        # are |0> below overflow, so free CX from the push flag writes r
        # exactly (overflow shots are the E2 class)
        for bp in bitlist(r, TAGB):
            E.cx(pqc, STK[0][B + bp])

    def erase_dw(self, ctl):
        """Erase Dw.  Post-cascade Dw == ctl * Delta on every branch (d<0:
        pushed, Dw = 0; ctl=0, d>=0: Delta = 0), and ctl=1 left sigma_0 =
        1/Delta: Dw == ctl * inv(sigma_0), one chain on SIG[0] and a B-term
        claim (stack overflow is the E2 class).
        """
        E = self.E
        with opened(E, "wreg", B) as wr, chain_live(E, self.SIG[0], wr):
            E.free([norm_term((ctl,), 0, (wr[t_],), 0, (self.Dw[t_],)) for t_ in range(B)])

    def close_ctl(self, r, ctl):
        """d complement if ctl, then erase ctl (the caller closes its
        window).  Post-complement dW[SGN] = [ctl OR pushed], and a push at
        this round iff the top tag reads r,
        so ctl == dW[SGN] AND NOT [top tag == r].  Rounds the DP proves
        push-free (and r=0) anchor on dW[SGN] alone, for free.  On a stack
        overflow the rotated-out tag corrupts the compare -- overflow shots
        are already the E2 class.
        """
        E, dW = self.E, self.dW
        for i in range(DWB):
            E.cx(ctl, dW[i])
        if CAPS_FULL[r] == 0:
            E.free([norm_term((dW[SGN],), 0, (), 1, (ctl,))])
        else:
            tag = self.STK[0][B:]  # materialized at round_head
            with opened(E, "eqt", TAGB - 1) as eqw:
                flips = bitlist(~r & TAGMASK, TAGB)
                for bp in flips:
                    E.x(tag[bp])
                eqr = E.ladder([((tag[i],), 0) for i in range(TAGB)], eqw)
                E.free([norm_term((dW[SGN],), 0, (eqw[TAGB - 2],), 1, (ctl,))])
                E.frees(eqr)
                for bp in flips:
                    E.x(tag[bp])

    def inc_d(self):
        """d += 1: the constant +1 ripple on dW (DWB-2 CCX, MBU-erased
        journaled carries).
        local: gadgets.arith.add has no constant-operand form (an all-None
        operand is a no-op) and no journal hook.
        """
        with opened(self.E, "inc", DWB - 2) as ic:
            self.E.inc1(self.dW, ic)

    def round0(self, bb_l):
        """r=0 needs no working discrepancy register: D(0)=S_1 lives in
        syn[0].  ctl = [S_1 != 0] off syn[0], and the round quotient is
        m = inv(S_1) on both branches (the chain maps 0 to 0).
        """
        E, SIG, syn0 = self.E, self.SIG, self.syn[0]
        with self.b.phase("bm/r00"):
            bb_l = self.round_head(0, bb_l)
            with opened(E, "ctl", 1) as (ctl,):
                flag_test(E, syn0, (), ctl)
                with opened(E, "m", B) as mq:
                    wins, rts = chain_fwd(E, syn0, mq)
                    # SIG[0]=|1>, SIG[1]=|0>, bb_l[0]=|0>, bb_l[1]=|1> are KNOWN basis
                    # states: the two register cswaps collapse to 4 free CX.
                    assert KMSV[0] == 1
                    E.cx(ctl, SIG[0][0])
                    E.cx(ctl, bb_l[0][0])
                    E.cx(ctl, SIG[1][0])
                    E.cx(ctl, bb_l[1][0])
                    # post-swap bb_l[0] value = ctl, bb_l[1] value = 1^ctl; m nonzero
                    # only when ctl=1, hence m*bb0 = m (free CX) and m*bb1 = 0 (skip).
                    for t_ in range(B):
                        E.cx(mq[t_], SIG[0][t_])
                    # erase m: post-update m == ctl * sigma_0 on both branches; the
                    # chain intermediates are functions of syn[0]
                    E.free([norm_term((ctl,), 0, (SIG[0][t_],), 0, (mq[t_],)) for t_ in range(B)])
                    chain_undo(E, wins, rts[:-1])
                self.close_ctl(0, ctl)
            self.inc_d()
        return bb_l

    def round(self, r, bb_l):
        """Emit a general round 0 < r < N-1."""
        with self.b.phase(f"bm/r{r:02d}"):
            bb_l = self.round_head(r, bb_l)
            with opened(self.E, "ctl", 1) as (ctl,):
                self.discrepancy(r)
                # ctl is set iff Dw != 0 and d >= 0
                flag_test(self.E, self.Dw, (self.dW[SGN],), ctl)
                with self.quotient(ctl, bb_l) as mq:
                    self.bank_swaps(r, ctl, bb_l)
                    self.update(r, 0, mq, bb_l)
                self.push(r)
                self.erase_dw(ctl)
                self.close_ctl(r, ctl)
            if r < N - 2:
                # +1 always -> d = ctl ? -d : d+1 (at r=N-2 the increment is
                # dropped: the counter has no later consumer and the complement
                # alone arms the erase anchor)
                self.inc_d()
        return bb_l

    def round_last(self, bb_l):
        """r=N-1: no ctl.  m = p, computed straight into the m window; the
        update skips slot 0 (with a length change due (w=W mass) zB occupies
        slots [N-1-r*, W] and never wraps to 0; otherwise M=0); the push is
        unconditional (push(ctl=False)), and r=N-1 writes no tag (no later
        round reads it).
        """
        r = N - 1
        with self.b.phase(f"bm/r{r:02d}"):
            bb_l = self.round_head(r, bb_l)
            self.discrepancy(r)
            with opened(self.E, "m", B) as mq:
                rtM = self.E.mac(prod_rt(mq, self.Dw, self.SIG[0]))
                self.update(r, 1, mq, bb_l)
                # m is a function of Dw and the untouched sigma_0
                self.E.free(rtM)
            assert CAPS[r] == STK_CAP
            self.push(r, ctl=False)

    def run(self):
        """Stage 2: the seed, round 0, rounds 1..N-2, round N-1; the journal
        is frozen afterwards (stages 3..9 are mirrored explicitly).
        """
        E, J, A = self.E, self.J, self.A
        J.recording = True
        A.round = -1  # seed segment: banks touched here free last
        A.log = self.pools.setdefault(-1, set())
        with E.using(J=J, A=A):
            with self.b.phase("bm/seed"):
                E.x(self.SIG[0][0])
                E.x(self.BB[0][0])
            bb_l = self.round0(list(self.BB))
            for r in range(1, N - 1):
                bb_l = self.round(r, bb_l)
            self.round_last(bb_l)
        self.final_bb_l = [bb_l[W + 1]] + bb_l[:W + 1]
        J.recording = False
        self.JROUND.append(J.mark())
        A.round = None
        A.log = None
        # every lazy register must have materialized inside BM: a first touch
        # in a later stage would have no mirror round to free it at
        assert all(reg.wires is not None for reg in self.SIG + self.BB + self.STK), (
            "lazy register untouched during BM"
        )

    def reverse(self):
        """Stage 10, the journaled measurement-based reverse pass.  Every BM
        step went through J with its conjugate (macs record
        release-or-recompute per the emit-time pebbling choice), so
        replaying the conjugates in reverse re-materializes each round's
        intermediates at the mirrored points and the replayed Fredkin
        cascades pop the stack at exactly the right moments.  The journal
        replays closures over hard wire ids: each round's window pool is
        re-materialized with reclaim right before its replay and freed
        right after, and every bank slot is freed after the replay of its
        first-touch round (its state there equals the forward state before
        that round: |0>).  The availability lines up by time symmetry -- an
        id a bank slot took at forward round f is released before the
        replays of rounds < f, exactly when their pools want it back -- and
        reclaim fails closed on any gap.
        """
        b, A, JR = self.b, self.A, self.JROUND
        freed = 0
        for k in range(len(JR) - 2, -1, -1):
            rr = k - 1  # round -1 = the seed
            with b.phase("reverse/seed" if rr < 0 else f"reverse/r{rr:02d}"):
                if JR[k + 1] > JR[k]:
                    with A.replay(self.pools.get(rr, ())):
                        self.J.reverse(since=JR[k], until=JR[k + 1])
                for reg in reversed(A.bank_log.get(rr, [])):
                    A.close(reg.name, reg.wires)
                    freed += 1
        assert freed == len(self.SIG) + len(self.BB) + len(self.STK), (
            "lazy registers leaked past the reverse",
            freed,
        )


# =========================================================================
# (Un)monicize (stages 3 and 9) and the in-place omega70 (stages 4 and 8)
# =========================================================================
# ARMORED: zflag = [sigma_0 == 0] XORed into W0[0] and SIG[0][0] around the
# slot loop makes the scaling invertible for ALL data (scale defaults to 1
# when sigma_0 = 0), so truncation deviations can never make phase garbage.


def w0_pass(E, SIG, mid, W0):
    """One w0 window: open the chain scratch, run the SIG[0] inversion
    chain intermediates forward, apply mid to the W0 term list (emit_mac
    to create W0, emit_free to erase it), then erase the intermediates
    and close the chain scratch.  The create and destroy windows differ
    only in mid.
    """
    wins, rts = chain_open(E, SIG[0], W0)
    E.macs(rts[:-1])
    mid(rts[-1])
    E.frees(rts[:-1])
    chain_close(E, wins)


def mon_slot(E, sig_i, scale, eraser, ymon):
    """sig_i <- scale * sig_i: ymon = scale * sig_i, a demolition erase of
    sig_i against eraser x ymon through Em's measurement-erase core
    (per-target hmr + conditioned-CZ repayment), then swap ymon into
    sig_i.  monicize: (scale, eraser) = (W0, SIG[0]); unmonicize: (SIG[0],
    W0).
    """
    E.emit_mac(prod_rt(ymon, scale, sig_i))
    E.emit_free(
        [
            ("P", (eraser[a_],), 0, (ymon[b_],), 0, (sig_i[k2],))
            for k2 in range(B)
            for (a_, b_) in MONB[k2]
        ]
    )
    for t_ in range(B):
        E.b.swap(ymon[t_], sig_i[t_])


def mon_stage(E, SIG, prefix, slot_range, forward):
    """One whole (un)monicize stage: create W0, arm the zflag armor,
    run the W scaling slots, disarm, erase W0.  monicize and
    unmonicize are the same block up to phase prefix, slot order, and
    the (scale, eraser) pair.
    """
    b = E.b
    # W0 and the zflag ladder (eq_test's two halves) outlive the phase that
    # opens them, so they open and close by hand in their first/last phase.
    with b.phase(prefix + "/w0+"):
        W0 = E.A.open("w0", B)
        w0_pass(E, SIG, E.emit_mac, W0)
    with b.phase(prefix + "/zflag+"):
        tr = E.A.open("tree", B - 1)
        rts = E.ladder(eq_rungs(SIG[0], 0), tr)
        top = tr[B - 2]
        b.cx(top, W0[0])
        b.cx(top, SIG[0][0])
    scale, eraser = (W0, SIG[0]) if forward else (SIG[0], W0)
    for i in slot_range:
        with b.phase(f"{prefix}/s{i:02d}"), opened(E, "ymon", B) as ym:
            mon_slot(E, SIG[i], scale, eraser, ym)
    with b.phase(prefix + "/zflag-"):
        b.cx(top, SIG[0][0])
        b.cx(top, W0[0])
        E.frees(rts)
        E.A.close("tree", tr)
    with b.phase(prefix + "/w0-"):
        w0_pass(E, SIG, E.emit_free, W0)
        E.A.close("w0", W0)


def omega_pass(E, macrt, prefix, order):
    """Emit the omega70 block term lists in order, one phase block each."""
    for bi in order:
        with E.b.phase(f"{prefix}/m{bi:02d}"):
            E.emit_mac(macrt[bi])


def build():
    """Build the circuit and assert its pins.  Returns (builder, Toffoli
    count); the occupancy/liveness profilers consume its wire_log.
    """
    # ---- registers, held scratch, allocators --------------------------------
    b = Builder()
    with b.phase("init"):  # profile-stage blocks (metadata; emit nothing)
        syn = list(declare_registers(b, [(f"s{r:02d}", B) for r in range(N)]).values())
        assert all(list(syn[r]) == list(range(B * r, B * r + B)) for r in range(N)), (
            "syn ids not contiguous"
        )  # qubits Br..Br+B-1

        ANC = []

        def An(n):
            qs = b.qubits(n)
            ANC.extend(qs)
            return qs

        Dw = An(B)  # fused-Q working discrepancy register
        dW = An(DWB)  # two's-complement d counter
        OMW = [syn[i] for i in range(W)]  # omega lives IN PLACE in syn[0..W-1]

        # Ownership wire-sets (claim/release bookkeeping; emits no op).  Every
        # long-lived wire joins exactly one named set with its full id list, so
        # the anonymous pool live - sum(owned) is empty from init to cleanup;
        # tools/occupancy_profile.py validates the discipline fail-closed.
        b.claim("syn", [w for reg in syn for w in reg])
        b.claim("dw", Dw)
        b.claim("dcount", dW)

        # Transient scratch round-trips through the allocator (A.open /
        # A.close).  Only state that persists between windows (discrepancies,
        # stack, controls, counters, the Em pivots) stays held; everything that
        # returns to |0> mid-circuit is freed there, so peak_qubits is the
        # measured max co-live width and the liveness profile shows the true
        # windows.  Journal replays (the BM reverse, the Psi route) run
        # closures over hard wire ids, so A logs the window ids a journal
        # touches and re-materializes exactly that set before its replay
        # (A.replay; reclaim fails closed when any id is not free).
        A = Alloc(b)

        sA = An(1)[0]
        b.claim("pivA", [sA])
        sB = An(1)[0]
        b.claim("pivB", [sB])
        sT = An(1)[0]
        b.claim("fanT", [sT])
        E = Em(b, sA, sB, sT, A)
        bm = BM(E, A, syn, Dw, dW)
        SIG = bm.SIG

    # Width ledger: N * B syn + (B + DWB + 3) held (Dw, dW, pivots) + the
    # lazily-materialized wires ((2W+3) B bank + STK_CAP tagged stack slots)
    # + the late BM/reverse rounds' window tiling.  The peak assert below
    # enforces the total.

    # ---- 1. zlin: free Z part sum_i Tr(u_i S_i) -----------------------------
    with b.phase("zlin"):
        for i in range(N):
            for bb in []:  # variable masks: no fixed syndrome-linear split
                b.z(syn[i][bb])

    # ---- 2. bm: BM forward (q-form conditional-swap, single-chain m) --------
    bm.run()
    BORROW = borrow_template(b, E, A, bm, dW) if os.environ.get("DQI_BORROW") else None

    # ---- 3. monicize: SIG[i] <- inv(sigma_0)*SIG[i] for i=1..W --------------
    mon_stage(E, SIG, "monicize", range(1, W + 1), True)

    # ---- 4. omega70 = sigma_m*S mod z^N, IN PLACE in syn[0..N-1] ------------
    # sigma_m is monic: the triangles' diagonals span sigma_1..sigma_{W-1},
    # the off-diagonal tail block's sigma_1..sigma_W (omega_rts).
    # Rows W..N-1 (the TAIL) end at 0 on every correct decode and host the
    # Bezout pair (stage 5).
    MACRT = omega_rts(SIG, syn)
    omega_pass(E, MACRT, "omega", range(len(MACRT)))

    # ---- 5. euclid: the half-degree Bezout pass (Psi route) -----------------
    # sigma_m = E(z)^2 + z O(z)^2 with E_i = sqrt(sigma_2i), O_i =
    # sqrt(sigma_2i+1): SIG[1..35] are square-rooted IN PLACE (a free linear
    # map) and the Euclid bank EB is E = [ONE, SIG2, .., SIG34], O = [SIG1, ..,
    # SIG35] (ONE is a fresh word holding 1 = E_0).  m = floor((35 - L)/2) =
    # (dW + 2) >> 2 (dW = the BM counter, +2 in place), and the barrel pad
    # shifts both halves up by m words so deg E' = 17 on every correct decode.
    # JSET journals the setup (ONE, the in-place square roots, dW += 2) so
    # stage 7 undoes it with one reverse.
    # PSI_IDS logs every scratch id the Psi journals touch; stages R and 7
    # hold exactly that set around their replays.
    with b.phase("euclid/setup"):
        ONE = A.open("one", B)
        EPS = A.open("zprec", 6)  # eps + 5-bit idx
        PRE = A.open("pre", 1)[0]
        PSI_IDS = set()
        JSET, JPAD, JPRE, JF, JSEED = (Journal(), Journal(), Journal(), Journal(), Journal())
        with E.using(J=JSET), A.logging(PSI_IDS):
            E.x(ONE[0])
            for i in range(1, W + 1):
                JSET.step(
                    lambda r=list(SIG[i]): linear_in_place(b, r, SQRT_ROWS),
                    lambda r=list(SIG[i]): linear_in_place(b, r, SQUARE_ROWS),
                )
            # dW += 2: a constant increment on bits 1..7 (ladder carries, MBU-erased)
            with opened(E, "inc", DWB - 3) as tr:
                E.inc1(dW[1:DWB], tr)
        EW = [ONE] + [list(SIG[2 * i]) for i in range(1, NH)]
        OWD = [list(SIG[2 * i + 1]) for i in range(NH)]
        MBITS = dW[2:7]  # m = (dW + 2) >> 2
    with b.phase("euclid/pad"), E.using(J=JPAD):
        pad_barrel(E, MBITS, EW, OWD)
    with b.phase("euclid/pre"), E.using(J=JPRE), A.logging(PSI_IDS):
        psi_pre(E, EW, OWD, PRE)
    with b.phase("euclid/F"), E.using(J=JF), A.logging(PSI_IDS):
        F_FIN, _G_FIN, RECS = psi_forward(E, EW, OWD, EPS[0], EPS[1:6])

    # ---- seed + R: the Bezout pair (a, b) grows in the TAIL + one EXT word --
    # a[(m+1) % 18] <- inv(h) (h = the final F pivot), then R = the journal
    # reverse of F interleaved with the cofactor recurrence: on exit EB holds
    # (E', O') again and (a, b) = (A, B) with A E' + B O' = z^m, deg <= 17.
    with b.phase("euclid/seed"):
        TAILW = [list(syn[i]) for i in range(W, N)]
        AW = TAILW[:NH]
        with E.using(J=JSEED), A.logging(PSI_IDS):
            psi_seed(E, F_FIN[NH - 1], MBITS, AW)
    with b.phase("euclid/R"):
        with A.replay(PSI_IDS, "psipool"):
            EXT = A.open("ext", B)  # opened under the replay hold: never a PSI_IDS id
            BWD = TAILW[NH:] + [EXT]
            JR, AFIN, BFIN = build_R(E, JF, RECS, AW, BWD, PRE, PSI_WIN, (EW, OWD))
            JR.forward()
        A.close("zprec", EPS)

    # ---- 6. sweep: root flag + rank-12 Psi pairing over the M points --------
    with b.phase("sweep"):
        emit_sweep(b, E, A, OMW, EW + OWD, AFIN + BFIN)

    # ---- 7. euclid_inv: R^-1, unseed, F^-1, un-pre, un-pad, un-setup --------
    # The zprec and psipool holds span R..setup, so they open and close by
    # hand (A.replay's two halves) in the first and last phase.
    PSI_POOL = sorted(PSI_IDS)
    with b.phase("euclid_inv/R"):
        A.reopen("zprec", EPS)
        A.reopen("psipool", PSI_POOL)
        JR.reverse()
        A.close("ext", EXT)
    with b.phase("euclid_inv/seed"):
        JSEED.reverse()
    with b.phase("euclid_inv/F"):
        JF.reverse()
    with b.phase("euclid_inv/pre"):
        JPRE.reverse()
    with b.phase("euclid_inv/pad"):
        JPAD.reverse()
    with b.phase("euclid_inv/setup"):
        JSET.reverse()
        A.close("psipool", PSI_POOL)
        A.close("pre", [PRE])
        A.close("zprec", EPS)
        A.close("one", ONE)

    # ---- 8. omega_restore: exact reversed block list ------------------------
    # each block's targets are disjoint from its operands, so re-emitting the
    # same term list un-XORs it; the reversed block order restores S exactly.
    omega_pass(E, MACRT, "omega_restore", reversed(range(len(MACRT))))

    # ---- 9. unmonicize: SIG[i] <- sigma_0*SIG[i] for i=W..1 -----------------
    mon_stage(E, SIG, "unmonicize", range(W, 0, -1), False)

    # ---- 10. reverse: the journaled BM reverse pass -------------------------
    if BORROW is not None:
        BORROW()
    bm.reverse()

    # ---- 11. cleanup ---------------------------------------------------------
    # Release-before-free through the only teardown path: every workspace set
    # returns its wires (full id lists, reverse claim order) ahead of the
    # frees; the N * B syndrome wires stay owned to the end as the declared
    # registers.
    with b.phase("cleanup"):
        b.release("fanT", [sT])
        b.release("pivB", [sB])
        b.release("pivA", [sA])
        b.release("dcount", dW)
        b.release("dw", Dw)
        b.recycle_all(reversed(ANC))

    # =========================================================================
    # Metrics gate, pre-emission: the pins under the measured window
    # schedule and STK_CAP = 1.
    # =========================================================================
    TOF = assert_metrics(b, __file__)

    return b, TOF


if __name__ == "__main__":
    emit_and_report(build()[0])

In [ ]:
%%writefile src/gen_dqi_v6/round2_gadgets.py
# SPDX-License-Identifier: Apache-2.0
"""Round-two reversible scheduling changes; original algebra: Gu et al.

The quotient swap schedule, early Euclid quotient parking, and early linear
root erasure were developed in this workspace (OpenAI Codex, 2026-10-23).
"""
from contextlib import contextmanager


def install(G,top_first=False,pebble_last=False,norm_pebble=False,compact_from=70):
    @contextmanager
    def quotient(self, ctl, bb_l):
        E=self.E
        if self.A.round>=compact_from:
            with compact_quotient(G,self,ctl,bb_l) as mq:yield mq
            return
        if pebble_last and self.A.round==68:
            with pebble_quotient(G,self,ctl,bb_l,full=norm_pebble) as mq:yield mq
            return
        with G.opened(E,'pq',G.B) as pv:
            E.mac(G.prod_rt(pv,self.Dw,self.SIG[0]))
            with G.opened(E,'dinv',G.B) as dv, G.chain_live(E,pv,dv):
                # The quotient occupies p's register; swap back before erasing
                # the inversion scaffold, whose source is the original p.
                pairs=G.word_pairs([pv],[dv])
                E.swaps(ctl,pairs)
                yield pv
                E.swaps(ctl,pairs)
            rt=[]
            G.bil_terms(rt,G.KT,G.spec_plain(self.Dw),
                [frozenset((bb_l[0][j],self.SIG[0][j])) for j in range(G.B)],pv)
            rt.append(('L',(ctl,),0,(pv[0],)))
            E.free(rt)
    G.BM.quotient=quotient
    if pebble_last or compact_from<70:
        original_run=G.BM.run
        def run(self):
            original_run(self)
            # An erased scratch window may be reused by a bank later in the
            # SAME round. Replay already owns that bank and restores it to
            # zero before the earlier scratch operations execute. Hold only
            # the other ids; free the bank once at its normal reverse point.
            for rr,regs in self.A.bank_log.items():
                for reg in regs:self.pools[rr].difference_update(reg.wires)
        G.BM.run=run

    # Preserve every journal mark's meaning: q_hi is parked in g17 before
    # q_lo's temporary window opens, and the recorded cofactor operand follows
    # its new location. No polynomial operation is dropped.
    import inspect
    if norm_pebble:
        G.inv_word=lambda E,src:compact_inverse(G,E,src)
    source=inspect.getsource(G.psi_forward)
    source=source.replace('''            with opened(E, "qh", B) as qh, opened(E, "ql", B) as ql:
                E.mac(prod_rt(qh, g[17], fi))''','''            with opened(E, "qh", B) as qh:
                E.mac(prod_rt(qh, g[17], fi))
                E.free(prod_rt(g[17], qh, f[17]))
                E.swaps(None, word_pairs([qh], [g[17]]))
            qh = g[17]
            with opened(E, "ql", B) as ql:''')
    source=source.replace('''                E.free(prod_rt(g[17], qh, f[17]))  # claim g17 == qh f17
''','')
    source=source.replace('''                E.swaps(None, word_pairs([qh, ql], [g[17], g[16]]))''',
                          '''                E.swaps(None, word_pairs([ql], [g[16]]))''')
    if top_first:
        G.top_poly_rt=lambda qh,ql,fw,tgt,high:top_poly_rt(G,qh,ql,fw,tgt,high)
        source=source.replace('''            qh = g[17]
            with opened(E, "ql", B) as ql:
                if j <= 16:
                    with opened(E, "t", B) as t:
                        trt = E.mac(prod_rt(t, qh, f[16]))
                        rtl = []
                        bil_terms(
                            rtl,
                            KT,
                            [frozenset((g[16][i], t[i])) for i in range(B)],
                            spec_plain(fi),
                            ql,
                        )
                        E.mac(rtl)
                        E.free(trt)
                else:
                    E.mac(prod_rt(ql, g[16], fi))''','''            qh = g[17]
            if j <= 16:
                targets = [[g[j - 1 + n]] for n in range(18 - j)]
                fw = [f[k] for k in range(j,17)]
                E.xor(top_poly_rt(qh,None,fw,targets,True))
            with opened(E, "ql", B) as ql:
                E.mac(prod_rt(ql, g[16], fi))''')
        source=source.replace('''                        polyprod_rt(
                            [ql, qh],
                            [f[k] for k in range(j, 17)],
                            [[g[j - 1 + n]] for n in range(18 - j)],
                        ),''','''                        top_poly_rt(qh,ql,fw,targets,False),''')
        assert 'with opened(E, "t", B)' not in source
    ns=dict(vars(G));exec(source,ns)
    G.psi_forward=ns['psi_forward']


@contextmanager
def compact_quotient(G,bm,ctl,bb_l):
    E=bm.E
    with G.opened(E,'pq',12) as pv:
        E.mac(G.prod_rt(pv,bm.Dw,bm.SIG[0]))
        with G.inv_word(E,pv) as dv:
            pairs=G.word_pairs([pv],[dv]);E.swaps(ctl,pairs)
            yield pv
            E.swaps(ctl,pairs)
        rt=[]
        G.bil_terms(rt,G.KT,G.spec_plain(bm.Dw),
            [frozenset((bb_l[0][j],bm.SIG[0][j])) for j in range(12)],pv)
        rt.append(('L',(ctl,),0,(pv[0],)));E.free(rt)


@contextmanager
def pebble_quotient(G,bm,ctl,bb_l,full=False):
    E=bm.E;A=bm.A
    with G.opened(E,'pq',12) as pv:
        E.mac(G.prod_rt(pv,bm.Dw,bm.SIG[0]))
        dv=A.open('dinv',12)
        wins,rts=G.chain_fwd(E,pv,dv)
        nq,m3,s3,yd=wins
        E.free(rts[2]);E.free(rts[1])
        A.close('s3d',s3);A.close('m3d',m3)
        if full:
            E.free(rts[0]);A.close('nqd',nq)
        pairs=G.word_pairs([pv],[dv]);E.swaps(ctl,pairs)
        yield pv
        E.swaps(ctl,pairs)
        # The inverse output needs only p and yd for MBU. Release its twelve
        # wires BEFORE recomputing the two erased three-bit norm windows.
        E.free(rts[4]);A.close('dinv',dv)
        if full:nq=A.open('nqd_re',6)
        nm=A.open('m3d_re',3);ns=A.open('s3d_re',3)
        nr=G.chain_rts(pv,[nq,nm,ns,yd],dv)
        if full:E.mac(nr[0])
        E.mac(nr[1]);E.mac(nr[2]);E.free(nr[3])
        E.free(nr[2]);E.free(nr[1]);E.free(nr[0] if full else rts[0])
        A.close('s3d_re',ns);A.close('m3d_re',nm)
        A.close('yd',yd);A.close('nqd',nq)
        rt=[]
        G.bil_terms(rt,G.KT,G.spec_plain(bm.Dw),
            [frozenset((bb_l[0][j],bm.SIG[0][j])) for j in range(12)],pv)
        rt.append(('L',(ctl,),0,(pv[0],)));E.free(rt)


def norm_rt(G,src,out):
    rt=[]
    G.bil_terms(rt,G.KT6,G.spec_rows(G.A0ROWS,src),G.spec_rows(G.A1ROWS,src),out)
    G.lin_rows_terms(rt,G.LNROWS,src,out)
    return rt


@contextmanager
def compact_inverse(G,E,src):
    """18-wire inverse schedule using Norm(x^-1) = Norm(x)^-1.

    Forward inverse is still 66 Toffolis. Erasing its output costs 39 rather
    than36, but no twelve-bit output overlaps an eighteen-bit scaffold.
    """
    A=E.A
    wins=[A.open(name,n) for name,n in G.CHAIN_WINS]
    nq,m3,s3,yd=wins
    # The dummy output is only used to construct term lists, never emitted.
    rts=G.chain_rts(src,wins,src)
    E.macs(rts[:4])
    E.free(rts[2]);E.free(rts[1]);E.free(rts[0])
    A.close('s3d',s3);A.close('m3d',m3);A.close('nqd',nq)
    out=A.open('inv',12)
    inv=G.chain_rts(src,wins,out)[4];E.mac(inv)
    E.free(norm_rt(G,out,yd));A.close('yd',yd)
    yield out
    y=A.open('yd_un',6);E.mac(norm_rt(G,out,y))
    dummy=[out[:6],out[:3],out[:3],y]
    E.free(G.chain_rts(src,dummy,out)[4]);A.close('inv',out)
    n=A.open('nqd_un',6);m=A.open('m3d_un',3);s=A.open('s3d_un',3)
    rt=G.chain_rts(src,[n,m,s,y],src)
    E.macs(rt[:3]);E.free(rt[3]);E.frees(rt[:3])
    A.close('s3d_un',s);A.close('m3d_un',m);A.close('nqd_un',n);A.close('yd_un',y)


_TOP_V={}

def certify_norm_identity(G):
    """Exhaust all4096 field elements over the actual emitted term lists."""
    src=list(range(12));wins=[list(range(12,18)),list(range(18,21)),list(range(21,24)),list(range(24,30))]
    out=list(range(30,42));normout=list(range(42,48))
    rts=G.chain_rts(src,wins,out);nr=norm_rt(G,out,normout)
    def emit(memory,rt):
        for term in rt:
            if term[0]=='L':v=term[2]^sum(memory[q] for q in term[1])%2
            else:v=(term[2]^sum(memory[q] for q in term[1])%2)&(term[4]^sum(memory[q] for q in term[3])%2)
            for q in term[-1]:memory[q]^=v
    for v in range(4096):
        memory=[(v>>q)&1 for q in range(12)]+[0]*36
        for rt in rts:emit(memory,rt)
        got=sum(memory[q]<<i for i,q in enumerate(out))
        assert got==G.ginv(v),(v,got,G.ginv(v))
        emit(memory,nr)
        assert [memory[q] for q in wins[-1]]==[memory[q] for q in normout],v
    return {'field_elements':4096,'inverse_correct':True,'norm_inverse_identity':True,
            'method':'evaluate exact Gu term lists and norm_rt on every GF4096 element, including zero'}

def top_poly_rt(G,qh,ql,fw,tgt,high):
    """Evaluate a linear-times-polynomial product at finite points + infinity.

    Emit infinity first so its high coefficient changes g16 before q_lo is
    divided from g16. This removes the redundant twelve-bit top-product copy.
    The remaining interpolation columns never write the high coefficient.
    """
    n=len(fw)+1
    if n not in _TOP_V:
        pts=[G.EXP[t] for t in range(n-1)]
        matrix=[[G.gpow(p,k) for k in range(n)] for p in pts]+[[0]*(n-1)+[1]]
        _TOP_V[n]=(pts,G.gf_matinv(matrix))
    pts,iv=_TOP_V[n]
    rt=[]
    for t in ([n-1] if high else range(n-1)):
        if high:
            sa=G.spec_plain(qh);sb=G.spec_plain(fw[-1])
        else:
            sa=[G.xfold([(ql,1<<b),(qh,G.cmrows(pts[t])[b])]) for b in range(G.B)]
            cf=[G.cmrows(G.gpow(pts[t],k)) for k in range(len(fw))]
            sb=[G.xfold([(fw[k],cf[k][b]) for k in range(len(fw))]) for b in range(G.B)]
        def targets(tb,t=t):
            acc={};u=G.tmask(tb)
            for k in range(n):
                v=G.gmul(iv[k][t],u)
                for word in tgt[k]:
                    for b in G.bitlist(v,G.B):acc[word[b]]=acc.get(word[b],0)^1
            return sorted(q for q,odd in acc.items() if odd)
        G.bil_terms(rt,G.KT,sa,sb,tws_of=targets)
    return [term for term in rt if term[-1]]


def lean_sweep(b,E,A,omw,ebw,abw,stream=False):
    import gu_port as G
    import kickmix as km
    from port_runtime import linear_in_place
    raw=b.k;gf=km.GF2Field(12)
    def arr(qs):return km.array([b.physical[q] for q in qs])
    def sum_into(words,target):
        raw.cx(arr([q for word in words for q in word]),arr([q for _ in words for q in target]))
    words=omw+ebw+abw
    logs=list(range(35))+G.LOG_EB+G.LOG_AB
    beta=gf.pow(gf.primitive_element,4094)
    steps=[gf.pow(beta,e) for e in logs]
    arrays=[arr(reg) for reg in words]
    for j in range(4095):
        val=A.open('scan_root',12);sum_into(ebw,val)
        tr=A.open('tree',11);rts=E.ladder(G.eq_rungs(val,0),tr);root=tr[-1]
        linear=[('L',tuple(w[k] for w in ebw),0,(val[k],)) for k in range(12)]
        E.free(linear);A.close('scan_root',val)
        om=A.open('scan_omega',1 if stream else 12);ps=A.open('scan_psi',12);loc=A.open('scan_located',1 if stream else 12)
        sum_into(abw,ps)
        linear_in_place(b,ps,G.SQUARE_ROWS)
        if stream:
            # Bilinearity permits one evaluator bit at a time. The AND
            # ancilla is repaid before reuse: still twelve Toffolis/point.
            for k in range(12):
                raw.cx(arr([w[k] for w in omw]),arr(om*len(omw)))
                raw.init_and(b.physical[root],arr(om),arr(loc))
                for h in range(12):
                    coeff=gf.mul(1<<k,1<<h)
                    for c in range(12):
                        if (coeff>>c)&1:
                            raw.push_condition(b.errors[j][c])
                            raw.cz(b.physical[loc[0]],b.physical[ps[h]])
                            raw.pop_condition()
                raw.del_and(b.physical[root],arr(om),arr(loc))
                raw.cx(arr([w[k] for w in omw]),arr(om*len(omw)))
        else:
            sum_into(omw,om)
            raw.init_and(b.physical[root],arr(om),arr(loc))
            raw.gf2_phase_by_product(arr(loc),arr(ps),mask=b.errors[j],field=gf)
            raw.del_and(b.physical[root],arr(om),arr(loc))
            sum_into(omw,om)
        linear_in_place(b,ps,G.SQRT_ROWS)
        sum_into(abw,ps)
        A.close('scan_located',loc);A.close('scan_psi',ps);A.close('scan_omega',om)
        A.reopen('scan_root',val);sum_into(ebw,val)
        E.frees(rts);A.close('tree',tr)
        sum_into(ebw,val);A.close('scan_root',val)
        if j+1<4095:
            for reg,factor in zip(arrays,steps):
                if factor!=1:raw.gf2_imul(factor,target=reg,field=gf)
        if j%512==0:
            import json,time
            p={'phase':'lean_sweep','position':j,'elapsed':time.monotonic()-b.started}
            print(json.dumps(p),flush=True)
            if b.progress:b.progress.write_text(json.dumps(p))
    for reg,factor in zip(arrays,steps):
        if factor!=1:raw.gf2_idiv(gf.pow(factor,4094),target=reg,field=gf)

def stream_sweep(b,E,A,omw,ebw,abw):
    return lean_sweep(b,E,A,omw,ebw,abw,stream=True)

In [ ]:
%%writefile src/gen_dqi_v6/run_gu_port.py
# SPDX-License-Identifier: Apache-2.0
"""Build the paper-source port with arbitrary classical phase masks.

The published source's fixed mask and loop emitter are replaced by an
in-place Chien scan. The decoder and all constants are generated from
source for the competition field, never translated from binary circuits.
"""
import argparse, hashlib, json, os, pathlib, time, traceback
os.environ['OMP_NUM_THREADS']='1'
import kickmix as km


def borrow_template(b,E,A,bm,dW):
    """Route the final template's two high zero words into a scratch loan.

    The common completed recurrence leaves words d and d+1, modulo 37,
    zero in the cyclic template bank. Routing these into fixed words 35
    and 36 permits a 24-qubit loan. This is an approximate support claim,
    tested by check_borrow.py and the official verifier, not an assertion
    for all inputs. The inverse route restores the remaining template.
    """
    import gu_port as G
    bb=bm.final_bb_l
    def swaps(v):
        m=(v+2)%37
        first=(m+35)%37;second=(m+36)%37
        pairs=[]
        if first!=35:pairs.append((35,first))
        if second==35:second=first
        if second!=36:pairs.append((36,second))
        return pairs
    def route(reverse=False):
        with b.phase('borrow/unroute' if reverse else 'borrow/route'):
            if os.environ.get('DQI_BORROW')=='affine':
                # Two 8-bit values form an affine line: choose a differing
                # pivot, XOR it into the other differing coordinates and
                # test the remaining seven coordinates. Thus each modulo-37
                # selector needs one six-AND ladder, no stored control bank.
                for m in range(37):
                    pairs=swaps(m-2)
                    if not pairs:continue
                    values=[v&255 for v in range(-1,69) if (v+2)%37==m]
                    assert len(values) in (1,2)
                    if len(values)==2:
                        u,v=values;diff=u^v;pivot=(diff&-diff).bit_length()-1
                        changed=[i for i in range(8) if i!=pivot and (diff>>i)&1]
                        for i in changed:E.cx(dW[pivot],dW[i])
                        bits=[dW[i] for i in range(8) if i!=pivot]
                        target=sum((((u>>i)&1)^(((u>>pivot)&1) if i in changed else 0))<<k
                                   for k,i in enumerate(i for i in range(8) if i!=pivot))
                    else:
                        changed=[];bits=dW;target=values[0]
                    with G.eq_test(E,bits,target) as ctl:
                        for x,y in (reversed(pairs) if reverse else pairs):
                            E.swaps(ctl,G.word_pairs([bb[x]],[bb[y]]))
                    for i in reversed(changed):E.cx(dW[pivot],dW[i])
            elif os.environ.get('DQI_BORROW')=='trie':
                mods=[m for m in range(37) if swaps(m-2)]
                ctlwords=A.open('loan_controls',len(mods))
                controls=dict(zip(mods,ctlwords))
                entries={v&255:(v+2)%37 for v in range(-1,69) if (v+2)%37 in controls}
                def visit(values,bit,parent=None):
                    if bit<0:
                        assert len(values)==1
                        E.cx(parent,controls[entries[values[0]]])
                        return
                    one=[v for v in values if (v>>bit)&1]
                    zero=[v for v in values if not (v>>bit)&1]
                    node=A.open('loan_trie',1)[0]
                    parity=0 if one else 1
                    rt=G.norm_term(() if parent is None else (parent,),1 if parent is None else 0,
                                   (dW[bit],),parity,(node,))
                    E.mac([rt])
                    if one:visit(one,bit-1,node)
                    if zero:
                        if one:
                            if parent is None:E.x(node)
                            else:E.cx(parent,node)
                            rt=G.norm_term(() if parent is None else (parent,),1 if parent is None else 0,
                                           (dW[bit],),1,(node,))
                        visit(zero,bit-1,node)
                    E.free([rt]);A.close('loan_trie',[node])
                visit(list(entries),7)
                for m,ctl in controls.items():
                    pairs=swaps(m-2)
                    for x,y in (reversed(pairs) if reverse else pairs):
                        E.swaps(ctl,G.word_pairs([bb[x]],[bb[y]]))
                visit(list(entries),7)
                A.close('loan_controls',ctlwords)
            else:
                for v in range(-1,69):
                    pairs=swaps(v)
                    if not pairs:continue
                    with G.eq_test(E,dW,v&255) as ctl:
                        for x,y in (reversed(pairs) if reverse else pairs):
                            E.swaps(ctl,G.word_pairs([bb[x]],[bb[y]]))
    route()
    loan=list(bb[35])+list(bb[36])
    A.close('borrowed_template',loan)
    def restore():
        A.reopen('borrowed_template',loan)
        route(reverse=True)
    return restore


def dynamic_sweep(b,E,A,omw,ebw,abw):
    import gu_port as G
    from port_runtime import linear_in_place
    raw=b.k;gf=km.GF2Field(12)
    def arr(qs):return km.array([b.physical[q] for q in qs])
    def sum_into(words,target):
        raw.cx(arr([q for word in words for q in word]),
               arr([q for _ in words for q in target]))
    names=[('scan_root',12),('scan_omega',12),('scan_psi',12),('scan_located',12)]
    val,om,ps,loc=[A.open(name,n) for name,n in names]
    all_words=omw+ebw+abw
    logs=list(range(35))+G.LOG_EB+G.LOG_AB
    beta=gf.pow(gf.primitive_element,4094)
    steps=[gf.pow(beta,e) for e in logs]
    arrays=[arr(reg) for reg in all_words]
    for j in range(4095):
        sum_into(omw,om);sum_into(ebw,val);sum_into(abw,ps)
        linear_in_place(b,ps,G.SQUARE_ROWS)
        with G.eq_test(E,val,0) as root:
            raw.init_and(b.physical[root],arr(om),arr(loc))
            raw.gf2_phase_by_product(arr(loc),arr(ps),mask=b.errors[j],field=gf)
            raw.del_and(b.physical[root],arr(om),arr(loc))
        linear_in_place(b,ps,G.SQRT_ROWS)
        sum_into(abw,ps);sum_into(ebw,val);sum_into(omw,om)
        if j+1<4095:
            for reg,factor in zip(arrays,steps):
                if factor!=1:raw.gf2_imul(factor,target=reg,field=gf)
        if j%256==0:
            p={'phase':'sweep','position':j,'positions':4095,'elapsed':time.monotonic()-b.started,
               'physical_qubits':len(b.physical)}
            print(json.dumps(p),flush=True)
            if b.progress:b.progress.write_text(json.dumps(p,indent=2))
    for reg,factor in zip(arrays,steps):
        if factor!=1:raw.gf2_idiv(gf.pow(factor,4094),target=reg,field=gf)
    for (name,_),qs in reversed(list(zip(names,[val,om,ps,loc]))):A.close(name,qs)


def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--name',required=True)
    ap.add_argument('--shots',type=int,default=2048)
    ap.add_argument('--salts',nargs='+',default=['dqi-gu-source-smoke-20261023'])
    ap.add_argument('--load',action='store_true')
    ap.add_argument('--derive-only',action='store_true')
    ap.add_argument('--borrow',action='store_true')
    ap.add_argument('--loan-routing',choices=['direct','trie','affine'],default='direct')
    ap.add_argument('--build-only',action='store_true')
    ap.add_argument('--expected-sha256')
    ap.add_argument('--lean',action='store_true')
    ap.add_argument('--stack',type=int,default=1)
    ap.add_argument('--wide',action='store_true')
    ap.add_argument('--stream',action='store_true')
    ap.add_argument('--top-first',action='store_true')
    ap.add_argument('--pebble-last',action='store_true')
    ap.add_argument('--norm-pebble',action='store_true')
    ap.add_argument('--wide-late',action='store_true')
    ap.add_argument('--compact-bm-from',type=int,default=70)
    a=ap.parse_args()
    out=pathlib.Path(os.environ['BRIDGE_OUT'])
    host=pathlib.Path('/home/admin_pc4/quantum/runs/ws_dqi')/a.name
    host.mkdir(parents=True,exist_ok=True)
    def report(obj,name='summary.json'):
        (out/name).write_text(json.dumps(obj,indent=2))
        print(json.dumps(obj),flush=True)
    from dqi import verify_submission_dqi as V
    t=time.monotonic()
    if a.load:
        circuit=km.Circuit.from_file(str(host/'dqi.kmb'),format='kmb')
    else:
        import gu_formula_derivation as F
        tables,timing=F.run_b12()
        assert F.certify_kt(tables['KT12'],F.FIELD)
        constants=json.loads((pathlib.Path(__file__).parent/'ref/gu_constants.json').read_text())
        constants.update(tables)
        if a.wide or a.wide_late:
            win=constants['bm_windows']
            for r in range(70):
                if a.wide_late and not a.wide and r<64:continue
                win['IMAXV'][r]=min(r,35,(r+1)//2+1)
                win['KMSV'][r]=min(r+1,35,r//2+2)
        constpath=out/'constants_competition.json'
        constpath.write_text(json.dumps(constants,indent=2))
        report({'field_modulus':F.CONTRACT['gf_poly'],'formulas':{k:len(v) for k,v in tables.items()},
                'derivation_seconds':timing},'formula_summary.json')
        if a.derive_only:return
        os.environ['DQI_CONSTANTS']=str(constpath)
        import gu_port as G
        G.emit_sweep=dynamic_sweep
        G.borrow_template=borrow_template
        if a.stack!=1:
            G.STK_CAP=a.stack
            G.CAPS=[min(c,a.stack) for c in G.CAPS_FULL]
        if a.lean:
            import round2_gadgets as R2
            if a.norm_pebble:report(R2.certify_norm_identity(G),'norm_certificate.json')
            R2.install(G,top_first=a.top_first,pebble_last=a.pebble_last,norm_pebble=a.norm_pebble,
                       compact_from=a.compact_bm_from)
            G.emit_sweep=R2.lean_sweep
            if a.stream:G.emit_sweep=R2.stream_sweep
        if a.borrow:os.environ['DQI_BORROW']=a.loan_routing
        b,tof=G.build()
        (out/'phase_stats.json').write_text(json.dumps(b.phase_stats,indent=2))
        (out/'liveness.json').write_text(json.dumps(b.live_profile,indent=2))
        b.k.free(b.measurement)
        circuit=b.k.finish_circuit()
        circuit.to_file(str(host/'dqi.kmb'),format='kmb')
    summary={'name':a.name,'args':vars(a),'path':str(host/'dqi.kmb'),
             'build_seconds':time.monotonic()-t,'qubits':circuit.num_qubits,
             'static_toffoli':circuit.operation_statistics().num_toffoli_like,
             'num_ops':len(circuit),'bytes':(host/'dqi.kmb').stat().st_size,'verification':{}}
    if a.expected_sha256:
        digest=hashlib.sha256()
        with (host/'dqi.kmb').open('rb') as f:
            for chunk in iter(lambda:f.read(8<<20),b''):digest.update(chunk)
        summary['rebuilt_sha256']=digest.hexdigest()
        summary['expected_sha256']=a.expected_sha256
        summary['reproducible']=digest.hexdigest()==a.expected_sha256
        report(summary)
        assert summary['reproducible'], 'Rebuilt circuit differs from verified artifact'
    report(summary)
    if a.build_only:return
    spec=V.ProblemSpecDQI.benchmark_instance(num_shots=a.shots)
    for salt in a.salts:
        report({'stage':'verify','salt':salt,'shots':a.shots},'progress.json')
        t=time.monotonic()
        try:r=V.verify(circuit,spec,salt=salt,max_workers=int(os.environ.get('BRIDGE_JOBS','1')))
        except Exception as e:
            traceback.print_exc();r={'status':'FAIL','error':str(e)}
        r.update(salt=salt,seconds=time.monotonic()-t)
        summary['verification'][salt]=r
        report(summary)
    report({'stage':'complete'},'progress.json')


if __name__=='__main__':main()

In [ ]:
%%writefile src/gen_dqi_v6/symbolic_runtime.py
"""Exact bounded symbolic partial evaluation for reversible kickmix emission.

Every form is an XOR of independent formal atoms, including constant atom 0.
An AND of forms is interned as a fresh atom; no independence assumption is
used to infer equalities. Oversized forms are forgotten into unique atoms,
which loses optimization opportunities but cannot create false equalities.
HMR returns a wire to zero and permits immediate physical reuse. Declared
input registers remain pinned. No input examples enter this compiler.
SPDX-License-Identifier: Apache-2.0
"""
import kickmix as km
import port_runtime as R

ZERO=frozenset()
ONE=frozenset((0,))
Base=R.Builder

class SymbolicBuilder(Base):
    def __init__(self):
        super().__init__()
        self.forms=[];self.pinned=set();self.atom_id=0;self.products={}
        self.elided={};self.folded={};self.max_native=0
        self.limit=48

    def atom(self):
        self.atom_id+=1
        return frozenset((self.atom_id,))

    def xor(self,a,c):
        v=a^c
        return v if len(v)<=self.limit else self.atom()

    def product(self,a,c):
        if not a or not c:return ZERO
        if a==ONE:return c
        if c==ONE or a==c:return a
        if a^c==ONE:return ZERO
        # Canonical complemented inputs share the same nonlinear atom.
        ac=0 in a;cc=0 in c
        aa=a-ONE;bb=c-ONE
        key=(aa,bb) if tuple(sorted(aa))<=tuple(sorted(bb)) else (bb,aa)
        if key not in self.products:self.products[key]=self.atom()
        v=self.products[key]
        if ac:v=self.xor(v,bb)
        if cc:v=self.xor(v,aa)
        if ac and cc:v=self.xor(v,ONE)
        return v

    def wire(self,q):
        if self.physical[q] is None:
            self.physical[q]=self.k.alloc_qubits(1)[0]
            if self.forms[q]==ONE:self.k.x(self.physical[q])
        return self.physical[q]

    def qubit(self):
        if self.free_qubits:
            q=self.free_qubits.pop();self.free_set.remove(q)
            assert self.forms[q]==ZERO
        else:
            q=len(self.physical);self.physical.append(None);self.forms.append(ZERO)
        assert q not in self.active
        self.active.add(q)
        return q

    def _constant(self,q):
        if q in self.pinned or self.forms[q] not in (ZERO,ONE):return
        if self.physical[q] is not None:
            if self.forms[q]==ONE:self.k.x(self.physical[q])
            self.k.free(self.physical[q]);self.physical[q]=None

    def recycle(self,q):
        assert q in self.active
        if self.forms[q]:
            # Retain actual reset semantics on every approximate path.
            self.k.reset(self.wire(q))
        if self.physical[q] is not None:self.k.free(self.physical[q])
        self.physical[q]=None;self.forms[q]=ZERO
        self.active.remove(q);self.free_qubits.append(q);self.free_set.add(q)
        self.owners.pop(q,None)

    def _native(self,name,qs,cond):
        if cond is not None:self.k.push_condition(cond)
        self.gate_counts[name]=self.gate_counts.get(name,0)+1
        getattr(self.k,name)(*[self.wire(q) for q in qs])
        if cond is not None:self.k.pop_condition()

    def _gate(self,name,*qs,cond=None):
        if getattr(self,'suspended',False):
            self._native(name,qs,cond)
            if name not in ('z','cz','ccz'):
                changed=qs[-2:] if name in ('swap','cswap') else qs[-1:]
                self.invalidate(changed)
            return
        fs=[self.forms[q] for q in qs]
        def skip():self.elided[name]=self.elided.get(name,0)+1
        def fold(n,qq):
            self.folded[name+'->'+n]=self.folded.get(name+'->'+n,0)+1
            self._gate(n,*qq,cond=cond)
        if name in ('z','cz','ccz'):
            if any(f==ZERO for f in fs):skip();return
            keep=[q for q,f in zip(qs,fs) if f!=ONE]
            if len(keep)<len(qs):
                if not keep:
                    if cond is None:
                        # Global half-turn retained explicitly using one bit.
                        q=self.k.alloc_qubits(1)[0];self.k.x(q);self.k.z(q);self.k.x(q);self.k.free(q)
                    else:self.k.neg_if(cond)
                else:fold(('z','cz','ccz')[len(keep)-1],keep)
                return
            self._native(name,qs,cond);return
        if name in ('cx','ccx','cswap'):
            n=1 if name in ('cx','cswap') else 2
            if any(f==ZERO for f in fs[:n]):skip();return
            if name=='cx' and fs[0]==ONE:fold('x',qs[1:]);return
            if name=='ccx':
                if fs[0]==ONE:fold('cx',qs[1:]);return
                if fs[1]==ONE or fs[0]==fs[1]:fold('cx',(qs[0],qs[2]));return
                if fs[0]^fs[1]==ONE:skip();return
            if name=='cswap':
                if fs[1]==fs[2]:skip();return
                if fs[0]==ONE:fold('swap',qs[1:]);return
        if name=='swap' and fs[0]==fs[1]:skip();return
        if name=='swap' and cond is None and not any(q in self.pinned for q in qs):
            a,c=qs;self.physical[a],self.physical[c]=self.physical[c],self.physical[a]
            self.forms[a],self.forms[c]=fs[1],fs[0]
            skip();return
        self._native(name,qs,cond)
        if name=='x':updates={qs[0]:self.xor(fs[0],ONE)}
        elif name=='cx':updates={qs[1]:self.xor(fs[1],fs[0])}
        elif name=='ccx':updates={qs[2]:self.xor(fs[2],self.product(fs[0],fs[1]))}
        elif name=='swap':updates={qs[0]:fs[1],qs[1]:fs[0]}
        elif name=='cswap':
            delta=self.product(fs[0],self.xor(fs[1],fs[2]))
            updates={qs[1]:self.xor(fs[1],delta),qs[2]:self.xor(fs[2],delta)}
        else:raise ValueError(name)
        for q,f in updates.items():
            self.forms[q]=f if cond is None else self.atom()
            self._constant(q)

    def hmr(self,q):
        self.k.hmr(self.wire(q),self.measurement)
        self.forms[q]=ZERO
        self._constant(q)
        return self.measurement

    def invalidate(self,qs):
        # Direct native field operations bypass this adapter. Conservatively
        # forget their outputs; no value-based optimization may cross them.
        for q in qs:self.forms[q]=self.atom()

def declare_registers(b,decls):
    regs=R.declare_registers(b,decls)
    assert not b.forms
    b.forms=[b.atom() for _ in b.physical]
    b.pinned.update(q for v in regs.values() for q in v)
    return regs


In [ ]:
%%writefile src/gen_dqi_v6/ref/gu_classical.py
"""uncompute.classical — classical math shared by circuits and their checks.

Nothing here touches a Builder: these are the pure-Python models and
table-builders whose OUTPUTS enter emitted circuits (lookup tables, linear
layers, test vectors, reference values): GF(2^n) scalar arithmetic and
GF(2) linear-map helpers. One exp/log-table field (``GF2n``) serves every
finite-field circuit. ``poly`` is always the full reduction polynomial
including the x^n term (e.g. 0x13 for GF(16)).
"""


# ---- GF(2^n) scalar arithmetic ---------------------------------------------


def _clmul_mod(a, b, n, poly):
    """Return a * b in GF(2^n) mod ``poly`` (LSB-first schoolbook + reduce)."""
    prod = 0
    for i in range(n):
        if (a >> i) & 1:
            prod ^= b << i
    for bit in range(2 * n - 2, n - 1, -1):
        if (prod >> bit) & 1:
            prod ^= poly << (bit - n)
    return prod & ((1 << n) - 1)


class GF2n:
    """GF(2^n) mod ``poly``, by exp/log tables over a primitive element.

    ``gen`` is the smallest primitive element, so ``gen == 2`` (the element
    x) whenever x is primitive. ``exp[k] = gen^k`` for k in [0, 2*order):
    the table repeats once, so ``exp[log[a] + log[b]]`` needs no modulo.
    ``log[a]`` is the exponent k with gen^k = a, for nonzero a. ``log[0]`` is 0, a
    placeholder: callers test for zero first. Hot loops may index ``exp``
    and ``log`` directly. The constructor raises ValueError when ``poly``
    is not irreducible of degree n. Tables cost O(2^n), so n stays small.
    """

    __slots__ = ("exp", "gen", "log", "n", "order")

    def __init__(self, n, poly):
        self.n = n
        self.order = order = (1 << n) - 1  # multiplicative group order
        for gen in range(1, 1 << n) if poly >> n == 1 else ():
            exp, x = [1], gen
            while x != 1 and len(exp) < order:
                exp.append(x)
                x = _clmul_mod(x, gen, n, poly)
            if x == 1 and len(exp) == order:
                break
        else:
            raise ValueError(f"poly {poly:#x} is not irreducible of degree {n}")
        self.gen = gen
        self.exp = exp + exp
        self.log = [0] * (order + 1)
        for k, v in enumerate(exp):
            self.log[v] = k

    def mul(self, a, b):
        """Return a * b."""
        if a == 0 or b == 0:
            return 0
        return self.exp[self.log[a] + self.log[b]]

    def pow(self, a, e):
        """Return a^e for any integer e. 0^0 is 1; 0^e is 0 otherwise."""
        if a == 0:
            return int(e == 0)
        return self.exp[self.log[a] * e % self.order]

    def inv(self, a):
        """Return a^-1. The inverse is totalized: 0 maps to 0."""
        return self.exp[self.order - self.log[a]] if a else 0

    def sqrt(self, a):
        """Return the unique square root, a^(2^(n-1))."""
        return self.exp[(self.log[a] << (self.n - 1)) % self.order] if a else 0

    def trace(self, a, k=None):
        """Return a + a^2 + a^4 + ... + a^(2^(k-1)).

        ``k=None`` means k = n: the absolute trace, 0 or 1 for every a.
        For a in the subfield GF(2^k), k dividing n, the same sum is the
        subfield trace to GF(2).
        """
        t = 0
        for _ in range(self.n if k is None else k):
            t ^= a
            a = self.mul(a, a)
        return t


# ---- GF(2) linear-map helpers ----------------------------------------------


def parity(x):
    """GF(2) parity of an int's bits."""
    return x.bit_count() & 1


def f2_rows_of(fn, n):
    """Return the rows of the GF(2)-linear map ``fn`` on n-bit ints.

    Row i is a packed int: bit j is set when input bit j feeds output
    bit i (LSB-first, the layout ``gadgets.linear_xor`` takes). The call
    checks linearity over all 2^n inputs and raises AssertionError on a
    nonlinear ``fn``, which would otherwise yield a silently wrong
    matrix.
    """
    cols = [fn(1 << j) for j in range(n)]
    for v in range(1 << n):
        image = 0
        for j in range(n):
            if (v >> j) & 1:
                image ^= cols[j]
        assert fn(v) == image, "map is not GF(2)-linear"
    return [sum((((cols[j] >> i) & 1) << j) for j in range(n)) for i in range(n)]


def f2_matinv(rows, n):
    """Inverse of the n x n GF(2) matrix ``rows`` (LSB-first packed ints)
    by partial-pivot Gauss-Jordan; asserts invertibility.
    """
    A = list(rows)
    I = [1 << i for i in range(n)]
    for col in range(n):
        piv = None
        for r in range(col, n):
            if (A[r] >> col) & 1:
                piv = r
                break
        assert piv is not None, "singular matrix"
        A[col], A[piv] = A[piv], A[col]
        I[col], I[piv] = I[piv], I[col]
        for r in range(n):
            if r != col and ((A[r] >> col) & 1):
                A[r] ^= A[col]
                I[r] ^= I[col]
    return I


def f2_transpose(rows, n):
    """Transpose of the n x n GF(2) matrix ``rows`` (LSB-first packed
    ints).
    """
    return [sum(((rows[r] >> c) & 1) << r for r in range(n)) for c in range(n)]

In [ ]:
%%writefile src/gen_dqi_v6/ref/gu_linear.py
"""GF(2) linear-layer synthesis.

Source: conventional Gauss-Jordan row elimination, emitted as the reversed
elimination schedule — the textbook in-place CNOT/SWAP realization of an
invertible F2 matrix, as used across the finite-field literature. ``linear_xor`` is the textbook
out-of-place form: one CNOT per set matrix entry.
"""


def linear_in_place(b, qs, rows):
    """Apply the invertible GF(2) matrix ``rows`` to register ``qs`` in
    place: qs <- rows * qs. ``rows[i]`` is the packed-int row i (bit j =
    column j, little-endian to match register order).

    Synthesis is leftmost-pivot Gauss-Jordan elimination recorded and
    emitted in reverse: 0 Toffoli, only CX and SWAP; no ancillas. Raises
    (via the identity assert) if the matrix is singular.
    """
    n = len(qs)
    a = list(rows)
    assert len(a) == n
    ops = []
    for col in range(n):
        pivot = next(r for r in range(col, n) if (a[r] >> col) & 1)
        if pivot != col:
            a[col], a[pivot] = a[pivot], a[col]
            ops.append(("swap", col, pivot))
        for r in range(n):
            if r != col and (a[r] >> col) & 1:
                a[r] ^= a[col]
                ops.append(("cx", col, r))
    assert a == [1 << i for i in range(n)], "singular matrix"
    for kind, i, j in reversed(ops):
        (b.swap if kind == "swap" else b.cx)(qs[i], qs[j])


def linear_xor(b, src, dst, rows):
    """XOR the GF(2) matrix-vector product onto a second register:
    dst ^= rows * src. ``rows[i]`` is the packed-int row i (bit j =
    column j), for example from :func:`uncompute.classical.f2_rows_of`.

    Emits one CX per set matrix entry in row-major order: for each dst
    bit i, for each src bit j. 0 Toffoli; no ancillas. ``src`` and
    ``dst`` must be disjoint.
    """
    for i, row in enumerate(rows):
        for j in range(len(src)):
            if (row >> j) & 1:
                b.cx(src[j], dst[i])

In [ ]:
%%writefile src/gen_dqi_v6/ref/gu_constants.json
{
 "KT12": [
  [
   [
    0,
    8,
    9
   ],
   [
    0,
    8,
    9
   ],
   [
    0,
    7,
    10,
    11
   ]
  ],
  [
   [
    6,
    7,
    9,
    10
   ],
   [
    6,
    7,
    9,
    10
   ],
   [
    1,
    2,
    3,
    4,
    8,
    9,
    11
   ]
  ],
  [
   [
    0,
    6,
    7,
    8,
    10
   ],
   [
    0,
    6,
    7,
    8,
    10
   ],
   [
    0,
    1,
    2,
    3,
    4,
    7,
    8,
    9,
    10
   ]
  ],
  [
   [
    0,
    1,
    2,
    3,
    4,
    5,
    7,
    8,
    10,
    11
   ],
   [
    0,
    1,
    2,
    3,
    4,
    5,
    7,
    8,
    10,
    11
   ],
   [
    0,
    1,
    4,
    5,
    7,
    9,
    11
   ]
  ],
  [
   [
    6,
    8,
    9
   ],
   [
    6,
    8,
    9
   ],
   [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    11
   ]
  ],
  [
   [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    9,
    10,
    11
   ],
   [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    9,
    10,
    11
   ],
   [
    0,
    2,
    3,
    6,
    9
   ]
  ],
  [
   [
    0,
    2,
    3,
    5,
    7,
    8,
    10
   ],
   [
    0,
    2,
    3,
    5,
    7,
    8,
    10
   ],
   [
    0,
    1,
    3,
    5,
    8,
    9,
    10,
    11
   ]
  ],
  [
   [
    1,
    2,
    4,
    5,
    11
   ],
   [
    1,
    2,
    4,
    5,
    11
   ],
   [
    2,
    4,
    6,
    8,
    9,
    10
   ]
  ],
  [
   [
    0,
    1,
    3,
    4,
    7,
    8,
    10,
    11
   ],
   [
    0,
    1,
    3,
    4,
    7,
    8,
    10,
    11
   ],
   [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    11
   ]
  ],
  [
   [
    0,
    1,
    3,
    4,
    6,
    8,
    11
   ],
   [
    0,
    1,
    3,
    4,
    6,
    8,
    11
   ],
   [
    1,
    4,
    8
   ]
  ],
  [
   [
    1,
    2,
    4,
    5,
    6,
    10
   ],
   [
    1,
    2,
    4,
    5,
    6,
    10
   ],
   [
    3,
    10,
    11
   ]
  ],
  [
   [
    0,
    2,
    3,
    5,
    8,
    10,
    11
   ],
   [
    0,
    2,
    3,
    5,
    8,
    10,
    11
   ],
   [
    1,
    3,
    4,
    8,
    10,
    11
   ]
  ],
  [
   [
    5,
    6,
    7,
    9,
    11
   ],
   [
    5,
    6,
    7,
    9,
    11
   ],
   [
    0,
    3,
    5,
    6,
    7,
    8,
    11
   ]
  ],
  [
   [
    7,
    8,
    11
   ],
   [
    7,
    8,
    11
   ],
   [
    0,
    1,
    2,
    4,
    6,
    7,
    9,
    10
   ]
  ],
  [
   [
    5,
    6,
    8,
    9
   ],
   [
    5,
    6,
    8,
    9
   ],
   [
    1,
    2,
    3,
    4,
    5,
    8,
    9,
    10,
    11
   ]
  ],
  [
   [
    0,
    4,
    6,
    7,
    9,
    11
   ],
   [
    0,
    4,
    6,
    7,
    9,
    11
   ],
   [
    1,
    3,
    6,
    8,
    9,
    11
   ]
  ],
  [
   [
    2,
    3,
    5,
    8,
    9,
    10
   ],
   [
    2,
    3,
    5,
    8,
    9,
    10
   ],
   [
    1,
    2,
    4,
    6,
    10,
    11
   ]
  ],
  [
   [
    0,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    10,
    11
   ],
   [
    0,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    10,
    11
   ],
   [
    2,
    3,
    4,
    8,
    9,
    10
   ]
  ],
  [
   [
    1,
    2,
    3,
    4,
    7,
    8,
    10,
    11
   ],
   [
    1,
    2,
    3,
    4,
    7,
    8,
    10,
    11
   ],
   [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    9,
    10
   ]
  ],
  [
   [
    3,
    7,
    8,
    9,
    10
   ],
   [
    3,
    7,
    8,
    9,
    10
   ],
   [
    0,
    1,
    2,
    6,
    11
   ]
  ],
  [
   [
    1,
    2,
    4,
    9,
    11
   ],
   [
    1,
    2,
    4,
    9,
    11
   ],
   [
    3,
    4,
    5,
    9,
    10,
    11
   ]
  ],
  [
   [
    0,
    1,
    2,
    3,
    6,
    8,
    9,
    10
   ],
   [
    0,
    1,
    2,
    3,
    6,
    8,
    9,
    10
   ],
   [
    1,
    4,
    5,
    6,
    8,
    10
   ]
  ],
  [
   [
    2,
    5,
    7
   ],
   [
    2,
    5,
    7
   ],
   [
    0,
    3,
    5,
    9,
    11
   ]
  ],
  [
   [
    0,
    1,
    3,
    5,
    6,
    7,
    8,
    9,
    10
   ],
   [
    0,
    1,
    3,
    5,
    6,
    7,
    8,
    9,
    10
   ],
   [
    0,
    1,
    3,
    4,
    6,
    8,
    9,
    10,
    11
   ]
  ],
  [
   [
    0,
    2,
    3,
    4,
    5
   ],
   [
    0,
    2,
    3,
    4,
    5
   ],
   [
    0,
    1,
    3,
    5,
    8,
    10,
    11
   ]
  ],
  [
   [
    2,
    3,
    5,
    6,
    8,
    11
   ],
   [
    2,
    3,
    5,
    6,
    8,
    11
   ],
   [
    6,
    9,
    10
   ]
  ],
  [
   [
    0,
    4,
    6,
    8,
    11
   ],
   [
    0,
    4,
    6,
    8,
    11
   ],
   [
    0,
    1,
    3,
    5,
    6,
    8,
    9,
    11
   ]
  ],
  [
   [
    1,
    2,
    4,
    6,
    7,
    8,
    9,
    10,
    11
   ],
   [
    1,
    2,
    4,
    6,
    7,
    8,
    9,
    10,
    11
   ],
   [
    1,
    3,
    5,
    6,
    10,
    11
   ]
  ],
  [
   [
    3,
    6,
    9,
    10,
    11
   ],
   [
    3,
    6,
    9,
    10,
    11
   ],
   [
    1,
    2,
    3,
    4,
    5,
    7,
    8,
    9
   ]
  ],
  [
   [
    1,
    2,
    3,
    4,
    7,
    8
   ],
   [
    1,
    2,
    3,
    4,
    7,
    8
   ],
   [
    2,
    4,
    6,
    7,
    8,
    9,
    10,
    11
   ]
  ],
  [
   [
    0,
    1,
    3,
    5,
    6,
    7,
    8,
    9,
    10,
    11
   ],
   [
    0,
    1,
    3,
    5,
    6,
    7,
    8,
    9,
    10,
    11
   ],
   [
    0,
    2,
    4,
    7,
    9,
    10,
    11
   ]
  ],
  [
   [
    2,
    5,
    8,
    9,
    10
   ],
   [
    2,
    5,
    8,
    9,
    10
   ],
   [
    2,
    4,
    7,
    8,
    11
   ]
  ],
  [
   [
    0,
    1,
    2,
    3,
    6,
    7,
    11
   ],
   [
    0,
    1,
    2,
    3,
    6,
    7,
    11
   ],
   [
    0,
    8,
    9,
    10
   ]
  ],
  [
   [
    0,
    2,
    5,
    8,
    9,
    10
   ],
   [
    0,
    2,
    5,
    8,
    9,
    10
   ],
   [
    1,
    3,
    4,
    8,
    11
   ]
  ],
  [
   [
    3,
    4,
    11
   ],
   [
    3,
    4,
    11
   ],
   [
    1,
    3,
    4,
    5,
    8,
    9
   ]
  ],
  [
   [
    0,
    2,
    3,
    4,
    5,
    8,
    9,
    10,
    11
   ],
   [
    0,
    2,
    3,
    4,
    5,
    8,
    9,
    10,
    11
   ],
   [
    5,
    9,
    11
   ]
  ],
  [
   [
    1,
    4,
    7,
    8,
    9
   ],
   [
    1,
    4,
    7,
    8,
    9
   ],
   [
    1,
    2,
    4,
    5,
    8,
    9,
    11
   ]
  ],
  [
   [
    2,
    3,
    10,
    11
   ],
   [
    2,
    3,
    10,
    11
   ],
   [
    0,
    1,
    5,
    7
   ]
  ],
  [
   [
    1,
    2,
    3,
    4,
    7,
    8,
    9,
    10,
    11
   ],
   [
    1,
    2,
    3,
    4,
    7,
    8,
    9,
    10,
    11
   ],
   [
    0,
    2,
    4,
    7,
    8,
    9,
    11
   ]
  ],
  [
   [
    0,
    1,
    2,
    4,
    5,
    7,
    10
   ],
   [
    0,
    1,
    2,
    4,
    5,
    7,
    10
   ],
   [
    0,
    2,
    4,
    5,
    7,
    8
   ]
  ],
  [
   [
    2,
    4,
    10
   ],
   [
    2,
    4,
    10
   ],
   [
    2,
    3,
    5,
    9
   ]
  ],
  [
   [
    0,
    1,
    5,
    7
   ],
   [
    0,
    1,
    5,
    7
   ],
   [
    0,
    3,
    4,
    7,
    8,
    9
   ]
  ]
 ],
 "KT6": [
  [
   [
    0,
    1,
    2,
    5
   ],
   [
    0,
    1,
    2,
    5
   ],
   [
    3
   ]
  ],
  [
   [
    1,
    2,
    3,
    5
   ],
   [
    1,
    2,
    3,
    5
   ],
   [
    0,
    2,
    3,
    4,
    5
   ]
  ],
  [
   [
    0,
    3
   ],
   [
    0,
    3
   ],
   [
    0,
    2,
    4,
    5
   ]
  ],
  [
   [
    0,
    4,
    5
   ],
   [
    0,
    4,
    5
   ],
   [
    0
   ]
  ],
  [
   [
    3
   ],
   [
    3
   ],
   [
    1,
    2,
    3
   ]
  ],
  [
   [
    0,
    3,
    4,
    5
   ],
   [
    0,
    3,
    4,
    5
   ],
   [
    0,
    1,
    2,
    3
   ]
  ],
  [
   [
    0,
    2,
    3,
    5
   ],
   [
    0,
    2,
    3,
    5
   ],
   [
    2,
    3
   ]
  ],
  [
   [
    2,
    4,
    5
   ],
   [
    2,
    4,
    5
   ],
   [
    3,
    4
   ]
  ],
  [
   [
    0,
    3,
    4
   ],
   [
    0,
    3,
    4
   ],
   [
    2,
    4
   ]
  ],
  [
   [
    0,
    1,
    3,
    4,
    5
   ],
   [
    0,
    1,
    3,
    4,
    5
   ],
   [
    1,
    3
   ]
  ],
  [
   [
    1,
    4
   ],
   [
    1,
    4
   ],
   [
    1,
    3,
    5
   ]
  ],
  [
   [
    0,
    3,
    5
   ],
   [
    0,
    3,
    5
   ],
   [
    5
   ]
  ],
  [
   [
    2,
    3,
    5
   ],
   [
    2,
    3,
    5
   ],
   [
    0,
    4
   ]
  ],
  [
   [
    1,
    3,
    5
   ],
   [
    1,
    3,
    5
   ],
   [
    0,
    1,
    2,
    5
   ]
  ],
  [
   [
    1,
    2
   ],
   [
    1,
    2
   ],
   [
    1,
    2,
    4,
    5
   ]
  ]
 ],
 "KT3": [
  [
   [
    0
   ],
   [
    0
   ],
   [
    0,
    1,
    2
   ]
  ],
  [
   [
    1
   ],
   [
    1
   ],
   [
    0,
    2
   ]
  ],
  [
   [
    2
   ],
   [
    2
   ],
   [
    0
   ]
  ],
  [
   [
    0,
    1
   ],
   [
    0,
    1
   ],
   [
    1
   ]
  ],
  [
   [
    1,
    2
   ],
   [
    1,
    2
   ],
   [
    0,
    1
   ]
  ],
  [
   [
    0,
    2
   ],
   [
    0,
    2
   ],
   [
    2
   ]
  ]
 ],
 "CUBE3Q": [
  [
   [
    1,
    2,
    6
   ],
   [
    1,
    4,
    2
   ],
   [
    2,
    4,
    1
   ]
  ],
  [
   7,
   2,
   4
  ]
 ],
 "PAIR3": [
  [
   1,
   1
  ],
  [
   2,
   6
  ],
  [
   4,
   4
  ]
 ],
 "LIN": {
  "E12": [
   1,
   2677,
   1970,
   228,
   566,
   1180,
   2041,
   919,
   3758,
   2733,
   3851,
   1630
  ],
  "A0ROWS": [
   1217,
   456,
   680,
   912,
   508,
   3032
  ],
  "A1ROWS": [
   1386,
   2892,
   742,
   1696,
   3470,
   1906
  ],
  "LNROWS": [
   1769,
   472,
   320,
   3540,
   3998,
   604
  ],
  "SQ3R": [
   1,
   4,
   6
  ],
  "LMROWS": [
   41,
   52,
   14
  ]
 },
 "bm_windows": {
  "IMAXV": [
   0,
   1,
   2,
   2,
   3,
   3,
   4,
   4,
   5,
   5,
   6,
   6,
   7,
   7,
   8,
   8,
   9,
   9,
   10,
   10,
   11,
   11,
   12,
   12,
   13,
   13,
   14,
   14,
   15,
   15,
   16,
   16,
   17,
   17,
   18,
   18,
   19,
   19,
   20,
   20,
   21,
   21,
   22,
   22,
   23,
   23,
   24,
   24,
   25,
   25,
   26,
   26,
   27,
   27,
   28,
   28,
   29,
   29,
   30,
   30,
   31,
   31,
   32,
   32,
   33,
   33,
   33,
   34,
   34,
   35
  ],
  "KMSV": [
   1,
   2,
   2,
   3,
   3,
   4,
   4,
   5,
   5,
   6,
   6,
   7,
   7,
   8,
   8,
   9,
   9,
   10,
   10,
   11,
   11,
   12,
   12,
   13,
   13,
   14,
   14,
   15,
   15,
   16,
   16,
   17,
   17,
   18,
   18,
   19,
   19,
   20,
   20,
   21,
   21,
   22,
   22,
   23,
   23,
   24,
   24,
   25,
   25,
   26,
   26,
   27,
   27,
   28,
   28,
   29,
   29,
   30,
   30,
   31,
   31,
   32,
   32,
   33,
   33,
   33,
   34,
   34,
   35,
   35
  ],
  "KMMV": [
   0,
   1,
   1,
   2,
   2,
   3,
   3,
   4,
   4,
   5,
   5,
   6,
   6,
   7,
   7,
   8,
   8,
   9,
   9,
   10,
   10,
   11,
   11,
   12,
   12,
   13,
   13,
   14,
   14,
   15,
   15,
   16,
   16,
   17,
   17,
   18,
   18,
   19,
   19,
   20,
   20,
   21,
   21,
   22,
   22,
   23,
   23,
   24,
   24,
   25,
   25,
   26,
   26,
   27,
   27,
   28,
   28,
   29,
   29,
   30,
   30,
   31,
   31,
   32,
   32,
   33,
   33,
   34,
   34,
   35
  ]
 },
 "psi_windows": {
  "mul": {
   "17": [
    0,
    0
   ],
   "16": [
    1,
    1
   ],
   "15": [
    2,
    3
   ],
   "14": [
    3,
    4
   ],
   "13": [
    4,
    5
   ],
   "12": [
    5,
    6
   ],
   "11": [
    6,
    7
   ],
   "10": [
    7,
    8
   ],
   "9": [
    8,
    9
   ],
   "8": [
    9,
    10
   ],
   "7": [
    10,
    11
   ],
   "6": [
    11,
    12
   ],
   "5": [
    12,
    13
   ],
   "4": [
    13,
    14
   ],
   "3": [
    14,
    15
   ],
   "2": [
    15,
    16
   ],
   "1": [
    16,
    17
   ],
   "0": [
    0,
    18
   ]
  },
  "rz": {
   "17": [
    0,
    0
   ],
   "16": [
    0,
    0
   ],
   "15": [
    2,
    2
   ],
   "14": [
    3,
    3
   ],
   "13": [
    4,
    4
   ],
   "12": [
    5,
    5
   ],
   "11": [
    6,
    6
   ],
   "10": [
    7,
    7
   ],
   "9": [
    8,
    8
   ],
   "8": [
    9,
    9
   ],
   "7": [
    10,
    10
   ],
   "6": [
    11,
    11
   ],
   "5": [
    12,
    12
   ],
   "4": [
    13,
    13
   ],
   "3": [
    14,
    14
   ],
   "2": [
    15,
    15
   ],
   "1": [
    16,
    16
   ],
   "0": [
    17,
    17
   ]
  },
  "swap": {
   "17": [
    1,
    1
   ],
   "16": [
    2,
    2
   ],
   "15": [
    3,
    4
   ],
   "14": [
    4,
    5
   ],
   "13": [
    5,
    6
   ],
   "12": [
    6,
    7
   ],
   "11": [
    7,
    8
   ],
   "10": [
    8,
    9
   ],
   "9": [
    9,
    10
   ],
   "8": [
    10,
    11
   ],
   "7": [
    11,
    12
   ],
   "6": [
    12,
    13
   ],
   "5": [
    13,
    14
   ],
   "4": [
    14,
    15
   ],
   "3": [
    15,
    16
   ],
   "2": [
    16,
    17
   ],
   "1": [
    0,
    18
   ],
   "0": [
    0,
    17
   ]
  }
 }
}

In [ ]:
%%writefile src/gen_dqi_v6/ref/gu_contract.json
{
 "_provenance": "Task-defining derived constants shared by grader.py and every circuit. grader.py re-derives all of them from its inline definitions and asserts equality with this file at import. c_stream_hex = counter-mode SHA-256 over c_tag (block i = SHA-256(tag || u32le(i))), truncated to 6143 bytes = the 49140-bit fixup vector c (bit k = bit k%8 of byte k/8; the top 4 bits of the last byte are unused); weight_cum_2p16 = the exact DQI weight law |w_k|^2 of Jordan et al. (2025), arXiv:2408.08292, at this instance (m=4095, l=35, p=2016/4096), quantized to a /2^16 grid by largest-remainder rounding (entry k = P(weight <= k) * 2^16).",
 "gf_poly": 4331,
 "m": 4095,
 "n_synd": 70,
 "b": 12,
 "w_max": 35,
 "c_tag": "kickmix/rs-decode-4095/fixup-c/v1",
 "c_stream_hex": "0ab02275411d99401427df155f38df3d98dc8cd245edf6c545c44fbdfa580586048e10430b242a866358f792f5d4bf309aa3b5df457ec1be714f477bb3674d87bfe6f1e5be30e94aebbc685caf06703c256d6fde56cf1a71a44b26dd3a7f4d39f6ee5a39ed36d49a8e7c160a82ce95c221a9fbc6191b503d5c06976bb53878ace4318e6992c5f4932f6671be9cb705bf20019f44b90345f3b313d959710822c65345a2e2ec8bc270038e5582285d76fe1f53be1cccfe381d130665f33d1269c7aa3dd34878ea8d00369625db16895818731b0694f61f915d05c3f9b62b3008ce551248aa11fb39a8ea189aadb70f573e5f0ef48b8bb73acd01aad27897b0d5b18284284238aca81eed501a557a884a7091224220db6414a2b7c794c187ef6a883e7b375c5b0b0b1497d28aa0f35cfce019afeba54fbb98dc521625e100829b51b15495d8fdf1cfe2175d92668f20a62456ce4ef718b26360f2e83f44de3d452b8a131aac6dca7ef463c96510ca780ff5f76dd3e45894651052e7fae51261b65be4d020647db7f45b233d3bbb4d5941e0b7c303ceb613f3c2ed9b11127a1858f12416d6c957bc5cc58e7f1cb655790578fc64db0720c70190f95ed1c662b81b5e29405e9fa3d8e3ed252853155b5b487dd1c3c8b72771712cb8980b4844211d343db065d60d8e31040e0ebd116bd0a434b38eeede887faad8f4f2a616040b6d605648772342a892f8c571728216671979de87050e3d6a76888b24c3111f257fdb13e5ee92fcc2d851abe85ee3bd80c57404144c2a6fe4049a4a9e8ec74a4642cc8c8eb760b801f14692371add6298127eda4508e4ceb9cfd363d1b55bee8157b116f12c938d5b3f37e147c57f065155e7a4c290fbb4f616acabf0a8ade65294356083304276d56d35790d8bd9b7103f70276ff89d187c0fff73a8fe23cdab99d92f4a176725d43ee8043ce2bb852ec84f834419c4a2ef968aab729a62f09ca081e5041af453a53d7ab399c41cff68e56d30e04bdd4aca3eae09ea16ff96fc3f00fe200aee77e275a3b05f2d6c623b746950fd294fc60bb42826113566cab8358d1a435bdd025cb1905a301f7dd71957c9359f73feb93af6325b84a6628b5097d1ef7d6a8c05949ec66b7a682fcc69ce76c3d736d2e1db88c89c012af8565fd7cfbdcc1f9cb26c73d6af71a4020d6b5edab892539c62e98d81bd7ba114f960e45cc78ca2946d77cec0d232a8bf397e99d9e112c5a576887f718850303d3aba9d958b55b913c9d310a20ff7e946688e5f8f9f01230dc26900594e93581a93a08a2d294d7c9349dc211ef66b729810e3bac545d94aec168270882dbd225bce7af21585fd6d572ccb3ded7903f0211145fb0132d56c07a4834fdbe997c99ed61d9bf391754c75726fcf4896af7d205cabebafee5d8d34a9f7d091ce5557a7c984cbd73076dbc76526668476a9b55c610bb68614be2662ae4e9d4cd823c6fabd7ccf627e672e56dff4555987168ba7b87be4b69cfcd78f6e3a6a2d6faf054b2c5787bbcc92688bdbe59ba6f0925b9d38922df8398022f27dac378b1e19b616eb65215350707341357c5cc4f006b3e12b073a03a1bd8edd06af7e31562effd6ba689130ea72f39106d075d09312625897741a7a82f00a0f5f0b91618ed22da815ad6a3e446e7621b9889c96eedc287785751985ee5622ecd80bfe2324893ba8656c290b3faa2b7c8726515f5e605809a7ae4b6e04fb3fd6ef0165eb01b0b0e6d8dc38b3bb7e7f18a6c983611e8ee601ce55a8897aa244a68d3ddcc1241a28d039874c1c352fbd99e505eb00d5e490d2b033fd4e8472a94a8ef947766e5881e1bba7513e899785b82148119f06676350d4ddb8f35905c53b0e05781d5ea6aa819c5152e4b9d3e4662dfbb5295467af2246af9b3743229f5f4a13edaae3860a99309a7fbd7a5c91d9b373f0cc597e30b9194cb5b3d491c27c6cb227f71939d0d836c588830e39e3e6b8436bcc9af6c76f07513b37cdf63ef37e1c1045a5d333a0c19c3960d02e9cbc2c0b1c196bab8c8ebdb7267dc1daa55e4820a3f351c0aa0e97850d9c8bc887faf4805cff561e93b4c1a25bdb8dd100180360855298b7f7e64bdcfc158a5fef4bdd811832c8e6266accde6f332df161e7ddb2d1627546c966da622d0a6a1a045251dda529b4e10fdd14f9f3fc94ea45ea3b29a56564ec9d30fd8c0f736b5ab7d7a1a45175347d391328170d559dc0da5c6d661a4513bfab803fa1eed3d2c658575ac5de52e6ae74a1addfdcb2d598e96826fd625147491b9bfa2ba270d531600ce2791e32e7fdf4fd7ec1152ee6596894c6e917c6f94e1e79094a82bdac1d5ae4b10c7062256cbc754c2fdc6546080c7242eeca80418cb10fb089f721e9e3552b2f9d0686533d9113d338883f246ac8e5e8028108797952b041ed934c65ec9d541c0d4766682e260e735627b2463d899c1bffe99c9337cf959e2947d01a91cd2d88951029cbc4726991190d1fae04117573905c8619b0afaca7163ef533e9c3bf8c279602488cdc0636c114e28697799b0242cbb310c3f557b0d9e6428ebe3d635d201d5ee183ece5400056943ba890e490a219f4e33bed5809bfea2c74138bfcb3aec67fca982aca9890de419c6054cb0c95f0504c01000fd05e86efab211b4fc7a722712c4f483d72b440065ed3869230623924e3254b2ebb7ffa29f4beea6bba9dc32f3572f78b749fba1540009a0b657aae24fea3d77241ce786ac68618c44c84a19b5f650c1217be19568c28ce5a95316de929f113b876c480456e73eaf94d1c5e1abe81d1f70ade42b87df32f0f7dc56520e7cca6679efe320c0ec1340e9a3f249871f8c7bc36c2e57920e5a2fe96917adf1c9c56809a733e7179246429092cac1a9ca5ffe668fb23a69579de104d3e08e86ce1b0f7bed3abf800608fd02853221cdcafab1ff8dd6eb9aa6577af77df9b3e774a62094f23a6cda60b64a54d1577c1fe0526087b86b3aac84d67db4a937bac13469a330e1fc28974cd2d60c07221de0980b1d46a7102317a381f951c8f0029ab2199b0f5a3fbbb5947cd25c291bff0dd37b635628d2882ff3ed73ad8bdeff3c9444ff97ebe21bb543cc5c3f7f1a0608c5ae28b6a5b591f43895cc088a078ed465f0dd8b289219ee1d1d4df353087b2e96d18c400ce7a8ac5d1e30e3920c5aba753d67e198bd51c57fb90f20ccd518f44a3162213e50f67f712e16d5fab5cbe32f585a817b10cb3e68f53c71c04d9c5a400be6701178f1681a9dc073511118eb668819290697a180380eaae3b68af08fddefdfe233f8cdda37eb90ae6d625b0119d54722ba4ccf62d2b44b272aae1d98ab954130e59e7240cc72dd9566459a2af5dca2ad2663dc4b6b8a48dc68b33ac575ffbd46da82f1bc31223c7e280853a6a2c6667152a3875a6d77f41a54a1f9f6e518f50e6a4e1ae06e81e77fd5461fe87516818a9a67a388e6b8ef41b03d521fb1034a59ce4ada4037844b815f00fe1560aa615052a02c2ec5d288bb441b905aa8fd78c1978632a4288d010b4d2357ec48a2535b827ebab108f2eba026a136043874363916cbd300a02b7624163bd7d2dc4688de111a94d5bce2c0f9b332d3b0f647ca94359d80c7785709942fb553165eb24e09489ba8fcbcb8992b89ab24f715dab250fdb51ccfcf0c2f5aded499a106c45fabd22542baa0c722e78c3e8196136a8ff93c46a1bda8222b6119a90c61fce032b382c52b6f308160d6190c85dabd93b633291969ddcb30b13e3bbf401dcd656944523181086bf5cede9ce66e756c3190df3dcdf268fb9b07dc642d2497625c7f87625780a537318fb5f719d12c384d3baa14094de38c440784b0036ef5e4ec873486ccb35baeab9185f9f5a53227e2e61540c8df2db7428a74aab497e16f4ec9c6ff63083339ad26d2e674b33d7f737f3a391302fa8fa367e8b035d78bc9db74145c5e6d338c8706bf0f70f7f39402d6c29d69760985f4ab54c30b1df29c6d26d3d7ecfd02749718c1decc5ecd7f8b5b007d3a533568217aba921f1bb56bf3c0ec82f1f4ae28acab500338de9a704893b5a33965a295a37f11649a90ce06b37ca0086a7f600e92a3d92a0462726bef2b5cf16818d28ff44cbc6742c4d8c5db383d3e35767c0849797bc2d1d54b3d733128bcf61a2c1f81534abce1170909592c2a4318ab53dcdf6b2c722cf992502ee3d08d460540359df9120a36e0414cbd9e144ad4c62c51f8b5873c3d9c6d6c1ee2e7e58bc40a9eabed857f9c53e5c5174237f5ba8dbe14e6219fb2f85b6364f4c81c295852a54401b57ee9354589283509e880decb0e6aa301a1def95506622fc0d37b6b91ebf5483cf4ea53ec2d452a37b440825beb96070c9d5b9c01485ffa779ba8b06c308b5e50185e335cb4016c4ea86c77878b08fdda99c0bc7376c3845b8c563ea2baff72fe7bf4e5bf2a7522dc4f9527baeeaff5c77dc5b7faa9a731e15d261e264b5ed30c278553b620c43e75addc70145f0c08d5db037d14057e143e5ae55b3fd973f818b008c45b063ee7f0c4f28554af92d6881f1efbc612e6af829525b2dd0f1fe2dc8ab2e4c31deb697699e88ea4fb4d43eea3519bf91730d96a1b554e484d8d95b58be290897ff641b71fd2d786d5ce4062e78b28545b30874b9eb30d9d8b4189919a29ecbc470c907b21ad5964034bbbbcc608d665b028b75627d00c411477fa3fbc78c34727cad31de8db8789d547fbe4a9431c9e03bb61f9e2b993cf127d0bb8402d2da817e9ae35e86134a57d5ce7061f2628b4db18f5d61c4ca8e5955fa126419b45f108dd81969e197473ec06ec736b48b9dd8bb838149042cfd9d9fabfd6b75282936f2ba24ee84b85751a0b1f8892b7a29f0e4c4d35f4d3d6efad602f8794954811d6d59145b2cd0d3d9ee5a2bd88c82664a98aa6a137a6d5046c7c6b23ab544478771b14b8e147e6ecc9a3146b438fffd9c86c708eaf915afb0b05f7b33e666780a4345ee7782624cd3db51ae1dce6e079662ff726a2d54527407eb4b983b9b5592a765a74f5f35cd608b97dc534dec3cbefc1430182a568386ff09755a5034aa2a3b6ea291a0483651f75e8b92bbc3260097ab5121708df9369315e86cb1f1a0a666252263294569115fd5169f034c9ce46897d736d6616cc11d407866bdb0af5c826c534763e810b70b0cae816603fa99ff3148ae17ad2122048cd2384950842d8d4608b32aacc304d5ab4b93f038db83fc45fbc2205d33a780717fc75aaf463f3793952d9a01cc9bb5c7755e4817b2d81d6f0a7a069c2e8d034f26e75ff8f36fa35e82613048e899c022b3b5abe3e2228f41edb25dc7235090644bbcff1287861e6cb545f5903839132f9e397768fc5e3e9160bb188a10c79f511e201e12de36e1b237a47d46bd41293f61e53fa0706ca447d2883ccdea8bc2f8b53eda7d61523bc576ecb381b1139dd286eaaa79fcd22a911554391573776c3a8fbae6ebac203bd993787145b13540582a3e2150de032895e61f3544e1a32ba84bbf95e240d38fae0080528fb1ce9dfdba196e3939b5045c9d6e2ca3a4867e4facc9ef2d620330a235550a489692818542786e256264f3aeb72e565ae46e83f73354573122d7ebf9fe345bdb3d67069e2664c4cdc128bdc43809b1660239411809d4f1cad42a99821bdb111c6c4a6ba41b0145db04198f034803f56c3deee1f3a4334bd9ec37db9bea24c66f342be2512d07a959d1c00a7d36f4c44963df8b22b6b788fca5fccef9871d555af82b8b27a06497a60bf24052cfdbdca7d18731c4cc5c561ede0102fc3679699111c8def6ea7df0701983c3a3722b3fdd6438de336cf7421e2029e40ecbd63c8a2c2479607df55d68b8fcf62941fb3c0a3bc3c21e86fb4e6b4c058b5a5282607640000c2adffb79de8f25d3a25dabebc26b9fe5be6943cd854be634a4e07556a68dc7c999312d4e0f77eded93dc454d44028fe10345e778444bda4c7af6593b650ff63909740c66a34329453ee46eb5014b6ee339ae734eec6c065218f0f22eb5eb4fd2836b7175b9bffe4a85c23dad53e6ab3625ef316e34c31718f3171d38328a143bc28af58184054690e4f16e279a1e5de044c1ac8a1ff5518c13242ddbcbcb97e30c580356bd466f4156a98c66f9a24976765aa396043f3f76e2184e963892e5bb4288c181a48d50aca0da0e99642692948934fdd30d892a3ed225438253abce27472fe1e62c3f2e9bcd01dc34ada72f0ba0ee50210881e25a522ba2b40e5b55695f0e9a0e094957d33e326df57c3fa48ee6931253939b782b015dac6e591fbda9b78d5b83e08d3ac8cc638e0a802762292ce095e51b16ecceee95b5323ea8fc2254a74b5c737ec217301ea14cc44703dec852603056d6dab998e6c34f8f3276216bf10a2e114c56f11d471c49858a95c272dddfcd5c86ae19e4f516b5f7a0a0ab7f7d414de68c0167ea71bbfb1afcec51614a1c0e2c51a4196c6ed44834776d14eda80fc0b446c8c7870990a0c6d1840f3d224d23151bf7fd36bf7e859c6c4c75b1f83bd01171514a438971ffbe5691a9d5b26a121270243eb6183ee60ac4c0b386ad3ce4b5a6d8f2cdc7c61de41d278aa9c473470a54aa27a80d78e9e22e51225f7b47dc3ad97017cc571350a622a9f895f66690773f8770f724f5cd86af9ded84d26972c80b88afd8e02684ac49c41b7d8875daa700f75b56909eb0e4d7c07ce7fbc92d5686aaaab86d53e3b5e5ff626c9b15052f919071c095b044cc3c3ddf194abcc128c07012584c735058b93544f4f736059c37ef3b1c1203c82c33396523b5b3a48b193e6eaff66cf2c88ab8506cfcc90d1e84011d78aa201b63f75ab435eb3326a81edb1531f95dbc19defdce72a66db0530b290fd42d194abd3961af6af54e9ee4ce9ccf5f37f08ab3bb6a9187e4da74675ee1ed963be6dbafb355a106e1656073e433302ca784d59cc10285000cfda8466c27247c1fc4f35bd5daaf2727c6c8e15b10713ac46528be2bba8bb0c4c9bf57109750cef993e2691d12c15d226923027d6f38b735caf4842d2dd8328519591248e5e5e23a533975ffc71276678415c397bdf9b99aee7d853c66fb6ceb3fab015b05e5c51bc8adb2064c629ae9acdead63edde2f1f35f95f40dce4bbe537121356477ef6135da2039e08a1af7db68903ca2bcb14bba9963c5d9df629a0d80dd13162016ecec5ca8fb0794315e2f4293fb3e652e6ca659a13cf45286983da01110f5731577bd14ffb461ace093b5170b405bfed241a957950cc1f5fe3e3914293c86c7e9e786fd464637b4b5980cb99b2fd06e43a535dd665df1c903e9a059ae9e71562cf1ee5fa018b7b8f341a2cd19580d79f915f3fa1ae22ced811a628535f60b823969a3e9b99a926adb1f3696035876d07f149aee05d5b73a714854aa6a1d26ccc926eabd32fbe985e881ba461280c98eee75e87b9af58b447ef6b01da8b6703f832963ecb4ac70fc43098274fbc58e417bdcc17056eb2359cb2d81e439d658a81ba959147d385cd207b10c7d73cd59494d2e4c33f22e6e573d1ea6ec390f7f9c218df196536b5c71fbda6807f8847c6fb5a88ced5e53cac46ac1a9dce71328ce65d41e50d084cc585c431699be3af7587ff6ae1a0aa76da0c70cc0dc4aa7cf3bdf767cc40a23e38e4e2a307214bd3025fbb84876f9d764c08bce44776088ae9b02a502bddb463cfbb8b2eec454e0c1e5159ab5b7a872ec35cce3461ccebc03a666fbcea8d3afce5decc4f61375bee4125eaaf2386655ae2554f7323b1dc22b200da6d6cdb77b64bff6676aa9c45386050d8f6a7025e5b7e765661a06f6eee5e57a330601196a6476a76bca9c2563a4e8738fc183a44b541cc1c42d64856007af7fa2f71cae4504078d187ba875b01f3826e15e98c48946e5b68812268860126a31fc6661f8608d61f5bc44b556f5fd9223aed2f61d55c31c3569716bbc473711812563e566163a93be589a35e469711e1f3b356e5357def24ba087e77b16adf5df3054251731c318a97dd1494ac9b81dbed2050295f336515f4ef346e0ed172f519824a95319f3f6a017ddf495b014e24f37420c7b7125ea9d6766bdc2dafacc2d7aa8363894dc73c5d36008f563a692622c35a55fd100c1e59580fa75589d4fb77f27088b4257b0cb33a400d1be219b69f230b24b3397caaf5ead6d74de2493e5259ed4b1696ea8a9c47c9a1196e8b107282d30ee31527f5b5c7c8e225641b8d78228727e915080279b56c2687cc4b87734c17560911323eeb6dc0c79fa914166968f54099075349fd578dbde760b4642004f9276e5c0fd2a2e4e6f0107c463784c7d42f085232994bd70e924767bc8ccba72c7f4d19ebf8c7839f72911f5354f22b2a305da91ef5fb5c623eb1d55566b4e4d781e87a4df56eea7bd89d0d8c4e3e1ca647baa1c106751cdb66ddb72da2a69e9bb4b366515f3a6db3fa4c9d2bf097e3b5f14caadbf64b2bf119b1b951cd1379299e3af7823a8a4e03c12ef861ad48b13839867a8c3bdb7119cc79b0f0521533f52475dfab5d10c430cb1b57ee6ece34c3b085fbbdb4ae080c7f63a3596cc3088cad73e5fd2fa90d02311540ed63b07c1c3a0be14f88fbf5f5280d9c5ca02c0408f1a2be67837f1af3eb134e57fc8b1b30ac1be1",
 "c_popcount": 24492,
 "c_stream_sha256": "8fcd44e5a419ea29d84bb443d75f10dc9ac85dde2085b7851d92cd198eb3785a",
 "weight_cum_2p16": [
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  4,
  13,
  41,
  121,
  333,
  843,
  1963,
  4199,
  8230,
  14737,
  24027,
  35550,
  47605,
  57694,
  63723,
  65536
 ]
}

In [ ]:
%%writefile src/gen_dqi_v6/ref/GU_LICENSE

                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

In [ ]:
%%writefile src/gen_dqi_v6/ref/GU_COMMIT.txt
64b5a13aa249f1360846657392e6c66817b26d0f

In [ ]:
# notebook: hide_input
import contextlib
import selected_decoder
with (REPORTS/'dqi_source_build.log').open('w') as stream:
    with contextlib.redirect_stdout(stream):
        dqi_build_report=selected_decoder.build_selected(FINAL/'dqi.kmb',PUBLIC/'dqi')
save_json(REPORTS/'dqi_build.json',dqi_build_report)
assert dqi_build_report['num_qubits']==1789 and dqi_build_report['toffoli_count']==445229
assert dqi_build_report['sha256']=='529b075009f7758abeda83a63407f0c1feeb51dd70631ae4335dfd28721b7e08'
print(f"Decoder built from source: {dqi_build_report['num_qubits']}Q, {dqi_build_report['toffoli_count']:,} static Toffolis.",flush=True)
gc.collect()


In [ ]:
# notebook: hide_input
EXPECTED_CIRCUIT_MEMBERS={'dfthcsos/angle_k_0.kmb': '6cc00e27b56f78bb69ca40f45fe57b4a8328e0c4cc37f8af0b938a4b634c1d70', 'dfthcsos/angle_k_1.kmb': '108a225439c916348496d374e5ce5452a140cd9abeaa176a48ccd02acbf9c455', 'dfthcsos/angle_k_2.kmb': 'e3091b0c2779a398c998586c43eb734fdc4309c2243917f2f338e4547174fa53', 'dfthcsos/angle_k_3.kmb': 'a48d7f26fc1b170452142d3aa6938f4554e9c317f288cff376356565e456fa1a', 'dfthcsos/angle_k_4.kmb': 'b639446b1e3efc8c3eefc630ea6388d24ae47c771cfea29790f45796621cceb2', 'dfthcsos/angle_k_5.kmb': '2140c1061a2f2e01db067656818847b4e4ba892816cdb824fb5a523d80d3c896', 'dfthcsos/angle_k_6.kmb': '211ce282122182aeafd3ce43dff5c5e039a4dbbbbc577839e5e769287f948cf3', 'dfthcsos/angle_k_7.kmb': '579d46c3131c46e346bf8a9db0ea83f565fdf00036b84990ff0a113ce6258f83', 'dfthcsos/angle_k_8.kmb': 'f13bf934f9c2296da4274e1d70e76d0572f52ad7d2191ec20ae7b347bada01d3', 'dfthcsos/angle_k_9.kmb': '951a31c552bf3fd2ee02dbe83169a4d83f4bfda0f623b1886e4f36136032569d', 'dfthcsos/angle_k_10.kmb': '8d731cb690f1319109593d72e39a5cdf2ef1c1f08d8846441a589341ac9d00e1', 'dfthcsos/angle_k_11.kmb': '395283ed0d2216cfd5a9a5d03c47e63bc467b95df176686239a4a548035472be', 'dfthcsos/angle_k_12.kmb': 'ae22d3c2030c06fe7f95739933e7a2c54556eda5ef94fd7a224d8a18913d4c63', 'dfthcsos/angle_k_13.kmb': '9bff1f9855937a6a301dd588bad8997dec213bcdcbb678ac190678b227d6b53b', 'dfthcsos/angle_k_14.kmb': 'cc1a49a71cd866d9f47a5973723208e937baf3996e0b90ef8d904f0b58ca8223', 'dfthcsos/angle_k_15.kmb': '20a6ddcf3cb0b0a35e44f35b15de2ae4886e7d4a95ad3add97154f7712faec37', 'dfthcsos/angle_k_16.kmb': 'fd83f84b5147f3f19d8ca352daa069891a23a4694047784d28f02454110456e0', 'dfthcsos/angle_k_17.kmb': '2a0f204ded1c90d4d59acf4ee38e1f3897bf2aa4adf1c877dcf3f848b9f049b3', 'dfthcsos/angle_k_18.kmb': '80366632dbf956a576176552fac45f5b2a889bcc047582fb99e658942784e228', 'dfthcsos/angle_k_19.kmb': '86a060743dbc76ee0132a7e8a1e1a826c92dbe6e3498cd3392aee3d42f3909dc', 'dfthcsos/angle_k_20.kmb': '93dea10520740578e157449197e08f10013767dbe6be2a934ca6990513d241e2', 'dfthcsos/angle_k_21.kmb': '3833ed6285cd5b33dea84add2570f9ba902ba291495668286d49d5fc54045a31', 'dfthcsos/angle_k_22.kmb': '81c3212cd0bb1065645089799fcdc67a294558268cc0b62aa0ef356c6286f7de', 'dfthcsos/angle_k_23.kmb': '119377ffd98218f55cfa7e816411d5bc11fccd1f20636d75913d88e4e9715a6d', 'dfthcsos/angle_k_24.kmb': '45eef871de2aecdb13b17bd96cdc2ec2a6993b887dba24fa7ecc2c6cddcd6066', 'invsqrt.kmb': '97f12b109eb17346991ca33e5604963a11eff1b2098586e048d1fd103de55b7c', 'dqi.kmb': '529b075009f7758abeda83a63407f0c1feeb51dd70631ae4335dfd28721b7e08'}
for name,expected_sha in EXPECTED_CIRCUIT_MEMBERS.items():
    assert digest(FINAL/name)==expected_sha,name
print('All 27 deterministic circuit files reproduce the selected source build.',flush=True)


## Full verification and the complete objective

Each task is checked once with the unchanged official verifier and its full shot count: **200,540,928** for inverse square root, **102,400** for DQI, and **254,464** for chemistry. The portfolio is the geometric mean of reference-cost gains.

The ZIP is created only after all three checks pass. Every one of its **28 members** is rehashed from the archive. Producing the ZIP does not submit it: the native competition submission is a separate action linked to this exact successful notebook version.



In [ ]:
%%writefile metric_preflight.py
"""Official Kaggle metric v10 readers, unchanged except formatting.
Source: https://www.kaggle.com/code/metric/google-quantum-ai-metric
Adapter preflight_directory only checks the file contract; task verification is separate.
"""
import json
import math
import pathlib
import random
import stat
import sys
import time
import tokenize
import zipfile
import zlib
from collections.abc import Callable, Mapping
from typing import Any
import numpy as np
import pandas as pd
ALLOWED_INSTRUCTIONS = {'X', 'CX', 'CCX', 'Z', 'CZ', 'CCZ', 'Z_POW', 'NEG', 'SWAP', 'CSWAP', 'R', 'HMR', 'BIT_INVERT', 'BIT_STORE0', 'BIT_STORE1', 'PUSH_CONDITION', 'POP_CONDITION', 'INFO_TAG', 'DEBUG_PRINT'}
MAX_QUBIT_OR_BIT_ID = 2 ** 30 // 1536
MAX_NPZ_BYTES = 2 ** 26

class ParticipantVisibleError(Exception):
    pass

def read_circuit(path: pathlib.Path, max_registers: int) -> Any:
    """Return a submitted .kmx or .kmb circuit as a kickmix.Circuit; raise ValueError unless it is fit to verify.

    kickmix validates a file as it parses it, within limits that keep parsing cheap, but simulating
    takes memory that grows with the largest qubit and bit ids, which kickmix allows up to ~5e8,
    so those are capped at MAX_QUBIT_OR_BIT_ID. kickmix also allows a million registers, at
    ~110 bytes each even if empty, so a circuit may have at most `max_registers`, the most its
    benchmark needs. Every register must be named: the dfthcsos verifier crashes on an unnamed one.
    """
    import kickmix
    circuit = kickmix.Circuit.from_file(path, format=path.suffix.removeprefix('.'))
    unsupported = set(circuit.operation_statistics()) - ALLOWED_INSTRUCTIONS
    if unsupported:
        raise ValueError(f'unsupported instructions {sorted(unsupported)}')
    largest_id = max(circuit.num_qubits, circuit.num_bits) - 1
    if largest_id >= MAX_QUBIT_OR_BIT_ID:
        raise ValueError(f'qubit or bit id {largest_id} is above the limit of {MAX_QUBIT_OR_BIT_ID - 1}')
    if circuit.num_registers > max_registers:
        raise ValueError(f'{circuit.num_registers} registers, above the limit of {max_registers} for this benchmark')
    if not all((isinstance(name, str) for name in circuit.register_data)):
        raise ValueError('every register needs a name, given by its REGISTER line')
    return circuit

def read_npz(path: pathlib.Path, spec: Mapping[str, tuple[str, tuple[int | str, ...]]]) -> dict[str, np.ndarray]:
    """Return the arrays `spec` names from an .npz; raise ValueError unless each is a plain number array as specified.

    `spec` maps each key to the numpy dtype kinds it may have, such as 'f' for floats, and its shape,
    in which a string names a dimension of any positive size that must agree wherever it recurs.

    Each array's header is a Python literal of its dtype and shape, which np.load trusts: a few KB
    of deflated zeros can claim a terabyte-sized array, and the dtype can be anything numpy can
    describe, from structured and nested types to Python objects that need unpickling. So the
    headers are read first, and the arrays loaded only if each is a plain number type of its kind,
    not big-endian, with its shape, and they total at most MAX_NPZ_BYTES. Only format 1.0 headers,
    which numpy.savez writes, are accepted, and numpy refuses any over 10 KB before parsing it.
    """
    dimensions: dict[str, int] = {}
    total = 0
    with zipfile.ZipFile(path) as archive:
        members = {name.removesuffix('.npy'): name for name in archive.namelist()}
        missing = [key for key in spec if key not in members]
        if missing:
            raise ValueError(f'{path.name} has no {missing}')
        for key, (kinds, expected_shape) in spec.items():
            with archive.open(members[key]) as member:
                if np.lib.format.read_magic(member) != (1, 0):
                    raise ValueError(f'{key} in {path.name} is not in .npy format 1.0, which numpy.savez writes')
                shape, _, dtype = np.lib.format.read_array_header_1_0(member)
            if dtype.kind not in kinds or dtype.names is not None or dtype.subdtype is not None or (dtype.byteorder == '>'):
                raise ValueError(f'{key} in {path.name} has dtype {dtype}, not a plain number type of numpy kind {kinds!r}')
            matches = len(shape) == len(expected_shape) and all((size > 0 and size == (dimensions.setdefault(expected, size) if isinstance(expected, str) else expected) for size, expected in zip(shape, expected_shape, strict=True)))
            if not matches:
                raise ValueError(f'{key} in {path.name} has shape {shape}, not {expected_shape} with its named dimensions as in {dimensions}')
            total += math.prod(shape) * dtype.itemsize
    if total > MAX_NPZ_BYTES:
        raise ValueError(f'the arrays in {path.name} take {total:,} bytes, above the limit of {MAX_NPZ_BYTES:,}')
    with np.load(path, allow_pickle=False) as npz:
        return {key: np.array(npz[key]) for key in spec}

def find_one(root: pathlib.Path, name: str, description: str, is_dir: bool=False) -> pathlib.Path:
    """The one path called `name` (a glob) under `root`, where the submission unpacked; raise ParticipantVisibleError otherwise."""
    paths = [path for path in root.rglob(name) if path.is_dir() == is_dir and '__MACOSX' not in path.relative_to(root).parts]
    if len(paths) != 1:
        raise ParticipantVisibleError(f'The submission must contain exactly one {description}, found {len(paths)}.')
    return paths[0]


def preflight_directory(directory):
    root = pathlib.Path(directory).resolve()
    inv_path = find_one(root, 'invsqrt.km[xb]', 'inverse circuit')
    dqi_path = find_one(root, 'dqi.km[xb]', 'DQI circuit')
    chemistry = find_one(root, 'dfthcsos', 'chemistry directory', is_dir=True)
    inv = read_circuit(inv_path, 1)
    inv_qubits = inv.num_qubits
    del inv
    decoder = read_circuit(dqi_path, 4095 + 70)
    decoder_qubits = decoder.num_qubits
    del decoder
    angles = sorted(chemistry.glob('angle_k_*.km[xb]'))
    if not angles or {p.stem for p in angles} != {f'angle_k_{i}' for i in range(len(angles))}:
        raise ValueError('Angle circuits must have consecutive numbering with no duplicate formats')
    angle_widths = []
    for path in angles:
        circuit = read_circuit(path, 128)
        if circuit.num_qubits > 128:
            raise ValueError('ANGLE circuit exceeds the 128-qubit cap')
        angle_widths.append(circuit.num_qubits)
        del circuit
    specification = {
        'U': ('f', ('R', 'B', 76)), 'W': ('f', ('R', 'B', 'C')),
        'H2_I': ('f', ('R', 'C')), 'H1_I': ('f', ()),
        'Hsym_lower': ('f', (2926,)), 'tree_edges': ('iu', (75, 2))
    }
    npz_path = find_one(chemistry, 'submission.npz', 'factorization')
    arrays = read_npz(npz_path, specification)
    paths = [inv_path, dqi_path, *sorted(angles, key=lambda p: int(p.stem.rsplit('_', 1)[1])), npz_path]
    return {'status': 'PASS', 'scope': 'Official file readers and archive layout; separate task PASS required',
            'invsqrt_qubits': inv_qubits, 'dqi_qubits': decoder_qubits,
            'angle_qubits': angle_widths, 'factorization_shape': list(arrays['W'].shape),
            'archive_members': [str(p.relative_to(root)).replace('\\', '/') for p in paths]}


In [ ]:
if not RUN_FULL_VERIFICATION:
    raise RuntimeError('This release requires full verification before producing a submission archive.')

# Verify the changed arithmetic first; each complete official task check runs once.
inv_result = measured('invsqrt_full', lambda: IV.verify(inv_candidate,
    IV.ProblemSpecInvSqrt.for_n_p(10), salt=SALT, max_workers=WORKERS))
assert inv_result['status'] == 'PASS' and inv_result['verified_shots'] == 200_540_928
del inv_candidate; gc.collect()
print(f'Full inverse-square-root PASS: {inv_result["verified_shots"]:,} shots; {inv_result["wall_seconds"]:.1f}s.', flush=True)

chemistry_result = measured('chemistry_full', lambda: verify_chemistry(FINAL / 'dfthcsos', salt=SALT))
assert chemistry_result['status'] == 'PASS'
assert chemistry_result['verified_shots'] == 254_464
assert chemistry_result['num_circuits'] == EXPECTED_ANGLE_COUNT
print(f'Full chemistry PASS: {chemistry_result["verified_shots"]:,} shots; {chemistry_result["wall_seconds"]:.1f}s.', flush=True)

decoder = km.Circuit.from_file(str(FINAL / 'dqi.kmb'), format='kmb')
dqi_result = measured('dqi_full', lambda: DV.verify(decoder,
    DV.ProblemSpecDQI.benchmark_instance(), salt=SALT, max_workers=WORKERS))
assert dqi_result['status'] == 'PASS' and dqi_result['verified_shots'] == 102_400
del decoder; gc.collect()
print(f'Full DQI PASS: {dqi_result["verified_shots"]:,} shots; {dqi_result["wall_seconds"]:.1f}s.', flush=True)

implementation_path = Path('metric_preflight.py').resolve()
module_spec = importlib.util.spec_from_file_location('metric_preflight', implementation_path)
preflight = importlib.util.module_from_spec(module_spec)
sys.modules[module_spec.name] = preflight
module_spec.loader.exec_module(preflight)
file_contract = measured('archive_preflight', lambda: preflight.preflight_directory(FINAL))
assert file_contract['status'] == 'PASS'

results = {'invsqrt': inv_result, 'dqi': dqi_result, 'dfthcsos': chemistry_result}
cost_rows = []
for name, result in results.items():
    cost_rows.append({'Task': name, 'Status': result['status'], 'Qubits': result['num_qubits'],
                      'Measured cost': result['score'], 'Reference cost': REFERENCE_COSTS[name],
                      'Gain vs reference': REFERENCE_COSTS[name] / result['score'],
                      'Verifier seconds': result['wall_seconds'], 'Shots': result['verified_shots']})
cost_table = pd.DataFrame(cost_rows)
display(cost_table.style.format({'Measured cost': '{:,.3f}', 'Reference cost': '{:,.3f}',
                                'Gain vs reference': '{:.4f}x', 'Verifier seconds': '{:.1f}'}))
projected_score = math.prod(row['Gain vs reference'] for row in cost_rows) ** (1/3)
print(f'Projected combined score from this run: {projected_score:.6f}')
print('This is not an independently observed Kaggle leaderboard score.')
save_json(REPORTS / 'portfolio.json', {'projected_score': projected_score, 'task_costs': cost_rows,
                                    'observed_kaggle_score': None,
                                    'projection_salt': SALT, 'environment': environment})


In [ ]:
task_labels = {'invsqrt': 'Inverse square root', 'dqi': 'DQI decoder', 'dfthcsos': 'Chemistry'}
task_order = ['invsqrt', 'dqi', 'dfthcsos']
measured_gains = []
for task in task_order:
    result = results[task]
    assert result['status'] == 'PASS' and math.isfinite(result['score']) and result['score'] > 0
    measured_gains.append({'task': task, 'label': task_labels[task],
        'measured_cost': float(result['score']), 'reference_cost': REFERENCE_COSTS[task],
        'gain': REFERENCE_COSTS[task] / result['score'], 'qubits': int(result['num_qubits']),
        'verified_shots': int(result['verified_shots']),
        'mean_toffoli_count': float(result['toffoli_count']),
        'max_toffoli_count': result.get('max_toffoli_count'),
        'rotation_count': result.get('rotation_count'),
        'toffoli_equivalent': result.get('toffoli_equivalent')})

checked_projection = math.prod(row['gain'] for row in measured_gains) ** (1/3)
assert math.isclose(checked_projection, projected_score, rel_tol=1e-12)

fig, ax = plt.subplots(figsize=(9, 3.5))
bars = ax.barh([row['label'] for row in measured_gains], [row['gain'] for row in measured_gains],
    color=[COLORS['blue'], COLORS['green'], COLORS['orange']], height=0.56)
ax.axvline(1, color=COLORS['muted'], linewidth=1, linestyle='--')
for bar, row in zip(bars, measured_gains):
    ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height()/2,
        f"{row['gain']:.3f}x  |  {row['qubits']:,}Q", va='center')
ax.set(xlabel='Reference task cost / full-verified task cost',
    xlim=(0, max(row['gain'] for row in measured_gains) * 1.23),
    title='Measured component gains in this Run All')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.15)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURES / 'portfolio_measured_gains.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)

inverse_runtime_resources = {
    'Qubits': int(results['invsqrt']['num_qubits']),
    'Mean Toffolis': float(results['invsqrt']['toffoli_count']),
    'Max static Toffolis': int(results['invsqrt']['max_toffoli_count']),
    'Synthesized rotations': int(results['invsqrt']['rotation_count']),
    'Toffoli-equivalent cost per qubit': float(results['invsqrt']['toffoli_equivalent']),
    'Measured task cost': float(results['invsqrt']['score']),
}
print('Selected arithmetic: fresh full-official-verifier measurements')
display(pd.DataFrame([inverse_runtime_resources]).style.format(precision=3).hide(axis='index'))


comparison_snapshot_utc='2026-10-24'
score_evidence=pd.DataFrame([
    {'Revision':'Our separate Signed-Digit Memory','Value':5.77242,'Evidence':'Observed competition score','Run':'361502374'},
    {'Revision':'Our previous source V3','Value':6.280713692702279,'Evidence':'Full-verifier projection; not submitted','Run':'361586741'},
    {'Revision':'Mizero calibration V1','Value':6.37294,'Evidence':'Observed competition score','Run':'361675142'},
    {'Revision':'Kozykorpesh signed-gauge V3','Value':6.68108,'Evidence':'Observed competition score','Run':'361677445'},
    {'Revision':'This source V4','Value':checked_projection,'Evidence':'Current full-verifier projection before native submission','Run':'Current'}
])
display(score_evidence.style.format({'Value':'{:.6f}'}).hide(axis='index'))
print(f'V4 full-verifier projection: {checked_projection:.6f}. A competition score requires the separate native submission.')
print("The notebook badge may initially show V1's 5.67004; it identifies a different archive.")
save_json(REPORTS/'portfolio_comparison.json',{'revision':'V4','component_gains':measured_gains,
    'selected_inverse_resources':inverse_runtime_resources,'source_comparison_snapshot_utc':comparison_snapshot_utc,
    'score_evidence':score_evidence.to_dict('records'),'new_projected_score':checked_projection,
    'new_observed_competition_score':None,'new_submission_id':None})


In [ ]:
expected_angle_members = {f'dfthcsos/angle_k_{index}.kmb' for index in range(EXPECTED_ANGLE_COUNT)}
expected_archive_members = {'invsqrt.kmb', 'dqi.kmb', 'dfthcsos/submission.npz'} | expected_angle_members
assert set(file_contract['archive_members']) == expected_archive_members
assert len(file_contract['archive_members']) == 3 + EXPECTED_ANGLE_COUNT

manifest = []
for name in file_contract['archive_members']:
    path = FINAL / name
    manifest.append({'file': name, 'bytes': path.stat().st_size, 'sha256': digest(path)})
save_json(REPORTS / 'submission_manifest.json', manifest)
archive_path = WORK / 'submission.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for row in manifest:
        archive.write(FINAL / row['file'], row['file'])
with zipfile.ZipFile(archive_path) as archive:
    assert sorted(archive.namelist()) == sorted(row['file'] for row in manifest)
    for row in manifest:
        with archive.open(row['file']) as stream:
            assert hashlib.file_digest(stream, 'sha256').hexdigest() == row['sha256'], row['file']
print(f'Verified submission.zip: {archive_path.stat().st_size / 2**20:.2f} MiB; {len(manifest)} entries.')
print('Creating this archive does not submit it to the competition.')

# The verified archive and final members are reviewable outputs. Only the
# temporary build directory is removed, after every archived hash matched.
if PUBLIC.exists():
    resolved_work = WORK.resolve()
    resolved_build = PUBLIC.resolve()
    if (PUBLIC.is_symlink() or PUBLIC.name != 'build'
            or resolved_build.parent != resolved_work or resolved_build == resolved_work):
        raise ValueError('Temporary build cleanup must stay in the direct WORK/build directory.')
    if PUBLIC.is_dir():
        shutil.rmtree(resolved_build)
gc.collect()


In [ ]:
# notebook: hide_input
required_shots={'invsqrt':200540928,'dqi':102400,'dfthcsos':254464}
member_hashes={row['file']:row['sha256'] for row in manifest}
expected_members={'invsqrt.kmb','dqi.kmb','dfthcsos/submission.npz'} | {
    f'dfthcsos/angle_k_{i}.kmb' for i in range(EXPECTED_ANGLE_COUNT)}
assert set(member_hashes)==expected_members and len(member_hashes)==28
for name,row in results.items():
    assert row['status']=='PASS' and row['verified_shots']==required_shots[name]
    if name!='dfthcsos':assert row['circuit_sha256']==member_hashes[name+'.kmb']
assert results['dfthcsos']['num_circuits']==25
with zipfile.ZipFile(archive_path) as archive:
    assert set(archive.namelist())==expected_members and len(archive.namelist())==28
    for name in archive.namelist():
        with archive.open(name) as stream:
            assert hashlib.file_digest(stream,'sha256').hexdigest()==member_hashes[name]
release_receipt={'revision':'V4','revision_status':'FULL_VERIFIED_BEFORE_NATIVE_SUBMISSION',
    'new_projected_score':projected_score,'new_observed_competition_score':None,
    'new_submission_id':None,'competition_submission_performed_by_notebook':False,
    'all_full_checks':{key:{'status':r['status'],'score':r['score'],'verified_shots':r['verified_shots']}
        for key,r in results.items()},
    'development_salt':SALT,'archive_file':archive_path.name,'archive_bytes':archive_path.stat().st_size,
    'archive_sha256':digest(archive_path),'member_count':28,'member_sha256':member_hashes,
    'archive_member_hashes_rechecked':True,
    'phase_calibration_report_sha256':digest(REPORTS/'rotation_calibration.json'),
    'signed_gauge_report_sha256':digest(REPORTS/'signed_gauge.json'),
    'historical_badge_v1':5.67004,
    'comparison_snapshot':{'date_utc':'2026-10-24','kozy_v3_actual':6.68108,
        'kozy_submission_id':57583144,'mizero_v1_actual':6.37294,'mizero_submission_id':57582994},
    'source_only_inputs':'Official competition plus public FeMoCo numerical dataset V2; no Toolbox circuits or binary oracle.'}
save_json(REPORTS/'release_receipt.json',release_receipt)
print('V4 complete: three fresh full official PASS checks and all 28 archived hashes rechecked.')
print(f'Full-verifier projection: {projected_score:.6f}. Competition score is obtained only by native submission.')



In [ ]:
# Retain the license and attribution beside the generated artifacts.
(WORK / 'LICENSE').write_text('\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', encoding='utf-8')
(WORK / 'NOTICE').write_text("Quantum Circuits resource labs (2026)\nOriginal experiments, explanations and added implementations: anazemcev / ViberTeam.\nReleased under the Apache License, Version 2.0, subject to upstream notices.\n\nAttribution and dependencies\n\nGoogle Quantum AI competition authors: task specifications, reference circuits,\nKickmix simulator and unchanged official verification/file-reader code.\nhttps://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation\n\nRudra Bhaskar: direct polynomial phasing, retained coefficient table and\nlow-memory arithmetic adapted by the inverse-square-root implementation.\nhttps://www.kaggle.com/code/rudrabhaskar/direct-polynomial-phasing-verified-search?scriptVersionId=361364488\n\nLightningV08: compressed normalization flags.\nhttps://www.kaggle.com/code/lightningv08/quantum-circuits-memory-algebra\n\nSumit Chongder: open centered signed-digit inverse-square-root generator,\ncoefficient fitting, integer pipeline and arithmetic, retained and credited.\nhttps://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149\nAdded adaptation: split high-word phasing, allocation after normalization,\nfolded final carry, and borrowing a proven constant input bit as clean carry.\nThe later 92-qubit revision reduces curvature precision F2 from 10 to 9,\nregenerates the coefficient table, and certifies the complete promised input\ndomain. Its same-table ordinary-schedule comparison has 102 qubits; the\noriginal Sumit coefficient table and 103-qubit circuit remain credited above.\n\nKozykappa: physical ensemble compression, sparse column rotations, and prior\npublic work on reachable-control simplification and clean-wire allocation.\nhttps://www.kaggle.com/code/kozykappa/quantum-circuits-sparse-sos-factors-and-circuit\n\nKozykorpesh / Kozykappa: two-address banked XOR QROM construction, adapted\nfrom Sumit's public arithmetic and used for the chemistry angle lookup.\nhttps://www.kaggle.com/code/kozykappa/quantum-ai-compact-arithmetic-banked-qrom?scriptVersionId=361433617\nKirill Tymchishin v10: integration of our public arithmetic/decoder/factors\nwith that banked lookup, which supplies the comparison baseline for this release.\nhttps://www.kaggle.com/code/kirilltym/quantum-circuits-qrom-tower42?scriptVersionId=361453242\nAdded adaptation: omit zero difference-bank columns and apply an exact\nreversible CNOT change of output basis for dependent columns. Both tables\nare transformed, preserving the XOR oracle on arbitrary input targets.\n\nk256.dev: public compiled DQI decoder and starting chemistry factorization.\nThe decoder's original generator is not included in that publication.\nhttps://www.kaggle.com/code/k256net/claude?scriptVersionId=361337610\n\nAccessible public dependency mirror: Kirill Tymchishin, version 7, labelled\nApache 2.0 by Kaggle. Both reused artifacts are pinned by SHA-256. This mirror\nlabel alone does not establish the full upstream ownership chain.\nhttps://www.kaggle.com/code/kirilltym/quantum-circuits-qrom-tower42?scriptVersionId=361358116\n\nAdded changes: split-phase scheduling and separate stage caches for invsqrt;\nindependent bounded Boolean-polynomial circuit simplification for DQI;\nalternating sparse rotations and effective-norm optimization for chemistry;\nlayout/precision experiments, resource explanations and full verification reports.\n\nThe scored archive, generated candidates and public dependencies are identified\nseparately in the notebook's reports. No endorsement by upstream authors is implied.\n\nCalibrated arithmetic revision\n\nKozykorpesh: published explicit calibrated integer coefficients and open\nexact-domain minimax regeneration, plus transitive AND implications.\nhttps://www.kaggle.com/code/kozykappa/quantum-ai-compact-arithmetic-banked-qrom?scriptVersionId=361456406\n\nAlex Chilton: shared prefix normalization and empty-target temporary-AND\nshift moves with measurement-based inverse and phase correction.\nhttps://www.kaggle.com/code/alexchilton/where-the-toffolis-go-anatomy-a-cheaper-invsqrt?scriptVersionId=361472684\n\nOur additions in this revision: use the promised input bound to replace\nmutually exclusive inverted-top-bit ANDs by affine logic; algebraic-normal-\nform lookup and measurement-phase cleanup for the exact chemistry words;\na fast C++ binary implementation of Kozykorpesh's implication pass with\nvalidated globally-clean conditional facts. The resulting DQI binary is\nidentical to Kozykorpesh's published output, not a new decoder design.\nThe original generator/compiled-input limits and notices above still apply.\n\nFinal derivative-split integration\n\nRudra Bhaskar: derivative residual/coarse representation, top-carry reuse,\nand static-high phase checkpoint generator, published under Apache-2.0.\nhttps://www.kaggle.com/code/rudrabhaskar/highest-public-score-direct-polynomial-phasing?scriptVersionId=361445283\nKirill Tymchishin v11: integration of that generator with Alex Chilton's\nnormalization and empty-target shift schedules; the 95-qubit baseline.\nhttps://www.kaggle.com/code/kirilltym/quantum-circuits-qrom-tower42?scriptVersionId=361491814\nOur final integration generalizes the derivative-split widths for the\nexplicit Kozykorpesh table, borrows one normalized constant input bit as\nan adder carry, and applies the promised-input top-bit affine identity.\nThe three ablation rows retain the same coefficient model.\n\nSource & Exact Compilation release\n\nThe sections above retain the history and notices of the earlier resource labs.\nThis release targets the existing public vqcc-banked-clean-staging notebook.\nIts decoder is generated from the newly credited sources below; it does not\nrequire the earlier compiled-decoder dependency or a previous submission ZIP.\n\nA. Gu, A. H. Kemeny, S. F. Yelin and M. D. Lukin: reversible Reed-Solomon\ndecoder, shared-path Berlekamp-Massey recurrence, in-place evaluator,\nhalf-degree Bezout pass, tower-field arithmetic, compressed reversal history,\nand approximation-window policy. Apache License, Version 2.0; the original\nsource notices and separate GU_LICENSE are retained.\nhttps://arxiv.org/abs/2610.22367\nhttps://github.com/andigu/uncompute/tree/64b5a13aa249f1360846657392e6c66817b26d0f\n\nDmitrii Gluzdov: competition-field formula derivation, Kickmix adapter,\nrecord-reading phase sweep and readable source-built portfolio. Apache-2.0.\nAdapted source V3, run 361529794:\nhttps://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361529794\nThe separately scored V1, run 361509836, returned 6.12847 for submission\n57567302. V1 and V3 contain identical embedded generator modules.\nhttps://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361509836\n\nDmitrii's public FeMoCo numerical input, dataset V2 / source 21143008,\nis published under Apache-2.0. Original factor refinement traces to k256.dev,\nKozykorpesh and Artem1410/anazemcev; the input's provenance credits remain.\nhttps://www.kaggle.com/datasets/dmitriigluzdov/vqcc-femoco-public-factorization/versions/2\nfemoco_public_words.npz SHA-256:\n9afd0ca5e90dd1740fe713e4ac1c26736184ea9bc4d4b57a6eeefafd608ee193\n\nOur added implementations in this release, Apache-2.0:\n- Full-domain arithmetic compiler: exhaustive Boolean truth vectors over the\n  original promised input domain certify five Clifford replacements. The\n  credited coefficient table, phases, measurements and integer model remain.\n- Numerical gauge generator: simultaneous orthogonal transformations of W\n  and H2_I reduce the normalization objective, with a bound on augmented-Gram\n  drift and unchanged angle words under the original physical checks.\n- DQI exact simplification and symbolic scratch-lifetime packing: unregistered\n  wires are reused only after conservative proof of a globally clean state;\n  the placement pass preserves register IDs, gate order and phase corrections.\n  The decoder algorithm and field port remain credited to Gu et al. and\n  Dmitrii Gluzdov. Compiler reports distinguish structural certificates from\n  full official candidate verification.\n\nHistorical Kaggle scores identify their exact upstream or earlier archives.\nNew cost-derived projections are labelled separately; creating an archive\ndoes not submit it to the competition or establish an observed score.\n\nSource & Exact Compilation: deferred arithmetic and phase calibration\n\nDmitrii Gluzdov, source V4 / run361569612: deferred coefficient-register\ndemolition and shared algebraic-normal-form measurement phase repair.\nThe scheduling origin is credited there to the Gaussian circuit of Gu et al.\nOur adaptation keeps the narrower fixed coefficient/carry schedule and its\nexisting normalization, applies the full-domain compiler, and combines the\ntwo coefficient repairs after their measurement phases have commuted through\nthe intervening reversible arithmetic. Apache License, Version2.0.\nhttps://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361569612\n\nmizeroluckygall, Quantum Circuits | Rotation Calibration, V3/run361537580:\nfull-input rotation activation collection, cutting-plane linear programming,\nexact integer phase-margin checks and removal of small rotations. That\npublication also credits lightningv08's Memory & Algebra work for exhaustive\nphase calibration. Apache License, Version2.0. Original function bodies and\nsource references are retained beside the adapted portable module.\nOur changes give the routines an explicit circuit API, add a conservative\nsupported-circuit guard, bound matrix-scan memory, retain exact dyadic\ncorrections and require independent phase replay and full official checking.\nThe phase optimization uses the full legal input domain rather than a\nfavorable verifier salt or circuit hash.\nhttps://www.kaggle.com/code/mizeroluckygall/quantum-circuits-rotation-calibration-39-char?scriptVersionId=361537580\nhttps://www.kaggle.com/code/lightningv08/quantum-circuits-memory-algebra\n\n\nSource & Exact Compilation V4: signed-gauge and curvature schedule\n\nKozykorpesh V3/run361677445, Apache2.0:\nhttps://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445\nGray-curvature checkpoints and reload plans, signed-Givens gauge,\nfour-address difference banks, selective scratch pebbling, symbolic DQI\nlowering, and the source context for the portable adapters used in V4.\nThe attributed Sumit/Rudra/Alex/Kirill/Artem arithmetic lineage is retained.\n\nDmitrii Gluzdov V6/run361592338, Apache2.0:\nhttps://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361592338\nCompact inverse-norm schedules, template-word loans and streamed sweep,\nbuilding on the credited Gu decoder and earlier competition-field port.\n\nOur V4 modifications: explicit small-public-input arithmetic adapters;\nfive full-domain Clifford reductions on the new schedule; newly hash-bound\ninteger rotation calibration; small-input signed-vector reconstruction\ncertificate; integration with our orthogonal factor gauge and ANF cleanup;\nfresh full-verifier and complete28-member source/archive receipts.\nNo upstream algorithm is claimed as our invention.\n", encoding='utf-8')
print('LICENSE and NOTICE written beside the reports and submission archive.')


## Sources, attribution and reproducibility

- [Competition, official verifiers and rules](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/rules).
- Kozykorpesh, [signed-gauge portfolio V3/run361677445](https://www.kaggle.com/code/kozykappa/quantum-ai-lb-score-6-68108?scriptVersionId=361677445), Apache 2.0. The Gray-curvature checkpoint, Boolean reload plans, signed Givens gauge, four-address banks, selective five-scratch pebbling, and exact symbolic decoder runtime originate there. Its observed score was6.68108, submission57583144. We retain the original notices and identify our changes.
- Dmitrii Gluzdov, [source-built portfolio V6/run361592338](https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source?scriptVersionId=361592338), Apache 2.0, for the competition-field port, compact inverse schedules, template loans and streaming sweep. Its observed score was6.35100.
- Gu et al., [paper](https://arxiv.org/abs/2610.22367) and [andigu/uncompute, pinned commit64b5a13](https://github.com/andigu/uncompute/tree/64b5a13aa249f1360846657392e6c66817b26d0f), Apache 2.0. The reversible decoder, field algorithms and approximate support policy are credited there; the separate GU_LICENSE is included.
- Mizero Lucky Gall, [rotation calibration V3/run361537580](https://www.kaggle.com/code/mizeroluckygall/quantum-circuits-rotation-calibration-39-char?scriptVersionId=361537580), Apache 2.0, for the full-input activation collection, cutting-plane LP and integer angle corrections. The source credits lightningv08's [Memory & Algebra](https://www.kaggle.com/code/lightningv08/quantum-circuits-memory-algebra). Our port adds explicit circuit APIs, bounded memory, structural applicability guards and independent phase replay. Mizero's [new V1/run361675142](https://www.kaggle.com/code/mizeroluckygall/quantum-circuits-rotation-calibration?scriptVersionId=361675142) scored6.37294; it is a comparison, not a circuit-binary input.
- Sumit Chongder, [signed-digit arithmetic](https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149); Rudra Bhaskar, [derivative split](https://www.kaggle.com/code/rudrabhaskar/highest-public-score-direct-polynomial-phasing?scriptVersionId=361445283); Alex Chilton, [normalization and shifts](https://www.kaggle.com/code/alexchilton/where-the-toffolis-go-anatomy-a-cheaper-invsqrt?scriptVersionId=361472684); Kirill Tymchishin, [schedule integration](https://www.kaggle.com/code/kirilltym/quantum-circuits-qrom-tower42?scriptVersionId=361491814). Their notices remain in the arithmetic modules.
- Dmitrii's [public numerical dataset V2](https://www.kaggle.com/datasets/dmitriigluzdov/vqcc-femoco-public-factorization/versions/2), Apache 2.0, builds on k256.dev, Kozykorpesh and Artem1410's FeMoCo factorization work. Input SHA256 is checked before use.
- Our [separate scored release5.77242](https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory?scriptVersionId=361502374) and [previous source V3](https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation?scriptVersionId=361586741) document the borrowed carry, promised-input reductions, full-domain compiler, polynomial ANF cleanup, numerical gauge and guarded phase-calibration integration. V4 ports the new public schedules to this small-input source build and combines them with five exact arithmetic reductions and newly bound calibrated angles.
- Gidney, [Halving the cost of quantum addition](https://quantum-journal.org/papers/q-2018-06-18-74/), for temporary ANDs and measurement-based erasure.

The code and changes are Apache2.0; original attribution and license texts are retained. AI assistants helped with research, implementation and review. All inputs are public and every final byte is hashed. The source and reports remain available beside the archive. Default verification is fresh, full and official; a notebook projection is distinct from an observed competition score.
